
<div dir="rtl" align="right">

# NB22E-v4 — ادامهٔ ارزیابی Retrieval فاز دوم

**پروژه:** RAG_finance  
**شاخهٔ مبنا:** `rag-phase2`  
**زیرکار:** R11.2  
**نقطهٔ ادامه:** خروجی موفق `NB22E-v3:C342`

## هدف

ادامهٔ آماده‌سازی و اجرای کنترل‌شدهٔ ارزیابی جمعیت Test، با مصرف شواهد تاریخی، بدون بازاجرای کارهای موفق.

## وضعیت انتقال

- جمعیت تاریخی: ۱۱۴۹ سؤال.
- ۲۹ اجرای دقیق هدف احراز شده‌اند.
- ۲ مورد با هویت اجرا نامحرز باقی مانده‌اند.
- ۲ مورد خطای ثبت‌شده محفوظ دارند.
- ۱۱۱۶ سؤال نیازمند تعیین و تثبیت مرز رتبه‌بندی و context هستند.
- R11.2 باز است و Gate نسخهٔ هدف BLOCKED باقی می‌ماند.

## ورودی‌ها

- `C342.summary` و `C342.upstream_candidate_manifest`
- `C341.question_bindings`
- قراردادهای فریز‌شده و رسیدهای Producer ارجاع‌شده در C342

## مسیر اجرا

۱. بارگذاری و اتصال دقیق Artifactهای انتقال.  
۲. آماده‌سازی مرز نسخهٔ جمعیتی مستقل.  
۳. کنترل محدود Index و Cache مدل.  
۴. طراحی Pilot محلی و اجرای آن فقط پس از مجوز.  
۵. تولید رتبه و context نسخه‌دار.  
۶. آماده‌سازی ورودی پاسخ و کنترل Cache و دفتر تلاش.  
۷. ادامهٔ ارزیابی، Regression و Gateهای قرارداد.

## مرزهای ممنوع

- تغییر مستقیم مخزن یا Artifactهای تاریخی.
- بازاجرای نوت‌بوک v3 یا سل‌های موفق C340 تا C342.
- تغییر Gold، مدل یا تنظیمات فریز‌شده بدون تصمیم قراردادی.
- اجرای خودکار API، Retrieval یا مدل در مرحلهٔ انتقال.

## خروجی‌های جدید

فقط در مسیر مستقل کنار مخزن:

`_RAG_finance_NB22E_v4_outputs`

## شرط پایان

Artifactهای رتبه، context، پاسخ، شواهد ارزیابی و Gateها باید مطابق قرارداد نسخه‌دار تولید، Reload و اعتبارسنجی شوند. موفقیت Loader به‌تنهایی پایان R11.2 نیست.

**وضعیت اولیه:** `OBJECTIVE_INCOMPLETE`

</div>



<div dir="rtl" align="right">

### NB22E-v4:C001–C002 — R11.2: انتقال مستقل و فقط‌خواندنی

**هدف:** ساخت Kernel سبک و مصرف مستقیم سه Artifact قطعی ادامه.

**ورودی‌ها:**
- Summary و Manifest نسخهٔ C342
- Question Bindings نسخهٔ C341
- هش‌های SHA-256 بایتی ثبت‌شده

**عملیات:**
- C001: واردکردن وابستگی‌های استاندارد و تنظیمات پایه.
- C002: یافتن محدود ریشهٔ پروژه، خواندن سه فایل مشخص، بررسی هش، Schema، تعداد ردیف‌ها، یکتایی شناسه و اتصال سؤال به Manifest.

**کنترل:**
- دقیقاً ۱۱۱۶ شناسهٔ یکتا در هر دو JSONL.
- تطبیق هش متن هر سؤال.
- کنترل وضعیت Producerها از Summary ثبت‌شده.
- حفظ موارد نامحرز و وضعیت BLOCKED.
- عدم تغییر مخزن و Artifactها.

**نتیجهٔ مورد انتظار:** `TRANSFER_LOADER_PASS` در صورت موفقیت تمام کنترل‌های محدود.

**API و هزینه:** صفر درخواست؛ صفر هزینهٔ API.

**شرط پایان اقدام:** موفقیت کنترل ورودی‌ها و آماده‌بودن شواهد برای تصمیم نسخهٔ جمعیتی؛ بدون ادعای آمادگی اجرای مدل.

</div>


In [17]:
# NB22E-v4 | C001 | R11.2
# Centralized imports and deferred optional dependencies; no network or writes.

import ast
import collections
import dataclasses
import hashlib
import html
import importlib
import importlib.metadata
import io
import json
import math
import os
import pickle
import platform
import re
import resource
import statistics
import subprocess
import sys
import tarfile
import tempfile
import time
import unicodedata
import urllib.error
import urllib.parse
import urllib.request
import uuid
import warnings

from collections import Counter, OrderedDict, defaultdict
from collections.abc import Mapping
from difflib import SequenceMatcher
from itertools import combinations
from pathlib import Path
from urllib.parse import urlsplit

NB22E_V4_NOTEBOOK_ID = "NB22E-v4"
NB22E_V4_REPOSITORY_NAME = "RAG_finance"
NB22E_V4_BRANCH = "rag-phase2"
NB22E_V4_SUBTASK = "R11.2"

NB22E_V4_READ_ONLY = True
NB22E_V4_ALLOW_API = False
NB22E_V4_ALLOW_MODEL_EXECUTION = False
NB22E_V4_ALLOW_RETRIEVAL = False
NB22E_V4_ALLOW_REPOSITORY_WRITES = False

NB22E_V4_PREVIOUS_NOTEBOOK = "NB22E-v3"
NB22E_V4_PREVIOUS_SUCCESSFUL_CELL = "C342"

_metadata = importlib.metadata
_platform = platform
_subprocess = subprocess

NB22E_IMPORT_SPECS = [
    ("qdrant_client", "QdrantClient", "QdrantClient"),
    ("dotenv", "dotenv_values", "dotenv_values"),
    ("numpy", None, "np"),
    ("onnxruntime", None, "ort"),
    ("torch", None, "torch"),
    ("transformers", "AutoTokenizer", "AutoTokenizer"),
    (
        "transformers",
        "AutoModelForSequenceClassification",
        "AutoModelForSequenceClassification",
    ),
    ("IPython.display", "display", "display"),
    ("IPython.display", "HTML", "HTML"),
    ("threadpoolctl", "threadpool_limits", "threadpool_limits"),
    ("sklearn", None, "sklearn"),
    (
        "sklearn.ensemble",
        "RandomForestClassifier",
        "RandomForestClassifier",
    ),
    ("lightgbm", None, "lgb"),
    ("xgboost", None, "xgb"),
    (
        "sentence_transformers",
        "SentenceTransformer",
        "SentenceTransformer",
    ),
]

NB22E_IMPORT_STATUS = {}
NB22E_V4_IMPORT_REGISTRY = {}

for module_name, attribute_name, binding_name in NB22E_IMPORT_SPECS:
    if binding_name in NB22E_V4_IMPORT_REGISTRY:
        raise RuntimeError(
            f"Duplicate optional import binding: {binding_name}"
        )

    NB22E_V4_IMPORT_REGISTRY[binding_name] = (
        module_name,
        attribute_name,
    )

    NB22E_IMPORT_STATUS[binding_name] = {
        "status": "DEFERRED",
        "module": module_name,
    }

    globals().pop(binding_name, None)


def nb22e_v4_require_imports(*binding_names):
    """Load only explicitly requested registered dependencies."""

    loaded = {}

    for binding_name in binding_names:
        if binding_name not in NB22E_V4_IMPORT_REGISTRY:
            raise KeyError(
                f"Unregistered optional dependency: {binding_name}"
            )

        module_name, attribute_name = (
            NB22E_V4_IMPORT_REGISTRY[binding_name]
        )

        if NB22E_IMPORT_STATUS[binding_name]["status"] == "AVAILABLE":
            loaded[binding_name] = globals()[binding_name]
            continue

        try:
            module = importlib.import_module(module_name)

            value = (
                module
                if attribute_name is None
                else getattr(module, attribute_name)
            )

            globals()[binding_name] = value

            NB22E_IMPORT_STATUS[binding_name] = {
                "status": "AVAILABLE",
                "module": module_name,
            }

            loaded[binding_name] = value

        except Exception as exc:
            globals().pop(binding_name, None)

            NB22E_IMPORT_STATUS[binding_name] = {
                "status": "UNAVAILABLE",
                "module": module_name,
                "error_type": type(exc).__name__,
                "error": str(exc),
            }

            raise RuntimeError(
                f"Optional dependency unavailable: {binding_name}: "
                f"{type(exc).__name__}: {exc}"
            ) from exc

    return loaded


assert NB22E_V4_READ_ONLY is True
assert NB22E_V4_ALLOW_API is False
assert NB22E_V4_ALLOW_MODEL_EXECUTION is False
assert NB22E_V4_ALLOW_RETRIEVAL is False
assert NB22E_V4_ALLOW_REPOSITORY_WRITES is False

assert len(NB22E_IMPORT_SPECS) == len(
    NB22E_V4_IMPORT_REGISTRY
)

assert hasattr(uuid, "uuid5")
assert hasattr(uuid, "UUID")

NB22E_V4_BOOTSTRAP_STATUS = {
    "cell": "NB22E-v4:C001",
    "status": "LIGHTWEIGHT_BOOTSTRAP_READY",
    "repository_policy": "READ_ONLY",
    "uuid_import": "AVAILABLE",
    "optional_dependencies_registered": len(
        NB22E_V4_IMPORT_REGISTRY
    ),
    "optional_dependencies_deferred": len(
        NB22E_V4_IMPORT_REGISTRY
    ),
    "api_calls": 0,
    "model_runs": 0,
    "repository_modified": False,
}

print(json.dumps(
    NB22E_V4_BOOTSTRAP_STATUS,
    ensure_ascii=False,
    indent=2,
))

{
  "cell": "NB22E-v4:C001",
  "status": "LIGHTWEIGHT_BOOTSTRAP_READY",
  "repository_policy": "READ_ONLY",
  "uuid_import": "AVAILABLE",
  "optional_dependencies_registered": 15,
  "optional_dependencies_deferred": 15,
  "api_calls": 0,
  "model_runs": 0,
  "repository_modified": false
}


In [18]:
# NB22E-v4 | C002 | R11.2
# Read-only C341/C342 transfer loader: exact hashes, schemas and candidate join.

if not NB22E_V4_READ_ONLY:
    raise RuntimeError("C002_REQUIRES_READ_ONLY_MODE")

def c002_require(condition, message):
    if not condition:
        raise RuntimeError("C002_" + message)

# 1. Bounded project-root resolution; no recursive directory search.

c002_candidates = []

c002_env_root = os.environ.get("RAG_FINANCE_ROOT")
if c002_env_root:
    c002_candidates.append(Path(c002_env_root).expanduser())

c002_cwd = Path.cwd().resolve()
c002_candidates.extend((c002_cwd, *c002_cwd.parents))

# Known project layout, relative to the user's home directory.
c002_candidates.append(
    Path.home()
    / "AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance"
)

c002_root = None
for candidate in c002_candidates:
    candidate = candidate.resolve()
    if (
        candidate.name == NB22E_V4_REPOSITORY_NAME
        and (candidate / "notebooks/phase2").is_dir()
    ):
        c002_root = candidate
        break

c002_require(c002_root is not None, "PROJECT_ROOT_NOT_FOUND")

c002_parent = c002_root.parent.resolve()

# 2. Exact receipts from the successful historical producer outputs.

c002_specs = {
    "summary": {
        "folder": "_RAG_finance_C342_rank_route_preflight_outputs",
        "stem": "C342.summary",
        "sha": "fe34926715265f4595dc10084ab18a3dcaca0fbbaf72b37de323466648616b3b",
        "size": 8623,
        "extension": "json",
    },
    "manifest": {
        "folder": "_RAG_finance_C342_rank_route_preflight_outputs",
        "stem": "C342.upstream_candidate_manifest",
        "sha": "4167708704c10ae1b556c0ee82dfdb517d8a9511d61b0723b1f8b0b1c8649dcc",
        "size": 1365688,
        "extension": "jsonl",
    },
    "questions": {
        "folder": "_RAG_finance_C341_question_binding_outputs",
        "stem": "C341.question_bindings",
        "sha": "9a4f0c044b459e78930ab6f0aded8f7d1bb75b677dbe170679333011cd94b5e3",
        "size": 654033,
        "extension": "jsonl",
    },
}

c002_paths = {}

for role, spec in c002_specs.items():
    expected_parent = (c002_parent / spec["folder"]).resolve()
    name = f'{spec["stem"]}.{spec["sha"]}.{spec["extension"]}'
    path = (expected_parent / name).resolve()

    c002_require(
        expected_parent.parent == c002_parent
        and path.parent == expected_parent
        and not path.is_relative_to(c002_root),
        role.upper() + "_UNSAFE_PATH",
    )

    c002_require(path.is_file(), role.upper() + "_FILE_MISSING")
    c002_require(
        path.stat().st_size == spec["size"],
        role.upper() + "_SIZE_MISMATCH",
    )
    c002_paths[role] = path


def c002_read_summary(path, spec):
    before = path.stat()
    raw = path.read_bytes()
    after = path.stat()

    c002_require(
        before.st_size == len(raw) == spec["size"]
        and (before.st_size, before.st_mtime_ns)
        == (after.st_size, after.st_mtime_ns),
        "SUMMARY_STABILITY_CONFLICT",
    )

    c002_require(
        hashlib.sha256(raw).hexdigest() == spec["sha"],
        "SUMMARY_SHA256_CONFLICT",
    )

    result = json.loads(raw)
    c002_require(isinstance(result, dict), "SUMMARY_SCHEMA")
    return result


def c002_read_jsonl(role, consumer, expected_rows=1116):
    spec = c002_specs[role]
    path = c002_paths[role]

    before = path.stat()
    digest = hashlib.sha256()
    count = 0

    with path.open("rb") as handle:
        for line_number, raw in enumerate(handle, 1):
            c002_require(
                line_number <= expected_rows
                and len(raw) <= 512 * 1024
                and raw.endswith(b"\n")
                and bool(raw.strip()),
                role.upper() + "_JSONL_LIMIT_OR_FORMAT",
            )

            digest.update(raw)
            row = json.loads(raw)

            c002_require(
                isinstance(row, dict),
                role.upper() + "_ROW_SCHEMA",
            )

            consumer(row, line_number)
            count += 1

    after = path.stat()

    c002_require(
        (before.st_size, before.st_mtime_ns)
        == (after.st_size, after.st_mtime_ns),
        role.upper() + "_CHANGED_DURING_READ",
    )
    c002_require(
        digest.hexdigest() == spec["sha"],
        role.upper() + "_SHA256_CONFLICT",
    )
    c002_require(
        count == expected_rows,
        role.upper() + "_ROW_COUNT_CONFLICT",
    )

    return count


# 3. Validate the exact C342 summary and its historical status.

c002_summary = c002_read_summary(
    c002_paths["summary"],
    c002_specs["summary"],
)

c002_require(
    c002_summary.get("cell") == "C342"
    and c002_summary.get("roadmap_subtask") == "R11.2"
    and c002_summary.get("status")
    == "FROZEN_RANK_PRODUCER_ROUTE_PREFLIGHT_NO_EXECUTION",
    "SUMMARY_IDENTITY_CONFLICT",
)

c002_expected_preserved = {
    "EXACT_PRESERVED": 29,
    "MODEL_IDENTITY_HELD": 2,
    "RECORDED_ERROR_HELD": 2,
    "OUTSIDE_PACKET": 1116,
}

c002_require(
    c002_summary.get("candidate_count") == 1116
    and c002_summary.get("historical_input_candidates_verified") == 1116
    and c002_summary.get("preserved") == c002_expected_preserved
    and c002_summary.get("grouped_candidate_input_blockers") == [],
    "SUMMARY_COUNTS_OR_BLOCKERS_CONFLICT",
)

c002_require(
    c002_summary.get("new_requests_ready_for_dispatch") == 0
    and c002_summary.get("new_api_calls") == 0
    and c002_summary.get("new_retrieval_runs") == 0
    and c002_summary.get("new_rank_runs") == 0
    and c002_summary.get("original_target_identity_reused") is False
    and c002_summary.get("new_population_requires_version_decision") is True,
    "SUMMARY_EXECUTION_OR_VERSION_CONFLICT",
)

c002_receipt = c002_summary.get("manifest_receipt", {})

c002_require(
    isinstance(c002_receipt, dict)
    and Path(c002_receipt.get("path", "")).resolve()
    == c002_paths["manifest"]
    and c002_receipt.get("file_bytes_sha256")
    == c002_specs["manifest"]["sha"]
    and c002_receipt.get("size_bytes")
    == c002_specs["manifest"]["size"],
    "MANIFEST_RECEIPT_CONFLICT",
)

c002_producers = c002_summary.get("producer_contract_checks", {})
c002_sources = c002_summary.get("source_checks", {})
c002_inputs = c002_summary.get("input_file_checks", {})

c002_require(
    isinstance(c002_producers, dict)
    and bool(c002_producers)
    and all(value is True for value in c002_producers.values()),
    "PRODUCER_CONTRACT_STATUS",
)

c002_require(
    isinstance(c002_sources, dict)
    and len(c002_sources) == 12
    and all(
        isinstance(item, dict)
        and item.get("status") == "FILE_BYTES_VERIFIED"
        for item in c002_sources.values()
    ),
    "SOURCE_RECEIPT_STATUS",
)

c002_roles = ("GOLDEN", "HYBRID", "RERANKER")

c002_require(
    isinstance(c002_inputs, dict)
    and set(c002_inputs) == {*c002_roles, "CHUNKS"}
    and all(
        isinstance(item, dict)
        and item.get("status") == "FILE_BYTES_VERIFIED"
        for item in c002_inputs.values()
    ),
    "INPUT_RECEIPT_STATUS",
)

# 4. Stream C341 question bindings and verify text/hash identity.

c002_questions = {}

def c002_consume_question(row, line_number):
    cid = row.get("candidate_id")
    question = row.get("question")
    recorded_sha = row.get("question_sha256")

    c002_require(
        isinstance(cid, str) and bool(cid)
        and cid not in c002_questions
        and isinstance(question, str) and bool(question)
        and isinstance(recorded_sha, str)
        and row.get("question_only_not_response_or_context") is True
        and isinstance(row.get("source_line_number"), int)
        and row["source_line_number"] > 0,
        "QUESTION_SCHEMA_OR_DUPLICATE",
    )

    actual_sha = hashlib.sha256(
        question.encode("utf-8")
    ).hexdigest()

    c002_require(
        actual_sha == recorded_sha,
        "QUESTION_TEXT_HASH_CONFLICT",
    )

    c002_questions[cid] = {
        "question": question,
        "question_sha256": recorded_sha,
    }

c002_question_count = c002_read_jsonl(
    "questions",
    c002_consume_question,
)

# 5. Stream C342 manifest and perform the exact 1116-ID join.

c002_remaining = set(c002_questions)
c002_manifest_status_count = 0

def c002_consume_manifest(row, line_number):
    global c002_manifest_status_count

    cid = row.get("candidate_id")

    c002_require(
        isinstance(cid, str)
        and cid in c002_remaining,
        "MANIFEST_UNKNOWN_OR_DUPLICATE_ID",
    )

    c002_remaining.remove(cid)

    c002_require(
        row.get("question_sha256")
        == c002_questions[cid]["question_sha256"],
        "MANIFEST_QUESTION_HASH_CONFLICT",
    )

    c002_require(
        row.get("historical_input_status")
        == "CANDIDATE_FOR_FROZEN_REPRODUCTION"
        and row.get("blockers") == []
        and row.get("target_char_rrf_rank_proven") is False
        and row.get("target_identity_8d5875_reused") is False
        and row.get("send_authorized") is False,
        "MANIFEST_VERSION_OR_READINESS_CONFLICT",
    )

    receipts = row.get("source_row_receipts")

    c002_require(
        isinstance(receipts, dict)
        and set(receipts) == set(c002_roles),
        "MANIFEST_SOURCE_RECEIPT_SCHEMA",
    )

    for role in c002_roles:
        receipt = receipts[role]
        pinned = c002_inputs[role]

        c002_require(
            isinstance(receipt, dict)
            and receipt.get("path") == pinned.get("path")
            and receipt.get("file_bytes_sha256")
            == pinned.get("file_bytes_sha256")
            and type(receipt.get("line_number")) is int
            and receipt["line_number"] > 0
            and isinstance(
                receipt.get("row_sha256_excluding_line_ending"),
                str,
            )
            and len(receipt["row_sha256_excluding_line_ending"]) == 64,
            "MANIFEST_SOURCE_RECEIPT_CONFLICT",
        )

    c002_manifest_status_count += 1


c002_manifest_count = c002_read_jsonl(
    "manifest",
    c002_consume_manifest,
)

c002_require(
    not c002_remaining
    and c002_manifest_status_count == 1116,
    "QUESTION_MANIFEST_JOIN_INCOMPLETE",
)

# 6. Preserve the pending Index/Cache gates without executing them.

c002_index = c002_summary.get("dense_index_precheck", {})
c002_vectors = c002_summary.get("query_vector_precheck", {})
c002_model_cache = c002_summary.get("model_cache_precheck", {})

c002_require(
    isinstance(c002_index, dict)
    and c002_index.get("frozen_index_hash_reverified") is False
    and isinstance(c002_vectors, dict)
    and c002_vectors.get("vector_candidate_id_coverage") == 1116
    and isinstance(c002_model_cache, dict),
    "PENDING_RUNTIME_PRECHECK_SCHEMA",
)

NB22E_V4_TRANSFER = {
    "status": "TRANSFER_LOADER_PASS",
    "candidate_count": c002_manifest_count,
    "questions_bound": c002_question_count,
    "artifact_paths": {
        role: str(path)
        for role, path in c002_paths.items()
    },
    "artifact_sha256": {
        role: spec["sha"]
        for role, spec in c002_specs.items()
    },
    "version_decision": c002_summary["version_decision"],
    "index_status": c002_index.get("status"),
    "model_cache_status": c002_model_cache.get("status"),
    "send_authorized": False,
}

print(json.dumps({
    "cell": "NB22E-v4:C002",
    "status": NB22E_V4_TRANSFER["status"],
    "file_bytes_verified": list(c002_specs),
    "question_count": c002_question_count,
    "manifest_count": c002_manifest_count,
    "joined_ids": 1116 - len(c002_remaining),
    "producer_contract_checks": c002_producers,
    "preserved": c002_expected_preserved,
    "dense_index_status": c002_index.get("status"),
    "dense_index_hash_reverified": False,
    "model_cache_status": c002_model_cache.get("status"),
    "version_decision": c002_summary["version_decision"],
    "target_release_gate": "BLOCKED",
    "api_calls": 0,
    "retrieval_runs": 0,
    "rank_runs": 0,
    "repository_modified": False,
    "next": "VERSION_BOUNDARY_PREPARATION",
}, ensure_ascii=False, indent=2))


{
  "cell": "NB22E-v4:C002",
  "status": "TRANSFER_LOADER_PASS",
  "file_bytes_verified": [
    "summary",
    "manifest",
    "questions"
  ],
  "question_count": 1116,
  "manifest_count": 1116,
  "joined_ids": 1116,
  "producer_contract_checks": {
    "C118_runtime_matches": true,
    "C119_parity_verified": true,
    "C139_caller_bound": true,
    "C186_input_pins_match": true,
    "C188_33_only": true,
    "NB22C_functions_pinned": true,
    "NB22D_functions_pinned": true
  },
  "preserved": {
    "EXACT_PRESERVED": 29,
    "MODEL_IDENTITY_HELD": 2,
    "RECORDED_ERROR_HELD": 2,
    "OUTSIDE_PACKET": 1116
  },
  "dense_index_status": "DIRECTORY_PRESENT_VERSION_UNREVALIDATED",
  "dense_index_hash_reverified": false,
  "model_cache_status": "PRESENT_REQUIRES_RECHECK_BEFORE_LOCAL_EXECUTION",
  "version_decision": "NEW_VERSIONED_POPULATION_AND_TARGET_GENERATION_BOUNDARY_RECEIPTS_REQUIRED_BEFORE_LABELLING_NEW_RESULTS_AS_TARGET",
  "target_release_gate": "BLOCKED",
  "api_calls": 0,
  "r


<div dir="rtl" align="right">

### NB22E-v4:C003 — R11.2: آماده‌سازی مرز نسخهٔ جمعیتی

**هدف:** ایجاد پیش‌نویس مستقل و قابل‌ردیابی از مرز نسخهٔ جدید برای ۱۱۱۶ سؤال، بدون اعطای هویت نهایی یا اجرای محاسبات Retrieval.

**ورودی‌ها:**
- Artifactهای تأییدشدهٔ C002.
- نسخهٔ فریز‌شدهٔ قالب درخواست C340.
- وضعیت Producerها، ورودی‌ها و محدودیت‌های C342.

**عملیات:**
۱. کنترل اتصال هویت تاریخی و تنظیمات واقعی مدل.  
۲. ثبت منابع فریز‌شده و مرز جمعیت جدید.  
۳. تفکیک رتبه و context تاریخی از خروجی‌های هنوز تولیدنشده.  
۴. محاسبهٔ Fingerprint پیشنهادی از محتوای واقعی.  
۵. ذخیرهٔ Atomic پیش‌نویس فقط در پوشهٔ خروجی مستقل کنار مخزن.

**خروجی:** فایل JSON نسخه‌دار با وضعیت `DRAFT_AWAITING_VERSION_APPROVAL`.

**کنترل:** هش بایتی، کنترل نسخه، عدم بازنویسی Artifact موجود و Reload فایل نهایی.

**محدودیت:** Fingerprint پیشنهادی، هویت نسخهٔ نهایی پاسخ نیست. Index، Cache مدل، تولید رتبه و context همچنان نیازمند اقدامات بعدی‌اند.

**API:** صفر.  
**تغییر مخزن:** صفر.

**شرط پایان:** ثبت موفق و بازخوانی پیش‌نویس معتبر؛ بدون تغییر وضعیت Gate اصلی.

</div>


In [3]:
# NB22E-v4 | C003 | R11.2
# Prepare a content-bound population version proposal; no repository writes or execution.

# ============================================================
# 1. Require the successful C002 transfer state
# ============================================================

if (
    not NB22E_V4_READ_ONLY
    or NB22E_V4_TRANSFER.get("status") != "TRANSFER_LOADER_PASS"
):
    raise RuntimeError("C003_REQUIRES_SUCCESSFUL_READ_ONLY_C002")

if (
    NB22E_V4_TRANSFER.get("candidate_count") != 1116
    or NB22E_V4_TRANSFER.get("questions_bound") != 1116
    or NB22E_V4_TRANSFER.get("send_authorized") is not False
):
    raise RuntimeError("C003_TRANSFER_BOUNDARY_CONFLICT")

c003_root = c002_root.resolve()
c003_parent = c003_root.parent.resolve()

if c003_root.name != "RAG_finance":
    raise RuntimeError("C003_PROJECT_ROOT_CONFLICT")

for role in ("summary", "manifest", "questions"):
    if (
        NB22E_V4_TRANSFER["artifact_sha256"].get(role)
        != c002_specs[role]["sha"]
        or Path(NB22E_V4_TRANSFER["artifact_paths"][role]).resolve()
        != c002_paths[role]
    ):
        raise RuntimeError("C003_TRANSFER_RECEIPT_CONFLICT:" + role)

# ============================================================
# 2. Read only the exact C340 frozen-template receipt
# ============================================================

c003_template_sha = (
    "f72301a2309c5c66aa84b0a5ba3eda96dcee703e1d93d67c9df71e40942240d9"
)

c003_template_path = (
    c003_parent
    / "_RAG_finance_C340_preflight_outputs"
    / ("C340.frozen_template." + c003_template_sha + ".json")
).resolve()

if (
    c003_template_path.parent
    != (c003_parent / "_RAG_finance_C340_preflight_outputs").resolve()
    or not c003_template_path.is_file()
):
    raise RuntimeError("C003_FROZEN_TEMPLATE_MISSING_OR_UNSAFE")

c003_before = c003_template_path.stat()

if c003_before.st_size != 47393:
    raise RuntimeError("C003_FROZEN_TEMPLATE_SIZE_CONFLICT")

c003_raw = c003_template_path.read_bytes()
c003_after = c003_template_path.stat()

if (
    c003_before.st_size != c003_after.st_size
    or c003_before.st_mtime_ns != c003_after.st_mtime_ns
    or hashlib.sha256(c003_raw).hexdigest() != c003_template_sha
):
    raise RuntimeError("C003_FROZEN_TEMPLATE_SHA256_CONFLICT")

c003_template = json.loads(c003_raw)

def c003_canonical(value):
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        allow_nan=False,
    ).encode("utf-8")

def c003_sha(raw):
    return hashlib.sha256(raw).hexdigest()

# ============================================================
# 3. Verify frozen model and historical identity
# ============================================================

c003_original_identity = c002_summary.get(
    "target_identity_sha256_original"
)

c003_request = c003_template.get("reference_request", {})
c003_identity = c003_template.get("target_identity", {})

if not (
    isinstance(c003_request, dict)
    and isinstance(c003_identity, dict)
    and isinstance(c003_request.get("messages"), list)
    and len(c003_request["messages"]) == 2
):
    raise RuntimeError("C003_TEMPLATE_SCHEMA_CONFLICT")

c003_profile = {
    key: value
    for key, value in c003_request.items()
    if key != "messages"
}

c003_expected_profile = {
    "model": "DeepSeek-V4-Flash",
    "temperature": 0,
    "max_tokens": 600,
    "thinking": {"type": "disabled"},
}

c003_messages = c003_request["messages"]

if not (
    c003_profile == c003_expected_profile
    and c003_identity.get("profile") == c003_expected_profile
    and c003_messages[0].get("role") == "system"
    and c003_messages[1].get("role") == "user"
    and isinstance(c003_messages[0].get("content"), str)
):
    raise RuntimeError("C003_FROZEN_MODEL_PROFILE_CONFLICT")

c003_prompt_sha = c003_sha(
    c003_messages[0]["content"].encode("utf-8")
)

if not (
    c003_sha(c003_canonical(c003_identity))
    == c003_template.get("target_identity_sha256")
    == c003_original_identity
    == "8d5875bfee0cfe5f2dc0ce9bdf67e5f14301191b60891047a630c3fe65442065"
    and c003_prompt_sha
    == c003_template.get("system_prompt_sha256")
    == c003_identity.get("system_prompt_sha256")
):
    raise RuntimeError("C003_HISTORICAL_TARGET_IDENTITY_CONFLICT")

# ============================================================
# 4. Verify source coverage and pending version conditions
# ============================================================

c003_sources = c002_summary["source_checks"]
c003_inputs = c002_summary["input_file_checks"]
c003_producers = c002_summary["producer_contract_checks"]

if not (
    all(v is True for v in c003_producers.values())
    and all(
        value.get("status") == "FILE_BYTES_VERIFIED"
        for value in c003_sources.values()
    )
    and all(
        value.get("status") == "FILE_BYTES_VERIFIED"
        for value in c003_inputs.values()
    )
    and c002_summary.get("existing_final_rank_coverage")
    == "C189_ONLY_33_VERIFIED"
    and c002_summary.get("frozen_final_char_rrf_rank_newly_bound") == 0
    and c002_summary.get("original_target_identity_reused") is False
    and c002_summary.get("new_population_requires_version_decision") is True
):
    raise RuntimeError("C003_PRODUCER_OR_VERSION_STATUS_CONFLICT")

c003_index = c002_summary["dense_index_precheck"]
c003_vector = c002_summary["query_vector_precheck"]
c003_cache = c002_summary["model_cache_precheck"]

if not (
    c003_index.get("frozen_index_hash_reverified") is False
    and c003_vector.get("vector_candidate_id_coverage") == 1116
    and c003_cache.get("status")
    == "PRESENT_REQUIRES_RECHECK_BEFORE_LOCAL_EXECUTION"
):
    raise RuntimeError("C003_RUNTIME_PENDING_STATE_CONFLICT")

# ============================================================
# 5. Build a proposal anchored to verified artifact content
# ============================================================

c003_core = {
    "schema": "NB22E_V4_R11_2_POPULATION_BOUNDARY_DRAFT_V1",
    "cell": "NB22E-v4:C003",
    "subtask": "R11.2",
    "status": "DRAFT_AWAITING_VERSION_APPROVAL",
    "population": {
        "historical_test_count": 1149,
        "new_candidate_count": 1116,
        "manifest_file_bytes_sha256": c002_specs["manifest"]["sha"],
        "questions_file_bytes_sha256": c002_specs["questions"]["sha"],
        "source_summary_file_bytes_sha256": c002_specs["summary"]["sha"],
        "preserved": c002_summary["preserved"],
    },
    "historical_target_lineage": {
        "identity_sha256": c003_original_identity,
        "rank_input_mode": c002_summary["original_rank_input_mode"],
        "existing_rank_coverage": "C189_ONLY_33_VERIFIED",
        "frozen_template_file_bytes_sha256": c003_template_sha,
    },
    "frozen_generation_profile": {
        "request_profile": c003_profile,
        "system_prompt_sha256": c003_prompt_sha,
        "request_wrapper_source": "C340_FROZEN_TEMPLATE",
        "historical_identity_only": True,
    },
    "producer_evidence": {
        "source_checks": c003_sources,
        "input_file_checks": c003_inputs,
        "producer_contract_checks": c003_producers,
        "upstream_candidate_manifest_sha256": (
            c002_specs["manifest"]["sha"]
        ),
    },
    "proposed_new_boundary": {
        "rank_input_mode": "PENDING_VERSION_BOUND_RANK_RECEIPTS",
        "retrieval_input_identity": "PENDING_VERSION_VALIDATION",
        "rank_receipts_for_new_population": "NOT_YET_PRODUCED",
        "ordered_context_receipts": "NOT_YET_PRODUCED",
        "rendered_context_receipts": "NOT_YET_PRODUCED",
        "generation_request_receipts": "NOT_YET_PRODUCED",
        "new_target_identity": "NOT_ASSIGNED",
        "historical_identity_transfer_authorized": False,
        "configuration_selection_on_test_authorized": False,
        "generation_dispatch_authorized": False,
    },
    "runtime_prechecks": {
        "dense_index": c003_index,
        "query_vectors": c003_vector,
        "model_cache": c003_cache,
    },
    "planned_execution_sequence": (
        c002_summary["next_execution_sequence_after_explicit_version_approval"]
    ),
    "pending": [
        "SUPERVISOR_VERSION_BOUNDARY_DECISION",
        "DENSE_INDEX_EXACT_VERSION_VERIFICATION",
        "LOCAL_MODEL_CACHE_EXACT_VERSION_VERIFICATION",
        "HISTORICAL_POOL50_VERSION_MEMBERSHIP_CHECK",
        "NEW_RANK_AND_CONTEXT_RECEIPTS",
        "NEW_GENERATION_REQUEST_AND_ATTEMPT_CONTROL",
        "R11_2_EVALUATION_AND_REGRESSION_GATES",
    ],
    "constraints": {
        "repository_read_only": True,
        "historical_artifacts_immutable": True,
        "no_retrieval_in_this_cell": True,
        "no_ranking_in_this_cell": True,
        "no_model_execution_in_this_cell": True,
        "no_api_in_this_cell": True,
        "proposal_is_not_target_release_identity": True,
    },
}

c003_proposal_sha = c003_sha(c003_canonical(c003_core))

c003_document = {
    **c003_core,
    "proposal_content_sha256": c003_proposal_sha,
    "fingerprint_kind": "CANONICAL_JSON_PROPOSAL_CONTENT",
}

# ============================================================
# 6. Publish an immutable draft outside the repository
# ============================================================

c003_out = (
    c003_parent / "_RAG_finance_NB22E_v4_outputs"
).resolve()

if (
    c003_out == c003_root
    or c003_out.is_relative_to(c003_root)
):
    raise RuntimeError("C003_OUTPUT_INSIDE_REPOSITORY")

c003_out.mkdir(parents=True, exist_ok=True)

c003_bytes = c003_canonical(c003_document) + b"\n"
c003_file_sha = c003_sha(c003_bytes)

c003_target = c003_out / (
    "C003.boundary_proposal."
    + c003_file_sha
    + ".json"
)

# Atomic, no-overwrite publication.
# The temporary file and final file are on the same filesystem.

if not c003_target.exists():
    with tempfile.NamedTemporaryFile(
        mode="wb",
        dir=c003_out,
        prefix=".C003.boundary.",
        suffix=".tmp",
        delete=False,
    ) as c003_handle:
        c003_temp = Path(c003_handle.name)
        c003_handle.write(c003_bytes)
        c003_handle.flush()
        os.fsync(c003_handle.fileno())

    try:
        if c003_temp.read_bytes() != c003_bytes:
            raise RuntimeError("C003_TEMP_RELOAD_CONFLICT")

        try:
            os.link(c003_temp, c003_target)
        except FileExistsError:
            pass
    finally:
        c003_temp.unlink(missing_ok=True)

# ============================================================
# 7. Independent reload and fingerprint verification
# ============================================================

c003_reloaded_bytes = c003_target.read_bytes()

if (
    c003_reloaded_bytes != c003_bytes
    or c003_sha(c003_reloaded_bytes) != c003_file_sha
):
    raise RuntimeError("C003_FINAL_FILE_BYTES_CONFLICT")

c003_reloaded = json.loads(c003_reloaded_bytes)

c003_reloaded_core = {
    key: value
    for key, value in c003_reloaded.items()
    if key not in (
        "proposal_content_sha256",
        "fingerprint_kind",
    )
}

if not (
    c003_reloaded["proposal_content_sha256"]
    == c003_sha(c003_canonical(c003_reloaded_core))
    == c003_proposal_sha
    and c003_reloaded["status"]
    == "DRAFT_AWAITING_VERSION_APPROVAL"
    and c003_reloaded["population"]["new_candidate_count"] == 1116
):
    raise RuntimeError("C003_RELOAD_VALIDATION_CONFLICT")

NB22E_V4_BOUNDARY_PROPOSAL = {
    "status": "DRAFT_RELOAD_PASS",
    "path": str(c003_target),
    "file_bytes_sha256": c003_file_sha,
    "proposal_content_sha256": c003_proposal_sha,
    "new_target_identity_assigned": False,
    "approved": False,
}

print(json.dumps({
    "cell": "NB22E-v4:C003",
    "status": "DRAFT_RELOAD_PASS",
    "population": 1116,
    "historical_identity": c003_original_identity,
    "historical_rank_mode": c002_summary["original_rank_input_mode"],
    "proposed_rank_mode": "PENDING_VERSION_BOUND_RANK_RECEIPTS",
    "model": c003_profile["model"],
    "frozen_profile_verified": True,
    "proposal_content_sha256": c003_proposal_sha,
    "output_receipt": {
        "path": str(c003_target),
        "file_bytes_sha256": c003_file_sha,
        "size_bytes": len(c003_bytes),
    },
    "new_target_identity_assigned": False,
    "version_approval": "PENDING",
    "dense_index_version": "PENDING",
    "local_model_cache_version": "PENDING",
    "target_release_gate": "BLOCKED",
    "api_calls": 0,
    "retrieval_runs": 0,
    "rank_runs": 0,
    "repository_modified": False,
    "next": "INDEX_AND_MODEL_CACHE_LIMITED_VERIFICATION",
}, ensure_ascii=False, indent=2))


{
  "cell": "NB22E-v4:C003",
  "status": "DRAFT_RELOAD_PASS",
  "population": 1116,
  "historical_identity": "8d5875bfee0cfe5f2dc0ce9bdf67e5f14301191b60891047a630c3fe65442065",
  "historical_rank_mode": "VERSION_BOUND_STORED_C189_RANKS",
  "proposed_rank_mode": "PENDING_VERSION_BOUND_RANK_RECEIPTS",
  "model": "DeepSeek-V4-Flash",
  "frozen_profile_verified": true,
  "proposal_content_sha256": "72a6dbc8de189b4add1032f9d9c0330b75935bfb7ff95eda982a332ea1edb327",
  "output_receipt": {
    "path": "/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/_RAG_finance_NB22E_v4_outputs/C003.boundary_proposal.0dbfd6e4c00de3d4e4ec75f616f2cbc09a1b63d025a4648249684f43593fb492.json",
    "file_bytes_sha256": "0dbfd6e4c00de3d4e4ec75f616f2cbc09a1b63d025a4648249684f43593fb492",
    "size_bytes": 9588
  },
  "new_target_identity_assigned": false,
  "version_approval": "PENDING",
  "dense_index_version": "PENDING",
  "local_model_cache_version": "PENDING",
  "target_release_gate": "BLOCKED",
  "a

<div dir="rtl" align="right">

### NB22E-v4:C004 — R11.2: کنترل محدود Index و Cache مدل

**هدف:** بررسی اعتبار فعلی فایل‌های قرارداد Index و مدل محلی، بدون اجرای Retrieval یا مدل.

**ورودی‌ها:**
- خروجی موفق C002 و پیش‌نویس معتبر C003.
- چهار فایل فریز‌شدهٔ Index با هش‌های ثبت‌شده.
- مدل ONNX و تنظیمات ثبت‌شده در C342.

**عملیات:**
۱. بررسی SHA-256 قراردادهای Index و تطبیق شناسهٔ نسخه، Collection و مسیر ذخیره‌سازی.  
۲. کنترل وجود مسیر فیزیکی Index، بدون بازکردن Qdrant.  
۳. محاسبهٔ قطعه‌ای هش مدل ONNX و مقایسه با نسخهٔ فریز‌شده.  
۴. بررسی فایل‌های Tokenizer و نسخهٔ نصب‌شدهٔ ONNX Runtime.  
۵. گزارش جداگانهٔ کنترل‌های موفق و موارد هنوز نامحرز.

**محدودیت:** تطبیق Metadata اثبات سلامت محتوای زندهٔ Qdrant نیست. این کنترل جداگانه و بدون امکان تغییر Index اصلی انجام خواهد شد.

**خروجی:** گزارش وضعیت در همان Kernel؛ بدون نوشتن فایل.

**API:** صفر.  
**تغییر مخزن:** صفر.  
**شرط پایان:** مشخص‌شدن اعتبار قراردادهای Index، هش مدل و موانع واقعی اجرای محلی.

</div>

In [4]:
# NB22E-v4 | C004 | R11.2
# Repair: verified historical SHA-256 pins; read-only Index and ONNX preflight.

# ============================================================
# 1. Previous cell and repository safety
# ============================================================

if not (
    NB22E_V4_READ_ONLY is True
    and NB22E_V4_TRANSFER.get("status") == "TRANSFER_LOADER_PASS"
    and NB22E_V4_BOUNDARY_PROPOSAL.get("status") == "DRAFT_RELOAD_PASS"
):
    raise RuntimeError("C004_REQUIRES_VERIFIED_C002_C003")

c004_root = c002_root.resolve()

if c004_root.name != "RAG_finance":
    raise RuntimeError("C004_PROJECT_ROOT_CONFLICT")

# ============================================================
# 2. Exact historical file-byte pins from NB22E-v3:C148
# ============================================================

c004_pins = {
    "release": (
        "artifacts/phase2/dense_index/dense_index_release_v1.json",
        "706c020caa2bda77c767fda8cd1b57498818afdbdbbbdd2a72f898b55ca6ca50",
    ),
    "handoff": (
        "artifacts/phase2/dense_index/nb16_dense_index_handoff_v1.json",
        "da7f812a79a21b1876c74668a548badb0f271b2d18ac2f1b2cf9663f789f301f",
    ),
    "manifest": (
        "data/phase2/index/dense_index_v1/dense_index_manifest.json",
        "739539f7abdfd0df435389e103658e28244b078f425f33878458034058bfae47",
    ),
    "lifecycle": (
        "data/phase2/index/dense_index_v1/index_lifecycle_evidence.json",
        "bcea3bda1ee7ad86c9c6ca40ab674aa9bc23394a668dce86a99e5b03a1a3d2ef",
    ),
}

for role, (relative, expected_sha) in c004_pins.items():
    if (
        len(expected_sha) != 64
        or any(ch not in "0123456789abcdef" for ch in expected_sha)
    ):
        raise RuntimeError(f"C004_INVALID_PIN:{role}")

# ============================================================
# 3. Strict read-only file verification
# ============================================================

def c004_read_pinned(relative, expected_sha):
    relative_path = Path(relative)
    path = (c004_root / relative_path).resolve()

    if (
        relative_path.is_absolute()
        or not path.is_relative_to(c004_root)
        or not path.is_file()
    ):
        raise RuntimeError(f"C004_FILE_MISSING_OR_UNSAFE:{relative}")

    before = path.stat()

    if before.st_size > 16 * 1024 * 1024:
        raise RuntimeError(f"C004_METADATA_TOO_LARGE:{relative}")

    raw = path.read_bytes()
    after = path.stat()

    if (
        before.st_size != after.st_size
        or before.st_size != len(raw)
        or before.st_mtime_ns != after.st_mtime_ns
    ):
        raise RuntimeError(f"C004_FILE_STABILITY_CONFLICT:{relative}")

    actual_sha = hashlib.sha256(raw).hexdigest()

    if actual_sha != expected_sha:
        raise RuntimeError(
            f"C004_FILE_HASH_CONFLICT:{relative}:"
            f"expected={expected_sha}:actual={actual_sha}"
        )

    document = json.loads(raw)

    if not isinstance(document, dict):
        raise RuntimeError(f"C004_JSON_SCHEMA_CONFLICT:{relative}")

    return document, {
        "path": relative,
        "status": "FILE_BYTES_VERIFIED",
        "file_bytes_sha256": actual_sha,
        "size_bytes": len(raw),
    }

c004_docs = {}
c004_file_checks = {}

for role, (relative, expected_sha) in c004_pins.items():
    document, receipt = c004_read_pinned(
        relative,
        expected_sha,
    )
    c004_docs[role] = document
    c004_file_checks[role] = receipt

# ============================================================
# 4. Index boundary: metadata is not live storage validation
# ============================================================

c004_dense_precheck = c002_summary["dense_index_precheck"]

c004_expected_index_relative = (
    "data/phase2/index/dense_index_v1/qdrant"
)

if (
    c004_dense_precheck.get("path")
    != c004_expected_index_relative
    or c004_dense_precheck.get("frozen_index_hash_reverified")
    is not False
):
    raise RuntimeError("C004_DENSE_INDEX_BOUNDARY_CONFLICT")

c004_index_path = (
    c004_root / c004_expected_index_relative
).resolve()

c004_index_present = (
    c004_index_path.is_relative_to(c004_root)
    and c004_index_path.is_dir()
)

c004_index_status = (
    "PINNED_METADATA_VERIFIED_LIVE_INDEX_PENDING"
    if c004_index_present
    else "PINNED_METADATA_VERIFIED_STORAGE_MISSING"
)

c004_index_observed_fields = {
    role: {
        "status": doc.get("status"),
        "final_status": doc.get("final_status"),
        "logical_index_sha256": doc.get("logical_index_sha256"),
    }
    for role, doc in c004_docs.items()
}

# Do not assume that all documents use the same schema.
# Historical release semantics remain bound to their original evidence.
# This cell verifies file bytes, not live Qdrant integrity.

# ============================================================
# 5. Verify exact ONNX model from the C342 cache precheck
# ============================================================

c004_cache = c002_summary["model_cache_precheck"]

c004_expected_model_sha = (
    "6c2513767fb63d008a4377bef7a7a3555433d9436342bb53e35a3a72ffc52d4b"
)

if (
    not isinstance(c004_cache, dict)
    or c004_cache.get("expected_onnx_sha256")
    != c004_expected_model_sha
    or not isinstance(c004_cache.get("path"), str)
    or not c004_cache["path"]
):
    raise RuntimeError("C004_MODEL_CONTRACT_CONFLICT")

c004_model_path = Path(
    c004_cache["path"]
).expanduser()

c004_model_actual_sha = None
c004_model_status = "MODEL_FILE_MISSING"

if c004_model_path.is_file():
    before = c004_model_path.stat()
    digest = hashlib.sha256()

    with c004_model_path.open("rb") as handle:
        while True:
            chunk = handle.read(1024 * 1024)
            if not chunk:
                break
            digest.update(chunk)

    after = c004_model_path.stat()

    c004_model_actual_sha = digest.hexdigest()

    if (
        before.st_size != after.st_size
        or before.st_mtime_ns != after.st_mtime_ns
    ):
        c004_model_status = "MODEL_CHANGED_DURING_READ"

    elif c004_model_actual_sha == c004_expected_model_sha:
        c004_model_status = "ONNX_FILE_HASH_MATCHED"

    else:
        c004_model_status = "ONNX_FILE_HASH_MISMATCH"

# ============================================================
# 6. Lightweight tokenizer and runtime inspection
# ============================================================

c004_snapshot = c004_model_path.parent.parent

c004_tokenizer_files = {
    name: (c004_snapshot / name).is_file()
    for name in (
        "config.json",
        "tokenizer.json",
        "tokenizer_config.json",
        "special_tokens_map.json",
        "vocab.txt",
    )
}

try:
    c004_onnxruntime_version = importlib.metadata.version(
        "onnxruntime"
    )
except importlib.metadata.PackageNotFoundError:
    c004_onnxruntime_version = None

c004_expected_runtime_version = "1.16.3"

c004_runtime_version_matches = (
    c004_onnxruntime_version
    == c004_expected_runtime_version
)

c004_historical_parity_preserved = (
    c002_summary["producer_contract_checks"].get(
        "C119_parity_verified"
    ) is True
)

# ============================================================
# 7. Store read-only preflight result in the current kernel
# ============================================================

c004_model_ready_for_later_validation = (
    c004_model_status == "ONNX_FILE_HASH_MATCHED"
    and c004_runtime_version_matches
)

NB22E_V4_RUNTIME_PREFLIGHT = {
    "cell": "NB22E-v4:C004",
    "status": "LIMITED_READ_ONLY_PREFLIGHT_COMPLETE",
    "index_file_checks": c004_file_checks,
    "index_metadata_files_verified": len(c004_file_checks),
    "index_status": c004_index_status,
    "index_directory_present": c004_index_present,
    "index_observed_fields": c004_index_observed_fields,
    "index_live_storage_reverified": False,
    "index_opened": False,
    "model_file_status": c004_model_status,
    "model_file_sha256": c004_model_actual_sha,
    "tokenizer_file_presence": c004_tokenizer_files,
    "tokenizer_runtime_loadability_reverified": False,
    "onnxruntime_version": c004_onnxruntime_version,
    "onnxruntime_version_matches": c004_runtime_version_matches,
    "historical_four_pair_parity_preserved": (
        c004_historical_parity_preserved
    ),
    "model_ready_for_later_validation": (
        c004_model_ready_for_later_validation
    ),
    "local_execution_ready": False,
    "target_release_gate": "BLOCKED",
    "api_calls": 0,
    "retrieval_runs": 0,
    "rank_runs": 0,
    "model_runs": 0,
    "repository_modified": False,
}

print(json.dumps(
    NB22E_V4_RUNTIME_PREFLIGHT,
    ensure_ascii=False,
    indent=2,
))

{
  "cell": "NB22E-v4:C004",
  "status": "LIMITED_READ_ONLY_PREFLIGHT_COMPLETE",
  "index_file_checks": {
    "release": {
      "path": "artifacts/phase2/dense_index/dense_index_release_v1.json",
      "status": "FILE_BYTES_VERIFIED",
      "file_bytes_sha256": "706c020caa2bda77c767fda8cd1b57498818afdbdbbbdd2a72f898b55ca6ca50",
      "size_bytes": 3592
    },
    "handoff": {
      "path": "artifacts/phase2/dense_index/nb16_dense_index_handoff_v1.json",
      "status": "FILE_BYTES_VERIFIED",
      "file_bytes_sha256": "da7f812a79a21b1876c74668a548badb0f271b2d18ac2f1b2cf9663f789f301f",
      "size_bytes": 1061
    },
    "manifest": {
      "path": "data/phase2/index/dense_index_v1/dense_index_manifest.json",
      "status": "FILE_BYTES_VERIFIED",
      "file_bytes_sha256": "739539f7abdfd0df435389e103658e28244b078f425f33878458034058bfae47",
      "size_bytes": 8493
    },
    "lifecycle": {
      "path": "data/phase2/index/dense_index_v1/index_lifecycle_evidence.json",
      "status": 

<div dir="rtl" align="right">

### NB22E-v4:C005 — R11.2: کنترل ایمن Tokenizer و Collection محلی

**هدف:** تکمیل کنترل بارگذاری Tokenizer و آزمایش خواندن Collection روی کپی موقت Index، بدون تغییر مخزن.

**ورودی‌ها:**
- خروجی‌های معتبر C002 تا C004.
- Snapshot مدل ONNX با هش تأییدشده.
- مسیر Qdrant و قراردادهای فریز‌شدهٔ Index.

**عملیات:**
۱. کنترل تطبیق تنظیمات Release و Handoff.
۲. بارگذاری Tokenizer فقط از Cache محلی، بدون دانلود.
۳. بررسی محدود اندازه و تعداد فایل‌های Index.
۴. در صورت کافی‌بودن منابع، تهیهٔ کپی موقت بیرون مخزن.
۵. بازکردن فقط کپی Qdrant و کنترل Collection و تعداد Pointها.
۶. بستن Qdrant و حذف خودکار کپی موقت.

**خروجی:** گزارش وضعیت Tokenizer، قابلیت خواندن Collection و موانع باقی‌مانده.

**محدودیت‌ها:**
- از اصل Qdrant هیچ Client ساخته نمی‌شود.
- حداکثر حجم کپی: ۵۱۲ مگابایت.
- هیچ Retrieval، رتبه‌بندی یا استنتاج مدل اجرا نمی‌شود.
- کنترل Collection به‌تنهایی معادل تأیید کامل Logical Index Hash نیست.
- هیچ نسخهٔ جدیدی تصویب نمی‌شود.

**API:** صفر.

**تغییر مخزن:** صفر.

**شرط پایان:** ثبت وضعیت واقعی هر دو کنترل و تعیین امکان طراحی Pilot محدود، بدون تغییر Gate رسمی.

</div>

In [5]:
# NB22E-v4 | C005 | R11.2
# Local tokenizer and isolated Qdrant-clone verification; no retrieval, inference or repository writes.

# ============================================================
# 1. Validate the successful prior checkpoints
# ============================================================

if not (
    NB22E_V4_READ_ONLY is True
    and NB22E_V4_TRANSFER.get("status") == "TRANSFER_LOADER_PASS"
    and NB22E_V4_BOUNDARY_PROPOSAL.get("status") == "DRAFT_RELOAD_PASS"
    and NB22E_V4_RUNTIME_PREFLIGHT.get("status")
    == "LIMITED_READ_ONLY_PREFLIGHT_COMPLETE"
    and NB22E_V4_RUNTIME_PREFLIGHT.get("index_metadata_files_verified") == 4
    and NB22E_V4_RUNTIME_PREFLIGHT.get("model_file_status")
    == "ONNX_FILE_HASH_MATCHED"
    and NB22E_V4_RUNTIME_PREFLIGHT.get("onnxruntime_version_matches") is True
):
    raise RuntimeError("C005_REQUIRES_SUCCESSFUL_C002_C003_C004")

c005_root = c002_root.resolve()
c005_out = c003_out.resolve()
c005_source = c004_index_path.resolve()

if (
    c005_out.is_relative_to(c005_root)
    or not c005_source.is_relative_to(c005_root)
    or not c005_source.is_dir()
):
    raise RuntimeError("C005_UNSAFE_REPOSITORY_BOUNDARY")

# ============================================================
# 2. Bind the previously verified release and handoff
# ============================================================

c005_release = c004_docs["release"]
c005_handoff = c004_docs["handoff"]

c005_backend = c005_release.get("backend")

if not isinstance(c005_backend, dict):
    raise RuntimeError("C005_RELEASE_BACKEND_SCHEMA")

c005_collection = c005_backend.get("collection_name")
c005_storage_relative = c005_backend.get("storage_path")

c005_logical_sha = (
    "2fc5e50ebbd1e1e66d59c703eb48f1367b2d3deadfed62efb9d5a5e6959f7ef4"
)

if not (
    isinstance(c005_collection, str)
    and bool(c005_collection)
    and c005_collection == c005_handoff.get("collection_name")
    and c005_storage_relative == c005_handoff.get("storage_path")
    and c005_storage_relative == c004_expected_index_relative
    and c005_release.get("logical_index_sha256") == c005_logical_sha
    and c005_handoff.get("logical_index_sha256") == c005_logical_sha
    and c005_release.get("status") == "FROZEN_FOR_DOWNSTREAM_RETRIEVAL"
    and c005_release.get("final_status") == "PASS_COMPLETE_FROZEN"
    and c005_handoff.get("status") == "READY_FOR_NB16"
):
    raise RuntimeError("C005_INDEX_RELEASE_HANDOFF_CONFLICT")

# ============================================================
# 3. Load the exact local tokenizer without loading the model
# ============================================================

c005_tokenizer_result = {
    "status": "NOT_ATTEMPTED",
    "snapshot": str(c004_snapshot),
    "local_files_only": True,
    "trust_remote_code": False,
    "model_loaded": False,
    "inference_calls": 0,
}

try:
    c005_auto_tokenizer = nb22e_v4_require_imports(
        "AutoTokenizer"
    )["AutoTokenizer"]

    c005_tokenizer = c005_auto_tokenizer.from_pretrained(
        str(c004_snapshot),
        local_files_only=True,
        trust_remote_code=False,
    )

    # Structural smoke input, not a Test-dataset evaluation.
    c005_encoded = c005_tokenizer(
        "Local runtime check",
        "Local context check",
        truncation=True,
        max_length=512,
        return_attention_mask=True,
    )

    c005_input_ids = c005_encoded.get("input_ids")
    c005_attention = c005_encoded.get("attention_mask")

    if not (
        isinstance(c005_input_ids, list)
        and isinstance(c005_attention, list)
        and len(c005_input_ids) > 0
        and len(c005_input_ids) == len(c005_attention)
        and len(c005_input_ids) <= 512
    ):
        raise RuntimeError("C005_TOKENIZER_ENCODING_CONFLICT")

    c005_tokenizer_result.update({
        "status": "LOCAL_TOKENIZER_LOAD_AND_ENCODE_PASS",
        "tokenizer_class": type(c005_tokenizer).__name__,
        "is_fast": bool(c005_tokenizer.is_fast),
        "smoke_token_count": len(c005_input_ids),
        "max_length_used": 512,
    })

except Exception as exc:
    c005_tokenizer_result.update({
        "status": "TOKENIZER_RUNTIME_CHECK_FAILED",
        "error_type": type(exc).__name__,
        "error": str(exc)[:500],
    })

# ============================================================
# 4. Inventory only the pinned Qdrant directory
# ============================================================

c005_max_copy_bytes = 512 * 1024 * 1024
c005_max_files = 4000

c005_files = []
c005_total_bytes = 0
c005_inventory_error = None

try:
    for directory, dirs, files in os.walk(
        c005_source,
        topdown=True,
        followlinks=False,
    ):
        directory_path = Path(directory)

        for name in dirs + files:
            entry = directory_path / name

            if entry.is_symlink():
                raise RuntimeError("INDEX_CONTAINS_SYMLINK")

        for name in files:
            source_file = directory_path / name

            if not source_file.is_file():
                raise RuntimeError("INDEX_CONTAINS_NONREGULAR_FILE")

            stat = source_file.stat()
            relative = source_file.relative_to(c005_source)

            c005_files.append({
                "relative": relative,
                "size": stat.st_size,
                "mtime_ns": stat.st_mtime_ns,
            })

            c005_total_bytes += stat.st_size

            if len(c005_files) > c005_max_files:
                raise RuntimeError("INDEX_FILE_COUNT_EXCEEDS_LIMIT")

            if c005_total_bytes > c005_max_copy_bytes:
                raise RuntimeError("INDEX_SIZE_EXCEEDS_COPY_LIMIT")

except Exception as exc:
    c005_inventory_error = {
        "error_type": type(exc).__name__,
        "error": str(exc)[:500],
    }

c005_index_result = {
    "status": "NOT_OPENED",
    "original_index_opened": False,
    "original_index_modified": False,
    "copy_location": "TEMPORARY_OUTSIDE_REPOSITORY",
    "copy_limit_bytes": c005_max_copy_bytes,
    "inventory_files": len(c005_files),
    "inventory_bytes": c005_total_bytes,
    "inventory_error": c005_inventory_error,
    "collection_name": c005_collection,
    "collection_readable": False,
    "exact_count": None,
    "reported_points_count": None,
    "logical_index_hash_reverified": False,
    "qdrant_queries": 0,
    "temporary_copy_removed": True,
}

# ============================================================
# 5. Check disk capacity before creating any temporary copy
# ============================================================

c005_copy_allowed = (
    c005_inventory_error is None
    and len(c005_files) > 0
    and c005_total_bytes > 0
)

if c005_copy_allowed:
    c005_fs = os.statvfs(str(c005_out))
    c005_free_bytes = c005_fs.f_bavail * c005_fs.f_frsize

    c005_required_free = max(
        c005_total_bytes * 2,
        256 * 1024 * 1024,
    )

    c005_index_result["free_bytes_before_copy"] = c005_free_bytes
    c005_index_result["required_free_bytes"] = c005_required_free

    if c005_free_bytes < c005_required_free:
        c005_copy_allowed = False
        c005_index_result["status"] = "INSUFFICIENT_SPACE_FOR_SAFE_COPY"

elif c005_inventory_error is not None:
    c005_index_result["status"] = "INDEX_INVENTORY_BLOCKED"

else:
    c005_index_result["status"] = "INDEX_DIRECTORY_EMPTY"

# ============================================================
# 6. Verify Collection using an isolated temporary working copy
# ============================================================

if c005_copy_allowed:
    c005_client = None

    try:
        with tempfile.TemporaryDirectory(
            prefix="NB22E-v4-C005-qdrant-",
            dir=str(c005_out),
        ) as temporary_root:

            c005_working = Path(temporary_root) / "qdrant"
            c005_working.mkdir()

            print(
                "C005 isolated index copy:",
                len(c005_files),
                "files,",
                c005_total_bytes,
                "bytes"
            )

            # Copy source files only; no Qdrant client touches the source.
            for item in c005_files:
                source_file = c005_source / item["relative"]
                target_file = c005_working / item["relative"]

                stat_before = source_file.stat()

                if (
                    stat_before.st_size != item["size"]
                    or stat_before.st_mtime_ns != item["mtime_ns"]
                    or source_file.is_symlink()
                ):
                    raise RuntimeError(
                        "INDEX_CHANGED_BEFORE_COPY:"
                        + item["relative"].as_posix()
                    )

                target_file.parent.mkdir(
                    parents=True,
                    exist_ok=True,
                )

                copied_bytes = 0
                source_digest = hashlib.sha256()

                with (
                    source_file.open("rb") as source_handle,
                    target_file.open("xb") as target_handle,
                ):
                    while True:
                        block = source_handle.read(1024 * 1024)

                        if not block:
                            break

                        source_digest.update(block)
                        target_handle.write(block)
                        copied_bytes += len(block)

                stat_after = source_file.stat()

                if (
                    copied_bytes != item["size"]
                    or stat_after.st_size != item["size"]
                    or stat_after.st_mtime_ns != item["mtime_ns"]
                    or source_digest.hexdigest()
                    != hashlib.file_digest(
                        target_file.open("rb"),
                        "sha256",
                    ).hexdigest()
                ):
                    raise RuntimeError(
                        "INDEX_COPY_INTEGRITY_CONFLICT:"
                        + item["relative"].as_posix()
                    )

            # The only opened Qdrant storage is the isolated copy.
            c005_qdrant_class = nb22e_v4_require_imports(
                "QdrantClient"
            )["QdrantClient"]

            try:
                c005_client = c005_qdrant_class(
                    path=str(c005_working)
                )

                c005_info = c005_client.get_collection(
                    collection_name=c005_collection
                )

                c005_count = c005_client.count(
                    collection_name=c005_collection,
                    exact=True,
                )

                c005_exact_count = int(c005_count.count)
                c005_reported_count = c005_info.points_count

                c005_count_consistent = (
                    c005_reported_count is None
                    or int(c005_reported_count) == c005_exact_count
                )

                if not c005_count_consistent:
                    raise RuntimeError(
                        "QDRANT_REPORTED_AND_EXACT_COUNT_DIFFER"
                    )

                c005_index_result.update({
                    "status": "ISOLATED_COLLECTION_READ_PASS",
                    "collection_readable": True,
                    "exact_count": c005_exact_count,
                    "reported_points_count": c005_reported_count,
                    "collection_status": str(c005_info.status),
                    "snapshot_files_copied": len(c005_files),
                    "snapshot_bytes_copied": c005_total_bytes,
                    "logical_index_hash_reverified": False,
                })

            finally:
                if c005_client is not None:
                    c005_client.close()
                    c005_client = None

    except Exception as exc:
        c005_index_result.update({
            "status": "ISOLATED_INDEX_CHECK_FAILED",
            "error_type": type(exc).__name__,
            "error": str(exc)[:700],
            "logical_index_hash_reverified": False,
        })

# ============================================================
# 7. Report the exact remaining readiness boundaries
# ============================================================

c005_tokenizer_pass = (
    c005_tokenizer_result["status"]
    == "LOCAL_TOKENIZER_LOAD_AND_ENCODE_PASS"
)

c005_clone_pass = (
    c005_index_result["status"]
    == "ISOLATED_COLLECTION_READ_PASS"
)

NB22E_V4_ISOLATED_RUNTIME_CHECK = {
    "cell": "NB22E-v4:C005",
    "status": (
        "LIMITED_RUNTIME_CHECK_PASS_WITH_VERSION_PENDING"
        if c005_tokenizer_pass and c005_clone_pass
        else "LIMITED_RUNTIME_CHECK_HAS_OPEN_ITEMS"
    ),
    "tokenizer": c005_tokenizer_result,
    "index": c005_index_result,
    "frozen_model_file_hash_verified_in_C004": True,
    "historical_parity_preserved": True,
    "new_population_version_approved": False,
    "live_original_index_opened": False,
    "logical_index_hash_reverified": False,
    "local_pilot_authorized": False,
    "full_population_execution_authorized": False,
    "target_release_gate": "BLOCKED",
    "api_calls": 0,
    "retrieval_runs": 0,
    "rank_runs": 0,
    "model_inference_calls": 0,
    "repository_modified": False,
    "next": (
        "VERSION_EQUIVALENCE_AND_BOUNDED_PILOT_DESIGN"
        if c005_tokenizer_pass and c005_clone_pass
        else "REPAIR_ONLY_REPORTED_RUNTIME_BLOCKERS"
    ),
}

print(json.dumps(
    NB22E_V4_ISOLATED_RUNTIME_CHECK,
    ensure_ascii=False,
    indent=2,
))

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C005 isolated index copy: 3 files, 46338588 bytes
{
  "cell": "NB22E-v4:C005",
  "status": "LIMITED_RUNTIME_CHECK_PASS_WITH_VERSION_PENDING",
  "tokenizer": {
    "status": "LOCAL_TOKENIZER_LOAD_AND_ENCODE_PASS",
    "snapshot": "/Users/macbookpro/.cache/huggingface/hub/models--cross-encoder--mmarco-mMiniLMv2-L12-H384-v1/snapshots/1427fd652930e4ba29e8149678df786c240d8825",
    "local_files_only": true,
    "trust_remote_code": false,
    "model_loaded": false,
    "inference_calls": 0,
    "tokenizer_class": "XLMRobertaTokenizerFast",
    "is_fast": true,
    "smoke_token_count": 11,
    "max_length_used": 512
  },
  "index": {
    "status": "ISOLATED_COLLECTION_READ_PASS",
    "original_index_opened": false,
    "original_index_modified": false,
    "copy_location": "TEMPORARY_OUTSIDE_REPOSITORY",
    "copy_limit_bytes": 536870912,
    "inventory_files": 3,
    "inventory_bytes": 46338588,
    "inventory_error": null,
    "collection_name": "rag_finance_phase2_dense_v1",
    "collecti

<div dir="rtl" align="right">

### NB22E-v4:C006 — R11.2: تثبیت طرح Pilot محدود و مرز نسخه

**نتیجهٔ قبلی:** در C005، بارگذاری Tokenizer و خواندن Collection روی کپی موقت Qdrant موفق بود. تعداد ۳۴۸۸ Point تأیید شد. اعتبار Logical Index Hash و نسخهٔ جمعیتی هنوز نهایی نشده‌اند.

**هدف:** ساخت بستهٔ اجرایی نسخه‌دار برای یک Pilot محدود، بدون شروع محاسبات یا استفاده از Test برای تنظیم الگوریتم.

**ورودی‌ها:**
- اتصال ۱۱۱۶ سؤال از C002.
- پیش‌نویس مرز نسخه از C003.
- قراردادها و هش مدل از C004.
- شواهد کنترل محلی C005.
- تنظیمات فریز‌شدهٔ C118، C119 و مسیر Producer ثبت‌شده در C342.

**عملیات:**

۱. انتخاب مکانیکی چهار شناسهٔ آزمایشی، مستقل از متن پاسخ، امتیاز و معیارهای کیفیت.

۲. تعیین دو مرحلهٔ مشروط Pilot:
- بررسی تطبیق Hybrid100 با نسخهٔ فریز‌شده.
- بررسی حداکثر ۲۰۰ جفت اضافی Local100، فقط پس از احراز شرایط مرحلهٔ اول.

۳. حفظ تنظیمات تاریخی:
- ONNX Runtime روی CPU
- چهار Thread محاسباتی
- Batch Size برابر ۴
- حداکثر طول توالی ۵۱۲

۴. تعیین محدودیت حافظه، مسیر Checkpoint، ثبت خطا و شرایط توقف.

۵. ثبت وابستگی به تأیید مرز نسخه و مجوز Pilot.

۶. نوشتن و Reload یک Artifact برنامه‌ریزی مستقل خارج از مخزن.

**موارد محفوظ:**
- ۲۹ اجرای دقیق تأییدشده.
- دو هویت نامحرز.
- دو خطای ثبت‌شده.
- نسخهٔ تاریخی و تمام فایل‌های فریز‌شده.

**خروجی:** پیش‌نویس `C006.pilot_boundary_plan` با هش معتبر.

**API و مدل:** هیچ اجرایی انجام نمی‌شود.

**تغییر مخزن:** صفر.

**شرط پایان:** ثبت طرح قابل‌بررسی و تعیین پیش‌نیازهای اجرای محدود؛ بدون اعطای هویت جدید یا تصویب Gate.

</div>

In [6]:
# NB22E-v4 | C006 | R11.2
# Prepare a bounded, version-linked pilot plan; no retrieval, inference, API or repository writes.

# ============================================================
# 1. Validate completed checkpoints
# ============================================================

def c006_require(condition, message):
    if not condition:
        raise RuntimeError("C006_" + message)

c006_require(
    NB22E_V4_READ_ONLY is True
    and NB22E_V4_TRANSFER["status"] == "TRANSFER_LOADER_PASS"
    and NB22E_V4_BOUNDARY_PROPOSAL["status"] == "DRAFT_RELOAD_PASS"
    and NB22E_V4_RUNTIME_PREFLIGHT["status"]
    == "LIMITED_READ_ONLY_PREFLIGHT_COMPLETE"
    and NB22E_V4_ISOLATED_RUNTIME_CHECK["status"]
    == "LIMITED_RUNTIME_CHECK_PASS_WITH_VERSION_PENDING",
    "PREVIOUS_CHECKPOINT_NOT_READY",
)

c006_require(
    len(c002_questions) == 1116
    and len(c002_remaining) == 0
    and c005_index_result["status"] == "ISOLATED_COLLECTION_READ_PASS"
    and c005_index_result["exact_count"] == 3488
    and c005_index_result["original_index_opened"] is False
    and c005_tokenizer_result["status"]
    == "LOCAL_TOKENIZER_LOAD_AND_ENCODE_PASS"
    and c004_model_status == "ONNX_FILE_HASH_MATCHED"
    and c004_runtime_version_matches is True,
    "RUNTIME_OR_POPULATION_CONFLICT",
)

c006_vector = c002_summary["query_vector_precheck"]

c006_require(
    c006_vector.get("vector_candidate_id_coverage") == 1116
    and c006_vector.get("cache_hashes_reverified") is True
    and all(
        value.get("status") == "FILE_BYTES_VERIFIED"
        for value in c006_vector["cache_file_checks"].values()
    ),
    "VECTOR_CACHE_RECEIPT_CONFLICT",
)

c006_require(
    all(
        value is True
        for value in c002_summary["producer_contract_checks"].values()
    ),
    "FROZEN_PRODUCER_CONTRACT_CONFLICT",
)

# ============================================================
# 2. Select four IDs without examining evaluation outcomes
# ============================================================

c006_population = sorted(c002_questions)

c006_selection_order = sorted(
    c006_population,
    key=lambda candidate_id: (
        hashlib.sha256(candidate_id.encode("utf-8")).hexdigest(),
        candidate_id,
    ),
)

c006_pilot_ids = c006_selection_order[:4]

c006_require(
    len(c006_pilot_ids) == len(set(c006_pilot_ids)) == 4,
    "PILOT_ID_SELECTION_CONFLICT",
)

c006_selected = [
    {
        "candidate_id": candidate_id,
        "question_sha256": c002_questions[candidate_id][
            "question_sha256"
        ],
    }
    for candidate_id in c006_pilot_ids
]

# Selection uses identifiers only, never scores or outcomes.

# ============================================================
# 3. Bind the verified frozen runtime and input lineage
# ============================================================

c006_expected_model = {
    "model_id": "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",
    "model_revision": "1427fd652930e4ba29e8149678df786c240d8825",
    "onnx_sha256": (
        "6c2513767fb63d008a4377bef7a7a3555433d9436342bb53e35a3a72ffc52d4b"
    ),
    "onnxruntime_version": "1.16.3",
    "batch_size": 4,
    "maximum_sequence_length": 512,
}

c006_require(
    c004_model_actual_sha == c006_expected_model["onnx_sha256"]
    and c004_onnxruntime_version
    == c006_expected_model["onnxruntime_version"]
    and c005_logical_sha
    == c004_docs["release"]["logical_index_sha256"],
    "FROZEN_RUNTIME_IDENTITY_CONFLICT",
)

c006_source_pins = {
    role: {
        "path": record["path"],
        "file_bytes_sha256": record["file_bytes_sha256"],
    }
    for role, record in c002_summary["source_checks"].items()
}

c006_input_pins = {
    role: {
        "path": record["path"],
        "file_bytes_sha256": record["file_bytes_sha256"],
    }
    for role, record in c002_summary["input_file_checks"].items()
}

# ============================================================
# 4. Freeze the planning boundary, not a target release
# ============================================================

c006_core = {
    "schema": "NB22E_V4_R11_2_BOUNDED_PILOT_PLAN_V1",
    "cell": "NB22E-v4:C006",
    "subtask": "R11.2",
    "status": "PILOT_PLAN_PREPARED_APPROVAL_PENDING",

    "population": {
        "total_test_questions": 1149,
        "new_candidate_count": 1116,
        "pilot_candidate_count": 4,
        "selection_method": (
            "FIRST_FOUR_SHA256_SORTED_CANDIDATE_IDS"
        ),
        "pilot_candidates": c006_selected,
        "selection_uses_test_scores": False,
        "selection_uses_test_answers": False,
        "question_binding_sha256": c002_specs["questions"]["sha"],
        "manifest_sha256": c002_specs["manifest"]["sha"],
    },

    "version_boundary": {
        "historical_target_identity": (
            c002_summary["target_identity_sha256_original"]
        ),
        "historical_rank_mode": (
            c002_summary["original_rank_input_mode"]
        ),
        "boundary_proposal_file_sha256": (
            NB22E_V4_BOUNDARY_PROPOSAL["file_bytes_sha256"]
        ),
        "boundary_proposal_content_sha256": (
            NB22E_V4_BOUNDARY_PROPOSAL["proposal_content_sha256"]
        ),
        "new_population_identity": "UNASSIGNED",
        "version_approval": "PENDING",
        "historical_hybrid100_equivalence": "NOT_YET_PROVEN",
        "pool50_version_membership": "NOT_YET_PROVEN",
        "new_rank_receipts": "NOT_YET_PRODUCED",
        "new_context_receipts": "NOT_YET_PRODUCED",
        "target_generation_authorized": False,
    },

    "verified_inputs": {
        "source_pins": c006_source_pins,
        "input_pins": c006_input_pins,
        "vector_cache_pointers": (
            c006_vector["cache_file_pointers"]
        ),
        "producer_contract_checks": (
            c002_summary["producer_contract_checks"]
        ),
    },

    "index": {
        "collection": c005_collection,
        "frozen_logical_index_sha256": c005_logical_sha,
        "verified_clone_point_count": 3488,
        "verified_clone_file_count": (
            c005_index_result["inventory_files"]
        ),
        "verified_clone_inventory_bytes": (
            c005_index_result["inventory_bytes"]
        ),
        "original_index_opened": False,
        "logical_index_hash_reverified": False,
        "pilot_index_policy": (
            "ISOLATED_WORKING_COPY_OUTSIDE_REPOSITORY"
        ),
    },

    "local_runtime": {
        **c006_expected_model,
        "provider": "CPUExecutionProvider",
        "intra_op_num_threads": 4,
        "inter_op_num_threads": 1,
        "execution_mode": "ORT_SEQUENTIAL",
        "candidate_parallelism": 1,
        "tokenizer_local_only": True,
        "model_download_allowed": False,
        "historical_four_pair_parity_preserved": True,
    },

    "pilot": {
        "stage_1": {
            "name": "VERIFY_FROZEN_HYBRID100_EQUIVALENCE",
            "candidate_count_max": 4,
            "retrieval_queries_max": 4,
            "historical_order_reuse_unconditional": False,
            "requires_version_approval": True,
            "status": "NOT_STARTED",
        },
        "stage_2": {
            "name": "LOCAL100_REMAINING_50_PAIR_CHECK",
            "candidate_count_max": 4,
            "additional_pairs_max": 200,
            "onnx_batch_size": 4,
            "requires_exact_hybrid100_membership": True,
            "requires_pool50_version_membership": True,
            "requires_stage_1_success": True,
            "status": "NOT_STARTED",
        },
        "after_pilot": {
            "rank_producer": "PINNED_NB22C_AND_NB22D",
            "context_boundary": "PINNED_C139_AND_PIPELINE",
            "configuration_tuning_on_test": False,
            "full_population_execution": "NOT_AUTHORIZED",
        },
    },

    "resource_controls": {
        "host_memory_gib_reported": 16,
        "proposed_pilot_rss_stop_threshold_gib": 6,
        "candidate_processing": "SEQUENTIAL",
        "checkpoint_frequency": "AFTER_EACH_COMPLETED_CANDIDATE",
        "checkpoint_write_policy": "ATOMIC_OUTSIDE_REPOSITORY",
        "unknown_outcomes_auto_retried": False,
        "pilot_runtime_estimate": "UNMEASURED",
        "pilot_peak_memory_estimate": "UNMEASURED",
        "full_population_runtime_estimate": "NOT_ESTIMATED",
        "pilot_time_and_memory_to_be_measured": True,
    },

    "approval_and_safety": {
        "version_boundary_approval": "PENDING_SUPERVISOR",
        "bounded_local_pilot_approval": "PENDING_SUPERVISOR",
        "execution_allowed_now": False,
        "generation_api_allowed": False,
        "existing_results_preserved": {
            "EXACT_PRESERVED": 29,
            "MODEL_IDENTITY_HELD": 2,
            "RECORDED_ERROR_HELD": 2,
            "OUTSIDE_PACKET": 1116,
        },
        "repository_read_only": True,
        "historical_artifacts_immutable": True,
        "target_release_gate": "BLOCKED",
    },
}

# ============================================================
# 5. Content-bound fingerprint
# ============================================================

c006_canonical_bytes = json.dumps(
    c006_core,
    ensure_ascii=False,
    sort_keys=True,
    separators=(",", ":"),
    allow_nan=False,
).encode("utf-8")

c006_content_sha = hashlib.sha256(
    c006_canonical_bytes
).hexdigest()

c006_document = {
    **c006_core,
    "plan_content_sha256": c006_content_sha,
    "fingerprint_kind": "CANONICAL_JSON_PLAN_CONTENT",
}

c006_bytes = json.dumps(
    c006_document,
    ensure_ascii=False,
    sort_keys=True,
    separators=(",", ":"),
    allow_nan=False,
).encode("utf-8") + b"\n"

c006_file_sha = hashlib.sha256(c006_bytes).hexdigest()

# ============================================================
# 6. Atomic, immutable output outside the repository
# ============================================================

c006_output_dir = c003_out.resolve()
c006_root = c002_root.resolve()

c006_require(
    c006_output_dir.is_dir()
    and not c006_output_dir.is_relative_to(c006_root),
    "OUTPUT_DIRECTORY_UNSAFE",
)

c006_target = c006_output_dir / (
    "C006.pilot_boundary_plan."
    + c006_file_sha
    + ".json"
)

c006_temp = None

if not c006_target.exists():
    with tempfile.NamedTemporaryFile(
        mode="wb",
        dir=c006_output_dir,
        prefix=".C006.pilot.",
        suffix=".tmp",
        delete=False,
    ) as handle:
        c006_temp = Path(handle.name)
        handle.write(c006_bytes)
        handle.flush()
        os.fsync(handle.fileno())

    try:
        c006_require(
            c006_temp.read_bytes() == c006_bytes,
            "TEMP_RELOAD_CONFLICT",
        )

        try:
            os.link(c006_temp, c006_target)
        except FileExistsError:
            pass
    finally:
        c006_temp.unlink(missing_ok=True)

# ============================================================
# 7. Independent output reload and report
# ============================================================

c006_reloaded_bytes = c006_target.read_bytes()

c006_require(
    c006_reloaded_bytes == c006_bytes
    and hashlib.sha256(c006_reloaded_bytes).hexdigest()
    == c006_file_sha,
    "FINAL_FILE_BYTES_CONFLICT",
)

c006_reloaded = json.loads(c006_reloaded_bytes)

c006_reloaded_core = {
    key: value
    for key, value in c006_reloaded.items()
    if key not in ("plan_content_sha256", "fingerprint_kind")
}

c006_reloaded_core_bytes = json.dumps(
    c006_reloaded_core,
    ensure_ascii=False,
    sort_keys=True,
    separators=(",", ":"),
    allow_nan=False,
).encode("utf-8")

c006_require(
    hashlib.sha256(c006_reloaded_core_bytes).hexdigest()
    == c006_reloaded["plan_content_sha256"]
    == c006_content_sha
    and c006_reloaded["pilot"]["stage_1"]["status"]
    == "NOT_STARTED"
    and c006_reloaded["pilot"]["stage_2"]["status"]
    == "NOT_STARTED"
    and c006_reloaded["approval_and_safety"]["execution_allowed_now"]
    is False,
    "PLAN_RELOAD_VALIDATION_CONFLICT",
)

NB22E_V4_PILOT_PLAN = {
    "status": "PILOT_PLAN_RELOAD_PASS",
    "path": str(c006_target),
    "file_bytes_sha256": c006_file_sha,
    "plan_content_sha256": c006_content_sha,
    "candidate_ids": c006_pilot_ids,
    "pilot_authorized": False,
}

print(json.dumps({
    "cell": "NB22E-v4:C006",
    "status": "PILOT_PLAN_RELOAD_PASS",
    "population": 1116,
    "pilot_candidate_count": 4,
    "pilot_candidate_ids": c006_pilot_ids,
    "hybrid100_queries_max_if_approved": 4,
    "local100_additional_pairs_max_if_approved": 200,
    "onnx_batch_size": 4,
    "onnx_max_sequence_length": 512,
    "onnx_intra_op_threads": 4,
    "candidate_parallelism": 1,
    "model_file_verified": True,
    "tokenizer_verified": True,
    "isolated_qdrant_collection_verified": True,
    "logical_index_hash_reverified": False,
    "version_approval": "PENDING",
    "pilot_execution_authorized": False,
    "output_receipt": {
        "path": str(c006_target),
        "file_bytes_sha256": c006_file_sha,
        "size_bytes": len(c006_bytes),
    },
    "target_release_gate": "BLOCKED",
    "api_calls": 0,
    "retrieval_runs": 0,
    "rank_runs": 0,
    "model_inference_calls": 0,
    "repository_writes_by_cell": 0,
    "next": "SUPERVISOR_VERSION_AND_BOUNDED_PILOT_DECISION",
}, ensure_ascii=False, indent=2))

{
  "cell": "NB22E-v4:C006",
  "status": "PILOT_PLAN_RELOAD_PASS",
  "population": 1116,
  "pilot_candidate_count": 4,
  "pilot_candidate_ids": [
    "golden_v2_candidate_ca5362d68c692d667fad3006",
    "golden_v2_candidate_15cb48c40c3d1e292f647ff9",
    "golden_v2_candidate_4032db31c53d0465590cf0ab",
    "golden_v2_candidate_ba8ca208df074fc579e48eb5"
  ],
  "hybrid100_queries_max_if_approved": 4,
  "local100_additional_pairs_max_if_approved": 200,
  "onnx_batch_size": 4,
  "onnx_max_sequence_length": 512,
  "onnx_intra_op_threads": 4,
  "candidate_parallelism": 1,
  "model_file_verified": true,
  "tokenizer_verified": true,
  "isolated_qdrant_collection_verified": true,
  "logical_index_hash_reverified": false,
  "version_approval": "PENDING",
  "pilot_execution_authorized": false,
  "output_receipt": {
    "path": "/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/_RAG_finance_NB22E_v4_outputs/C006.pilot_boundary_plan.fbfa555c8d3ab8e9f54e05dee701349ee63bc3c2a5aaee41d80e6d5

<div dir="rtl" align="right">

### NB22E-v4:C007 — R11.2: تطبیق Logical Index Hash با قرارداد فریز‌شده

**هدف:** احراز همسانی واقعی محتوای Index با نسخهٔ فریز‌شده، پیش از هر Query آزمایشی.

**قرارداد مرجع:** `NB14 / Code Cell 10`

**روش محاسبه:**
- ترتیب Pointها از ترتیب اصلی رکوردهای فایل Chunk گرفته می‌شود.
- شناسهٔ هر Point با UUIDv5 و Namespace فریز‌شده ساخته می‌شود.
- برای هر Point، شناسهٔ UTF-8، کل Payload به‌شکل JSON کانونی و بردار little-endian float32 مصرف می‌شوند.
- پیش از هر جزء، طول آن به‌صورت unsigned big-endian هشت‌بایتی به SHA-256 افزوده می‌شود.
- نتیجه با هش فریز‌شدهٔ Release، Handoff و Manifest مقایسه می‌شود.

**ایمنی:** Qdrant اصلی باز نمی‌شود. محاسبه روی کپی موقت خارج مخزن، با Batch برابر ۱۲۸ Point انجام می‌شود.

**خروجی:** Checkpoint مستقل با وضعیت `LOGICAL_INDEX_HASH_MATCH` یا `BLOCKED`.

**شرط ادامه:** فقط تطبیق دقیق هش و Reload موفق Checkpoint اجازهٔ ورود به C008 می‌دهد.

**API، مدل و Retrieval:** صفر.

</div>

In [7]:
# NB22E-v4 | C007 | R11.2
# Verify frozen NB14 logical Index hash on an isolated Qdrant copy.

if not (
    NB22E_V4_READ_ONLY is True
    and NB22E_V4_PILOT_PLAN["status"] == "PILOT_PLAN_RELOAD_PASS"
    and NB22E_V4_RUNTIME_PREFLIGHT["index_metadata_files_verified"] == 4
    and NB22E_V4_ISOLATED_RUNTIME_CHECK["index"]["status"]
    == "ISOLATED_COLLECTION_READ_PASS"
):
    raise RuntimeError("C007_PRIOR_GATE_NOT_PASSED")

c007_plan_sha = (
    "fbfa555c8d3ab8e9f54e05dee701349ee63bc3c2a5aaee41d80e6d568e1c6213"
)

c007_plan_path = Path(NB22E_V4_PILOT_PLAN["path"]).resolve()

if not (
    c007_plan_path.parent == c003_out.resolve()
    and NB22E_V4_PILOT_PLAN["file_bytes_sha256"] == c007_plan_sha
    and hashlib.sha256(c007_plan_path.read_bytes()).hexdigest()
    == c007_plan_sha
):
    raise RuntimeError("C007_SUPERVISOR_APPROVED_PLAN_HASH_CONFLICT")

c007_root = c002_root.resolve()
c007_out = c003_out.resolve()
c007_original = c004_index_path.resolve()

c007_release = c004_docs["release"]
c007_handoff = c004_docs["handoff"]
c007_manifest = c004_docs["manifest"]

c007_expected = (
    "2fc5e50ebbd1e1e66d59c703eb48f1367b2d3deadfed62efb9d5a5e6959f7ef4"
)

c007_chunks_sha = c007_release["inputs"]["chunk_sha256"]

c007_chunks = (
    c007_root / c007_release["inputs"]["chunk_path"]
).resolve()

c007_source_nb14 = (
    c007_root / "notebooks/phase2/14_dense_index.ipynb"
).resolve()

if not (
    c007_out.is_dir()
    and not c007_out.is_relative_to(c007_root)
    and c007_original.is_relative_to(c007_root)
    and c007_original.is_dir()
    and c007_chunks.is_relative_to(c007_root)
    and c007_chunks.is_file()
    and c007_source_nb14.is_file()
    and c007_source_nb14.is_relative_to(c007_root)
    and c007_release["logical_index_sha256"] == c007_expected
    and c007_handoff["logical_index_sha256"] == c007_expected
    and c007_manifest["index_content"]["logical_index_sha256"]
    == c007_expected
    and c007_release["point_count"] == 3488
    and c007_manifest["index_content"]["point_count"] == 3488
    and c007_chunks_sha
    == c002_summary["input_file_checks"]["CHUNKS"]["file_bytes_sha256"]
):
    raise RuntimeError("C007_FROZEN_REFERENCE_OR_PATH_CONFLICT")

# Verify the original NB14 producer Git blob.
c007_nb14_bytes = c007_source_nb14.read_bytes()

c007_nb14_blob = hashlib.sha1(
    b"blob "
    + str(len(c007_nb14_bytes)).encode("ascii")
    + b"\0"
    + c007_nb14_bytes
).hexdigest()

c007_nb14_expected_blob = (
    "1f8a8a82f2f1fbadc8e05977d8b129572381f2bf"
)

if c007_nb14_blob != c007_nb14_expected_blob:
    raise RuntimeError("C007_NB14_FROZEN_PRODUCER_GIT_BLOB_CONFLICT")

c007_started = time.perf_counter()

c007_result = {
    "cell": "NB22E-v4:C007",
    "status": "BLOCKED",
    "producer": "NB14_CODE_CELL_10",
    "producer_git_blob_sha1": c007_nb14_blob,
    "expected_logical_index_sha256": c007_expected,
    "observed_logical_index_sha256": None,
    "collection": c007_release["backend"]["collection_name"],
    "point_count": 0,
    "batches": 0,
    "original_qdrant_opened": False,
    "repository_writes": 0,
    "retrieval_queries": 0,
    "model_inference_calls": 0,
    "api_calls": 0,
    "pilot_started": False,
    "version_identity_final": False,
}

try:
    # Preserve exact source order; do not sort chunk IDs.
    c007_ids = []
    c007_chunk_ids = []
    c007_chunk_digest = hashlib.sha256()

    c007_namespace = uuid.UUID(
        c007_manifest["point_identity"]["namespace"]
    )

    if not (
        c007_namespace.hex == "4dd4798997bb5e13bc0ffae8fa884248"
        and c007_manifest["point_identity"]["algorithm"] == "UUIDv5"
    ):
        raise RuntimeError("POINT_ID_CONTRACT_MISMATCH")

    with c007_chunks.open("rb") as handle:
        for line_number, line in enumerate(handle, 1):
            if (
                line_number > 3488
                or not line.strip()
                or len(line) > 2 * 1024 * 1024
            ):
                raise RuntimeError("CHUNK_JSONL_BOUNDS_OR_FORMAT")

            c007_chunk_digest.update(line)
            chunk_id = json.loads(line)["chunk_id"]

            if not isinstance(chunk_id, str) or not chunk_id:
                raise RuntimeError("INVALID_CHUNK_ID")

            c007_chunk_ids.append(chunk_id)
            c007_ids.append(
                str(uuid.uuid5(c007_namespace, chunk_id))
            )

    if not (
        len(c007_ids) == len(set(c007_ids)) == 3488
        and c007_chunk_digest.hexdigest() == c007_chunks_sha
    ):
        raise RuntimeError("CHUNK_ORDER_COUNT_OR_SHA256_CONFLICT")

    # Bounded inventory of the existing Index files.
    c007_files = []
    c007_total_bytes = 0

    for directory, dirs, files in os.walk(
        c007_original,
        followlinks=False,
    ):
        if any(
            (Path(directory) / name).is_symlink()
            for name in dirs + files
        ):
            raise RuntimeError("ORIGINAL_INDEX_HAS_SYMLINK")

        for filename in files:
            source = Path(directory) / filename

            if not source.is_file():
                raise RuntimeError("ORIGINAL_INDEX_HAS_NONREGULAR_FILE")

            stat = source.stat()

            c007_files.append((
                source.relative_to(c007_original),
                stat.st_size,
                stat.st_mtime_ns,
            ))

            c007_total_bytes += stat.st_size

            if (
                len(c007_files) > 4000
                or c007_total_bytes > 512 * 1024 * 1024
            ):
                raise RuntimeError("INDEX_COPY_SIZE_LIMIT")

    if not (
        len(c007_files) == c005_index_result["inventory_files"]
        and c007_total_bytes == c005_index_result["inventory_bytes"]
    ):
        raise RuntimeError("INDEX_INVENTORY_CHANGED_SINCE_C005")

    c007_fs = os.statvfs(str(c007_out))
    c007_free = c007_fs.f_bavail * c007_fs.f_frsize

    if c007_free < max(
        256 * 1024 * 1024,
        c007_total_bytes * 2,
    ):
        raise RuntimeError("INSUFFICIENT_FREE_DISK")

    c007_np = nb22e_v4_require_imports("np")["np"]
    c007_qdrant_class = nb22e_v4_require_imports(
        "QdrantClient"
    )["QdrantClient"]

    with tempfile.TemporaryDirectory(
        prefix="NB22E-v4-C007-index-",
        dir=str(c007_out),
    ) as working_directory:

        clone = Path(working_directory) / "qdrant"
        clone.mkdir()

        # Copy and verify bytes; never instantiate Qdrant on original.
        for relative, size, modified in c007_files:
            source = c007_original / relative
            target = clone / relative

            before = source.stat()

            if (
                source.is_symlink()
                or before.st_size != size
                or before.st_mtime_ns != modified
            ):
                raise RuntimeError(
                    "ORIGINAL_CHANGED_BEFORE_COPY:" + str(relative)
                )

            target.parent.mkdir(parents=True, exist_ok=True)
            source_digest = hashlib.sha256()

            with (
                source.open("rb") as source_handle,
                target.open("xb") as target_handle,
            ):
                for block in iter(
                    lambda: source_handle.read(1024 * 1024),
                    b"",
                ):
                    source_digest.update(block)
                    target_handle.write(block)

            after = source.stat()

            if (
                after.st_size != size
                or after.st_mtime_ns != modified
            ):
                raise RuntimeError(
                    "ORIGINAL_CHANGED_DURING_COPY:" + str(relative)
                )

            target_digest = hashlib.sha256()

            with target.open("rb") as verify_handle:
                for block in iter(
                    lambda: verify_handle.read(1024 * 1024),
                    b"",
                ):
                    target_digest.update(block)

            if target_digest.digest() != source_digest.digest():
                raise RuntimeError(
                    "INDEX_COPY_BYTES_MISMATCH:" + str(relative)
                )

        client = None

        try:
            client = c007_qdrant_class(path=str(clone))
            collection = c007_result["collection"]

            if client.count(
                collection_name=collection,
                exact=True,
            ).count != 3488:
                raise RuntimeError("QDRANT_COUNT_MISMATCH")

            config = client.get_collection(
                collection_name=collection
            ).config.params.vectors

            if not (
                getattr(config, "size", None) == 384
                and str(
                    getattr(config, "distance", "")
                ).upper().split(".")[-1] == "COSINE"
            ):
                raise RuntimeError("QDRANT_VECTOR_CONFIG_MISMATCH")

            logical_digest = hashlib.sha256()

            # NB14 frozen algorithm: batches of 128, original chunk order.
            for start in range(0, 3488, 128):
                batch_ids = c007_ids[start:start + 128]

                points = client.retrieve(
                    collection_name=collection,
                    ids=batch_ids,
                    with_payload=True,
                    with_vectors=True,
                )

                by_id = {
                    str(point.id): point
                    for point in points
                }

                if (
                    len(by_id) != len(batch_ids)
                    or set(by_id) != set(batch_ids)
                ):
                    raise RuntimeError("QDRANT_POINT_ID_PARITY_CONFLICT")

                for point_id in batch_ids:
                    point = by_id[point_id]

                    if not isinstance(point.payload, dict):
                        raise RuntimeError("MISSING_POINT_PAYLOAD")

                    payload_bytes = json.dumps(
                        point.payload,
                        ensure_ascii=False,
                        sort_keys=True,
                        separators=(",", ":"),
                    ).encode("utf-8")

                    vector = c007_np.asarray(
                        point.vector,
                        dtype="<f4",
                    )

                    if not (
                        vector.shape == (384,)
                        and bool(c007_np.isfinite(vector).all())
                    ):
                        raise RuntimeError("INVALID_STORED_VECTOR")

                    vector_bytes = vector.tobytes(order="C")

                    for component in (
                        point_id.encode("utf-8"),
                        payload_bytes,
                        vector_bytes,
                    ):
                        logical_digest.update(
                            len(component).to_bytes(
                                8,
                                byteorder="big",
                                signed=False,
                            )
                        )
                        logical_digest.update(component)

                    c007_result["point_count"] += 1

                c007_result["batches"] += 1

                if (
                    c007_result["batches"] % 7 == 0
                    or c007_result["point_count"] == 3488
                ):
                    print(
                        "C007 logical hash progress:",
                        c007_result["point_count"],
                        "/ 3488",
                        flush=True,
                    )

            c007_result["observed_logical_index_sha256"] = (
                logical_digest.hexdigest()
            )

            if (
                c007_result["observed_logical_index_sha256"]
                != c007_expected
            ):
                raise RuntimeError(
                    "FROZEN_LOGICAL_INDEX_SHA256_MISMATCH"
                )

            c007_result["status"] = "LOGICAL_INDEX_HASH_MATCH"

        finally:
            if client is not None:
                client.close()

except Exception as error:
    c007_result["status"] = "BLOCKED"
    c007_result["blocker"] = {
        "type": type(error).__name__,
        "detail": str(error)[:600],
    }

c007_result["elapsed_seconds"] = round(
    time.perf_counter() - c007_started,
    3,
)

c007_result["pilot_stage_1_authorized_if_pass"] = (
    c007_result["status"] == "LOGICAL_INDEX_HASH_MATCH"
)

c007_result["pilot_stage_2_authorized"] = False
c007_result["target_release_gate"] = "BLOCKED"

# Immutable external checkpoint.
c007_bytes = json.dumps(
    c007_result,
    ensure_ascii=False,
    sort_keys=True,
    separators=(",", ":"),
    allow_nan=False,
).encode("utf-8") + b"\n"

c007_sha = hashlib.sha256(c007_bytes).hexdigest()

c007_target = c007_out / (
    "C007.logical_index_check."
    + c007_sha
    + ".json"
)

if not c007_target.is_file():
    with tempfile.NamedTemporaryFile(
        mode="wb",
        dir=c007_out,
        prefix=".C007-",
        suffix=".tmp",
        delete=False,
    ) as stream:
        c007_temp = Path(stream.name)
        stream.write(c007_bytes)
        stream.flush()
        os.fsync(stream.fileno())

    try:
        if c007_temp.read_bytes() != c007_bytes:
            raise RuntimeError("C007_TEMP_RELOAD_CONFLICT")

        try:
            os.link(c007_temp, c007_target)
        except FileExistsError:
            pass
    finally:
        c007_temp.unlink(missing_ok=True)

if (
    c007_target.read_bytes() != c007_bytes
    or hashlib.sha256(c007_target.read_bytes()).hexdigest()
    != c007_sha
):
    raise RuntimeError("C007_CHECKPOINT_RELOAD_CONFLICT")

NB22E_V4_LOGICAL_INDEX_CHECK = {
    **c007_result,
    "checkpoint_path": str(c007_target),
    "checkpoint_sha256": c007_sha,
}

print(json.dumps(
    NB22E_V4_LOGICAL_INDEX_CHECK,
    ensure_ascii=False,
    indent=2,
))

C007 logical hash progress: 896 / 3488
C007 logical hash progress: 1792 / 3488
C007 logical hash progress: 2688 / 3488
C007 logical hash progress: 3488 / 3488
{
  "cell": "NB22E-v4:C007",
  "status": "LOGICAL_INDEX_HASH_MATCH",
  "producer": "NB14_CODE_CELL_10",
  "producer_git_blob_sha1": "1f8a8a82f2f1fbadc8e05977d8b129572381f2bf",
  "expected_logical_index_sha256": "2fc5e50ebbd1e1e66d59c703eb48f1367b2d3deadfed62efb9d5a5e6959f7ef4",
  "observed_logical_index_sha256": "2fc5e50ebbd1e1e66d59c703eb48f1367b2d3deadfed62efb9d5a5e6959f7ef4",
  "collection": "rag_finance_phase2_dense_v1",
  "point_count": 3488,
  "batches": 28,
  "original_qdrant_opened": false,
  "repository_writes": 0,
  "retrieval_queries": 0,
  "model_inference_calls": 0,
  "api_calls": 0,
  "pilot_started": false,
  "version_identity_final": false,
  "elapsed_seconds": 4.472,
  "pilot_stage_1_authorized_if_pass": true,
  "pilot_stage_2_authorized": false,
  "target_release_gate": "BLOCKED",
  "checkpoint_path": "/Users/ma

<div dir="rtl" align="right">

### NB22E-v4:C008 — R11.2: Pilot مرحلهٔ اول، تطبیق Hybrid100

**پیش‌نیاز اجباری:** `C007.status = LOGICAL_INDEX_HASH_MATCH`

**هدف:** اجرای حداکثر چهار Query با همان چهار شناسهٔ C006 و مقایسهٔ دقیق ترتیب Hybrid100 با خروجی تاریخی.

**روش:**
- خواندن Cache بردار Test بدون تولید embedding.
- مصرف Sparse Index و Analyzer فریز‌شده.
- محاسبهٔ BM25 و Weighted Fusion با تنظیمات ثابت NB16.
- اجرای Dense Retrieval صرفاً روی کپی موقت Index خارج از مخزن.
- مقایسهٔ تمام ۱۰۰ رتبه و عضویت Pool50.
- ذخیرهٔ نتیجهٔ هر سؤال پیش از سؤال بعدی.

**خروجی:** Checkpointهای هر سؤال و Summary مرحلهٔ اول.

**حداکثر Query:** چهار.

**شرط ورود به Stage 2:** هر چهار ترتیب Hybrid100 دقیقاً برابر باشند، عضویت Pool50 تطبیق کند و Summary معتبر ذخیره شود.

**محدودیت:** موفقیت این مرحله به معنی تأیید کیفیت، تولید پاسخ، مجوز Full Run یا پذیرش R11.2 نیست.

</div>

In [8]:
# NB22E-v4 | C008 | R11.2
# Run approved Stage 1: up to four frozen Hybrid100 queries on an isolated Index copy.

if not (
    NB22E_V4_LOGICAL_INDEX_CHECK.get("status")
    == "LOGICAL_INDEX_HASH_MATCH"
    and NB22E_V4_LOGICAL_INDEX_CHECK.get("checkpoint_sha256")
    and NB22E_V4_PILOT_PLAN["file_bytes_sha256"]
    == "fbfa555c8d3ab8e9f54e05dee701349ee63bc3c2a5aaee41d80e6d568e1c6213"
):
    raise RuntimeError("C008_BLOCKED_UNTIL_C007_HASH_MATCH")

c008_root = c002_root.resolve()
c008_out = c003_out.resolve()
c008_ids = NB22E_V4_PILOT_PLAN["candidate_ids"]

if len(c008_ids) != 4 or len(set(c008_ids)) != 4:
    raise RuntimeError("C008_C006_FOUR_IDS_REQUIRED")

c008_np = nb22e_v4_require_imports("np")["np"]
c008_qdrant_class = nb22e_v4_require_imports(
    "QdrantClient"
)["QdrantClient"]

c008_started = time.perf_counter()
c008_attempts = 0
c008_results = []

def c008_require(condition, code):
    if not condition:
        raise RuntimeError("C008_" + code)

def c008_sha_file(path, expected):
    c008_require(
        path.is_file() and path.resolve().is_relative_to(c008_root),
        "UNSAFE_OR_MISSING_INPUT",
    )
    before = path.stat()
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    after = path.stat()

    c008_require(
        (before.st_size, before.st_mtime_ns)
        == (after.st_size, after.st_mtime_ns)
        and digest.hexdigest() == expected,
        "INPUT_SHA256_MISMATCH:" + path.name,
    )
    return path

def c008_read_file_pin(receipt):
    path = (c008_root / receipt["path"]).resolve()
    return c008_sha_file(path, receipt["file_bytes_sha256"])

def c008_write_once(label, value):
    raw = json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        allow_nan=False,
    ).encode("utf-8") + b"\n"

    digest = hashlib.sha256(raw).hexdigest()
    target = c008_out / (label + "." + digest + ".json")

    if not target.exists():
        with tempfile.NamedTemporaryFile(
            "wb",
            dir=c008_out,
            prefix=".C008-",
            suffix=".tmp",
            delete=False,
        ) as stream:
            tmp = Path(stream.name)
            stream.write(raw)
            stream.flush()
            os.fsync(stream.fileno())

        try:
            c008_require(
                tmp.read_bytes() == raw,
                "TEMP_RELOAD_MISMATCH",
            )
            try:
                os.link(tmp, target)
            except FileExistsError:
                pass
        finally:
            tmp.unlink(missing_ok=True)

    c008_require(
        target.read_bytes() == raw,
        "IMMUTABLE_FILE_CONFLICT",
    )

    return {
        "path": str(target),
        "file_bytes_sha256": digest,
        "size_bytes": len(raw),
    }

# Load only the pinned vector cache.
c008_vectors = c002_summary[
    "query_vector_precheck"
]["cache_file_pointers"]

c008_metadata_path = c008_read_file_pin(
    c008_vectors["metadata"]
)
c008_array_path = c008_read_file_pin(
    c008_vectors["array"]
)

c008_metadata = json.loads(
    c008_metadata_path.read_bytes()
)

c008_candidate_order = c008_metadata["candidate_ids"]

c008_require(
    len(c008_candidate_order)
    == len(set(c008_candidate_order))
    == 1149
    and set(c008_ids).issubset(c008_candidate_order),
    "VECTOR_CACHE_CANDIDATES",
)

c008_cache_array = c008_np.load(
    c008_array_path,
    mmap_mode="r",
    allow_pickle=False,
)

c008_require(
    c008_cache_array.shape == (1149, 384)
    and c008_cache_array.dtype == c008_np.float32,
    "VECTOR_CACHE_SHAPE",
)

c008_vector_positions = {
    cid: index
    for index, cid in enumerate(c008_candidate_order)
    if cid in c008_ids
}

# Pinned NB15 sparse contracts.
c008_sparse_handoff_ref = {
    "path": "artifacts/phase2/sparse_index/nb16_sparse_index_handoff_v1.json",
    "file_bytes_sha256": (
        "e4070684ba1662de35e2fbdf2737560aee1b7c67a694e6dab3bd98be2f578439"
    ),
}

c008_sparse_release_ref = {
    "path": "artifacts/phase2/sparse_index/sparse_index_release_v1.json",
    "file_bytes_sha256": (
        "879f52850a71629f02437c1ab1173e3f9e6c55ec5441ad4649784be63935c51b"
    ),
}

c008_handoff = json.loads(
    c008_read_file_pin(c008_sparse_handoff_ref).read_bytes()
)
c008_sparse_release = json.loads(
    c008_read_file_pin(c008_sparse_release_ref).read_bytes()
)

c008_require(
    c008_handoff["dense_pairing_contract"][
        "dense_logical_index_sha256"
    ] == NB22E_V4_LOGICAL_INDEX_CHECK[
        "observed_logical_index_sha256"
    ]
    and c008_handoff["sparse_release"]["sha256"]
    == c008_sparse_release_ref["file_bytes_sha256"]
    and c008_sparse_release["logical_index_sha256"]
    == c008_handoff["logical_index_sha256"]
    and c008_handoff["loader_contract"]["analyzer_id"]
    == "fa_join_zwnj_stopwords_v1",
    "SPARSE_VERSION_PAIRING_CONFLICT",
)

c008_loader = c008_handoff["loader_contract"]

c008_cfg_ref = {
    "path": c008_loader["configuration_path"],
    "file_bytes_sha256": c008_loader["configuration_file_sha256"],
}

c008_cfg = json.loads(
    c008_read_file_pin(c008_cfg_ref).read_bytes()
)

c008_require(
    c008_cfg["configuration_sha256"]
    == "96d66676d20bacd764ec9a163995dd48266c90c2acea1c8a5f77d95d9efb6ecc"
    and c008_cfg["configuration_status"]
    == "FROZEN_BEFORE_TEST_ACCESS"
    and float(c008_cfg["bm25"]["k1"]) == 1.2
    and float(c008_cfg["bm25"]["b"]) == 0.3,
    "SPARSE_CONFIG_CONFLICT",
)

c008_analyzer = c008_cfg["analyzer"]["definition"]
c008_stopwords = frozenset(
    c008_cfg["analyzer"]["stopwords"]
)

c008_require(
    c008_analyzer["analyzer_id"] == "fa_join_zwnj_stopwords_v1"
    and len(c008_stopwords) == 28,
    "SPARSE_ANALYZER_CONTRACT_CONFLICT",
)

# Frozen NB16 analyzer behavior.
def c008_tokens(text):
    config = c008_analyzer
    text = unicodedata.normalize(config["unicode_form"], text)

    if config["normalize_arabic_letters"]:
        text = text.translate(str.maketrans({
            "ي": "ی", "ى": "ی", "ك": "ک", "ة": "ه",
        }))

    if config["normalize_digits"]:
        text = text.translate(str.maketrans({
            "۰": "0", "۱": "1", "۲": "2", "۳": "3", "۴": "4",
            "۵": "5", "۶": "6", "۷": "7", "۸": "8", "۹": "9",
            "٠": "0", "١": "1", "٢": "2", "٣": "3", "٤": "4",
            "٥": "5", "٦": "6", "٧": "7", "٨": "8", "٩": "9",
        }))

    if config["lowercase_latin"]:
        text = text.lower()

    text = re.sub(r"(?<=\d)[,٬](?=\d)", "", text)
    text = re.sub(r"(?<=\d)٫(?=\d)", ".", text)
    text = text.replace("٪", " __percent__ ")
    text = text.replace("%", " __percent__ ")

    if config["zwnj_policy"] == "join":
        text = text.replace("\u200c", "")
    elif config["zwnj_policy"] == "split":
        text = text.replace("\u200c", " ")
    else:
        c008_require(
            config["zwnj_policy"] == "preserve",
            "UNKNOWN_ZWNJ_POLICY",
        )

    characters = []

    for character in text:
        if (
            config["remove_diacritics"]
            and unicodedata.category(character).startswith("M")
        ):
            continue

        if (
            character == "\u200c"
            or character == "."
            or character == "_"
            or character.isalpha()
            or character.isdigit()
        ):
            characters.append(character)
        else:
            characters.append(" ")

    words = [
        word.strip("._")
        for word in "".join(characters).split()
    ]

    return [
        word
        for word in words
        if word
        and not (
            config["remove_stopwords"]
            and word in c008_stopwords
        )
    ]

# Preserve the three frozen analyzer unit tests.
c008_require(
    c008_tokens("مي‌خواهم كسب‌وكار را بررسی کنم.")
    == ["میخواهم", "کسبوکار", "بررسی", "کنم"],
    "FROZEN_ANALYZER_UNIT_TEST_1",
)

c008_require(
    c008_tokens("بازدهی ۱۲٫۵٪ و مبلغ 1,250 تومان")
    == ["بازدهی", "12.5", "percent", "مبلغ", "1250", "تومان"],
    "FROZEN_ANALYZER_UNIT_TEST_2",
)

c008_require(
    c008_tokens("ETF، P/E و بهرهٔ مرکب چیست؟")
    == ["etf", "p", "e", "بهره", "مرکب", "چیست"],
    "FROZEN_ANALYZER_UNIT_TEST_3",
)

c008_query_tokens = {
    cid: c008_tokens(c002_questions[cid]["question"].strip())
    for cid in c008_ids
}

c008_relevant_tokens = set().union(
    *(set(tokens) for tokens in c008_query_tokens.values())
)

# Load only frozen sparse documents and relevant postings.
c008_document_ref = {
    "path": c008_loader["documents_path"],
    "file_bytes_sha256": c008_loader["documents_sha256"],
}
c008_postings_ref = {
    "path": c008_loader["postings_path"],
    "file_bytes_sha256": c008_loader["postings_sha256"],
}

c008_doc_path = c008_read_file_pin(c008_document_ref)
c008_postings_path = c008_read_file_pin(c008_postings_ref)

c008_docs = []

with c008_doc_path.open("rb") as handle:
    for line in handle:
        row = json.loads(line)
        c008_require(
            row["document_index"] == len(c008_docs),
            "SPARSE_DOCUMENT_ORDER",
        )
        c008_docs.append((
            row["document_id"],
            float(row["document_length"]),
        ))

c008_require(
    len(c008_docs) == 3488
    and [item[0] for item in c008_docs] == c007_chunk_ids,
    "SPARSE_DOCUMENTS_CHUNK_PARITY",
)

c008_doc_ids = [item[0] for item in c008_docs]

c008_doc_lengths = c008_np.asarray(
    [item[1] for item in c008_docs],
    dtype=c008_np.float64,
)

c008_average_doc_length = float(
    c008_cfg["index_statistics"]["average_document_length"]
)

c008_postings = {}

with c008_postings_path.open("rb") as handle:
    for line in handle:
        row = json.loads(line)

        if row["token"] not in c008_relevant_tokens:
            continue

        indexes = c008_np.asarray(
            row["document_indexes"],
            dtype=c008_np.int64,
        )

        frequencies = c008_np.asarray(
            row["term_frequencies"],
            dtype=c008_np.float64,
        )

        c008_require(
            len(indexes)
            == len(frequencies)
            == row["document_frequency"]
            and bool((indexes[:-1] < indexes[1:]).all()),
            "POSTINGS_DATA_CONFLICT",
        )

        c008_postings[row["token"]] = (
            indexes,
            frequencies,
        )

def c008_sparse_retrieve(candidate_id):
    scores = c008_np.zeros(
        3488,
        dtype=c008_np.float64,
    )

    for token, query_frequency in collections.Counter(
        c008_query_tokens[candidate_id]
    ).items():
        posting = c008_postings.get(token)

        if posting is None:
            continue

        indexes, frequencies = posting
        df = len(indexes)

        idf = math.log(
            1.0 + (3488 - df + 0.5) / (df + 0.5)
        )

        normalizer = (
            1.0 - 0.3
            + 0.3 * (
                c008_doc_lengths[indexes]
                / c008_average_doc_length
            )
        )

        scores[indexes] += (
            idf
            * (frequencies * 2.2)
            / (frequencies + 1.2 * normalizer)
        ) * float(query_frequency)

    positive = c008_np.flatnonzero(scores > 0)

    selected = positive[
        c008_np.lexsort((
            positive,
            -scores[positive],
        ))
    ][:100]

    return (
        [c008_doc_ids[i] for i in selected],
        [float(scores[i]) for i in selected],
    )

def c008_minmax(scores):
    arr = c008_np.asarray(
        scores,
        dtype=c008_np.float64,
    )

    if not arr.size:
        return arr

    if c008_np.isclose(
        float(arr.max()),
        float(arr.min()),
    ):
        return c008_np.ones_like(
            arr,
            dtype=c008_np.float64,
        )

    return (
        (arr - float(arr.min()))
        / (float(arr.max()) - float(arr.min()))
    )

def c008_fuse(dense_ids, dense_scores, sparse_ids, sparse_scores):
    combined = collections.defaultdict(float)

    for cid, value in zip(
        dense_ids[:100],
        c008_minmax(dense_scores[:100]),
    ):
        combined[str(cid)] += 0.4 * float(value)

    for cid, value in zip(
        sparse_ids[:100],
        c008_minmax(sparse_scores[:100]),
    ):
        combined[str(cid)] += 0.6 * float(value)

    return [
        cid
        for cid, _ in sorted(
            combined.items(),
            key=lambda item: (-item[1], item[0]),
        )
    ][:100]

# Bind historical rows without modifying them.
c008_hybrid_ref = c002_summary["input_file_checks"]["HYBRID"]
c008_reranker_ref = c002_summary["input_file_checks"]["RERANKER"]

def c008_select_rows(receipt, depth):
    path = c008_read_file_pin(receipt)
    selected = {}

    with path.open("rb") as handle:
        for line in handle:
            row = json.loads(line)
            candidate_id = row.get("candidate_id")

            if candidate_id not in c008_ids:
                continue

            valid_split = (
                row.get("split") == "test"
                if depth == 100
                else row.get("split") in (None, "test")
            )

            c008_require(
                candidate_id not in selected and valid_split,
                "HISTORICAL_CANDIDATE_DUPLICATE_OR_SPLIT",
            )

            entries = row["candidates"]

            c008_require(
                len(entries) == depth
                and [entry["rank"] for entry in entries]
                == list(range(1, depth + 1)),
                "HISTORICAL_RANK_SEQUENCE",
            )

            selected[candidate_id] = row

    c008_require(
        set(selected) == set(c008_ids),
        "HISTORICAL_FOUR_CANDIDATES_MISSING",
    )

    return selected

c008_historical_hybrid = c008_select_rows(
    c008_hybrid_ref,
    100,
)

c008_historical_reranker = c008_select_rows(
    c008_reranker_ref,
    50,
)

# Repeat the isolated-copy method, not the historical parity audit.
c008_source_files = []

for directory, dirs, files in os.walk(
    c007_original,
    followlinks=False,
):
    c008_require(
        not any(
            (Path(directory) / entry).is_symlink()
            for entry in dirs + files
        ),
        "INDEX_SYMLINK",
    )

    for name in files:
        source = Path(directory) / name
        stat = source.stat()

        c008_source_files.append((
            source.relative_to(c007_original),
            stat.st_size,
            stat.st_mtime_ns,
        ))

c008_require(
    len(c008_source_files) == c005_index_result["inventory_files"]
    and sum(item[1] for item in c008_source_files)
    == c005_index_result["inventory_bytes"],
    "INDEX_INVENTORY_CHANGED",
)

c008_stage1_status = "BLOCKED"
c008_error = None

try:
    with tempfile.TemporaryDirectory(
        prefix="NB22E-v4-C008-pilot-",
        dir=str(c008_out),
    ) as working_directory:

        clone = Path(working_directory) / "qdrant"
        clone.mkdir()

        for relative, size, modified in c008_source_files:
            source = c007_original / relative
            target = clone / relative

            before = source.stat()

            c008_require(
                not source.is_symlink()
                and (before.st_size, before.st_mtime_ns)
                == (size, modified),
                "INDEX_CHANGED",
            )

            target.parent.mkdir(parents=True, exist_ok=True)

            with (
                source.open("rb") as source_handle,
                target.open("xb") as target_handle,
            ):
                for block in iter(
                    lambda: source_handle.read(1024 * 1024),
                    b"",
                ):
                    target_handle.write(block)

            after = source.stat()

            c008_require(
                (after.st_size, after.st_mtime_ns)
                == (size, modified),
                "INDEX_CHANGED_DURING_COPY",
            )

        client = None

        try:
            client = c008_qdrant_class(path=str(clone))

            c008_require(
                client.count(
                    collection_name=c005_collection,
                    exact=True,
                ).count == 3488,
                "INDEX_CLONE_COUNT",
            )

            for candidate_id in c008_ids:
                started = time.perf_counter()

                vector = c008_np.asarray(
                    c008_cache_array[
                        c008_vector_positions[candidate_id]
                    ],
                    dtype=c008_np.float32,
                )

                c008_require(
                    vector.shape == (384,)
                    and bool(c008_np.isfinite(vector).all())
                    and bool(
                        c008_np.isclose(
                            c008_np.linalg.norm(vector),
                            1.0,
                        )
                    ),
                    "VECTOR_INVALID",
                )

                sparse_ids, sparse_scores = (
                    c008_sparse_retrieve(candidate_id)
                )

                c008_attempts += 1

                dense_result = client.query_points(
                    collection_name=c005_collection,
                    query=vector.tolist(),
                    limit=100,
                    with_payload=True,
                    with_vectors=False,
                )

                dense_points = getattr(
                    dense_result,
                    "points",
                    dense_result,
                )

                dense_ids = [
                    str(point.payload["chunk_id"])
                    for point in dense_points
                ]

                dense_scores = [
                    float(point.score)
                    for point in dense_points
                ]

                observed = c008_fuse(
                    dense_ids,
                    dense_scores,
                    sparse_ids,
                    sparse_scores,
                )

                expected = [
                    row["chunk_id"]
                    for row in c008_historical_hybrid[
                        candidate_id
                    ]["candidates"]
                ]

                pool = [
                    row["chunk_id"]
                    for row in c008_historical_reranker[
                        candidate_id
                    ]["candidates"]
                ]

                order_matches = observed == expected
                pool_matches = set(observed[:50]) == set(pool)

                result = {
                    "candidate_id": candidate_id,
                    "question_sha256": (
                        c002_questions[candidate_id][
                            "question_sha256"
                        ]
                    ),
                    "observed_hybrid100_order": observed,
                    "expected_hybrid100_order_sha256": (
                        hashlib.sha256(
                            json.dumps(
                                expected,
                                ensure_ascii=False,
                                separators=(",", ":"),
                            ).encode("utf-8")
                        ).hexdigest()
                    ),
                    "dense_retrieved": len(dense_ids),
                    "sparse_retrieved": len(sparse_ids),
                    "observed_depth": len(observed),
                    "full_order_matches": order_matches,
                    "pool50_membership_matches": pool_matches,
                    "first_mismatch_rank": next(
                        (
                            index + 1
                            for index, (left, right) in enumerate(
                                zip(observed, expected)
                            )
                            if left != right
                        ),
                        None,
                    ),
                    "elapsed_seconds": round(
                        time.perf_counter() - started,
                        3,
                    ),
                    "stage1_status": (
                        "PASS"
                        if (
                            order_matches
                            and pool_matches
                            and len(observed) == 100
                        )
                        else "BLOCKED"
                    ),
                    "qdrant_queries": 1,
                    "api_calls": 0,
                    "model_inference_calls": 0,
                }

                result["checkpoint"] = c008_write_once(
                    "C008.stage1_candidate",
                    result,
                )

                c008_results.append(result)

                print(
                    "C008 Stage 1:",
                    len(c008_results),
                    "/ 4",
                    result["stage1_status"],
                    flush=True,
                )

                if result["stage1_status"] != "PASS":
                    raise RuntimeError(
                        "STAGE1_HYBRID100_OR_POOL50_VERSION_CONFLICT:"
                        + candidate_id
                    )

        finally:
            if client is not None:
                client.close()

    c008_stage1_status = "STAGE1_PASS"

except Exception as error:
    c008_error = {
        "type": type(error).__name__,
        "detail": str(error)[:600],
    }

c008_peak_rss_raw = resource.getrusage(
    resource.RUSAGE_SELF
).ru_maxrss

c008_peak_rss_gib = (
    c008_peak_rss_raw
    / (1024 ** 3 if sys.platform == "darwin" else 1024 ** 2)
)

c008_report = {
    "cell": "NB22E-v4:C008",
    "status": c008_stage1_status,
    "stage": "STAGE1_FROZEN_HYBRID100_AND_POOL50_PARITY",
    "candidate_ids": c008_ids,
    "candidate_count_completed": len(c008_results),
    "results": c008_results,
    "qdrant_queries_attempted": c008_attempts,
    "elapsed_seconds": round(
        time.perf_counter() - c008_started,
        3,
    ),
    "process_peak_rss_gib": round(c008_peak_rss_gib, 3),
    "original_index_opened": False,
    "repository_writes": 0,
    "version_boundary_approved": True,
    "logical_index_hash_match_checkpoint_sha256": (
        NB22E_V4_LOGICAL_INDEX_CHECK["checkpoint_sha256"]
    ),
    "stage2_authorized": c008_stage1_status == "STAGE1_PASS",
    "stage2_started": False,
    "target_release_gate": "BLOCKED",
    "api_calls": 0,
    "model_inference_calls": 0,
    "blocker": c008_error,
}

c008_report["checkpoint"] = c008_write_once(
    "C008.stage1_summary",
    c008_report,
)

NB22E_V4_STAGE1_RESULT = c008_report

print(json.dumps(
    {
        key: value
        for key, value in c008_report.items()
        if key != "results"
    },
    ensure_ascii=False,
    indent=2,
))

C008 Stage 1: 1 / 4 PASS
C008 Stage 1: 2 / 4 PASS
C008 Stage 1: 3 / 4 PASS
C008 Stage 1: 4 / 4 PASS
{
  "cell": "NB22E-v4:C008",
  "status": "STAGE1_PASS",
  "stage": "STAGE1_FROZEN_HYBRID100_AND_POOL50_PARITY",
  "candidate_ids": [
    "golden_v2_candidate_ca5362d68c692d667fad3006",
    "golden_v2_candidate_15cb48c40c3d1e292f647ff9",
    "golden_v2_candidate_4032db31c53d0465590cf0ab",
    "golden_v2_candidate_ba8ca208df074fc579e48eb5"
  ],
  "candidate_count_completed": 4,
  "qdrant_queries_attempted": 4,
  "elapsed_seconds": 3.66,
  "process_peak_rss_gib": 0.851,
  "original_index_opened": false,
  "repository_writes": 0,
  "version_boundary_approved": true,
  "logical_index_hash_match_checkpoint_sha256": "d52e91db8f136f5b2145715f54d6aecc9e5bd5a2869cb1da719c5008c76a49b8",
  "stage2_authorized": true,
  "stage2_started": false,
  "target_release_gate": "BLOCKED",
  "api_calls": 0,
  "model_inference_calls": 0,
  "blocker": null,
  "checkpoint": {
    "path": "/Users/macbookpro/AMIR_DA

<div dir="rtl" align="right">

### NB22E-v4:C009 — R11.2: اجرای محدود Stage 2 / Local100

**پیش‌نیازهای تأییدشده:**
- C007: تطبیق کامل Logical Index Hash.
- C008: موفقیت Stage 1 برای چهار سؤال.
- مجوز ناظر: حداکثر ۲۰۰ جفت اضافی ONNX.

**روش اجرا:**
۱. حفظ همان چهار شناسهٔ C006 و ترتیب Hybrid100 تأییدشده در C008.
۲. استفادهٔ مجدد از ۵۰ امتیاز تاریخی Pool50 برای هر سؤال.
۳. محاسبهٔ فقط ۵۰ جفت باقی‌مانده برای هر سؤال، در Batchهای حداکثر چهارتایی.
۴. اجرای ONNX صرفاً روی CPU با چهار Thread و `max_length=512`.
۵. تشکیل رتبه‌بندی Local100 با امتیاز نزولی و در تساوی، `chunk_id` صعودی.
۶. ذخیرهٔ Checkpoint پیش از هر Batch، پس از تکمیل آن و پس از هر سؤال.
۷. بازیابی Batchهای تکمیل‌شده بدون استنتاج تکراری؛ توقف در صورت وجود Batch شروع‌شده با نتیجهٔ نامعلوم.
۸. ثبت زمان، حافظهٔ اوج پردازش، تعداد جفت‌ها، تعداد فراخوانی‌های مدل و خطاها.

**محدودیت‌ها:**
- بدون Qdrant Query تازه، API، تولید پاسخ یا تغییر مخزن.
- بدون تغییر تنظیمات براساس نتایج Test.
- بدون اجرای ۱۱۱۶ سؤال.
- موفقیت Stage 2، معادل تأیید کیفیت یا پذیرش نهایی R11.2 نیست.
- هویت نهایی نسخهٔ پاسخ همچنان جدا از هویت تاریخی تعیین خواهد شد.

**خروجی:** گزارش‌ها و Checkpointهای تغییرناپذیر در پوشهٔ مستقل `C009_stage2` خارج از مخزن.

**شرط پایان:** چهار Checkpoint معتبر با مجموع دقیق ۲۰۰ امتیاز جدید و Summary موفق؛ در غیر این صورت `STAGE2_BLOCKED`.

</div>

In [9]:
# NB22E-v4 | C009 | R11.2
# Stage 2 repair: preserve Hugging Face snapshot aliases while verifying actual ONNX file bytes.

def c009_require(condition, code):
    if not condition:
        raise RuntimeError("C009_" + code)

# ============================================================
# 1. Bind the approved plan and completed checkpoints
# ============================================================

c009_plan_sha = "fbfa555c8d3ab8e9f54e05dee701349ee63bc3c2a5aaee41d80e6d568e1c6213"
c009_stage1_sha = "ba56c485f014a2eb149b63bad85eec6c790d2eb8ebf36f4f8348a9768f7955c6"
c009_index_sha = "2fc5e50ebbd1e1e66d59c703eb48f1367b2d3deadfed62efb9d5a5e6959f7ef4"
c009_model_sha = "6c2513767fb63d008a4377bef7a7a3555433d9436342bb53e35a3a72ffc52d4b"

c009_root = c002_root.resolve()
c009_out = c003_out.resolve()

c009_require(
    NB22E_V4_READ_ONLY is True
    and c009_out.is_dir()
    and not c009_out.is_relative_to(c009_root)
    and NB22E_V4_PILOT_PLAN.get("status") == "PILOT_PLAN_RELOAD_PASS"
    and NB22E_V4_LOGICAL_INDEX_CHECK.get("status") == "LOGICAL_INDEX_HASH_MATCH"
    and NB22E_V4_STAGE1_RESULT.get("status") == "STAGE1_PASS"
    and NB22E_V4_STAGE1_RESULT.get("stage2_authorized") is True
    and NB22E_V4_STAGE1_RESULT.get("qdrant_queries_attempted") == 4,
    "PREREQUISITE_GATE",
)

def c009_verify_checkpoint(path_value, expected_sha):
    path = Path(path_value).resolve()
    c009_require(
        path.parent == c009_out and path.is_file(),
        "CHECKPOINT_LOCATION",
    )
    raw = path.read_bytes()
    c009_require(
        hashlib.sha256(raw).hexdigest() == expected_sha,
        "CHECKPOINT_FILE_BYTES",
    )
    return json.loads(raw)

c009_require(
    NB22E_V4_PILOT_PLAN["file_bytes_sha256"] == c009_plan_sha
    and NB22E_V4_STAGE1_RESULT["checkpoint"]["file_bytes_sha256"]
    == c009_stage1_sha
    and NB22E_V4_LOGICAL_INDEX_CHECK["observed_logical_index_sha256"]
    == c009_index_sha
    and c004_model_actual_sha == c009_model_sha
    and c004_onnxruntime_version == "1.16.3",
    "FROZEN_PIN_CONFLICT",
)

c009_plan = c009_verify_checkpoint(
    NB22E_V4_PILOT_PLAN["path"], c009_plan_sha
)
c009_stage1 = c009_verify_checkpoint(
    NB22E_V4_STAGE1_RESULT["checkpoint"]["path"],
    c009_stage1_sha,
)
c009_index = c009_verify_checkpoint(
    NB22E_V4_LOGICAL_INDEX_CHECK["checkpoint_path"],
    NB22E_V4_LOGICAL_INDEX_CHECK["checkpoint_sha256"],
)

c009_ids = list(NB22E_V4_PILOT_PLAN["candidate_ids"])

c009_require(
    len(c009_ids) == len(set(c009_ids)) == 4
    and [row["candidate_id"] for row in c009_plan["population"]["pilot_candidates"]]
    == c009_ids
    and c009_stage1["candidate_ids"] == c009_ids
    and c009_stage1["status"] == "STAGE1_PASS"
    and c009_stage1["stage2_authorized"] is True
    and c009_stage1["logical_index_hash_match_checkpoint_sha256"]
    == NB22E_V4_LOGICAL_INDEX_CHECK["checkpoint_sha256"]
    and c009_index["status"] == "LOGICAL_INDEX_HASH_MATCH"
    and c009_index["observed_logical_index_sha256"] == c009_index_sha,
    "APPROVED_SCOPE_CONFLICT",
)

c009_stage1_rows = c009_stage1["results"]

c009_require(
    len(c009_stage1_rows) == 4
    and [row["candidate_id"] for row in c009_stage1_rows] == c009_ids
    and all(
        row["stage1_status"] == "PASS"
        and row["full_order_matches"] is True
        and row["pool50_membership_matches"] is True
        and row["observed_depth"] == 100
        and row["qdrant_queries"] == 1
        for row in c009_stage1_rows
    ),
    "STAGE1_DETAIL_CONFLICT",
)

for row in c009_stage1_rows:
    receipt = row["checkpoint"]
    saved = c009_verify_checkpoint(
        receipt["path"], receipt["file_bytes_sha256"]
    )
    c009_require(
        saved["candidate_id"] == row["candidate_id"]
        and saved["stage1_status"] == "PASS"
        and saved["observed_hybrid100_order"]
        == row["observed_hybrid100_order"],
        "STAGE1_CANDIDATE_CHECKPOINT",
    )

# ============================================================
# 2. Bind 50 historical scores and 50 missing pairs per question
# ============================================================

c009_inputs = {}
c009_needed_chunk_ids = set()

for candidate_id, stage1_row in zip(c009_ids, c009_stage1_rows):
    hybrid_rows = c008_historical_hybrid[candidate_id]["candidates"]
    pool_rows = c008_historical_reranker[candidate_id]["candidates"]

    hybrid_ids = [row["chunk_id"] for row in hybrid_rows]
    pool_ids = [row["chunk_id"] for row in pool_rows]

    c009_require(
        len(hybrid_ids) == len(set(hybrid_ids)) == 100
        and len(pool_ids) == len(set(pool_ids)) == 50
        and [row["rank"] for row in hybrid_rows] == list(range(1, 101))
        and [row["rank"] for row in pool_rows] == list(range(1, 51))
        and hybrid_ids == stage1_row["observed_hybrid100_order"]
        and set(pool_ids) == set(hybrid_ids[:50]),
        "HISTORICAL_POOL_OR_RANK_CONFLICT:" + candidate_id,
    )

    saved_scores = {
        row["chunk_id"]: float(row["reranker_score"])
        for row in pool_rows
    }

    c009_require(
        len(saved_scores) == 50
        and all(math.isfinite(score) for score in saved_scores.values()),
        "HISTORICAL_SCORE_INVALID:" + candidate_id,
    )

    missing_ids = [
        chunk_id for chunk_id in hybrid_ids
        if chunk_id not in saved_scores
    ]

    c009_require(
        len(missing_ids) == len(set(missing_ids)) == 50,
        "MISSING50_IDENTITY_CONFLICT:" + candidate_id,
    )

    raw_question = c002_questions[candidate_id]["question"]
    question_sha = hashlib.sha256(
        raw_question.encode("utf-8")
    ).hexdigest()

    c009_require(
        bool(raw_question.strip())
        and question_sha == c002_questions[candidate_id]["question_sha256"]
        and question_sha == stage1_row["question_sha256"],
        "QUESTION_SHA_CONFLICT:" + candidate_id,
    )

    c009_inputs[candidate_id] = {
        "question": raw_question.strip(),
        "question_sha256": question_sha,
        "hybrid100": hybrid_ids,
        "pool50": pool_ids,
        "saved_scores": saved_scores,
        "missing50": missing_ids,
    }

    c009_needed_chunk_ids.update(missing_ids)

c009_require(
    sum(len(item["missing50"]) for item in c009_inputs.values()) == 200,
    "PAIR_BUDGET",
)

# ============================================================
# 3. Load exact chunk texts; repair the ONNX symlink check
# ============================================================

c009_chunks_receipt = c002_summary["input_file_checks"]["CHUNKS"]
c009_chunks_path = c008_read_file_pin(c009_chunks_receipt)

c009_contents = {}

with c009_chunks_path.open("rb") as stream:
    for line in stream:
        row = json.loads(line)
        chunk_id = row["chunk_id"]

        if chunk_id not in c009_needed_chunk_ids:
            continue

        c009_require(
            chunk_id not in c009_contents
            and isinstance(row.get("content"), str),
            "CHUNK_TEXT_INVALID_OR_DUPLICATE",
        )
        c009_contents[chunk_id] = row["content"]

c009_require(
    set(c009_contents) == c009_needed_chunk_ids,
    "CHUNK_TEXT_COVERAGE",
)

# Important: do not dereference the snapshot alias before checking
# its parent directory. Hugging Face model files may link to blobs/.
c009_model_path = Path(c004_model_path).expanduser()
c009_snapshot_alias = Path(c004_snapshot).expanduser()

c009_model_alias_parent = Path(
    os.path.abspath(str(c009_model_path.parent.parent))
)
c009_expected_snapshot_alias = Path(
    os.path.abspath(str(c009_snapshot_alias))
)

c009_model_cache_root = c009_snapshot_alias.parent.parent.resolve()
c009_model_target = c009_model_path.resolve()

c009_require(
    c009_model_path.is_file()
    and c009_model_alias_parent == c009_expected_snapshot_alias
    and c009_model_target.is_file()
    and c009_model_target.is_relative_to(c009_model_cache_root),
    "ONNX_SNAPSHOT_ALIAS_OR_CACHE_BOUNDARY_CONFLICT",
)

print(json.dumps({
    "cell": "NB22E-v4:C009",
    "model_alias": str(c009_model_path),
    "model_resolved_target": str(c009_model_target),
    "snapshot_alias_verified": True,
    "model_sha256_required_at_execution": c009_model_sha,
    "candidate_count": 4,
    "additional_pairs_max": 200,
    "qdrant_queries": 0,
    "api_calls": 0,
}, ensure_ascii=False, indent=2))

# ============================================================
# 4. Freeze Stage 2 identity
# ============================================================

def c009_canonical(value):
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        allow_nan=False,
    ).encode("utf-8")

c009_scope = {
    "schema": "NB22E_V4_R11_2_STAGE2_INPUTS_V1",
    "approved_plan_sha256": c009_plan_sha,
    "verified_stage1_sha256": c009_stage1_sha,
    "verified_c007_sha256": (
        NB22E_V4_LOGICAL_INDEX_CHECK["checkpoint_sha256"]
    ),
    "logical_index_sha256": c009_index_sha,
    "chunks_sha256": c009_chunks_receipt["file_bytes_sha256"],
    "hybrid_sha256": (
        c002_summary["input_file_checks"]["HYBRID"]["file_bytes_sha256"]
    ),
    "reranker_sha256": (
        c002_summary["input_file_checks"]["RERANKER"]["file_bytes_sha256"]
    ),
    "model_sha256": c009_model_sha,
    "model_revision": "1427fd652930e4ba29e8149678df786c240d8825",
    "onnxruntime_version": "1.16.3",
    "provider": "CPUExecutionProvider",
    "threads": 4,
    "batch_size": 4,
    "max_length": 512,
    "candidate_ids": c009_ids,
    "new_pairs_max": 200,
}

c009_scope_sha = hashlib.sha256(
    c009_canonical(c009_scope)
).hexdigest()

c009_dir = (c009_out / "C009_stage2").resolve()

c009_require(
    c009_dir.parent == c009_out,
    "STAGE2_DIRECTORY_UNSAFE",
)

c009_dir.mkdir(parents=True, exist_ok=True)

# ============================================================
# 5. Immutable checkpoint helpers
# ============================================================

def c009_record_bytes(core):
    core_sha = hashlib.sha256(
        c009_canonical(core)
    ).hexdigest()

    return c009_canonical({
        **core,
        "record_core_sha256": core_sha,
    }) + b"\n"

def c009_put_once(path, core):
    c009_require(
        path.parent.resolve() == c009_dir,
        "OUTPUT_PATH_UNSAFE",
    )

    raw = c009_record_bytes(core)

    if not path.exists():
        temporary = None

        try:
            with tempfile.NamedTemporaryFile(
                "wb",
                dir=c009_dir,
                prefix=".C009-",
                suffix=".tmp",
                delete=False,
            ) as stream:
                temporary = Path(stream.name)
                stream.write(raw)
                stream.flush()
                os.fsync(stream.fileno())

            try:
                os.link(temporary, path)
            except FileExistsError:
                pass

        finally:
            if temporary is not None:
                temporary.unlink(missing_ok=True)

    c009_require(
        path.read_bytes() == raw,
        "IMMUTABLE_CHECKPOINT_CONFLICT:" + path.name,
    )

    return hashlib.sha256(raw).hexdigest()

def c009_read_record(path):
    c009_require(
        path.parent.resolve() == c009_dir and path.is_file(),
        "CHECKPOINT_MISSING_OR_UNSAFE",
    )

    raw = path.read_bytes()
    record = json.loads(raw)
    recorded_core_sha = record.pop("record_core_sha256", None)

    c009_require(
        recorded_core_sha == hashlib.sha256(
            c009_canonical(record)
        ).hexdigest(),
        "CHECKPOINT_CORE_SHA_CONFLICT:" + path.name,
    )

    return record, hashlib.sha256(raw).hexdigest()

def c009_rss_gib():
    peak = resource.getrusage(
        resource.RUSAGE_SELF
    ).ru_maxrss

    divisor = (
        1024 ** 3 if sys.platform == "darwin"
        else 1024 ** 2
    )

    return round(peak / divisor, 3)

# ============================================================
# 6. Execute only missing batches; preserve completed batches
# ============================================================

c009_started = time.perf_counter()
c009_pairs_attempted_this_run = 0
c009_pairs_completed_this_run = 0
c009_pairs_reused_this_run = 0
c009_model_calls_this_run = 0

c009_candidate_results = []
c009_blocker = None
c009_current_candidate = None

c009_session = None
c009_tokenizer = c005_tokenizer

try:
    for candidate_id in c009_ids:
        c009_current_candidate = candidate_id
        candidate_started = time.perf_counter()
        item = c009_inputs[candidate_id]

        candidate_key = hashlib.sha256(
            candidate_id.encode("utf-8")
        ).hexdigest()[:24]

        candidate_identity = {
            "scope_sha256": c009_scope_sha,
            "candidate_id": candidate_id,
            "question_sha256": item["question_sha256"],
            "hybrid100": item["hybrid100"],
            "pool50": item["pool50"],
            "saved_scores": item["saved_scores"],
            "missing50": item["missing50"],
            "chunk_text_sha256": {
                chunk_id: hashlib.sha256(
                    c009_contents[chunk_id].encode("utf-8")
                ).hexdigest()
                for chunk_id in item["missing50"]
            },
        }

        candidate_input_sha = hashlib.sha256(
            c009_canonical(candidate_identity)
        ).hexdigest()

        candidate_path = c009_dir / (
            f"C009.candidate.{candidate_key}."
            f"{c009_scope_sha[:16]}.json"
        )

        def c009_batch_paths(start):
            key = (
                f"{candidate_key}.{start:02d}."
                f"{c009_scope_sha[:16]}"
            )

            return (
                c009_dir / f"C009.started.{key}.json",
                c009_dir / f"C009.completed.{key}.json",
            )

        # Reuse an already validated complete candidate.
        if candidate_path.is_file():
            saved, candidate_file_sha = c009_read_record(
                candidate_path
            )

            c009_require(
                saved["status"] == "CANDIDATE_PASS"
                and saved["scope_sha256"] == c009_scope_sha
                and saved["input_sha256"] == candidate_input_sha
                and saved["candidate_id"] == candidate_id
                and saved["original_hybrid100_order"]
                == item["hybrid100"]
                and saved["stored_pool50_order"] == item["pool50"]
                and saved["stored_pool50_scores"] == item["saved_scores"]
                and set(saved["new_scores_by_chunk_id"])
                == set(item["missing50"])
                and len(saved["batch_receipts"]) == 13,
                "EXISTING_CANDIDATE_IDENTITY_CONFLICT",
            )

            full_scores = {
                **item["saved_scores"],
                **saved["new_scores_by_chunk_id"],
            }

            c009_require(
                len(full_scores) == 100
                and all(math.isfinite(v) for v in full_scores.values())
                and saved["local100_order"] == sorted(
                    item["hybrid100"],
                    key=lambda chunk_id: (
                        -full_scores[chunk_id],
                        chunk_id,
                    ),
                ),
                "EXISTING_LOCAL100_ORDER_CONFLICT",
            )

            for index, receipt in enumerate(saved["batch_receipts"]):
                start = index * 4
                batch_ids = item["missing50"][start:start + 4]
                started_path, completed_path = c009_batch_paths(start)

                c009_require(
                    receipt["path"] == str(completed_path)
                    and receipt["batch_start"] == start
                    and started_path.is_file(),
                    "CANDIDATE_BATCH_RECEIPT_PATH",
                )

                started_row, _ = c009_read_record(started_path)
                completed_row, completed_sha = c009_read_record(
                    completed_path
                )

                c009_require(
                    started_row["status"] == "STARTED_NOT_VERIFIED"
                    and started_row["input_sha256"] == candidate_input_sha
                    and started_row["chunk_ids"] == batch_ids
                    and completed_sha == receipt["file_bytes_sha256"]
                    and completed_row["status"] == "BATCH_PASS"
                    and completed_row["input_sha256"] == candidate_input_sha
                    and completed_row["chunk_ids"] == batch_ids
                    and dict(zip(
                        completed_row["chunk_ids"],
                        completed_row["scores"],
                    )) == {
                        chunk_id: saved["new_scores_by_chunk_id"][chunk_id]
                        for chunk_id in batch_ids
                    },
                    "EXISTING_BATCH_RECEIPT_CONFLICT",
                )

            c009_pairs_reused_this_run += 50

            c009_candidate_results.append({
                "candidate_id": candidate_id,
                "status": "CANDIDATE_PASS_REUSED",
                "pairs_in_receipt": 50,
                "new_pairs_this_run": 0,
                "path": str(candidate_path),
                "file_bytes_sha256": candidate_file_sha,
            })

            print(
                "C009 candidate:",
                len(c009_candidate_results),
                "/4 REUSED",
                flush=True,
            )

            continue

        new_scores = {}
        batch_receipts = []
        candidate_new_calls = 0
        candidate_new_pairs = 0

        for start in range(0, 50, 4):
            batch_ids = item["missing50"][start:start + 4]
            started_path, completed_path = c009_batch_paths(start)

            attempt_identity = {
                "cell": "NB22E-v4:C009",
                "status": "STARTED_NOT_VERIFIED",
                "scope_sha256": c009_scope_sha,
                "input_sha256": candidate_input_sha,
                "candidate_id": candidate_id,
                "batch_start": start,
                "chunk_ids": batch_ids,
                "score_count_expected": len(batch_ids),
                "manual_resolution_required_if_completion_missing": True,
            }

            if completed_path.is_file():
                c009_require(
                    started_path.is_file(),
                    "COMPLETED_WITHOUT_STARTED",
                )

                started_record, _ = c009_read_record(started_path)
                completed_record, completed_sha = c009_read_record(
                    completed_path
                )

                c009_require(
                    started_record == attempt_identity
                    and completed_record["status"] == "BATCH_PASS"
                    and completed_record["scope_sha256"] == c009_scope_sha
                    and completed_record["input_sha256"]
                    == candidate_input_sha
                    and completed_record["candidate_id"] == candidate_id
                    and completed_record["chunk_ids"] == batch_ids,
                    "REUSED_BATCH_IDENTITY_CONFLICT",
                )

                values = completed_record["scores"]

                c009_require(
                    len(values) == len(batch_ids)
                    and all(math.isfinite(v) for v in values),
                    "REUSED_BATCH_SCORE_CONFLICT",
                )

                c009_pairs_reused_this_run += len(batch_ids)

            else:
                c009_require(
                    not started_path.exists(),
                    "UNRESOLVED_STARTED_BATCH_NO_AUTO_RETRY:"
                    + candidate_id + ":" + str(start),
                )

                c009_require(
                    c009_pairs_attempted_this_run + len(batch_ids) <= 200,
                    "AUTHORIZED_PAIR_LIMIT",
                )

                if c009_session is None:
                    c009_np = nb22e_v4_require_imports("np")["np"]
                    c009_ort = nb22e_v4_require_imports("ort")["ort"]

                    c009_require(
                        c009_ort.__version__ == "1.16.3",
                        "ONNX_RUNTIME_VERSION_CONFLICT",
                    )

                    # Hash the resolved file contents, not the symlink text.
                    before = c009_model_path.stat()
                    digest = hashlib.sha256()

                    with c009_model_path.open("rb") as stream:
                        for block in iter(
                            lambda: stream.read(1024 * 1024),
                            b"",
                        ):
                            digest.update(block)

                    after = c009_model_path.stat()

                    c009_require(
                        (before.st_size, before.st_mtime_ns)
                        == (after.st_size, after.st_mtime_ns)
                        and digest.hexdigest() == c009_model_sha,
                        "MODEL_FILE_SHA256_CONFLICT",
                    )

                    options = c009_ort.SessionOptions()
                    options.intra_op_num_threads = 4
                    options.inter_op_num_threads = 1
                    options.execution_mode = (
                        c009_ort.ExecutionMode.ORT_SEQUENTIAL
                    )
                    options.graph_optimization_level = (
                        c009_ort.GraphOptimizationLevel.ORT_ENABLE_ALL
                    )

                    c009_session = c009_ort.InferenceSession(
                        str(c009_model_path),
                        sess_options=options,
                        providers=["CPUExecutionProvider"],
                    )

                    c009_input_names = {
                        field.name
                        for field in c009_session.get_inputs()
                    }

                    c009_require(
                        c009_session.get_providers()
                        == ["CPUExecutionProvider"]
                        and {"input_ids", "attention_mask"}
                        <= c009_input_names,
                        "ONNX_CPU_PROVIDER_OR_INPUT_CONFLICT",
                    )

                c009_require(
                    c009_rss_gib() <= 6.0,
                    "PROCESS_RSS_OVER_LIMIT",
                )

                encoded = c009_tokenizer(
                    [item["question"]] * len(batch_ids),
                    [
                        c009_contents[chunk_id]
                        for chunk_id in batch_ids
                    ],
                    padding=True,
                    truncation=True,
                    max_length=512,
                    return_tensors="np",
                )

                c009_require(
                    c009_input_names <= set(encoded),
                    "TOKENIZER_INPUT_CONFLICT",
                )

                feed = {
                    name: c009_np.asarray(
                        encoded[name],
                        dtype=c009_np.int64,
                    )
                    for name in c009_input_names
                }

                c009_require(
                    all(
                        value.shape[0] == len(batch_ids)
                        and value.shape[1] <= 512
                        for value in feed.values()
                    ),
                    "TOKENIZER_BATCH_SHAPE",
                )

                # Durable intent is written before inference.
                c009_put_once(
                    started_path,
                    attempt_identity,
                )

                batch_started = time.perf_counter()

                c009_pairs_attempted_this_run += len(batch_ids)
                c009_model_calls_this_run += 1
                candidate_new_calls += 1

                outputs = c009_session.run(None, feed)

                scores = c009_np.asarray(
                    outputs[0],
                    dtype=c009_np.float64,
                ).reshape(-1)

                c009_require(
                    scores.shape == (len(batch_ids),)
                    and bool(c009_np.isfinite(scores).all()),
                    "ONNX_OUTPUT_SCORE_CONFLICT",
                )

                values = [float(score) for score in scores]

                completed_sha = c009_put_once(
                    completed_path,
                    {
                        "cell": "NB22E-v4:C009",
                        "status": "BATCH_PASS",
                        "scope_sha256": c009_scope_sha,
                        "input_sha256": candidate_input_sha,
                        "candidate_id": candidate_id,
                        "chunk_ids": batch_ids,
                        "scores": values,
                        "elapsed_seconds": round(
                            time.perf_counter() - batch_started,
                            4,
                        ),
                        "process_peak_rss_gib": c009_rss_gib(),
                        "model_inference_calls": 1,
                        "qdrant_queries": 0,
                        "api_calls": 0,
                    },
                )

                c009_pairs_completed_this_run += len(batch_ids)
                candidate_new_pairs += len(batch_ids)

            c009_require(
                not set(batch_ids).intersection(new_scores),
                "DUPLICATE_BATCH_IDS",
            )

            new_scores.update(dict(zip(batch_ids, values)))

            batch_receipts.append({
                "batch_start": start,
                "count": len(batch_ids),
                "path": str(completed_path),
                "file_bytes_sha256": completed_sha,
            })

        # ====================================================
        # 7. Finish this candidate before the next one
        # ====================================================

        c009_require(
            len(new_scores) == 50
            and set(new_scores) == set(item["missing50"]),
            "CANDIDATE_INCOMPLETE",
        )

        all_scores = {
            **item["saved_scores"],
            **new_scores,
        }

        c009_require(
            len(all_scores) == 100
            and set(all_scores) == set(item["hybrid100"])
            and all(math.isfinite(v) for v in all_scores.values()),
            "LOCAL100_SCORE_SET_CONFLICT",
        )

        local100_order = sorted(
            item["hybrid100"],
            key=lambda chunk_id: (
                -all_scores[chunk_id],
                chunk_id,
            ),
        )

        c009_require(
            len(local100_order) == len(set(local100_order)) == 100,
            "LOCAL100_RANK_CONFLICT",
        )

        candidate_report = {
            "cell": "NB22E-v4:C009",
            "status": "CANDIDATE_PASS",
            "candidate_id": candidate_id,
            "scope_sha256": c009_scope_sha,
            "input_sha256": candidate_input_sha,
            "question_sha256": item["question_sha256"],
            "original_hybrid100_order": item["hybrid100"],
            "stored_pool50_order": item["pool50"],
            "stored_pool50_scores": item["saved_scores"],
            "new_scores_by_chunk_id": new_scores,
            "all100_scores_by_chunk_id": all_scores,
            "local100_order": local100_order,
            "local100_order_sha256": hashlib.sha256(
                c009_canonical(local100_order)
            ).hexdigest(),
            "new_pair_count": 50,
            "reused_pool50_pair_count": 50,
            "new_model_calls_this_execution": candidate_new_calls,
            "new_pairs_this_execution": candidate_new_pairs,
            "batch_receipts": batch_receipts,
            "elapsed_seconds_this_execution": round(
                time.perf_counter() - candidate_started,
                3,
            ),
            "process_peak_rss_gib_since_kernel_start": c009_rss_gib(),
            "historical_local100_rank_equivalence": "NOT_ASSERTED",
            "target_release_gate": "BLOCKED",
            "qdrant_queries": 0,
            "api_calls": 0,
        }

        candidate_sha = c009_put_once(
            candidate_path,
            candidate_report,
        )

        c009_candidate_results.append({
            "candidate_id": candidate_id,
            "status": "CANDIDATE_PASS",
            "pairs_in_receipt": 50,
            "new_pairs_this_run": candidate_new_pairs,
            "path": str(candidate_path),
            "file_bytes_sha256": candidate_sha,
        })

        print(
            "C009 candidate:",
            len(c009_candidate_results),
            "/4 PASS;",
            "new pairs:",
            candidate_new_pairs,
            "peak RSS GiB:",
            c009_rss_gib(),
            flush=True,
        )

except Exception as error:
    c009_blocker = {
        "type": type(error).__name__,
        "detail": str(error)[:700],
        "candidate_id": c009_current_candidate,
    }

finally:
    c009_session = None

# ============================================================
# 8. Persist final Stage 2 status
# ============================================================

c009_success = (
    c009_blocker is None
    and len(c009_candidate_results) == 4
    and all(
        row["status"] in (
            "CANDIDATE_PASS",
            "CANDIDATE_PASS_REUSED",
        )
        for row in c009_candidate_results
    )
    and sum(
        row["pairs_in_receipt"]
        for row in c009_candidate_results
    ) == 200
)

c009_summary = {
    "cell": "NB22E-v4:C009",
    "status": (
        "STAGE2_PASS" if c009_success else "STAGE2_BLOCKED"
    ),
    "scope_sha256": c009_scope_sha,
    "approved_pilot_plan_file_sha256": c009_plan_sha,
    "verified_stage1_file_sha256": c009_stage1_sha,
    "logical_index_sha256": c009_index_sha,
    "candidate_ids": c009_ids,
    "candidate_results": c009_candidate_results,
    "candidates_completed": len(c009_candidate_results),
    "new_pairs_max_authorized": 200,
    "new_pairs_attempted_this_execution": (
        c009_pairs_attempted_this_run
    ),
    "new_pairs_completed_this_execution": (
        c009_pairs_completed_this_run
    ),
    "pairs_reused_from_stage2_checkpoints": (
        c009_pairs_reused_this_run
    ),
    "complete_stage2_new_pair_receipts": sum(
        row["pairs_in_receipt"]
        for row in c009_candidate_results
    ),
    "historical_pool50_scores_reused_if_all_complete": (
        200 if c009_success else None
    ),
    "model_inference_calls_this_execution": (
        c009_model_calls_this_run
    ),
    "qdrant_queries_this_execution": 0,
    "api_calls": 0,
    "elapsed_seconds_this_execution": round(
        time.perf_counter() - c009_started,
        3,
    ),
    "process_peak_rss_gib_since_kernel_start": c009_rss_gib(),
    "blocker": c009_blocker,
    "repository_modified": False,
    "version_identity_final": False,
    "target_release_gate": "BLOCKED",
    "full_population_authorized": False,
    "next": (
        "PILOT_EVIDENCE_REVIEW_AND_MEASURED_CONTINUATION_ESTIMATE"
        if c009_success
        else "REPAIR_REPORTED_BLOCKER_NO_UNVERIFIED_RETRIES"
    ),
}

summary_bytes = c009_canonical(c009_summary) + b"\n"
summary_sha = hashlib.sha256(summary_bytes).hexdigest()

summary_path = c009_dir / (
    "C009.stage2_summary." + summary_sha + ".json"
)

if not summary_path.exists():
    temporary = None

    try:
        with tempfile.NamedTemporaryFile(
            "wb",
            dir=c009_dir,
            prefix=".C009-summary-",
            suffix=".tmp",
            delete=False,
        ) as stream:
            temporary = Path(stream.name)
            stream.write(summary_bytes)
            stream.flush()
            os.fsync(stream.fileno())

        try:
            os.link(temporary, summary_path)
        except FileExistsError:
            pass

    finally:
        if temporary is not None:
            temporary.unlink(missing_ok=True)

c009_require(
    summary_path.read_bytes() == summary_bytes,
    "SUMMARY_RELOAD_CONFLICT",
)

NB22E_V4_STAGE2_RESULT = {
    **c009_summary,
    "checkpoint": {
        "path": str(summary_path),
        "file_bytes_sha256": summary_sha,
        "size_bytes": len(summary_bytes),
    },
}

print(json.dumps(
    {
        key: value
        for key, value in NB22E_V4_STAGE2_RESULT.items()
        if key != "candidate_results"
    },
    ensure_ascii=False,
    indent=2,
))

RuntimeError: C009_FROZEN_PIN_CONFLICT

<div dir="rtl" align="right">

### NB22E-v4:C010 — R11.2: ممیزی Pilot و برآورد ادامه

**هدف:** اعتبارسنجی مستقل خروجی‌های موفق C007 تا C009، بدون اجرای مجدد Retrieval یا مدل.

**کنترل‌ها:**
۱. Reload و تطبیق SHA-256 طرح مصوب C006، کنترل Index در C007 و نتایج دو Stage.
۲. بررسی چهار Checkpoint نهایی سؤال‌ها.
۳. اعتبارسنجی ۵۲ رسید شروع و ۵۲ رسید تکمیل Batch.
۴. تطبیق ۲۰۰ امتیاز جدید ONNX با خروجی‌های ذخیره‌شده.
۵. تأیید حفظ ۲۰۰ امتیاز تاریخی Pool50.
۶. بازسازی ترتیب Local100 از امتیازها و کنترل قانون مرتب‌سازی.
۷. استخراج زمان و حافظهٔ اندازه‌گیری‌شده و تهیهٔ برآورد مشروط برای ۱۱۱۶ سؤال.
۸. ثبت گزارش تغییرناپذیر و تأیید Reload آن.

**محدودیت:** هیچ Query، مدل، API یا اجرای جمعیتی انجام نمی‌شود. هویت نسخهٔ جدید نهایی نمی‌شود و Gate پذیرش R11.2 مسدود باقی می‌ماند.

**خروجی:** `C010.pilot_audit.<SHA256>.json` خارج از مخزن.

</div>

In [ ]:
# NB22E-v4 | C010 | R11.2
# Audit approved Pilot receipts; no retrieval, ONNX inference, API or repository writes.

def c010_require(ok, code):
    if not ok:
        raise RuntimeError("C010_" + code)

c010_root = c002_root.resolve()
c010_out = c003_out.resolve()
c010_stage2_dir = (c010_out / "C009_stage2").resolve()

c010_require(
    NB22E_V4_READ_ONLY is True
    and c010_out.is_dir()
    and not c010_out.is_relative_to(c010_root)
    and c010_stage2_dir.parent == c010_out
    and c010_stage2_dir.is_dir(),
    "EXTERNAL_OUTPUT_BOUNDARY",
)

c010_pins = {
    "plan": "fbfa555c8d3ab8e9f54e05dee701349ee63bc3c2a5aaee41d80e6d568e1c6213",
    "stage1": "ba56c485f014a2eb149b63bad85eec6c790d2eb8ebf36f4f8348a9768f7955c6",
    "stage2": "933a0bfdbe38991f9d6715d97e8b5517f7815a6f67a3e7562c4df5d67ecf8174",
    "scope": "fce1de6a63a9614fa05fd23014f8d7353f0767724d9351ea23ea30199f2cc6d6",
    "index": "2fc5e50ebbd1e1e66d59c703eb48f1367b2d3deadfed62efb9d5a5e6959f7ef4",
}

# ============================================================
# 1. Independently reload pinned stage artifacts
# ============================================================

def c010_read(path_value, expected_sha, allowed_parent):
    path = Path(path_value).resolve()
    c010_require(
        path.parent == allowed_parent and path.is_file(),
        "UNSAFE_RECEIPT",
    )

    before = path.stat()
    raw = path.read_bytes()
    after = path.stat()

    c010_require(
        before.st_size == after.st_size == len(raw)
        and before.st_mtime_ns == after.st_mtime_ns
        and hashlib.sha256(raw).hexdigest() == expected_sha,
        "RECEIPT_BYTES_OR_SHA_CONFLICT:" + path.name,
    )
    return json.loads(raw)

c010_require(
    NB22E_V4_PILOT_PLAN["file_bytes_sha256"] == c010_pins["plan"]
    and NB22E_V4_STAGE1_RESULT["checkpoint"]["file_bytes_sha256"]
    == c010_pins["stage1"]
    and NB22E_V4_STAGE2_RESULT["checkpoint"]["file_bytes_sha256"]
    == c010_pins["stage2"],
    "SUPERVISOR_OR_STAGE_PIN_CONFLICT",
)

c010_plan = c010_read(
    NB22E_V4_PILOT_PLAN["path"],
    c010_pins["plan"],
    c010_out,
)
c010_s1 = c010_read(
    NB22E_V4_STAGE1_RESULT["checkpoint"]["path"],
    c010_pins["stage1"],
    c010_out,
)
c010_s2 = c010_read(
    NB22E_V4_STAGE2_RESULT["checkpoint"]["path"],
    c010_pins["stage2"],
    c010_stage2_dir,
)
c010_index = c010_read(
    NB22E_V4_LOGICAL_INDEX_CHECK["checkpoint_path"],
    NB22E_V4_LOGICAL_INDEX_CHECK["checkpoint_sha256"],
    c010_out,
)

c010_ids = list(NB22E_V4_PILOT_PLAN["candidate_ids"])

c010_require(
    c010_s2 == {
        key: value
        for key, value in NB22E_V4_STAGE2_RESULT.items()
        if key != "checkpoint"
    }
    and len(c010_ids) == len(set(c010_ids)) == 4
    and c010_plan["population"]["new_candidate_count"] == 1116
    and [
        item["candidate_id"]
        for item in c010_plan["population"]["pilot_candidates"]
    ] == c010_ids
    and c010_index["status"] == "LOGICAL_INDEX_HASH_MATCH"
    and c010_index["observed_logical_index_sha256"]
    == c010_pins["index"]
    and c010_index["point_count"] == 3488
    and c010_s1["status"] == "STAGE1_PASS"
    and c010_s1["candidate_ids"] == c010_ids
    and c010_s1["qdrant_queries_attempted"] == 4
    and c010_s1["stage2_authorized"] is True
    and c010_s1["logical_index_hash_match_checkpoint_sha256"]
    == NB22E_V4_LOGICAL_INDEX_CHECK["checkpoint_sha256"]
    and c010_s2["status"] == "STAGE2_PASS"
    and c010_s2["candidate_ids"] == c010_ids
    and c010_s2["scope_sha256"] == c010_pins["scope"]
    and c010_s2["approved_pilot_plan_file_sha256"]
    == c010_pins["plan"]
    and c010_s2["verified_stage1_file_sha256"]
    == c010_pins["stage1"]
    and c010_s2["logical_index_sha256"]
    == c010_pins["index"]
    and c010_s2["target_release_gate"] == "BLOCKED"
    and c010_s2["full_population_authorized"] is False
    and c010_s2["version_identity_final"] is False
    and c010_s2["blocker"] is None
    and c010_s2["repository_modified"] is False
    and c010_s2["api_calls"] == 0
    and c010_s2["qdrant_queries_this_execution"] == 0
    and c010_s2["new_pairs_attempted_this_execution"] == 200
    and c010_s2["new_pairs_completed_this_execution"] == 200
    and c010_s2["pairs_reused_from_stage2_checkpoints"] == 0
    and c010_s2["complete_stage2_new_pair_receipts"] == 200
    and c010_s2["historical_pool50_scores_reused_if_all_complete"] == 200
    and c010_s2["model_inference_calls_this_execution"] == 52
    and c010_s2["candidates_completed"] == 4,
    "CHECKPOINT_CHAIN_OR_AGGREGATE_CONFLICT",
)

# ============================================================
# 2. Verify each candidate and all 52 batch receipts
# ============================================================

c010_verified = []
c010_batch_seconds = []
c010_expected_started = set()
c010_expected_completed = set()
c010_expected_candidates = set()
c010_total_new_pairs = 0
c010_total_old_scores = 0

for pos, receipt in enumerate(c010_s2["candidate_results"]):
    candidate_id = c010_ids[pos]
    candidate_path = Path(receipt["path"]).resolve()

    c010_require(
        receipt["candidate_id"] == candidate_id
        and receipt["status"] == "CANDIDATE_PASS"
        and receipt["pairs_in_receipt"] == 50
        and receipt["new_pairs_this_run"] == 50,
        "CANDIDATE_SUMMARY_CONFLICT",
    )

    candidate, observed_sha = c009_read_record(candidate_path)
    c010_expected_candidates.add(candidate_path)

    c010_require(
        observed_sha == receipt["file_bytes_sha256"]
        and candidate["status"] == "CANDIDATE_PASS"
        and candidate["scope_sha256"] == c010_pins["scope"]
        and candidate["candidate_id"] == candidate_id
        and candidate["question_sha256"]
        == c002_questions[candidate_id]["question_sha256"]
        and candidate["original_hybrid100_order"]
        == c010_s1["results"][pos]["observed_hybrid100_order"]
        and c010_s1["results"][pos]["stage1_status"] == "PASS"
        and c010_s1["results"][pos]["full_order_matches"] is True
        and c010_s1["results"][pos]["pool50_membership_matches"] is True
        and candidate["new_pair_count"] == 50
        and candidate["reused_pool50_pair_count"] == 50
        and candidate["new_model_calls_this_execution"] == 13
        and candidate["new_pairs_this_execution"] == 50
        and candidate["historical_local100_rank_equivalence"]
        == "NOT_ASSERTED"
        and len(candidate["batch_receipts"]) == 13,
        "CANDIDATE_VERSION_OR_COUNT_CONFLICT:" + candidate_id,
    )

    hybrid = candidate["original_hybrid100_order"]
    old_order = candidate["stored_pool50_order"]
    old_scores = candidate["stored_pool50_scores"]
    new_scores = candidate["new_scores_by_chunk_id"]
    all_scores = candidate["all100_scores_by_chunk_id"]

    c010_require(
        len(hybrid) == len(set(hybrid)) == 100
        and len(old_order) == len(set(old_order)) == 50
        and set(old_order) == set(hybrid[:50]) == set(old_scores)
        and set(new_scores) == set(hybrid[50:])
        and len(new_scores) == 50
        and all_scores == {**old_scores, **new_scores}
        and all(math.isfinite(score) for score in all_scores.values())
        and candidate["local100_order"] == sorted(
            hybrid,
            key=lambda chunk_id: (-all_scores[chunk_id], chunk_id),
        )
        and candidate["local100_order_sha256"] == hashlib.sha256(
            c009_canonical(candidate["local100_order"])
        ).hexdigest(),
        "LOCAL100_SCORE_OR_RANK_CONFLICT:" + candidate_id,
    )

    reassembled = {}
    candidate_batch_seconds = 0.0

    for batch_pos, batch_receipt in enumerate(
        candidate["batch_receipts"]
    ):
        start = batch_pos * 4
        expected_ids = hybrid[50:][start:start + 4]
        completed_path = Path(batch_receipt["path"]).resolve()
        started_path = completed_path.with_name(
            completed_path.name.replace(
                "C009.completed.",
                "C009.started.",
                1,
            )
        )

        c010_require(
            completed_path.name.startswith("C009.completed.")
            and batch_receipt["batch_start"] == start
            and batch_receipt["count"] == len(expected_ids)
            and completed_path not in c010_expected_completed
            and started_path not in c010_expected_started,
            "BATCH_PATH_OR_OFFSET_CONFLICT",
        )

        started, _ = c009_read_record(started_path)
        completed, actual_sha = c009_read_record(completed_path)

        c010_expected_started.add(started_path)
        c010_expected_completed.add(completed_path)

        c010_require(
            actual_sha == batch_receipt["file_bytes_sha256"]
            and started["status"] == "STARTED_NOT_VERIFIED"
            and started["scope_sha256"] == c010_pins["scope"]
            and started["input_sha256"] == candidate["input_sha256"]
            and started["candidate_id"] == candidate_id
            and started["batch_start"] == start
            and started["chunk_ids"] == expected_ids
            and started["score_count_expected"] == len(expected_ids)
            and completed["status"] == "BATCH_PASS"
            and completed["scope_sha256"] == c010_pins["scope"]
            and completed["input_sha256"] == candidate["input_sha256"]
            and completed["candidate_id"] == candidate_id
            and completed["chunk_ids"] == expected_ids
            and completed["model_inference_calls"] == 1
            and completed["qdrant_queries"] == 0
            and completed["api_calls"] == 0,
            "BATCH_RECEIPT_CHAIN_CONFLICT",
        )

        scores = completed["scores"]
        seconds = float(completed["elapsed_seconds"])

        c010_require(
            len(scores) == len(expected_ids)
            and all(math.isfinite(float(value)) for value in scores)
            and math.isfinite(seconds)
            and seconds >= 0,
            "BATCH_SCORE_OR_TIMING_CONFLICT",
        )

        reassembled.update(zip(expected_ids, scores))
        c010_batch_seconds.append(seconds)
        candidate_batch_seconds += seconds
        c010_total_new_pairs += len(scores)

    c010_require(
        reassembled == new_scores
        and len(reassembled) == 50,
        "BATCH_SCORE_REASSEMBLY_CONFLICT",
    )

    c010_total_old_scores += len(old_scores)

    c010_verified.append({
        "candidate_id": candidate_id,
        "status": "VERIFIED",
        "hybrid100_count": 100,
        "reused_pool50_scores": 50,
        "new_onnx_scores": 50,
        "verified_batches": 13,
        "local100_rank_count": 100,
        "local100_order_sha256": (
            candidate["local100_order_sha256"]
        ),
        "candidate_receipt_sha256": observed_sha,
        "batch_model_seconds_sum": round(
            candidate_batch_seconds, 4
        ),
    })

# ============================================================
# 3. Confirm exact scoped checkpoint coverage
# ============================================================

c010_short_scope = c010_pins["scope"][:16]

def c010_scope_files(prefix):
    return {
        path.resolve()
        for path in c010_stage2_dir.glob(
            prefix + "*." + c010_short_scope + ".json"
        )
    }

c010_require(
    len(c010_verified) == 4
    and len(c010_batch_seconds) == 52
    and c010_total_new_pairs == 200
    and c010_total_old_scores == 200
    and c010_scope_files("C009.started.")
    == c010_expected_started
    and c010_scope_files("C009.completed.")
    == c010_expected_completed
    and c010_scope_files("C009.candidate.")
    == c010_expected_candidates,
    "INCOMPLETE_OR_EXTRA_SCOPED_RECEIPTS",
)

# ============================================================
# 4. Measured workload and conditional projection
# ============================================================

c010_stage1_seconds = float(c010_s1["elapsed_seconds"])
c010_stage2_seconds = float(
    c010_s2["elapsed_seconds_this_execution"]
)
c010_batch_seconds_sum = sum(c010_batch_seconds)
c010_peak_rss = float(
    c010_s2["process_peak_rss_gib_since_kernel_start"]
)

c010_require(
    0 < c010_stage1_seconds
    and 0 < c010_stage2_seconds
    and 0 < c010_peak_rss
    and c010_batch_seconds_sum <= c010_stage2_seconds + 0.5,
    "PILOT_MEASUREMENT_CONFLICT",
)

c010_population = 1116

c010_projection = {
    "method": "CONDITIONAL_LINEAR_PROJECTION_FROM_4_QUESTIONS",
    "population_if_approved": c010_population,
    "stage1_queries_if_same_contract_holds": c010_population,
    "additional_onnx_pairs_if_all_have_pool50": (
        c010_population * 50
    ),
    "additional_onnx_batches_if_all_have_pool50": (
        c010_population * 13
    ),
    "stage1_linear_hours": round(
        (c010_stage1_seconds / 4 * c010_population) / 3600,
        3,
    ),
    "stage2_model_run_only_linear_hours": round(
        (
            c010_batch_seconds_sum
            / 52
            * c010_population
            * 13
        ) / 3600,
        3,
    ),
    "stage2_end_to_end_linear_hours": round(
        (c010_stage2_seconds / 4 * c010_population) / 3600,
        3,
    ),
    "both_stages_linear_hours": round(
        (
            (c010_stage1_seconds + c010_stage2_seconds)
            / 4
            * c010_population
        ) / 3600,
        3,
    ),
    "not_a_runtime_guarantee": True,
    "excludes_generation_and_all_other_R11_2_work": True,
    "not_authorized_to_run_full_population": True,
}

# ============================================================
# 5. Save immutable audit outside the repository
# ============================================================

c010_report = {
    "cell": "NB22E-v4:C010",
    "status": "PILOT_EVIDENCE_AUDIT_PASS",
    "approved_plan_sha256": c010_pins["plan"],
    "index_logical_sha256": c010_pins["index"],
    "stage1_summary_sha256": c010_pins["stage1"],
    "stage2_summary_sha256": c010_pins["stage2"],
    "stage2_scope_sha256": c010_pins["scope"],
    "candidates_verified": c010_verified,
    "started_batch_receipts_verified": 52,
    "completed_batch_receipts_verified": 52,
    "new_onnx_pair_receipts_verified": c010_total_new_pairs,
    "historical_pool50_scores_preserved": c010_total_old_scores,
    "stage1_queries_observed": 4,
    "stage2_onnx_calls_observed": 52,
    "stage1_elapsed_seconds_observed": c010_stage1_seconds,
    "stage2_elapsed_seconds_observed": c010_stage2_seconds,
    "stage2_batch_model_seconds_total_observed": round(
        c010_batch_seconds_sum, 4
    ),
    "stage2_batch_model_seconds_median_observed": round(
        statistics.median(c010_batch_seconds), 4
    ),
    "process_peak_rss_gib_since_kernel_start": c010_peak_rss,
    "peak_rss_not_incremental_stage2_only": True,
    "continuation_projection": c010_projection,
    "historical_target_identity_kept_separate": (
        "8d5875bfee0cfe5f2dc0ce9bdf67e5f14301191b60891047a630c3fe65442065"
    ),
    "new_population_identity_finalized": False,
    "R11_2_acceptance": "BLOCKED",
    "target_release_gate": "BLOCKED",
    "full_population_run_authorized": False,
    "repository_writes": 0,
    "qdrant_queries_by_C010": 0,
    "model_inference_calls_by_C010": 0,
    "api_calls_by_C010": 0,
    "next": "SUPERVISOR_VERSIONED_POPULATION_AND_CONTINUATION_DECISION",
}

c010_bytes = c009_canonical(c010_report) + b"\n"
c010_file_sha = hashlib.sha256(c010_bytes).hexdigest()

c010_target = c010_out / (
    "C010.pilot_audit." + c010_file_sha + ".json"
)

if not c010_target.is_file():
    c010_temp = None

    try:
        with tempfile.NamedTemporaryFile(
            mode="wb",
            dir=c010_out,
            prefix=".C010-",
            suffix=".tmp",
            delete=False,
        ) as stream:
            c010_temp = Path(stream.name)
            stream.write(c010_bytes)
            stream.flush()
            os.fsync(stream.fileno())

        try:
            os.link(c010_temp, c010_target)
        except FileExistsError:
            pass

    finally:
        if c010_temp is not None:
            c010_temp.unlink(missing_ok=True)

c010_require(
    c010_target.read_bytes() == c010_bytes,
    "AUDIT_ARTIFACT_RELOAD_CONFLICT",
)

NB22E_V4_PILOT_AUDIT = {
    **c010_report,
    "checkpoint": {
        "path": str(c010_target),
        "file_bytes_sha256": c010_file_sha,
        "size_bytes": len(c010_bytes),
    },
}

print(json.dumps({
    "cell": NB22E_V4_PILOT_AUDIT["cell"],
    "status": NB22E_V4_PILOT_AUDIT["status"],
    "candidate_count": 4,
    "stage1_queries": 4,
    "stage2_new_pairs": c010_total_new_pairs,
    "historical_pool50_scores": c010_total_old_scores,
    "batch_receipts_verified": 52,
    "inference_calls_observed": 52,
    "stage1_seconds": c010_stage1_seconds,
    "stage2_seconds": c010_stage2_seconds,
    "process_peak_rss_gib_since_kernel_start": c010_peak_rss,
    "continuation_projection": c010_projection,
    "new_population_identity_finalized": False,
    "target_release_gate": "BLOCKED",
    "checkpoint": NB22E_V4_PILOT_AUDIT["checkpoint"],
}, ensure_ascii=False, indent=2))

{
  "cell": "NB22E-v4:C010",
  "status": "PILOT_EVIDENCE_AUDIT_PASS",
  "candidate_count": 4,
  "stage1_queries": 4,
  "stage2_new_pairs": 200,
  "historical_pool50_scores": 200,
  "batch_receipts_verified": 52,
  "inference_calls_observed": 52,
  "stage1_seconds": 3.513,
  "stage2_seconds": 32.2,
  "process_peak_rss_gib_since_kernel_start": 1.019,
  "continuation_projection": {
    "method": "CONDITIONAL_LINEAR_PROJECTION_FROM_4_QUESTIONS",
    "population_if_approved": 1116,
    "stage1_queries_if_same_contract_holds": 1116,
    "additional_onnx_pairs_if_all_have_pool50": 55800,
    "additional_onnx_batches_if_all_have_pool50": 14508,
    "stage1_linear_hours": 0.272,
    "stage2_model_run_only_linear_hours": 2.214,
    "stage2_end_to_end_linear_hours": 2.496,
    "both_stages_linear_hours": 2.768,
    "not_a_runtime_guarantee": true,
    "excludes_generation_and_all_other_R11_2_work": true,
    "not_authorized_to_run_full_population": true
  },
  "new_population_identity_finalized":

<div dir="rtl" align="right">

### NB22E-v4:C011 — R11.2: ثبت هویت مستقل جمعیتی و Manifest اجرایی

**هدف:** ثبت هویت جدید برای جمعیت دقیق ۱۱۱۶ سؤال و تثبیت مرز اجرای مرحله‌ای با مجوز ناظر.

**کنترل‌ها:**

۱. تطبیق SHA-256 بایتی Artifact ممیزی C010 و زنجیرهٔ رسیدهای C006 تا C009.

۲. ثبت همهٔ ۱۱۱۶ شناسه همراه با هش متن سؤال و ترتیب قطعی عضویت.

۳. حفظ چهار سؤال Pilot به‌عنوان نتایج موفق قابل‌استفادهٔ مجدد، بدون قرار دادن آن‌ها در صف اجرای تازه.

۴. تعریف ۴۵ دستهٔ حداکثر ۲۵سؤالی برای ۱۱۱۲ سؤال باقی‌مانده.

۵. اتصال هویت مستقل به ورودی‌ها، قراردادهای Producer، Index فریز‌شده، بردارهای Query، ONNX، فایل‌های Tokenizer، نسخه‌های Runtime و تنظیمات واقعی.

۶. ایجاد SHA-256 مستقل برای قرارداد مرز اجرای جمعیتی؛ جدا از شناسهٔ تاریخی و هش Artifactهای رتبهٔ آینده.

۷. نوشتن Manifest تغییرناپذیر خارج مخزن و کنترل Reload آن.

**محدودیت:** هیچ Query، استنتاج مدل، API یا اجرای جمعیتی در این سل انجام نمی‌شود.

**وضعیت پذیرش:** R11.2 همچنان BLOCKED است. مجوز اجرای Stage 1/2، به‌معنای مجوز مراحل بعدی Rank، Context یا Generation نیست.

</div>

In [ ]:
# NB22E-v4 | C011 | R11.2
# Register immutable 1116-member population boundary; no model, retrieval, API or repository writes.

def c011_require(ok, code):
    if not ok:
        raise RuntimeError("C011_" + code)

c011_root = c002_root.resolve()
c011_out = c003_out.resolve()
c011_pilot_dir = (c011_out / "C009_stage2").resolve()

c011_require(
    NB22E_V4_READ_ONLY is True
    and c011_root.name == "RAG_finance"
    and c011_out.is_dir()
    and not c011_out.is_relative_to(c011_root)
    and c011_pilot_dir.is_dir()
    and c011_pilot_dir.parent == c011_out,
    "UNSAFE_BOUNDARY",
)

c011_expected = {
    "plan": "fbfa555c8d3ab8e9f54e05dee701349ee63bc3c2a5aaee41d80e6d568e1c6213",
    "index_receipt": "6e61431607fac16faa1295455670d016da88a3a9cf5e8951f242353232326d79",
    "stage1": "ba56c485f014a2eb149b63bad85eec6c790d2eb8ebf36f4f8348a9768f7955c6",
    "stage2": "933a0bfdbe38991f9d6715d97e8b5517f7815a6f67a3e7562c4df5d67ecf8174",
    "audit": "ae2360b309dc5ed6fe44c1735eecdb0407f96324ca394776739bd6fddbccc6a0",
    "logical_index": "2fc5e50ebbd1e1e66d59c703eb48f1367b2d3deadfed62efb9d5a5e6959f7ef4",
    "onnx": "6c2513767fb63d008a4377bef7a7a3555433d9436342bb53e35a3a72ffc52d4b",
    "historical": "8d5875bfee0cfe5f2dc0ce9bdf67e5f14301191b60891047a630c3fe65442065",
}

def c011_canon(obj):
    return json.dumps(
        obj,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        allow_nan=False,
    ).encode("utf-8")

def c011_sha_file(path, size_limit=None):
    path = Path(path)
    c011_require(path.is_file(), "FILE_MISSING:" + path.name)
    before = path.stat()

    if size_limit is not None:
        c011_require(
            before.st_size <= size_limit,
            "FILE_OVERSIZE:" + path.name,
        )

    digest = hashlib.sha256()

    with path.open("rb") as stream:
        for block in iter(
            lambda: stream.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    after = path.stat()

    c011_require(
        (before.st_size, before.st_mtime_ns)
        == (after.st_size, after.st_mtime_ns),
        "CHANGED_DURING_HASH:" + path.name,
    )

    return digest.hexdigest(), before.st_size

def c011_load_receipt(path_value, expected_sha, parent):
    path = Path(path_value).resolve()

    c011_require(
        path.parent == parent and path.is_file(),
        "RECEIPT_PATH_UNSAFE",
    )

    actual_sha, size = c011_sha_file(
        path,
        16 * 1024 * 1024,
    )

    c011_require(
        actual_sha == expected_sha,
        "RECEIPT_SHA_CONFLICT:" + path.name,
    )

    return json.loads(path.read_bytes()), {
        "path": str(path),
        "file_bytes_sha256": actual_sha,
        "size_bytes": size,
    }

# ============================================================
# 1. Independently verify C006 through C010
# ============================================================

c011_plan, c011_plan_pin = c011_load_receipt(
    NB22E_V4_PILOT_PLAN["path"],
    c011_expected["plan"],
    c011_out,
)

c011_index, c011_index_pin = c011_load_receipt(
    NB22E_V4_LOGICAL_INDEX_CHECK["checkpoint_path"],
    c011_expected["index_receipt"],
    c011_out,
)

c011_stage1, c011_stage1_pin = c011_load_receipt(
    NB22E_V4_STAGE1_RESULT["checkpoint"]["path"],
    c011_expected["stage1"],
    c011_out,
)

c011_stage2, c011_stage2_pin = c011_load_receipt(
    NB22E_V4_STAGE2_RESULT["checkpoint"]["path"],
    c011_expected["stage2"],
    c011_pilot_dir,
)

c011_audit, c011_audit_pin = c011_load_receipt(
    NB22E_V4_PILOT_AUDIT["checkpoint"]["path"],
    c011_expected["audit"],
    c011_out,
)

c011_pilot_ids = [
    item["candidate_id"]
    for item in c011_plan["population"]["pilot_candidates"]
]

c011_population_ids = sorted(c002_questions)

c011_require(
    c011_audit["status"] == "PILOT_EVIDENCE_AUDIT_PASS"
    and c011_audit["stage2_summary_sha256"]
    == c011_expected["stage2"]
    and c011_audit["stage1_summary_sha256"]
    == c011_expected["stage1"]
    and c011_audit["approved_plan_sha256"]
    == c011_expected["plan"]
    and c011_audit["started_batch_receipts_verified"] == 52
    and c011_audit["completed_batch_receipts_verified"] == 52
    and c011_audit["new_onnx_pair_receipts_verified"] == 200
    and c011_audit["historical_pool50_scores_preserved"] == 200
    and c011_index["status"] == "LOGICAL_INDEX_HASH_MATCH"
    and c011_index["observed_logical_index_sha256"]
    == c011_expected["logical_index"]
    and c011_stage1["status"] == "STAGE1_PASS"
    and c011_stage1["candidate_ids"] == c011_pilot_ids
    and c011_stage1["stage2_authorized"] is True
    and c011_stage2["status"] == "STAGE2_PASS"
    and c011_stage2["candidate_ids"] == c011_pilot_ids
    and c011_stage2["candidates_completed"] == 4
    and c011_stage2["complete_stage2_new_pair_receipts"] == 200
    and c011_plan["population"]["new_candidate_count"] == 1116
    and len(c011_population_ids) == 1116
    and len(set(c011_population_ids)) == 1116
    and len(c011_pilot_ids) == len(set(c011_pilot_ids)) == 4
    and set(c011_pilot_ids).issubset(c011_population_ids),
    "PILOT_AUDIT_OR_MEMBERSHIP_MISMATCH",
)

# ============================================================
# 2. Exact membership and exclusion of completed Pilot work
# ============================================================

c011_membership = []

for cid in c011_population_ids:
    record = c002_questions[cid]
    question = record["question"]
    recorded_sha = record["question_sha256"]

    c011_require(
        isinstance(cid, str)
        and bool(cid)
        and isinstance(question, str)
        and bool(question.strip())
        and hashlib.sha256(
            question.encode("utf-8")
        ).hexdigest() == recorded_sha,
        "QUESTION_BYTES_OR_SHA_CONFLICT:" + cid,
    )

    c011_membership.append({
        "candidate_id": cid,
        "question_sha256": recorded_sha,
    })

c011_membership_sha = hashlib.sha256(
    c011_canon(c011_membership)
).hexdigest()

c011_pilot_id_set = set(c011_pilot_ids)

c011_remaining = [
    cid
    for cid in c011_population_ids
    if cid not in c011_pilot_id_set
]

c011_batches = [
    {
        "batch_number": offset // 25 + 1,
        "candidate_ids": c011_remaining[offset:offset + 25],
    }
    for offset in range(0, len(c011_remaining), 25)
]

c011_require(
    len(c011_remaining) == 1112
    and sum(
        len(batch["candidate_ids"])
        for batch in c011_batches
    ) == 1112
    and len(c011_batches) == 45
    and all(
        1 <= len(batch["candidate_ids"]) <= 25
        for batch in c011_batches
    ),
    "BATCH_MEMBERSHIP_CONFLICT",
)

# ============================================================
# 3. Bind and recheck frozen source and vector file bytes
# ============================================================

c011_source_pins = {}

for label, record in sorted(
    c002_summary["source_checks"].items()
):
    c011_require(
        record.get("status") == "FILE_BYTES_VERIFIED",
        "SOURCE_NOT_PINNED:" + label,
    )

    c011_source_pins[label] = {
        "path": record["path"],
        "file_bytes_sha256": record["file_bytes_sha256"],
    }

c011_input_pins = {}

for label, record in sorted(
    c002_summary["input_file_checks"].items()
):
    c011_require(
        record.get("status") == "FILE_BYTES_VERIFIED",
        "INPUT_NOT_PINNED:" + label,
    )

    c011_input_pins[label] = {
        "path": record["path"],
        "file_bytes_sha256": record["file_bytes_sha256"],
    }

c011_vector_pins = {}

for label, record in sorted(
    c002_summary["query_vector_precheck"][
        "cache_file_pointers"
    ].items()
):
    c011_require(
        isinstance(record, dict)
        and isinstance(record.get("path"), str)
        and isinstance(record.get("file_bytes_sha256"), str),
        "VECTOR_PIN_INVALID:" + label,
    )

    c011_vector_pins[label] = {
        "path": record["path"],
        "file_bytes_sha256": record["file_bytes_sha256"],
    }

c011_all_pins = {
    **c011_source_pins,
    **c011_input_pins,
    **{
        "VECTOR_" + key: value
        for key, value in c011_vector_pins.items()
    },
}

for label, record in c011_all_pins.items():
    relative = Path(record["path"])
    path = (c011_root / relative).resolve()

    c011_require(
        not relative.is_absolute()
        and path.is_relative_to(c011_root),
        "INPUT_PATH_ESCAPE:" + label,
    )

    actual_sha, _ = c011_sha_file(path)

    c011_require(
        actual_sha == record["file_bytes_sha256"],
        "INPUT_CHANGED:" + label,
    )

c011_require(
    c002_summary["target_identity_sha256_original"]
    == c011_expected["historical"]
    and c002_summary["producer_contract_checks"]
    and all(
        value is True
        for value in c002_summary[
            "producer_contract_checks"
        ].values()
    )
    and c002_summary["query_vector_precheck"][
        "vector_candidate_id_coverage"
    ] == 1116,
    "FROZEN_LINEAGE_OR_CONTRACT_CONFLICT",
)

# ============================================================
# 4. Actual ONNX, Tokenizer and Runtime identities
# ============================================================

c011_snapshot = Path(c004_snapshot).expanduser()
c011_model_alias = Path(c004_model_path).expanduser()

c011_require(
    c011_model_alias.is_file()
    and Path(
        os.path.abspath(
            str(c011_model_alias.parent.parent)
        )
    ) == Path(
        os.path.abspath(str(c011_snapshot))
    )
    and c011_snapshot.name
    == "1427fd652930e4ba29e8149678df786c240d8825",
    "ONNX_SNAPSHOT_CONTRACT",
)

c011_model_actual, c011_model_bytes = c011_sha_file(
    c011_model_alias
)

c011_require(
    c011_model_actual == c011_expected["onnx"],
    "ONNX_HASH_CHANGED",
)

c011_tokenizer_files = {}

for filename in (
    "config.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
):
    file_path = c011_snapshot / filename

    file_sha, file_bytes = c011_sha_file(
        file_path,
        64 * 1024 * 1024,
    )

    c011_tokenizer_files[filename] = {
        "sha256": file_sha,
        "size_bytes": file_bytes,
    }

c011_tokenizer_identity = {
    "model_snapshot_revision": c011_snapshot.name,
    "tokenizer_class": c005_tokenizer_result[
        "tokenizer_class"
    ],
    "is_fast": c005_tokenizer_result["is_fast"],
    "files": c011_tokenizer_files,
    "local_files_only": True,
    "trust_remote_code": False,
}

c011_tokenizer_sha = hashlib.sha256(
    c011_canon(c011_tokenizer_identity)
).hexdigest()

c011_versions = {
    "python": platform.python_version(),
    "numpy": importlib.metadata.version("numpy"),
    "onnxruntime": importlib.metadata.version("onnxruntime"),
    "qdrant-client": importlib.metadata.version("qdrant-client"),
    "transformers": importlib.metadata.version("transformers"),
    "tokenizers": importlib.metadata.version("tokenizers"),
}

c011_require(
    c011_versions["onnxruntime"] == "1.16.3"
    and c004_docs["release"]["backend"][
        "qdrant_client_version"
    ] == c011_versions["qdrant-client"]
    and c004_docs["release"]["logical_index_sha256"]
    == c011_expected["logical_index"]
    and c011_index["point_count"] == 3488,
    "RUNTIME_OR_INDEX_VERSION_CONFLICT",
)

# ============================================================
# 5. Independent population execution identity
# ============================================================

c011_execution_contract = {
    "schema": "NB22E_V4_R11_2_POPULATION_EXECUTION_BOUNDARY_V1",
    "membership_order": "CANDIDATE_ID_ASCENDING",
    "member_count": 1116,
    "members": c011_membership,
    "membership_sha256": c011_membership_sha,
    "historical_lineage_identity_sha256": (
        c011_expected["historical"]
    ),
    "historical_identity_inherited": False,
    "question_bindings_sha256": (
        c011_plan["population"]["question_binding_sha256"]
    ),
    "transfer_manifest_sha256": (
        c011_plan["population"]["manifest_sha256"]
    ),
    "source_producer_pins": c011_source_pins,
    "input_pins": c011_input_pins,
    "query_vector_cache_pins": c011_vector_pins,
    "producer_contract_checks": (
        c002_summary["producer_contract_checks"]
    ),
    "dense_index": {
        "release_file_sha256": (
            c004_file_checks["release"]["file_bytes_sha256"]
        ),
        "manifest_file_sha256": (
            c004_file_checks["manifest"]["file_bytes_sha256"]
        ),
        "handoff_file_sha256": (
            c004_file_checks["handoff"]["file_bytes_sha256"]
        ),
        "lifecycle_file_sha256": (
            c004_file_checks["lifecycle"]["file_bytes_sha256"]
        ),
        "collection": c005_collection,
        "logical_index_sha256": (
            c011_expected["logical_index"]
        ),
        "point_count": 3488,
        "point_dimension": 384,
        "hash_method_producer": "NB14_CODE_CELL_10",
        "producer_git_blob_sha1": (
            c011_index["producer_git_blob_sha1"]
        ),
    },
    "onnx": {
        "model_id": (
            "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
        ),
        "revision": c011_snapshot.name,
        "model_alias_relative_inside_snapshot": (
            "onnx/model_quint8_avx2.onnx"
        ),
        "model_sha256": c011_model_actual,
        "model_size_bytes": c011_model_bytes,
        "tokenizer_identity_sha256": c011_tokenizer_sha,
        "tokenizer": c011_tokenizer_identity,
    },
    "runtime_versions": c011_versions,
    "frozen_parameters": {
        "dense_candidate_depth": 100,
        "hybrid_fusion": "WEIGHTED_SCORE_FUSION",
        "hybrid_normalization": "minmax",
        "dense_weight": 0.4,
        "sparse_weight": 0.6,
        "sparse_analyzer": "fa_join_zwnj_stopwords_v1",
        "sparse_k1": 1.2,
        "sparse_b": 0.3,
        "local100_onnx_provider": "CPUExecutionProvider",
        "onnx_intra_op_threads": 4,
        "onnx_inter_op_threads": 1,
        "onnx_batch_size": 4,
        "onnx_max_length": 512,
        "candidate_parallelism": 1,
        "rank_tiebreak": "(-score,chunk_id)",
    },
}

c011_boundary_sha = hashlib.sha256(
    c011_canon(c011_execution_contract)
).hexdigest()

c011_require(
    c011_boundary_sha != c011_expected["historical"],
    "IDENTITY_REUSE_FORBIDDEN",
)

# ============================================================
# 6. Assemble the approved execution queue
# ============================================================

c011_document = {
    "cell": "NB22E-v4:C011",
    "status": "POPULATION_BOUNDARY_REGISTERED_APPROVAL_SCOPED",
    "execution_boundary_sha256": c011_boundary_sha,
    "execution_contract": c011_execution_contract,
    "approved_pilot_provenance": {
        "C006": c011_plan_pin,
        "C007": c011_index_pin,
        "C008": c011_stage1_pin,
        "C009": c011_stage2_pin,
        "C010": c011_audit_pin,
        "pilot_ids": c011_pilot_ids,
        "pilot_outcome": (
            "STAGE1_STAGE2_PASS_C010_AUDIT_PASS"
        ),
        "pilot_is_subset_of_population": True,
        "pilot_will_not_be_rerun": True,
    },
    "execution_queue": {
        "remaining_count": 1112,
        "pilot_reused_count": 4,
        "max_batch_size": 25,
        "batch_count": len(c011_batches),
        "batches": c011_batches,
        "fresh_stage1_queries_max": 1112,
        "fresh_stage2_onnx_pairs_max": 55600,
    },
    "status_policy": {
        "no_automatic_retry_of_uncertain_batches": True,
        "no_download_or_install": True,
        "no_new_model_or_hybrid_configuration": True,
        "stage1_must_pass_before_stage2_per_candidate": True,
        "stop_on_first_input_version_rank_score_or_receipt_conflict": True,
        "upstream_repo_read_only": True,
        "rank_artifact_identity_distinct_from_execution_boundary": True,
        "generation_api_allowed": False,
        "derived_order_quota_char_rrf_authorized": False,
        "full_R11_2_quality_gate": "BLOCKED",
        "response_generation_gate": "BLOCKED",
    },
}

# ============================================================
# 7. Immutable atomic write and independent reload
# ============================================================

c011_raw = c011_canon(c011_document) + b"\n"
c011_file_sha = hashlib.sha256(c011_raw).hexdigest()

c011_target = c011_out / (
    "C011.population_boundary."
    + c011_file_sha
    + ".json"
)

if not c011_target.exists():
    c011_tmp = None

    try:
        with tempfile.NamedTemporaryFile(
            "wb",
            dir=c011_out,
            prefix=".C011-",
            suffix=".tmp",
            delete=False,
        ) as stream:
            c011_tmp = Path(stream.name)
            stream.write(c011_raw)
            stream.flush()
            os.fsync(stream.fileno())

        try:
            os.link(c011_tmp, c011_target)
        except FileExistsError:
            pass

    finally:
        if c011_tmp is not None:
            c011_tmp.unlink(missing_ok=True)

c011_require(
    c011_target.read_bytes() == c011_raw,
    "IMMUTABLE_MANIFEST_RELOAD",
)

c011_reloaded = json.loads(c011_target.read_bytes())

c011_require(
    c011_reloaded["execution_boundary_sha256"]
    == c011_boundary_sha
    and hashlib.sha256(
        c011_canon(c011_reloaded["execution_contract"])
    ).hexdigest() == c011_boundary_sha
    and len(
        c011_reloaded["execution_contract"]["members"]
    ) == 1116
    and c011_reloaded["execution_queue"]["remaining_count"]
    == 1112
    and c011_reloaded["approved_pilot_provenance"]["C010"][
        "file_bytes_sha256"
    ] == c011_expected["audit"],
    "MANIFEST_IDENTITY_RELOAD_CONFLICT",
)

NB22E_V4_POPULATION_BOUNDARY = {
    "status": "POPULATION_BOUNDARY_RELOAD_PASS",
    "execution_boundary_sha256": c011_boundary_sha,
    "manifest_path": str(c011_target),
    "manifest_file_sha256": c011_file_sha,
    "population_count": 1116,
    "pilot_reuse_count": 4,
    "remaining_count": 1112,
    "batch_count": len(c011_batches),
}

print(json.dumps(
    {
        "cell": "NB22E-v4:C011",
        **NB22E_V4_POPULATION_BOUNDARY,
        "runtime_versions": c011_versions,
        "tokenizer_identity_sha256": c011_tokenizer_sha,
        "new_target_quality_accepted": False,
        "repository_modified": False,
        "model_runs": 0,
        "retrieval_queries": 0,
        "api_calls": 0,
        "next": "C012_BATCHED_STAGE1_STAGE2_RESUMABLE_EXECUTION",
    },
    ensure_ascii=False,
    indent=2,
))

{
  "cell": "NB22E-v4:C011",
  "status": "POPULATION_BOUNDARY_RELOAD_PASS",
  "execution_boundary_sha256": "aa099a56e76054c30629a35a3acbd1ae9a86970367c9fb60677b907c2f0d2cbc",
  "manifest_path": "/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/_RAG_finance_NB22E_v4_outputs/C011.population_boundary.a3ffce372eab24e210bedae9b2b6087dc2493e45805c4c27836d223a6578fe80.json",
  "manifest_file_sha256": "a3ffce372eab24e210bedae9b2b6087dc2493e45805c4c27836d223a6578fe80",
  "population_count": 1116,
  "pilot_reuse_count": 4,
  "remaining_count": 1112,
  "batch_count": 45,
  "runtime_versions": {
    "python": "3.11.3",
    "numpy": "1.24.3",
    "onnxruntime": "1.16.3",
    "qdrant-client": "1.18.0",
    "transformers": "4.53.2",
    "tokenizers": "0.21.2"
  },
  "tokenizer_identity_sha256": "34ab0082bf9db105fafe2b0e581339e820c2a3748d4d826a2f8de6b17def9325",
  "new_target_quality_accepted": false,
  "repository_modified": false,
  "model_runs": 0,
  "retrieval_queries": 0,
  "api_call

<div dir="rtl" align="right">

### NB22E-v4:C012 — اصلاح کنترل مسیر ONNX پس از توقف ایمن

**علت توقف:** استفاده از `resolve()` روی مسیر ارجاعی ONNX، رابطهٔ ظاهری فایل با Snapshot را تغییر داده و کنترل `ONNX_ALIAS` را نامعتبر کرده است.

**اصلاح:** کنترل مسیر مطابق روش موفق C011 انجام می‌شود؛ مسیر نسبی `onnx/model_quint8_avx2.onnx` و Revision با قرارداد فریز‌شده تطبیق می‌یابند. کنترل SHA-256 کامل مدل همچنان برقرار است.

**نتایج محفوظ:** چهار Pilot، سه سؤال کامل، ۱۷۰ جفت ONNX موفق و رسید تاریخی Batch نامعلوم. اجرای ناموفق اخیر هیچ Query یا ONNX تازه‌ای انجام نداده است.

**دامنهٔ اجرای بعد:** یک Retry دستی مصوب برای چهار جفت نامعلوم و ادامهٔ حداکثر پنج سؤال تکمیل‌نشده، با همان چهار Thread و مکث حفاظتی.

**معیار بررسی:** رسیدهای معتبر، سقف تجمعی ۵۵٬۶۰۴ pair-attempt، زمان‌بندی تفکیکی، مصرف حافظه و گزارش گلوگاه واقعی.

**محدودیت:** بدون API، تغییر مدل، GPU، تغییر مخزن یا تکرار نتایج موفق.

</div>

In [15]:
# NB22E-v4 | C012 | R11.2
# One approved manual retry; then at most five unfinished candidates with timing instrumentation.

c012_execute_new_work = True
c012_max_new_candidates = 5
c012_onnx_pause_seconds = 0.8
c012_process_rss_limit_gib = 4.0
c012_query_slow_limit_seconds = 45.0
c012_onnx_slow_limit_seconds = 25.0

c012_retry_id = "golden_v2_candidate_0184f90d79f3dfc4744b6e29"
c012_retry_start = 20
c012_retry_original_start_sha = (
    "4745577680cbd38d3a212e27ec67e3af98a42088cd2f3aace077fca563e4daf2"
)
c012_boundary_sha = (
    "aa099a56e76054c30629a35a3acbd1ae9a86970367c9fb60677b907c2f0d2cbc"
)
c012_manifest_sha = (
    "a3ffce372eab24e210bedae9b2b6087dc2493e45805c4c27836d223a6578fe80"
)
c012_c010_sha = (
    "ae2360b309dc5ed6fe44c1735eecdb0407f96324ca394776739bd6fddbccc6a0"
)
c012_model_sha = (
    "6c2513767fb63d008a4377bef7a7a3555433d9436342bb53e35a3a72ffc52d4b"
)

c012_run_started = time.perf_counter()

c012_times = {
    key: 0.0
    for key in (
        "source_sha",
        "index_sha",
        "input_read",
        "qdrant_setup",
        "model_setup",
        "query",
        "tokenization",
        "onnx",
        "pause",
        "checkpoint",
    )
}

c012_work = {
    "query_attempts": 0,
    "onnx_attempted_pairs": 0,
    "onnx_done_pairs": 0,
    "onnx_calls": 0,
    "new_completed": 0,
    "manual_retry_pairs": 0,
}

c012_stop = None
c012_active = None


def c012_check(ok, label):
    if not ok:
        raise RuntimeError("C012_" + label)


def c012_bytes(value):
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        allow_nan=False,
    ).encode("utf-8")


def c012_stamp(path):
    stat = path.stat()
    return (
        stat.st_dev,
        stat.st_ino,
        stat.st_size,
        stat.st_mtime_ns,
    )


def c012_sha(path):
    before = c012_stamp(path)
    digest = hashlib.sha256()

    with path.open("rb") as stream:
        for data in iter(
            lambda: stream.read(1024 * 1024),
            b"",
        ):
            digest.update(data)

    c012_check(
        c012_stamp(path) == before,
        "HASHED_FILE_CHANGED:" + path.name,
    )
    return digest.hexdigest()


def c012_memory():
    current = None

    try:
        result = subprocess.run(
            [
                "/bin/ps",
                "-o",
                "rss=",
                "-p",
                str(os.getpid()),
            ],
            capture_output=True,
            text=True,
            timeout=3,
            check=True,
        )
        current = round(
            int(result.stdout.strip()) / (1024 ** 2),
            3,
        )
    except (
        OSError,
        ValueError,
        subprocess.SubprocessError,
    ):
        pass

    peak = resource.getrusage(
        resource.RUSAGE_SELF
    ).ru_maxrss

    divisor = (
        1024 ** 3
        if sys.platform == "darwin"
        else 1024 ** 2
    )

    return {
        "process_current_rss_gib": current,
        "process_peak_rss_gib": round(
            peak / divisor,
            3,
        ),
        "system_memory_pressure": "NOT_MEASURED",
    }


# ============================================================
# 1. Independent population boundary
# ============================================================

c012_root = c002_root.resolve()
c012_out = c003_out.resolve()

c012_manifest_path = (
    c012_out
    / (
        "C011.population_boundary."
        + c012_manifest_sha
        + ".json"
    )
).resolve()

c012_check(
    NB22E_V4_READ_ONLY is True
    and c012_out.is_dir()
    and not c012_out.is_relative_to(c012_root)
    and c012_manifest_path.parent == c012_out
    and c012_sha(c012_manifest_path) == c012_manifest_sha,
    "MANIFEST_BYTES",
)

c012_manifest = json.loads(
    c012_manifest_path.read_bytes()
)

c012_contract = c012_manifest["execution_contract"]
c012_batches = c012_manifest["execution_queue"]["batches"]

c012_ids = [
    cid
    for batch in c012_batches
    for cid in batch["candidate_ids"]
]

c012_members = {
    item["candidate_id"]: item["question_sha256"]
    for item in c012_contract["members"]
}

c012_check(
    c012_manifest["execution_boundary_sha256"]
    == c012_boundary_sha
    and hashlib.sha256(
        c012_bytes(c012_contract)
    ).hexdigest() == c012_boundary_sha
    and c012_contract["historical_identity_inherited"] is False
    and len(c012_ids) == len(set(c012_ids)) == 1112
    and len(c012_members) == 1116
    and len(c012_batches) == 45
    and len(
        c012_manifest[
            "approved_pilot_provenance"
        ]["pilot_ids"]
    ) == 4
    and c012_contract["onnx"]["model_sha256"]
    == c012_model_sha
    and c012_contract["frozen_parameters"][
        "onnx_intra_op_threads"
    ] == 4
    and c012_contract["frozen_parameters"][
        "onnx_batch_size"
    ] == 4
    and c012_contract["frozen_parameters"][
        "onnx_max_length"
    ] == 512
    and c012_contract["frozen_parameters"][
        "candidate_parallelism"
    ] == 1
    and 1 <= c012_max_new_candidates <= 5
    and c012_onnx_pause_seconds >= 0,
    "BOUNDARY_OR_FROZEN_PROFILE",
)

c012_audit_pin = c012_manifest[
    "approved_pilot_provenance"
]["C010"]

c012_audit_path = Path(
    c012_audit_pin["path"]
).resolve()

c012_check(
    c012_audit_pin["file_bytes_sha256"]
    == c012_c010_sha
    and c012_audit_path.parent == c012_out
    and c012_sha(c012_audit_path) == c012_c010_sha
    and json.loads(
        c012_audit_path.read_bytes()
    )["status"] == "PILOT_EVIDENCE_AUDIT_PASS",
    "PILOT_AUDIT_BYTES",
)

c012_dir = (
    c012_out
    / ("C012_population_" + c012_boundary_sha)
).resolve()

c012_check(
    c012_dir.parent == c012_out
    and c012_dir.is_dir()
    and not c012_dir.is_symlink(),
    "EXISTING_C012_RECEIPTS_MISSING",
)


# ============================================================
# 2. Existing receipt format and immutable writes
# ============================================================

def c012_path(cid, suffix):
    key = hashlib.sha256(
        cid.encode("utf-8")
    ).hexdigest()

    return c012_dir / (
        "C012." + key + "." + suffix + ".json"
    )


def c012_read(path):
    started = time.perf_counter()

    c012_check(
        path.parent.resolve() == c012_dir
        and path.is_file()
        and not path.is_symlink(),
        "RECEIPT_MISSING:" + path.name,
    )

    raw = path.read_bytes()
    row = json.loads(raw)
    signature = row.pop("core_sha256", None)

    c012_check(
        hashlib.sha256(
            c012_bytes(row)
        ).hexdigest() == signature,
        "RECEIPT_SELF_HASH:" + path.name,
    )

    c012_times["checkpoint"] += (
        time.perf_counter() - started
    )

    return row, hashlib.sha256(raw).hexdigest()


def c012_put(path, core):
    started = time.perf_counter()

    c012_check(
        path.parent.resolve() == c012_dir
        and not path.is_symlink(),
        "UNSAFE_OUTPUT",
    )

    raw = c012_bytes({
        **core,
        "core_sha256": hashlib.sha256(
            c012_bytes(core)
        ).hexdigest(),
    }) + b"\n"

    if not path.exists():
        temporary = None

        try:
            with tempfile.NamedTemporaryFile(
                "wb",
                dir=c012_dir,
                prefix=".C012-",
                suffix=".tmp",
                delete=False,
            ) as stream:
                temporary = Path(stream.name)
                stream.write(raw)
                stream.flush()
                os.fsync(stream.fileno())

            c012_check(
                temporary.read_bytes() == raw,
                "TEMP_RELOAD",
            )

            try:
                os.link(temporary, path)
            except FileExistsError:
                pass

        finally:
            if temporary is not None:
                temporary.unlink(missing_ok=True)

    c012_check(
        path.read_bytes() == raw,
        "IMMUTABLE_CONFLICT:" + path.name,
    )

    c012_times["checkpoint"] += (
        time.perf_counter() - started
    )

    return hashlib.sha256(raw).hexdigest()


# ============================================================
# 3. Receipt audit, including the one authorized retry
# ============================================================

def c012_audit():
    counts = {
        "complete": 0,
        "partial": 0,
        "not_started": 0,
        "uncertain": 0,
        "blocked": 0,
    }

    states = {}
    unknown = []

    query_attempts = 0
    pair_attempts = 0
    pair_done = 0

    for cid in c012_ids:
        s1 = c012_path(cid, "stage1_started")
        d1 = c012_path(cid, "stage1_done")
        d2 = c012_path(cid, "stage2_done")

        s1_data = None
        d1_data = None

        if s1.is_file():
            s1_data, _ = c012_read(s1)

            c012_check(
                s1_data["status"] == "STARTED_UNCONFIRMED"
                and s1_data["boundary"] == c012_boundary_sha
                and s1_data["candidate_id"] == cid,
                "S1_START:" + cid,
            )

            query_attempts += 1

        if d1.is_file():
            c012_check(
                s1_data is not None,
                "S1_WITHOUT_START:" + cid,
            )

            d1_data, _ = c012_read(d1)

            c012_check(
                d1_data["boundary"] == c012_boundary_sha
                and d1_data["candidate_id"] == cid
                and d1_data["input_sha256"]
                == s1_data["input_sha256"]
                and d1_data["status"] in (
                    "STAGE1_PASS",
                    "STAGE1_BLOCKED",
                ),
                "S1_DONE:" + cid,
            )

        has_done = {}
        unknown_batches = []

        for start in range(0, 50, 4):
            begun = c012_path(
                cid,
                "stage2_batch_%02d_started" % start,
            )
            ended = c012_path(
                cid,
                "stage2_batch_%02d_done" % start,
            )

            retry_begun = (
                c012_path(
                    cid,
                    "stage2_batch_20_retry1_started",
                )
                if cid == c012_retry_id and start == 20
                else None
            )

            retry_ended = (
                c012_path(
                    cid,
                    "stage2_batch_20_retry1_done",
                )
                if retry_begun is not None
                else None
            )

            original = None
            original_sha = None

            if begun.is_file():
                original, original_sha = c012_read(begun)

                c012_check(
                    original["status"] == "STARTED_UNCONFIRMED"
                    and original["candidate_id"] == cid
                    and original["boundary"] == c012_boundary_sha
                    and original["batch_start"] == start
                    and len(original["chunk_ids"])
                    == min(4, 50 - start),
                    "S2_START:" + cid,
                )

                pair_attempts += len(
                    original["chunk_ids"]
                )

            c012_check(
                not ended.is_file()
                or original is not None,
                "S2_DONE_WITHOUT_START:" + cid,
            )

            retry_intent = None

            if (
                retry_begun is not None
                and retry_begun.is_file()
            ):
                c012_check(
                    original is not None
                    and not ended.is_file()
                    and original_sha
                    == c012_retry_original_start_sha,
                    "RETRY_ORIGINAL_IDENTITY",
                )

                retry_intent, _ = c012_read(
                    retry_begun
                )

                c012_check(
                    retry_intent["status"]
                    == "STARTED_UNCONFIRMED"
                    and retry_intent["original_start_sha256"]
                    == c012_retry_original_start_sha
                    and retry_intent["candidate_id"] == cid
                    and retry_intent["input_sha256"]
                    == original["input_sha256"]
                    and retry_intent["chunk_ids"]
                    == original["chunk_ids"]
                    and retry_intent["batch_start"] == start
                    and retry_intent["boundary"]
                    == c012_boundary_sha
                    and retry_intent["manual_attempt"] == 1,
                    "RETRY_STARTED_CONTRACT",
                )

                pair_attempts += len(
                    retry_intent["chunk_ids"]
                )

            active_path = (
                retry_ended
                if retry_ended is not None
                and retry_ended.is_file()
                else ended
            )

            if active_path.is_file():
                c012_check(
                    original is not None,
                    "S2_DONE_MISSING_ORIGINAL_START",
                )

                finished, receipt_sha = c012_read(
                    active_path
                )

                c012_check(
                    finished["status"] == "BATCH_PASS"
                    and finished["boundary"]
                    == c012_boundary_sha
                    and finished["candidate_id"] == cid
                    and finished["input_sha256"]
                    == original["input_sha256"]
                    and finished["chunk_ids"]
                    == original["chunk_ids"]
                    and len(finished["scores"])
                    == len(original["chunk_ids"])
                    and all(
                        math.isfinite(value)
                        for value in finished["scores"]
                    )
                    and (
                        active_path == ended
                        or retry_intent is not None
                    ),
                    "S2_DONE_BAD:" + cid,
                )

                has_done[start] = (
                    finished,
                    receipt_sha,
                    active_path,
                )

                pair_done += len(
                    finished["scores"]
                )

            elif original is not None:
                unknown_batches.append({
                    "batch_start": start,
                    "kind": (
                        "RETRY_STARTED_NO_DONE"
                        if retry_intent is not None
                        else "ORIGINAL_STARTED_NO_DONE"
                    ),
                })

        if has_done:
            c012_check(
                d1_data is not None
                and d1_data["status"] == "STAGE1_PASS"
                and all(
                    item[0]["input_sha256"]
                    == d1_data["input_sha256"]
                    for item in has_done.values()
                ),
                "ONNX_WITHOUT_VALID_STAGE1:" + cid,
            )

        if d2.is_file():
            final, _ = c012_read(d2)

            merged = {
                **final["historical_pool50_scores"],
                **final["new_scores"],
            }

            hybrid = final["historical_hybrid100"]

            missing = [
                chunk_id
                for chunk_id in hybrid
                if chunk_id not in final[
                    "historical_pool50_scores"
                ]
            ]

            c012_check(
                final["status"] == "STAGE2_PASS"
                and final["boundary"] == c012_boundary_sha
                and final["candidate_id"] == cid
                and d1_data is not None
                and d1_data["status"] == "STAGE1_PASS"
                and final["input_sha256"]
                == d1_data["input_sha256"]
                and d1_data["observed_hybrid100"] == hybrid
                and len(hybrid)
                == len(set(hybrid))
                == len(merged)
                == 100
                and len(missing)
                == len(final["new_scores"])
                == 50
                and all(
                    math.isfinite(value)
                    for value in merged.values()
                )
                and final["local100_order"] == sorted(
                    hybrid,
                    key=lambda chunk_id: (
                        -merged[chunk_id],
                        chunk_id,
                    ),
                )
                and final[
                    "rank_artifact_content_sha256"
                ] == hashlib.sha256(
                    c012_bytes({
                        "boundary": c012_boundary_sha,
                        "candidate_id": cid,
                        "scores": merged,
                        "order": final["local100_order"],
                    })
                ).hexdigest()
                and len(final["batch_receipts"])
                == len(has_done)
                == 13,
                "FINAL_SCORE_CONFLICT:" + cid,
            )

            rebuilt = {}

            for index, receipt in enumerate(
                final["batch_receipts"]
            ):
                begin = index * 4

                c012_check(
                    begin in has_done
                    and receipt["batch_start"] == begin
                    and receipt["path"]
                    == str(has_done[begin][2])
                    and receipt["sha256"]
                    == has_done[begin][1]
                    and has_done[begin][0]["chunk_ids"]
                    == missing[begin:begin + 4],
                    "FINAL_BATCH_LINK:" + cid,
                )

                rebuilt.update(
                    zip(
                        has_done[begin][0]["chunk_ids"],
                        has_done[begin][0]["scores"],
                    )
                )

            c012_check(
                rebuilt == final["new_scores"],
                "FINAL_SCORE_RECONSTRUCT:" + cid,
            )

        if s1_data is not None and d1_data is None:
            status = "UNCERTAIN"
            unknown.append((cid, "QUERY", []))

        elif unknown_batches:
            status = "UNCERTAIN"
            unknown.append((
                cid,
                "ONNX",
                unknown_batches,
            ))

        elif (
            d1_data is not None
            and d1_data["status"] == "STAGE1_BLOCKED"
        ):
            status = "BLOCKED"

        elif d2.is_file():
            status = "COMPLETE"

        elif s1_data is not None or has_done:
            status = "PARTIAL"

        else:
            status = "NOT_STARTED"

        states[cid] = status
        counts[status.lower()] += 1

    c012_check(
        query_attempts <= 1112
        and pair_attempts <= 55604,
        "SUPERVISOR_CUMULATIVE_ATTEMPT_LIMIT",
    )

    return {
        "states": states,
        "counts": counts,
        "unknown": unknown,
        "query_attempts": query_attempts,
        "pair_attempts": pair_attempts,
        "pair_done": pair_done,
    }


# ============================================================
# 4. Audit before any work
# ============================================================

c012_before = c012_audit()

c012_authorized_unknown = [
    (
        c012_retry_id,
        "ONNX",
        [{
            "batch_start": 20,
            "kind": "ORIGINAL_STARTED_NO_DONE",
        }],
    )
]

c012_check(
    c012_before["unknown"] in (
        c012_authorized_unknown,
        [],
    ),
    "UNKNOWN_UNITS_OUTSIDE_APPROVED_RETRY",
)

c012_check(
    c012_before["counts"]["blocked"] == 0,
    "STAGE1_BLOCKED",
)

c012_selected = [
    cid
    for cid in c012_ids
    if c012_before["states"][cid] != "COMPLETE"
][:c012_max_new_candidates]

print(
    json.dumps({
        "cell": "NB22E-v4:C012",
        "phase": "BEFORE_LIMITED_EXECUTION",
        "counts": c012_before["counts"],
        "unknown": c012_before["unknown"],
        "pair_attempts": c012_before["pair_attempts"],
        "selected": c012_selected,
        "max_candidates": 5,
    }, ensure_ascii=False, indent=2),
    flush=True,
)


# ============================================================
# 5. Controlled execution
# ============================================================

def c012_run(selected):
    np = nb22e_v4_require_imports("np")["np"]

    pins = {
        **c012_contract["source_producer_pins"],
        **c012_contract["input_pins"],
        **{
            "VECTOR_" + key: value
            for key, value in c012_contract[
                "query_vector_cache_pins"
            ].items()
        },
    }

    loader = c008_loader

    pins.update({
        "SPARSE_CONFIG": {
            "path": loader["configuration_path"],
            "file_bytes_sha256": loader[
                "configuration_file_sha256"
            ],
        },
        "SPARSE_DOCS": {
            "path": loader["documents_path"],
            "file_bytes_sha256": loader[
                "documents_sha256"
            ],
        },
        "SPARSE_POSTINGS": {
            "path": loader["postings_path"],
            "file_bytes_sha256": loader[
                "postings_sha256"
            ],
        },
    })

    pinned_paths = {}

    for label, pin in pins.items():
        relative = Path(pin["path"])
        path = (c012_root / relative).resolve()

        c012_check(
            not relative.is_absolute()
            and path.is_relative_to(c012_root)
            and path.is_file(),
            "PIN_PATH:" + label,
        )

        if path in pinned_paths:
            c012_check(
                pinned_paths[path]
                == pin["file_bytes_sha256"],
                "PIN_CONFLICT:" + label,
            )

        pinned_paths[path] = pin[
            "file_bytes_sha256"
        ]

        # C012 / R11.2 — ONNX alias contract aligned with C011.
    # Preserve the snapshot alias path; do not resolve the model symlink.

    snapshot = Path(
        os.path.abspath(
            str(Path(c004_snapshot).expanduser())
        )
    )

    model = Path(
        os.path.abspath(
            str(Path(c004_model_path).expanduser())
        )
    )

    c012_check(
        model.is_file()
        and model.parent.parent == snapshot
        and model.relative_to(snapshot).as_posix()
        == c012_contract["onnx"][
            "model_alias_relative_inside_snapshot"
        ]
        and snapshot.name
        == c012_contract["onnx"]["revision"],
        "ONNX_ALIAS",
    )

    pinned_paths[model] = c012_model_sha

    c012_check(
        type(c005_tokenizer).__name__
        == c012_contract["onnx"]["tokenizer"][
            "tokenizer_class"
        ]
        and bool(c005_tokenizer.is_fast)
        == c012_contract["onnx"]["tokenizer"][
            "is_fast"
        ],
        "TOKENIZER_IDENTITY",
    )

    for name, record in c012_contract[
        "onnx"
    ]["tokenizer"]["files"].items():
        pinned_paths[snapshot / name] = record["sha256"]

    for name, version in c012_contract[
        "runtime_versions"
    ].items():
        actual = (
            platform.python_version()
            if name == "python"
            else importlib.metadata.version(name)
        )

        c012_check(
            actual == version,
            "RUNTIME_CHANGED:" + name,
        )

    # Full input and model SHA: exactly once per invocation.
    started = time.perf_counter()
    pin_stamps = {}

    for path, expected in pinned_paths.items():
        c012_check(
            c012_sha(path) == expected,
            "SOURCE_SHA_CHANGED:" + path.name,
        )
        pin_stamps[path] = c012_stamp(path)

    c012_times["source_sha"] += (
        time.perf_counter() - started
    )

    # ========================================================
    # 6. Original Qdrant inventory: one full verification
    # ========================================================

    original = c004_index_path.resolve()

    c012_check(
        original.is_dir()
        and original.is_relative_to(c012_root),
        "QDRANT_ORIGINAL_PATH",
    )

    inventory = c004_docs[
        "manifest"
    ]["storage_inventory"]

    frozen = inventory["files"]

    c012_check(
        inventory["inventory_sha256"]
        == c004_docs["release"][
            "storage_inventory_sha256"
        ]
        == hashlib.sha256(
            c012_bytes(frozen)
        ).hexdigest()
        and inventory["file_count"] == len(frozen),
        "FROZEN_STORAGE_MANIFEST",
    )

    by_path = {
        item["path"]: item
        for item in frozen
    }

    c012_check(
        len(by_path) == len(frozen),
        "FROZEN_STORAGE_DUPLICATE",
    )

    index_stamps = {}
    seen = set()
    size_total = 0

    started = time.perf_counter()

    for folder, dirs, files in os.walk(
        original,
        followlinks=False,
    ):
        c012_check(
            not any(
                (Path(folder) / name).is_symlink()
                for name in dirs + files
            ),
            "INDEX_SYMLINK",
        )

        for name in files:
            file = Path(folder) / name
            relative = file.relative_to(
                original
            ).as_posix()

            c012_check(
                file.is_file(),
                "INDEX_NONFILE",
            )

            size = file.stat().st_size

            if name != ".lock":
                c012_check(
                    relative in by_path
                    and by_path[relative][
                        "size_bytes"
                    ] == size
                    and c012_sha(file)
                    == by_path[relative]["sha256"],
                    "INDEX_SHA_CHANGED:" + relative,
                )

                seen.add(relative)

            index_stamps[file] = c012_stamp(file)
            size_total += size

    c012_times["index_sha"] += (
        time.perf_counter() - started
    )

    c012_check(
        seen == set(by_path)
        and len(index_stamps)
        == c005_index_result["inventory_files"]
        and size_total
        == c005_index_result["inventory_bytes"]
        and size_total <= 512 * 1024 * 1024,
        "INDEX_INVENTORY_DRIFT",
    )

    def guard():
        # Metadata checks only; original full-byte SHA already verified.
        for path, stamp in pin_stamps.items():
            c012_check(
                path.is_file()
                and c012_stamp(path) == stamp,
                "PIN_CHANGED_IN_INVOCATION",
            )

        for path, stamp in index_stamps.items():
            c012_check(
                path.is_file()
                and c012_stamp(path) == stamp,
                "INDEX_CHANGED_IN_INVOCATION",
            )

        # Historical process peak, NOT system memory pressure.
        peak = resource.getrusage(
            resource.RUSAGE_SELF
        ).ru_maxrss

        divisor = (
            1024 ** 3
            if sys.platform == "darwin"
            else 1024 ** 2
        )

        c012_check(
            peak / divisor
            <= c012_process_rss_limit_gib,
            "PROCESS_HISTORICAL_PEAK_LIMIT",
        )

    guard()

    # ========================================================
    # 7. Read data for at most five selected questions
    # ========================================================

    started = time.perf_counter()
    selected_set = set(selected)
    rows = {}

    for key, depth in (
        ("HYBRID", 100),
        ("RERANKER", 50),
    ):
        found = {}

        path = (
            c012_root
            / c012_contract["input_pins"][key]["path"]
        ).resolve()

        with path.open("rb") as source:
            for line in source:
                item = json.loads(line)
                cid = item.get("candidate_id")

                if cid not in selected_set:
                    continue

                sequence = item["candidates"]

                c012_check(
                    cid not in found
                    and len(sequence) == depth
                    and [
                        entry["rank"]
                        for entry in sequence
                    ] == list(range(1, depth + 1))
                    and (
                        item.get("split") == "test"
                        if depth == 100
                        else item.get("split")
                        in ("test", None)
                    ),
                    "HISTORIC_ROW:" + cid,
                )

                found[cid] = sequence

        c012_check(
            set(found) == selected_set,
            "HISTORIC_ROW_COVERAGE:" + key,
        )

        rows[key] = found

    needed = {
        item["chunk_id"]
        for cid in selected
        for item in rows["HYBRID"][cid]
    }

    chunk_text = {}

    chunk_path = (
        c012_root
        / c012_contract["input_pins"]["CHUNKS"]["path"]
    )

    with chunk_path.open("rb") as stream:
        for line in stream:
            item = json.loads(line)
            chunk_id = item["chunk_id"]

            if chunk_id not in needed:
                continue

            c012_check(
                chunk_id not in chunk_text
                and isinstance(
                    item["content"],
                    str,
                ),
                "CHUNK_CONTENT",
            )

            chunk_text[chunk_id] = item["content"]

    c012_check(
        set(chunk_text) == needed,
        "CHUNK_COVERAGE",
    )

    questions = {}

    for cid in selected:
        question = c002_questions[cid]["question"]

        question_sha = hashlib.sha256(
            question.encode("utf-8")
        ).hexdigest()

        c012_check(
            question_sha
            == c002_questions[cid]["question_sha256"]
            == c012_members[cid],
            "QUESTION_PIN:" + cid,
        )

        questions[cid] = question.strip()

    vector_pins = c012_contract[
        "query_vector_cache_pins"
    ]

    metadata = json.loads(
        (
            c012_root
            / vector_pins["metadata"]["path"]
        ).read_bytes()
    )

    vector_order = metadata["candidate_ids"]

    c012_check(
        len(vector_order)
        == len(set(vector_order))
        == 1149
        and selected_set.issubset(vector_order),
        "VECTOR_ID_COVERAGE",
    )

    positions = {
        cid: index
        for index, cid in enumerate(vector_order)
        if cid in selected_set
    }

    vectors = np.load(
        c012_root / vector_pins["array"]["path"],
        mmap_mode="r",
        allow_pickle=False,
    )

    c012_check(
        vectors.shape == (1149, 384)
        and vectors.dtype == np.float32,
        "VECTOR_SHAPE",
    )

    tokens = {
        cid: c008_tokens(questions[cid])
        for cid in selected
    }

    needed_terms = set().union(
        *(set(values) for values in tokens.values())
    )

    postings = {}

    with (
        c012_root / loader["postings_path"]
    ).open("rb") as stream:
        for line in stream:
            item = json.loads(line)

            if item["token"] not in needed_terms:
                continue

            ids = np.asarray(
                item["document_indexes"],
                dtype=np.int64,
            )
            freqs = np.asarray(
                item["term_frequencies"],
                dtype=np.float64,
            )

            c012_check(
                len(ids)
                == len(freqs)
                == item["document_frequency"]
                and bool(
                    (ids[:-1] < ids[1:]).all()
                ),
                "SPARSE_POSTINGS_CONFLICT",
            )

            postings[item["token"]] = (
                ids,
                freqs,
            )

    doc_ids = c008_doc_ids

    lengths = np.asarray(
        c008_doc_lengths,
        dtype=np.float64,
    )

    avg = float(c008_average_doc_length)

    c012_check(
        len(doc_ids)
        == len(lengths)
        == 3488
        and doc_ids == c007_chunk_ids
        and avg > 0,
        "SPARSE_DOCUMENT_CONTRACT",
    )

    c012_times["input_read"] += (
        time.perf_counter() - started
    )

    def sparse(cid):
        scores = np.zeros(
            3488,
            dtype=np.float64,
        )

        for term, qtf in collections.Counter(
            tokens[cid]
        ).items():
            if term not in postings:
                continue

            ids, freqs = postings[term]
            df = len(ids)

            idf = math.log(
                1.0
                + (3488 - df + 0.5) / (df + 0.5)
            )

            normal = (
                1.0 - 0.3
                + 0.3 * lengths[ids] / avg
            )

            scores[ids] += (
                idf
                * freqs
                * 2.2
                / (freqs + 1.2 * normal)
            ) * float(qtf)

        positive = np.flatnonzero(scores > 0)

        ranked = positive[
            np.lexsort((
                positive,
                -scores[positive],
            ))
        ][:100]

        return (
            [doc_ids[index] for index in ranked],
            [float(scores[index]) for index in ranked],
        )

    # ========================================================
    # 8. Lazy model and Qdrant setup
    # ========================================================

    temp_index = None
    client = None
    session = None
    input_names = None
    previous_new_onnx = False

    def model_session():
        nonlocal session, input_names

        if session is not None:
            return

        started = time.perf_counter()

        ort = nb22e_v4_require_imports("ort")["ort"]

        c012_check(
            ort.__version__ == "1.16.3",
            "ORT_VERSION",
        )

        options = ort.SessionOptions()
        options.intra_op_num_threads = 4
        options.inter_op_num_threads = 1
        options.execution_mode = (
            ort.ExecutionMode.ORT_SEQUENTIAL
        )
        options.graph_optimization_level = (
            ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        )

        session = ort.InferenceSession(
            str(model),
            sess_options=options,
            providers=["CPUExecutionProvider"],
        )

        input_names = {
            item.name
            for item in session.get_inputs()
        }

        c012_check(
            session.get_providers()
            == ["CPUExecutionProvider"]
            and {
                "input_ids",
                "attention_mask",
            } <= input_names,
            "ONNX_SESSION_CONTRACT",
        )

        c012_times["model_setup"] += (
            time.perf_counter() - started
        )

    def ensure_client():
        nonlocal client, temp_index

        if client is not None:
            return

        started = time.perf_counter()

        disk = os.statvfs(str(c012_out))

        c012_check(
            disk.f_bavail * disk.f_frsize
            >= max(
                256 * 1024 * 1024,
                2 * size_total,
            ),
            "INSUFFICIENT_CLONE_DISK",
        )

        temp_index = tempfile.TemporaryDirectory(
            prefix="NB22E-C012-",
            dir=str(c012_out),
        )

        clone = Path(temp_index.name) / "qdrant"
        clone.mkdir()

        for source, stamp in index_stamps.items():
            guard()

            target = clone / source.relative_to(original)
            target.parent.mkdir(
                parents=True,
                exist_ok=True,
            )

            with (
                source.open("rb") as source_stream,
                target.open("xb") as target_stream,
            ):
                for block in iter(
                    lambda: source_stream.read(
                        1024 * 1024
                    ),
                    b"",
                ):
                    target_stream.write(block)

            c012_check(
                c012_stamp(source) == stamp,
                "SOURCE_CHANGED_DURING_COPY",
            )

        client = nb22e_v4_require_imports(
            "QdrantClient"
        )["QdrantClient"](
            path=str(clone)
        )

        c012_check(
            client.count(
                collection_name=c005_collection,
                exact=True,
            ).count == 3488,
            "CLONE_POINT_COUNT",
        )

        c012_times["qdrant_setup"] += (
            time.perf_counter() - started
        )

    # ========================================================
    # 9. Sequential candidate execution
    # ========================================================

    try:
        for cid in selected:
            global c012_active
            c012_active = cid

            guard()

            ids100 = [
                item["chunk_id"]
                for item in rows["HYBRID"][cid]
            ]

            ids50 = [
                item["chunk_id"]
                for item in rows["RERANKER"][cid]
            ]

            saved = {
                item["chunk_id"]: float(
                    item["reranker_score"]
                )
                for item in rows["RERANKER"][cid]
            }

            missing50 = [
                chunk_id
                for chunk_id in ids100
                if chunk_id not in saved
            ]

            c012_check(
                len(set(ids100)) == 100
                and len(set(ids50))
                == len(saved)
                == 50
                and set(ids50) == set(ids100[:50])
                and len(missing50) == 50
                and all(
                    math.isfinite(value)
                    for value in saved.values()
                ),
                "FROZEN_POOL50:" + cid,
            )

            identity = hashlib.sha256(
                c012_bytes({
                    "boundary": c012_boundary_sha,
                    "candidate_id": cid,
                    "question_sha256": (
                        c002_questions[cid][
                            "question_sha256"
                        ]
                    ),
                    "historical_hybrid100": ids100,
                    "historical_pool50": ids50,
                    "historical_pool50_scores": saved,
                    "model_sha256": c012_model_sha,
                })
            ).hexdigest()

            s1 = c012_path(
                cid,
                "stage1_started",
            )
            d1 = c012_path(
                cid,
                "stage1_done",
            )
            d2 = c012_path(
                cid,
                "stage2_done",
            )

            s1_core = {
                "status": "STARTED_UNCONFIRMED",
                "boundary": c012_boundary_sha,
                "candidate_id": cid,
                "input_sha256": identity,
                "query_limit": 1,
                "manual_retry_only_if_uncertain": True,
            }

            # ================================================
            # Stage 1: reuse or one new Query
            # ================================================

            if d1.is_file():
                previous, _ = c012_read(d1)
                intent, _ = c012_read(s1)

                c012_check(
                    intent == s1_core
                    and previous["status"] == "STAGE1_PASS"
                    and previous["input_sha256"] == identity
                    and previous["observed_hybrid100"]
                    == ids100
                    and previous["full_order_matches"] is True
                    and previous[
                        "pool50_membership_matches"
                    ] is True,
                    "REUSED_STAGE1_CONFLICT:" + cid,
                )

            else:
                c012_check(
                    not s1.exists(),
                    "UNKNOWN_STAGE1_STARTED_NO_DONE",
                )

                # Only now copy/open Qdrant.
                ensure_client()

                vector = np.asarray(
                    vectors[positions[cid]],
                    dtype=np.float32,
                )

                c012_check(
                    vector.shape == (384,)
                    and bool(
                        np.isfinite(vector).all()
                    )
                    and bool(
                        np.isclose(
                            np.linalg.norm(vector),
                            1.0,
                        )
                    ),
                    "VECTOR_INVALID",
                )

                sparse_ids, sparse_scores = sparse(cid)

                guard()

                c012_check(
                    c012_before["query_attempts"]
                    + c012_work["query_attempts"]
                    + 1 <= 1112,
                    "QUERY_ATTEMPT_BUDGET",
                )

                c012_put(s1, s1_core)

                c012_work["query_attempts"] += 1

                started = time.perf_counter()

                response = client.query_points(
                    collection_name=c005_collection,
                    query=vector.tolist(),
                    limit=100,
                    with_payload=True,
                    with_vectors=False,
                )

                points = getattr(
                    response,
                    "points",
                    response,
                )

                dense_ids = [
                    str(item.payload["chunk_id"])
                    for item in points
                ]

                dense_scores = [
                    float(item.score)
                    for item in points
                ]

                observed = c008_fuse(
                    dense_ids,
                    dense_scores,
                    sparse_ids,
                    sparse_scores,
                )

                elapsed = time.perf_counter() - started
                c012_times["query"] += elapsed

                order_ok = (
                    observed == ids100
                    and len(observed) == 100
                )

                pool_ok = (
                    set(observed[:50]) == set(ids50)
                )

                c012_put(d1, {
                    "status": (
                        "STAGE1_PASS"
                        if order_ok and pool_ok
                        else "STAGE1_BLOCKED"
                    ),
                    "boundary": c012_boundary_sha,
                    "candidate_id": cid,
                    "input_sha256": identity,
                    "observed_hybrid100": observed,
                    "expected_hybrid100_sha256": (
                        hashlib.sha256(
                            c012_bytes(ids100)
                        ).hexdigest()
                    ),
                    "full_order_matches": order_ok,
                    "pool50_membership_matches": pool_ok,
                    "dense_count": len(dense_ids),
                    "sparse_count": len(sparse_ids),
                    "query_count": 1,
                    "elapsed_seconds": round(elapsed, 4),
                    "first_mismatch_rank": next(
                        (
                            index + 1
                            for index in range(
                                min(len(observed), 100)
                            )
                            if observed[index] != ids100[index]
                        ),
                        None,
                    ),
                })

                c012_check(
                    order_ok and pool_ok,
                    "HYBRID_ORDER_CONFLICT",
                )

                c012_check(
                    elapsed <= c012_query_slow_limit_seconds,
                    "QUERY_SLOW_STOP",
                )

            c012_check(
                not d2.is_file(),
                "COMPLETE_CANDIDATE_WAS_SELECTED",
            )

            # ================================================
            # Stage 2: immutable ONNX batch checkpoints
            # ================================================

            new_scores = {}
            receipts = []

            for start in range(0, 50, 4):
                batch_ids = missing50[
                    start:start + 4
                ]

                begun = c012_path(
                    cid,
                    "stage2_batch_%02d_started" % start,
                )
                done = c012_path(
                    cid,
                    "stage2_batch_%02d_done" % start,
                )

                core = {
                    "status": "STARTED_UNCONFIRMED",
                    "boundary": c012_boundary_sha,
                    "candidate_id": cid,
                    "input_sha256": identity,
                    "batch_start": start,
                    "chunk_ids": batch_ids,
                    "no_automatic_retry_if_uncertain": True,
                }

                if done.is_file():
                    intent, _ = c012_read(begun)
                    completed, done_sha = c012_read(done)

                    c012_check(
                        intent == core
                        and completed["status"] == "BATCH_PASS"
                        and completed["input_sha256"] == identity
                        and completed["chunk_ids"] == batch_ids
                        and len(completed["scores"])
                        == len(batch_ids)
                        and all(
                            math.isfinite(value)
                            for value in completed["scores"]
                        ),
                        "NORMAL_RECEIPT_REUSE:" + cid,
                    )

                    scores = completed["scores"]
                    selected_done = done

                else:
                    approved_retry = (
                        cid == c012_retry_id
                        and start == c012_retry_start
                    )

                    retry_begun = c012_path(
                        cid,
                        "stage2_batch_20_retry1_started",
                    )
                    retry_done = c012_path(
                        cid,
                        "stage2_batch_20_retry1_done",
                    )

                    if approved_retry:
                        original, original_sha = c012_read(
                            begun
                        )

                        c012_check(
                            original == core
                            and original_sha
                            == c012_retry_original_start_sha,
                            "ORIGINAL_STARTED_RECEIPT_CHANGED",
                        )

                        manual_intent = {
                            **core,
                            "manual_attempt": 1,
                            "original_start_sha256": (
                                c012_retry_original_start_sha
                            ),
                            "authorization": (
                                "SUPERVISOR_SINGLE_MANUAL_RETRY_MAX_55604"
                            ),
                            "total_pair_attempt_ceiling": 55604,
                        }

                        if retry_done.is_file():
                            retry_intent, _ = c012_read(
                                retry_begun
                            )

                            completed, done_sha = c012_read(
                                retry_done
                            )

                            c012_check(
                                retry_intent == manual_intent
                                and completed["status"]
                                == "BATCH_PASS"
                                and completed["input_sha256"]
                                == identity
                                and completed["chunk_ids"]
                                == batch_ids
                                and completed["manual_attempt"] == 1
                                and len(completed["scores"]) == 4
                                and all(
                                    math.isfinite(value)
                                    for value in completed["scores"]
                                ),
                                "RETRY_DONE_CONFLICT",
                            )

                            scores = completed["scores"]
                            selected_done = retry_done

                        else:
                            c012_check(
                                not retry_begun.exists(),
                                "RETRY_STARTED_WITHOUT_DONE_NO_AUTORETRY",
                            )

                    else:
                        c012_check(
                            not begun.exists(),
                            "UNKNOWN_BATCH_NO_AUTORETRY",
                        )

                    if not (
                        approved_retry
                        and retry_done.is_file()
                    ):
                        guard()
                        model_session()

                        c012_check(
                            c012_before["pair_attempts"]
                            + c012_work["onnx_attempted_pairs"]
                            + len(batch_ids)
                            <= 55604,
                            "PAIR_ATTEMPT_BUDGET_55604",
                        )

                        # Measure tokenization separately.
                        started = time.perf_counter()

                        encoded = c005_tokenizer(
                            [questions[cid]] * len(batch_ids),
                            [
                                chunk_text[chunk_id]
                                for chunk_id in batch_ids
                            ],
                            padding=True,
                            truncation=True,
                            max_length=512,
                            return_tensors="np",
                        )

                        c012_times["tokenization"] += (
                            time.perf_counter() - started
                        )

                        c012_check(
                            input_names <= set(encoded),
                            "TOKENIZER_INPUTS",
                        )

                        feed = {
                            name: np.asarray(
                                encoded[name],
                                dtype=np.int64,
                            )
                            for name in input_names
                        }

                        c012_check(
                            all(
                                value.shape[0] == len(batch_ids)
                                and value.shape[1] <= 512
                                for value in feed.values()
                            ),
                            "TOKEN_SHAPE",
                        )

                        # Preserve protective pauses.
                        if previous_new_onnx:
                            started = time.perf_counter()

                            time.sleep(
                                c012_onnx_pause_seconds
                            )

                            c012_times["pause"] += (
                                time.perf_counter() - started
                            )

                            guard()

                        # Historical STARTED stays immutable.
                        # Retry uses a separately named receipt.
                        c012_put(
                            retry_begun if approved_retry else begun,
                            manual_intent if approved_retry else core,
                        )

                        c012_work[
                            "onnx_attempted_pairs"
                        ] += len(batch_ids)

                        c012_work["onnx_calls"] += 1

                        if approved_retry:
                            c012_work[
                                "manual_retry_pairs"
                            ] += len(batch_ids)

                        started = time.perf_counter()

                        outputs = session.run(
                            None,
                            feed,
                        )

                        elapsed = (
                            time.perf_counter() - started
                        )

                        c012_times["onnx"] += elapsed

                        values = np.asarray(
                            outputs[0],
                            dtype=np.float64,
                        ).reshape(-1)

                        c012_check(
                            values.shape == (len(batch_ids),)
                            and bool(
                                np.isfinite(values).all()
                            ),
                            "NONFINITE_ONNX_OUTPUT",
                        )

                        scores = [
                            float(value)
                            for value in values
                        ]

                        selected_done = (
                            retry_done
                            if approved_retry
                            else done
                        )

                        done_core = {
                            "status": "BATCH_PASS",
                            "boundary": c012_boundary_sha,
                            "candidate_id": cid,
                            "input_sha256": identity,
                            "chunk_ids": batch_ids,
                            "scores": scores,
                            "elapsed_seconds": round(
                                elapsed,
                                4,
                            ),
                            "process_peak_rss_gib": (
                                c012_memory()[
                                    "process_peak_rss_gib"
                                ]
                            ),
                            "onnx_calls": 1,
                            "qdrant_queries": 0,
                            "api_calls": 0,
                        }

                        if approved_retry:
                            done_core.update({
                                "manual_attempt": 1,
                                "original_start_sha256": (
                                    c012_retry_original_start_sha
                                ),
                            })

                        done_sha = c012_put(
                            selected_done,
                            done_core,
                        )

                        c012_work[
                            "onnx_done_pairs"
                        ] += len(batch_ids)

                        previous_new_onnx = True

                        c012_check(
                            elapsed
                            <= c012_onnx_slow_limit_seconds,
                            "ONNX_SLOW_STOP",
                        )

                        guard()

                c012_check(
                    not set(batch_ids).intersection(
                        new_scores
                    ),
                    "REUSED_PAIR_DUPLICATE",
                )

                new_scores.update(
                    zip(batch_ids, scores)
                )

                receipts.append({
                    "path": str(selected_done),
                    "sha256": done_sha,
                    "batch_start": start,
                    "count": len(batch_ids),
                })

            # ================================================
            # 10. Complete Local100 without changing scores
            # ================================================

            merged = {
                **saved,
                **new_scores,
            }

            c012_check(
                len(new_scores) == 50
                and len(merged) == 100
                and set(merged) == set(ids100)
                and all(
                    math.isfinite(value)
                    for value in merged.values()
                ),
                "LOCAL100_INCOMPLETE",
            )

            rank = sorted(
                ids100,
                key=lambda chunk_id: (
                    -merged[chunk_id],
                    chunk_id,
                ),
            )

            rank_sha = hashlib.sha256(
                c012_bytes({
                    "boundary": c012_boundary_sha,
                    "candidate_id": cid,
                    "scores": merged,
                    "order": rank,
                })
            ).hexdigest()

            c012_put(d2, {
                "status": "STAGE2_PASS",
                "boundary": c012_boundary_sha,
                "candidate_id": cid,
                "input_sha256": identity,
                "historical_hybrid100": ids100,
                "historical_pool50_scores": saved,
                "new_scores": new_scores,
                "local100_order": rank,
                "rank_artifact_content_sha256": rank_sha,
                "batch_receipts": receipts,
                "new_pairs": 50,
                "historical_reused_pairs": 50,
                "derived_order_quota_char_rrf_executed": False,
                "context_generated": False,
                "target_gate": "BLOCKED",
            })

            c012_put(
                c012_path(cid, "progress"),
                {
                    "status": "CANDIDATE_CHECKPOINTED",
                    "boundary": c012_boundary_sha,
                    "candidate_id": cid,
                    "stage1_path": str(d1),
                    "stage2_path": str(d2),
                },
            )

            c012_work["new_completed"] += 1

            print(
                "C012 candidate",
                c012_work["new_completed"],
                "/",
                len(selected),
                cid,
                "RSS",
                c012_memory(),
                flush=True,
            )

        # ====================================================
        # 11. Close only fully completed original batches
        # ====================================================

        for batch in c012_batches:
            summary_path = c012_dir / (
                "C012.batch_%03d.done.json"
                % batch["batch_number"]
            )

            if summary_path.is_file():
                continue

            if not all(
                c012_path(
                    cid,
                    "stage2_done",
                ).is_file()
                for cid in batch["candidate_ids"]
            ):
                continue

            receipts = []

            for cid in batch["candidate_ids"]:
                stage1, stage1_sha = c012_read(
                    c012_path(
                        cid,
                        "stage1_done",
                    )
                )

                stage2, stage2_sha = c012_read(
                    c012_path(
                        cid,
                        "stage2_done",
                    )
                )

                c012_check(
                    stage1["status"] == "STAGE1_PASS"
                    and stage2["status"] == "STAGE2_PASS",
                    "BATCH_CLOSURE_NOT_VERIFIED",
                )

                receipts.append({
                    "candidate_id": cid,
                    "stage1_sha256": stage1_sha,
                    "stage2_sha256": stage2_sha,
                })

            c012_put(summary_path, {
                "status": "BATCH_COMPLETE",
                "boundary": c012_boundary_sha,
                "batch_number": batch["batch_number"],
                "candidate_ids": batch["candidate_ids"],
                "completed_in_batch": len(
                    batch["candidate_ids"]
                ),
                "receipts": receipts,
            })

    finally:
        if client is not None:
            client.close()

        session = None

        if temp_index is not None:
            temp_index.cleanup()


# ============================================================
# 12. Execute once, bounded by five unfinished candidates
# ============================================================

if c012_execute_new_work:
    try:
        if c012_selected:
            c012_run(c012_selected)

    except KeyboardInterrupt:
        c012_stop = {
            "kind": "KEYBOARD_INTERRUPT",
            "candidate_id": c012_active,
            "action": (
                "STARTED_WITHOUT_DONE_HELD_NO_AUTO_RETRY"
            ),
        }

    except Exception as error:
        sys.excepthook(type(error), error, error.__traceback__)
        c012_stop = {
            "kind": type(error).__name__,
            "detail": str(error)[:700],
            "candidate_id": c012_active,
        }


# ============================================================
# 13. Final audit, timing table and bounded diagnosis
# ============================================================

c012_after = c012_audit()

c012_elapsed = (
    time.perf_counter() - c012_run_started
)

c012_accounted = sum(
    c012_times.values()
)

c012_times["other_unattributed"] = max(
    0.0,
    c012_elapsed - c012_accounted,
)

c012_share = [
    {
        "part": key,
        "seconds": round(value, 3),
        "percent_of_elapsed": (
            round(
                100 * value / c012_elapsed,
                1,
            )
            if c012_elapsed
            else 0.0
        ),
    }
    for key, value in c012_times.items()
]

c012_share.sort(
    key=lambda row: -row["seconds"]
)

if c012_work["onnx_calls"] == 0:
    c012_diagnosis = (
        "INSUFFICIENT_NEW_ONNX_TIMING"
    )
    c012_next = (
        "CONTINUE_ONE_CONTROLLED_INVOCATION_AFTER_REPAIRING_ANY_STOP"
    )

else:
    dominant = c012_share[0]["part"]

    if dominant == "onnx":
        c012_diagnosis = (
            "ONNX_WALL_TIME_DOMINATES_THIS_INVOCATION;"
            " SYSTEM_PRESSURE_NOT_MEASURED"
        )
        c012_next = (
            "PREPARE_CPU_OR_GPU_NUMERIC_PARITY_FEASIBILITY_REVIEW_ONLY"
        )

    else:
        c012_diagnosis = (
            "LOCAL_OVERHEAD_DOMINANT:" + dominant
        )
        c012_next = (
            "OPTIMIZE_THIS_LOCAL_OVERHEAD_WITHOUT_CHANGING_FROZEN_SCORING"
        )

c012_report = {
    "cell": "NB22E-v4:C012",
    "status": (
        "STOPPED_WITH_CHECKPOINTS"
        if c012_stop
        else (
            "LIMITED_INVOCATION_COMPLETE"
            if c012_execute_new_work
            else "AUDIT_ONLY"
        )
    ),
    "execution_boundary_sha256": c012_boundary_sha,
    "selected_candidate_ids": c012_selected,
    "before": {
        key: value
        for key, value in c012_before.items()
        if key != "states"
    },
    "after": {
        key: value
        for key, value in c012_after.items()
        if key != "states"
    },
    "new_work_this_invocation": c012_work,
    "total_elapsed_seconds": round(
        c012_elapsed,
        3,
    ),
    "timing_table": c012_share,
    "bottleneck_assessment": c012_diagnosis,
    "single_next_action": c012_next,
    "stop_reason": c012_stop,
    "memory": c012_memory(),
    "api_calls": 0,
    "repository_modified": False,
    "four_pilot_reused": 4,
    "retried_historical_started_receipt_unchanged": True,
    "quality_gate": "BLOCKED",
    "GPU_transfer_authorized": False,
}


# ============================================================
# 14. Immutable run report with reload verification
# ============================================================

if c012_execute_new_work:
    raw = c012_bytes(c012_report) + b"\n"

    report_sha = hashlib.sha256(
        raw
    ).hexdigest()

    report_path = c012_dir / (
        "C012.run_summary."
        + report_sha
        + ".json"
    )

    if not report_path.exists():
        temporary = None

        try:
            with tempfile.NamedTemporaryFile(
                "wb",
                dir=c012_dir,
                prefix=".C012-report-",
                suffix=".tmp",
                delete=False,
            ) as stream:
                temporary = Path(stream.name)
                stream.write(raw)
                stream.flush()
                os.fsync(stream.fileno())

            try:
                os.link(temporary, report_path)
            except FileExistsError:
                pass

        finally:
            if temporary is not None:
                temporary.unlink(missing_ok=True)

    c012_check(
        report_path.read_bytes() == raw,
        "REPORT_RELOAD_CONFLICT",
    )

    c012_report["report_checkpoint"] = {
        "path": str(report_path),
        "sha256": report_sha,
    }

NB22E_V4_POPULATION_EXECUTION = c012_report

print(
    json.dumps(
        c012_report,
        ensure_ascii=False,
        indent=2,
    ),
    flush=True,
)

{
  "cell": "NB22E-v4:C012",
  "phase": "BEFORE_LIMITED_EXECUTION",
  "counts": {
    "complete": 4,
    "partial": 0,
    "not_started": 1108,
    "uncertain": 0,
    "blocked": 0
  },
  "unknown": [],
  "pair_attempts": 204,
  "selected": [
    "golden_v2_candidate_01b22bd8a25fb93d74033d78",
    "golden_v2_candidate_01bacc2bf4a88b206123a68c",
    "golden_v2_candidate_01d35d353c6e3c5a219fc1f2",
    "golden_v2_candidate_021a702f669446a411c22d0e",
    "golden_v2_candidate_0223fb3898c3681e9fd788a0"
  ],
  "max_candidates": 5
}


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

C012 candidate 1 / 5 golden_v2_candidate_01b22bd8a25fb93d74033d78 RSS {'process_current_rss_gib': 0.985, 'process_peak_rss_gib': 0.985, 'system_memory_pressure': 'NOT_MEASURED'}


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

C012 candidate 2 / 5 golden_v2_candidate_01bacc2bf4a88b206123a68c RSS {'process_current_rss_gib': 0.901, 'process_peak_rss_gib': 0.998, 'system_memory_pressure': 'NOT_MEASURED'}


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

... (4 outputs left out)

C012 candidate 5 / 5 golden_v2_candidate_0223fb3898c3681e9fd788a0 RSS {'process_current_rss_gib': 0.968, 'process_peak_rss_gib': 0.998, 'system_memory_pressure': 'NOT_MEASURED'}


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


{
  "cell": "NB22E-v4:C012",
  "status": "LIMITED_INVOCATION_COMPLETE",
  "execution_boundary_sha256": "aa099a56e76054c30629a35a3acbd1ae9a86970367c9fb60677b907c2f0d2cbc",
  "selected_candidate_ids": [
    "golden_v2_candidate_01b22bd8a25fb93d74033d78",
    "golden_v2_candidate_01bacc2bf4a88b206123a68c",
    "golden_v2_candidate_01d35d353c6e3c5a219fc1f2",
    "golden_v2_candidate_021a702f669446a411c22d0e",
    "golden_v2_candidate_0223fb3898c3681e9fd788a0"
  ],
  "before": {
    "counts": {
      "complete": 4,
      "partial": 0,
      "not_started": 1108,
      "uncertain": 0,
      "blocked": 0
    },
    "unknown": [],
    "query_attempts": 4,
    "pair_attempts": 204,
    "pair_done": 200
  },
  "after": {
    "counts": {
      "complete": 9,
      "partial": 0,
      "not_started": 1103,
      "uncertain": 0,
      "blocked": 0
    },
    "unknown": [],
    "query_attempts": 9,
    "pair_attempts": 454,
    "pair_done": 450
  },
  "new_work_this_invocation": {
    "query_attempts"

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


<div dir="rtl" align="right">

### NB22E-v4 | C012 | R11.2 — ادامهٔ خودکار و محافظت‌شدهٔ Stage 1/2

**هدف:** تکمیل ۱۱۰۳ سؤال باقی‌مانده در مرحله‌های متوالی حداکثر پنج‌سؤالی، بدون تأیید دستی برای هر مرحله و بدون تکرار محاسبات موفق.

**مبنای پذیرفته‌شده:** ۹ سؤال جدید کامل، چهار Pilot محفوظ، ۴۵۰ جفت یکتای موفق، ۴۵۴ pair-attempt و چهار تلاش تاریخیِ تکراریِ مصوب. سقف تجمعی تلاش‌ها ۵۵٬۶۰۴ است.

**حفاظت از نتایج:**
- خواندن و اعتبارسنجی Manifest فریز‌شدهٔ C011 و رسیدهای موجود.
- انتخاب نخستین سؤال ناتمام از ترتیب اصلی جمعیت.
- استفادهٔ مجدد از Stage 1، Batchهای ONNX و امتیازهای معتبر؛ بدون Retry جدید.
- ذخیرهٔ STARTED پیش از کار تازه و DONE فقط پس از خروجی معتبر.
- ذخیره و بازخوانی Checkpoint بعد از هر سؤال.
- حفظ مرز Batchهای اصلی ۲۵سؤالی و عدم تغییر فایل‌های تاریخی.

**کنترل منابع:**
- یک مرتبه هش کامل فایل‌های فریز‌شده در هر فراخوانی؛ سپس کنترل تغییر مشخصات فایل‌ها.
- ساخت یک کپی موقت Qdrant، فقط در صورت نیاز، برای کل فراخوانی.
- یک نشست ONNX و دسترسی انتخابی به داده‌های پنج سؤال جاری.
- حفظ CPU، چهار Thread، Batch=4، max_length=512 و مکث حفاظتی ۰٫۸ ثانیه.
- نمونه‌برداری دوره‌ای از RSS فرایند، شاخص حافظهٔ سیستم و محدودیت سرعت CPU در macOS. دادهٔ ناموجود به‌معنای سلامت تأییدشده نیست.

**گزارش هر مرحله:** تعداد سؤال و جفت‌های تکمیل‌شده، زمان Query، Tokenization، ONNX، مکث، هش و آماده‌سازی، Checkpoint، سربار، مصرف حافظه و برآورد تازهٔ زمان باقی‌مانده.

**شرایط توقف:** Interrupt، STARTED بدون DONE، ناسازگاری SHA یا نسخه، خروجی نامعتبر، شکست Checkpoint، افت شدید سرعت یا شاخص نامطلوب واقعی سیستم. خطا باعث Retry خودکار نمی‌شود.

**دامنهٔ مجاز:** فقط Stage 1 و Stage 2. بدون API، تغییر مدل، انتقال GPU، تغییر مخزن، derived_order، Quota، charRRF، ساخت Context یا Generation.

**پذیرش:** تکمیل فنی Stage 1/2 مستقل از پذیرش نهایی R11.2 است. فایل‌های خروجی فقط در پوشهٔ جدا از مخزن قرار می‌گیرند.

</div>

In [16]:
# NB22E-v4 | C012 | R11.2
# Resume only frozen Stage 1/2; automatic five-candidate stages, immutable checkpoints.

c012_stage_size = 5
c012_onnx_pause_seconds = 0.8
c012_process_rss_limit_gib = 4.0
c012_monitor_interval_seconds = 90.0
c012_query_slow_limit_seconds = 45.0
c012_onnx_slow_limit_seconds = 25.0
c012_retry_id = "golden_v2_candidate_0184f90d79f3dfc4744b6e29"
c012_retry_original_start_sha = "4745577680cbd38d3a212e27ec67e3af98a42088cd2f3aace077fca563e4daf2"
c012_boundary_sha = "aa099a56e76054c30629a35a3acbd1ae9a86970367c9fb60677b907c2f0d2cbc"
c012_manifest_sha = "a3ffce372eab24e210bedae9b2b6087dc2493e45805c4c27836d223a6578fe80"
c012_pilot_audit_sha = "ae2360b309dc5ed6fe44c1735eecdb0407f96324ca394776739bd6fddbccc6a0"
c012_model_sha = "6c2513767fb63d008a4377bef7a7a3555433d9436342bb53e35a3a72ffc52d4b"
c012_index_sha = "2fc5e50ebbd1e1e66d59c703eb48f1367b2d3deadfed62efb9d5a5e6959f7ef4"
c012_run_start = time.perf_counter()
c012_timing = {name: 0.0 for name in (
    "source_sha", "index_sha", "input_read", "model_setup", "qdrant_setup",
    "query", "tokenization", "onnx", "pause", "checkpoint",
    "system_monitor")}
c012_counts_run = {"new_queries": 0, "new_onnx_attempts": 0,
                   "new_onnx_completed": 0, "onnx_calls": 0,
                   "new_candidates_completed": 0, "new_retries": 0}
c012_stage_history = []
c012_stop = None
c012_active = None
c012_last_sample_at = 0.0
c012_system_samples = []
c012_recent_onnx = []
c012_slow_consecutive = 0


def c012_check(condition, code):
    if not condition:
        raise RuntimeError("C012_" + code)


def c012_canonical(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True,
                      separators=(",", ":"), allow_nan=False).encode("utf-8")


def c012_stamp(path):
    st = path.stat()
    return (st.st_dev, st.st_ino, st.st_size, st.st_mtime_ns)


def c012_sha(path):
    old = c012_stamp(path)
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    c012_check(old == c012_stamp(path), "SOURCE_CHANGED_DURING_SHA:" + path.name)
    return digest.hexdigest()


def c012_read(path):
    started = time.perf_counter()
    c012_check(path.parent.resolve() == c012_dir and path.is_file()
               and not path.is_symlink(), "RECEIPT_UNSAFE:" + path.name)
    raw = path.read_bytes()
    payload = json.loads(raw)
    signature = payload.pop("core_sha256", None)
    c012_check(signature == hashlib.sha256(c012_canonical(payload)).hexdigest(),
               "RECEIPT_SHA:" + path.name)
    c012_timing["checkpoint"] += time.perf_counter() - started
    return payload, hashlib.sha256(raw).hexdigest()


def c012_write(path, core):
    started = time.perf_counter()
    c012_check(path.parent.resolve() == c012_dir and not path.is_symlink(),
               "OUTPUT_UNSAFE:" + path.name)
    raw = c012_canonical({**core,
        "core_sha256": hashlib.sha256(c012_canonical(core)).hexdigest()}) + b"\n"
    if not path.exists():
        temporary = None
        try:
            with tempfile.NamedTemporaryFile("wb", dir=c012_dir, prefix=".C012-",
                                             suffix=".tmp", delete=False) as stream:
                temporary = Path(stream.name)
                stream.write(raw)
                stream.flush()
                os.fsync(stream.fileno())
            c012_check(temporary.read_bytes() == raw, "TEMP_WRITE_RELOAD")
            try:
                os.link(temporary, path)
            except FileExistsError:
                pass
        finally:
            if temporary is not None:
                temporary.unlink(missing_ok=True)
    c012_check(path.is_file() and path.read_bytes() == raw,
               "IMMUTABLE_RECEIPT_CONFLICT:" + path.name)
    c012_timing["checkpoint"] += time.perf_counter() - started
    return hashlib.sha256(raw).hexdigest()


def c012_path(cid, suffix):
    key = hashlib.sha256(cid.encode("utf-8")).hexdigest()
    return c012_dir / ("C012." + key + "." + suffix + ".json")


def c012_memory_peak():
    peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    return round(peak / (1024 ** 3 if sys.platform == "darwin" else 1024 ** 2), 3)


def c012_sample_system(force=False):
    global c012_last_sample_at
    now = time.perf_counter()
    if not force and now - c012_last_sample_at < c012_monitor_interval_seconds:
        return c012_system_samples[-1] if c012_system_samples else None
    started = now
    sample = {"process_current_rss_gib": None, "process_peak_rss_gib": c012_memory_peak(),
              "system_free_memory_percent": None, "thermal_cpu_speed_limit_percent": None,
              "memory_pressure_class": "NOT_VERIFIED", "thermal_throttling": "NOT_VERIFIED"}
    commands = (
        ("ps", ["/bin/ps", "-o", "rss=", "-p", str(os.getpid())]),
        ("memory_pressure", ["/usr/bin/memory_pressure", "-Q"]),
        ("thermal", ["/usr/bin/pmset", "-g", "therm"]),
    )
    for name, command in commands:
        if not Path(command[0]).is_file():
            continue
        try:
            reply = subprocess.run(command, capture_output=True, text=True,
                                   timeout=5, check=False)
            if reply.returncode != 0:
                continue
            if name == "ps":
                sample["process_current_rss_gib"] = round(
                    int(reply.stdout.strip()) / (1024 ** 2), 3)
            elif name == "memory_pressure":
                match = re.search(r"System-wide free memory percentage\s*:\s*(\d+)%",
                                  reply.stdout, re.IGNORECASE)
                if match:
                    sample["system_free_memory_percent"] = int(match.group(1))
            elif name == "thermal":
                match = re.search(r"CPU_Speed_Limit\s*=\s*(\d+)", reply.stdout)
                if match:
                    sample["thermal_cpu_speed_limit_percent"] = int(match.group(1))
                    sample["thermal_throttling"] = (
                        "MEASURED_LIMIT" if int(match.group(1)) < 100
                        else "NO_LIMIT_IN_THIS_SAMPLE")
        except (OSError, ValueError, subprocess.SubprocessError):
            continue
    c012_system_samples.append(sample)
    c012_last_sample_at = time.perf_counter()
    c012_timing["system_monitor"] += c012_last_sample_at - started
    current = sample["process_current_rss_gib"]
    c012_check(current is None or current <= c012_process_rss_limit_gib,
               "PROCESS_CURRENT_RSS_EXCESS")
    c012_check(sample["process_peak_rss_gib"] <= c012_process_rss_limit_gib,
               "PROCESS_HISTORICAL_PEAK_EXCESS")
    # The free-memory number is NOT itself a verified macOS pressure class.
    # A very low observed value is only a conservative stop trigger.
    free = sample["system_free_memory_percent"]
    c012_check(free is None or free > 5, "MEASURED_SYSTEM_FREE_MEMORY_VERY_LOW")
    speed = sample["thermal_cpu_speed_limit_percent"]
    c012_check(speed is None or speed >= 90, "MEASURED_THERMAL_CPU_SPEED_LIMIT")
    return sample


# ============================================================
# Validated frozen boundary and restartable disk receipts
# ============================================================
c012_root = c002_root.resolve()
c012_out = c003_out.resolve()
c012_manifest_path = (c012_out / (
    "C011.population_boundary." + c012_manifest_sha + ".json")).resolve()
c012_check(NB22E_V4_READ_ONLY is True and c012_out.is_dir()
           and not c012_out.is_relative_to(c012_root)
           and c012_manifest_path.parent == c012_out
           and c012_sha(c012_manifest_path) == c012_manifest_sha,
           "C011_MANIFEST_BYTES")
c012_manifest = json.loads(c012_manifest_path.read_bytes())
c012_contract = c012_manifest["execution_contract"]
c012_batches = c012_manifest["execution_queue"]["batches"]
c012_ids = [cid for batch in c012_batches for cid in batch["candidate_ids"]]
c012_members = {row["candidate_id"]: row["question_sha256"]
                for row in c012_contract["members"]}
c012_pilots = c012_manifest["approved_pilot_provenance"]["pilot_ids"]
c012_check(c012_manifest["execution_boundary_sha256"] == c012_boundary_sha
           and hashlib.sha256(c012_canonical(c012_contract)).hexdigest() == c012_boundary_sha
           and c012_contract["historical_identity_inherited"] is False
           and len(c012_ids) == len(set(c012_ids)) == 1112
           and len(c012_members) == 1116 and len(c012_pilots) == 4
           and set(c012_ids).isdisjoint(c012_pilots)
           and set(c012_ids) | set(c012_pilots) == set(c012_members)
           and len(c012_batches) == 45
           and all(batch["batch_number"] == i + 1
                   and 1 <= len(batch["candidate_ids"]) <= 25
                   for i, batch in enumerate(c012_batches))
           and c012_contract["dense_index"]["logical_index_sha256"] == c012_index_sha
           and c012_contract["onnx"]["model_sha256"] == c012_model_sha
           and c012_contract["frozen_parameters"]["onnx_intra_op_threads"] == 4
           and c012_contract["frozen_parameters"]["onnx_batch_size"] == 4
           and c012_contract["frozen_parameters"]["onnx_max_length"] == 512
           and c012_contract["frozen_parameters"]["candidate_parallelism"] == 1
           and c012_manifest["execution_queue"]["fresh_stage1_queries_max"] == 1112
           and c012_manifest["execution_queue"]["fresh_stage2_onnx_pairs_max"] == 55600
           and c012_stage_size == 5 and c012_onnx_pause_seconds == 0.8,
           "FROZEN_POPULATION_OR_EXECUTION_PROFILE")
c012_c010_pin = c012_manifest["approved_pilot_provenance"]["C010"]
c012_c010_path = Path(c012_c010_pin["path"]).resolve()
c012_check(c012_c010_path.parent == c012_out
           and c012_c010_pin["file_bytes_sha256"] == c012_pilot_audit_sha
           and c012_sha(c012_c010_path) == c012_pilot_audit_sha
           and json.loads(c012_c010_path.read_bytes())["status"]
               == "PILOT_EVIDENCE_AUDIT_PASS", "C010_PILOT_PROVENANCE")
c012_dir = (c012_out / ("C012_population_" + c012_boundary_sha)).resolve()
c012_check(c012_dir.parent == c012_out and c012_dir.is_dir()
           and not c012_dir.is_symlink(), "C012_EXISTING_CHECKPOINT_DIRECTORY")


def c012_audit():
    states = {}
    counts = {key: 0 for key in ("COMPLETE", "PARTIAL", "NOT_STARTED", "UNCERTAIN", "BLOCKED")}
    query_attempts = 0
    pair_attempts = 0
    pair_done = 0
    unknown = []
    for cid in c012_ids:
        s1 = c012_path(cid, "stage1_started")
        d1 = c012_path(cid, "stage1_done")
        d2 = c012_path(cid, "stage2_done")
        started1 = None
        finished1 = None
        if s1.is_file():
            started1, _ = c012_read(s1)
            c012_check(started1["status"] == "STARTED_UNCONFIRMED"
                       and started1["candidate_id"] == cid
                       and started1["boundary"] == c012_boundary_sha,
                       "STAGE1_STARTED_INVALID:" + cid)
            query_attempts += 1
        if d1.is_file():
            c012_check(started1 is not None, "STAGE1_DONE_WITHOUT_START:" + cid)
            finished1, _ = c012_read(d1)
            c012_check(finished1["status"] in ("STAGE1_PASS", "STAGE1_BLOCKED")
                       and finished1["candidate_id"] == cid
                       and finished1["boundary"] == c012_boundary_sha
                       and finished1["input_sha256"] == started1["input_sha256"],
                       "STAGE1_DONE_INVALID:" + cid)
            if finished1["status"] == "STAGE1_PASS":
                c012_check(finished1["full_order_matches"] is True
                           and finished1["pool50_membership_matches"] is True
                           and len(set(finished1["observed_hybrid100"])) == 100,
                           "STAGE1_PASS_INVALID:" + cid)
        done_batches = {}
        local_unknown = []
        for start in range(0, 50, 4):
            started_path = c012_path(cid, "stage2_batch_%02d_started" % start)
            done_path = c012_path(cid, "stage2_batch_%02d_done" % start)
            original = None
            original_sha = None
            if started_path.is_file():
                original, original_sha = c012_read(started_path)
                c012_check(original["status"] == "STARTED_UNCONFIRMED"
                           and original["candidate_id"] == cid
                           and original["boundary"] == c012_boundary_sha
                           and original["batch_start"] == start
                           and len(original["chunk_ids"]) == min(4, 50 - start),
                           "STAGE2_START_INVALID:" + cid)
                pair_attempts += len(original["chunk_ids"])
            c012_check(not done_path.is_file() or original is not None,
                       "STAGE2_DONE_WITHOUT_START:" + cid)
            if cid == c012_retry_id and start == 20:
                retry_s = c012_path(cid, "stage2_batch_20_retry1_started")
                retry_d = c012_path(cid, "stage2_batch_20_retry1_done")
                retry_intent = None
                if retry_s.is_file():
                    c012_check(original is not None and not done_path.is_file()
                               and original_sha == c012_retry_original_start_sha,
                               "HISTORICAL_RETRY_ORIGINAL_CHANGED")
                    retry_intent, _ = c012_read(retry_s)
                    c012_check(retry_intent["status"] == "STARTED_UNCONFIRMED"
                               and retry_intent["candidate_id"] == cid
                               and retry_intent["boundary"] == c012_boundary_sha
                               and retry_intent["manual_attempt"] == 1
                               and retry_intent["original_start_sha256"]
                                   == c012_retry_original_start_sha
                               and retry_intent["input_sha256"] == original["input_sha256"]
                               and retry_intent["chunk_ids"] == original["chunk_ids"],
                               "HISTORICAL_RETRY_INTENT_INVALID")
                    pair_attempts += len(retry_intent["chunk_ids"])
                if retry_d.is_file():
                    c012_check(retry_intent is not None, "RETRY_DONE_WITHOUT_START")
                    done_path = retry_d
            if done_path.is_file():
                done, sha = c012_read(done_path)
                c012_check(original is not None
                           and done["status"] == "BATCH_PASS"
                           and done["boundary"] == c012_boundary_sha
                           and done["candidate_id"] == cid
                           and done["input_sha256"] == original["input_sha256"]
                           and done["chunk_ids"] == original["chunk_ids"]
                           and len(done["scores"]) == len(original["chunk_ids"])
                           and all(math.isfinite(float(x)) for x in done["scores"]),
                           "STAGE2_DONE_INVALID:" + cid)
                if cid == c012_retry_id and start == 20:
                    c012_check(done_path.name.endswith("retry1_done.json")
                               and done["manual_attempt"] == 1
                               and done["original_start_sha256"]
                                   == c012_retry_original_start_sha,
                               "RETRY_DONE_CONTRACT")
                done_batches[start] = (done, sha, done_path)
                pair_done += len(done["scores"])
            elif original is not None:
                local_unknown.append(start)
        if done_batches:
            c012_check(finished1 is not None and finished1["status"] == "STAGE1_PASS"
                       and all(row[0]["input_sha256"] == finished1["input_sha256"]
                               for row in done_batches.values()),
                       "STAGE2_WITHOUT_VALID_STAGE1:" + cid)
        if d2.is_file():
            final, _ = c012_read(d2)
            historic = final["historical_pool50_scores"]
            fresh = final["new_scores"]
            hybrid = final["historical_hybrid100"]
            combined = {**historic, **fresh}
            missing = [x for x in hybrid if x not in historic]
            c012_check(finished1 is not None
                       and final["status"] == "STAGE2_PASS"
                       and final["candidate_id"] == cid
                       and final["boundary"] == c012_boundary_sha
                       and final["input_sha256"] == finished1["input_sha256"]
                       and final["historical_hybrid100"] == finished1["observed_hybrid100"]
                       and len(set(hybrid)) == len(combined) == 100
                       and len(historic) == len(fresh) == len(missing) == 50
                       and set(fresh) == set(missing)
                       and all(math.isfinite(float(x)) for x in combined.values())
                       and final["local100_order"] == sorted(
                           hybrid, key=lambda x: (-combined[x], x))
                       and final["rank_artifact_content_sha256"]
                           == hashlib.sha256(c012_canonical({
                               "boundary": c012_boundary_sha, "candidate_id": cid,
                               "scores": combined,
                               "order": final["local100_order"]})).hexdigest()
                       and len(final["batch_receipts"]) == len(done_batches) == 13,
                       "STAGE2_FINAL_CONFLICT:" + cid)
            rebuilt = {}
            for j, pin in enumerate(final["batch_receipts"]):
                start = j * 4
                c012_check(start in done_batches
                           and pin["batch_start"] == start
                           and pin["path"] == str(done_batches[start][2])
                           and pin["sha256"] == done_batches[start][1]
                           and done_batches[start][0]["chunk_ids"] == missing[start:start + 4],
                           "STAGE2_FINAL_BATCH_LINK:" + cid)
                rebuilt.update(zip(done_batches[start][0]["chunk_ids"],
                                   done_batches[start][0]["scores"]))
            c012_check(rebuilt == fresh, "STAGE2_SCORE_RECONSTRUCTION:" + cid)
            progress_path = c012_path(cid, "progress")
            c012_check(progress_path.is_file(), "CANDIDATE_PROGRESS_MISSING:" + cid)
            progress, _ = c012_read(progress_path)
            c012_check(progress["status"] == "CANDIDATE_CHECKPOINTED"
                       and progress["candidate_id"] == cid
                       and progress["boundary"] == c012_boundary_sha,
                       "CANDIDATE_PROGRESS_INVALID:" + cid)
        elif c012_path(cid, "progress").exists():
            c012_check(False, "ORPHAN_PROGRESS:" + cid)
        if started1 is not None and finished1 is None:
            state = "UNCERTAIN"
            unknown.append((cid, "QUERY"))
        elif local_unknown:
            state = "UNCERTAIN"
            unknown.append((cid, "ONNX", local_unknown))
        elif finished1 is not None and finished1["status"] == "STAGE1_BLOCKED":
            state = "BLOCKED"
        elif d2.is_file():
            state = "COMPLETE"
        elif started1 is not None or done_batches:
            state = "PARTIAL"
        else:
            state = "NOT_STARTED"
        states[cid] = state
        counts[state] += 1
    c012_check(query_attempts <= 1112 and pair_attempts <= 55604
               and pair_done <= 55600,
               "CUMULATIVE_APPROVED_ATTEMPTS_EXCEEDED")
    for batch in c012_batches:
        summary_path = c012_dir / ("C012.batch_%03d.done.json" % batch["batch_number"])
        if not summary_path.is_file():
            continue
        summary, _ = c012_read(summary_path)
        c012_check(summary["status"] == "BATCH_COMPLETE"
                   and summary["boundary"] == c012_boundary_sha
                   and summary["batch_number"] == batch["batch_number"]
                   and summary["candidate_ids"] == batch["candidate_ids"]
                   and summary["completed_in_batch"] == len(batch["candidate_ids"])
                   and len(summary["receipts"]) == len(batch["candidate_ids"]),
                   "ORIGINAL_BATCH_SUMMARY_INVALID")
        for cid, pin in zip(batch["candidate_ids"], summary["receipts"]):
            c012_check(states[cid] == "COMPLETE" and pin["candidate_id"] == cid
                       and c012_read(c012_path(cid, "stage1_done"))[1]
                           == pin["stage1_sha256"]
                       and c012_read(c012_path(cid, "stage2_done"))[1]
                           == pin["stage2_sha256"],
                       "ORIGINAL_BATCH_SUMMARY_LINK:" + cid)
    return {"states": states, "counts": counts, "unknown": unknown,
            "query_attempts": query_attempts, "pair_attempts": pair_attempts,
            "pair_done": pair_done}


c012_before = c012_audit()
c012_check(c012_before["counts"]["COMPLETE"] >= 9
           and c012_before["counts"]["PARTIAL"] >= 0
           and c012_before["counts"]["UNCERTAIN"] == 0
           and c012_before["counts"]["BLOCKED"] == 0
           and c012_before["pair_done"] >= 450
           and c012_before["pair_attempts"] >= 454
           and c012_before["pair_attempts"] == c012_before["pair_done"] + 4
           and c012_before["query_attempts"] >= c012_before["counts"]["COMPLETE"],
           "APPROVED_PROGRESS_OR_NO_RETRY_BASELINE_CHANGED")
c012_completed_before = c012_before["counts"]["COMPLETE"]
c012_attempts_before = c012_before["pair_attempts"]
c012_pairs_before = c012_before["pair_done"]
print(json.dumps({"cell": "NB22E-v4:C012", "phase": "AUTOMATIC_RESUME_PREFLIGHT",
    "completed": c012_completed_before, "remaining": 1112 - c012_completed_before,
    "pair_attempts": c012_attempts_before,
    "unique_pairs_remaining": 55600 - c012_pairs_before,
    "max_total_pair_attempts": 55604,
    "stage_size": c012_stage_size, "onnx_pause_seconds": c012_onnx_pause_seconds,
    "no_extra_retry": True}, ensure_ascii=False, indent=2), flush=True)


def c012_run_automatic():
    global c012_active, c012_slow_consecutive
    np_local = np
    ort_local = ort
    c012_check(np_local is not None and ort_local is not None,
               "CENTRALIZED_IMPORT_MISSING")
    sources = {**c012_contract["source_producer_pins"],
               **c012_contract["input_pins"],
               **{"VECTOR_" + name: pin for name, pin in
                  c012_contract["query_vector_cache_pins"].items()}}
    loader = c008_loader
    sources.update({
        "SPARSE_CONFIGURATION": {"path": loader["configuration_path"],
            "file_bytes_sha256": loader["configuration_file_sha256"]},
        "SPARSE_DOCUMENTS": {"path": loader["documents_path"],
            "file_bytes_sha256": loader["documents_sha256"]},
        "SPARSE_POSTINGS": {"path": loader["postings_path"],
            "file_bytes_sha256": loader["postings_sha256"]},
    })
    expected = {}
    for name, pin in sources.items():
        relative = Path(pin["path"])
        path = (c012_root / relative).resolve()
        c012_check(not relative.is_absolute() and path.is_relative_to(c012_root)
                   and path.is_file(), "PIN_PATH:" + name)
        if path in expected:
            c012_check(expected[path] == pin["file_bytes_sha256"],
                       "PIN_ALIAS_CONFLICT:" + name)
        expected[path] = pin["file_bytes_sha256"]
    # Preserve the original model alias, even when it is a symlink into HF cache.
    snapshot = Path(os.path.abspath(str(Path(c004_snapshot).expanduser())))
    model = Path(os.path.abspath(str(Path(c004_model_path).expanduser())))
    c012_check(model.is_file() and model.parent.parent == snapshot
               and model.relative_to(snapshot).as_posix()
                   == c012_contract["onnx"]["model_alias_relative_inside_snapshot"]
               and snapshot.name == c012_contract["onnx"]["revision"],
               "ONNX_ALIAS_FROZEN")
    expected[model] = c012_model_sha
    for name, row in c012_contract["onnx"]["tokenizer"]["files"].items():
        expected[snapshot / name] = row["sha256"]
    c012_check(type(c005_tokenizer).__name__
                   == c012_contract["onnx"]["tokenizer"]["tokenizer_class"]
               and bool(c005_tokenizer.is_fast)
                   == c012_contract["onnx"]["tokenizer"]["is_fast"],
               "TOKENIZER_FROZEN_IDENTITY")
    for name, version in c012_contract["runtime_versions"].items():
        actual = (platform.python_version() if name == "python"
                  else importlib.metadata.version(name))
        c012_check(actual == version, "RUNTIME_VERSION:" + name)
    started = time.perf_counter()
    pinned_stamps = {}
    for path, sha in expected.items():
        c012_check(c012_sha(path) == sha, "PIN_SHA:" + str(path))
        pinned_stamps[path] = c012_stamp(path)
    c012_timing["source_sha"] += time.perf_counter() - started
    original = c004_index_path.resolve()
    c012_check(original.is_dir() and original.is_relative_to(c012_root),
               "ORIGINAL_INDEX_PATH")
    inventory = c004_docs["manifest"]["storage_inventory"]
    frozen_files = inventory["files"]
    c012_check(inventory["file_count"] == len(frozen_files)
               and inventory["inventory_sha256"]
                   == c004_docs["release"]["storage_inventory_sha256"]
                   == hashlib.sha256(c012_canonical(frozen_files)).hexdigest(),
               "QDRANT_FROZEN_INVENTORY")
    index_by_name = {row["path"]: row for row in frozen_files}
    c012_check(len(index_by_name) == len(frozen_files), "INDEX_DUPLICATE")
    index_stamps = {}
    index_bytes = 0
    seen = set()
    started = time.perf_counter()
    for folder, dirs, files in os.walk(original, followlinks=False):
        c012_check(not any((Path(folder) / name).is_symlink()
                           for name in dirs + files), "INDEX_SYMLINK")
        for name in files:
            path = Path(folder) / name
            relative = path.relative_to(original).as_posix()
            c012_check(path.is_file(), "INDEX_NOT_REGULAR")
            if name != ".lock":
                c012_check(relative in index_by_name
                           and path.stat().st_size == index_by_name[relative]["size_bytes"]
                           and c012_sha(path) == index_by_name[relative]["sha256"],
                           "INDEX_STORAGE_SHA:" + relative)
                seen.add(relative)
            index_stamps[path] = c012_stamp(path)
            index_bytes += path.stat().st_size
    c012_timing["index_sha"] += time.perf_counter() - started
    c012_check(seen == set(index_by_name)
               and len(index_stamps) == c005_index_result["inventory_files"]
               and index_bytes == c005_index_result["inventory_bytes"]
               and index_bytes <= 512 * 1024 * 1024,
               "QDRANT_INDEX_DRIFT")

    def guard():
        for path, stamp in pinned_stamps.items():
            c012_check(path.is_file() and c012_stamp(path) == stamp,
                       "INPUT_CHANGED:" + path.name)
        for path, stamp in index_stamps.items():
            c012_check(path.is_file() and c012_stamp(path) == stamp,
                       "INDEX_CHANGED:" + path.name)
        c012_check(c012_memory_peak() <= c012_process_rss_limit_gib,
                   "PROCESS_PEAK_EXCESS")

    # Index the byte offsets just once. Each stage reads only its five candidates.
    started = time.perf_counter()
    index_sources = {}
    opened = []
    client = None
    temp_index = None
    session = None
    model_inputs = None
    previous_fresh_onnx = False
    try:
        for label in ("HYBRID", "RERANKER", "CHUNKS"):
            handle = (c012_root / c012_contract["input_pins"][label]["path"]).open("rb")
            opened.append(handle)
            positions = {}
            while True:
                position = handle.tell()
                line = handle.readline()
                if not line:
                    break
                row = json.loads(line)
                key = (row["candidate_id"] if label != "CHUNKS"
                       else row["chunk_id"])
                c012_check(key not in positions, "SOURCE_DUPLICATE:" + label)
                positions[key] = position
            index_sources[label] = (handle, positions)
        handle = (c012_root / loader["postings_path"]).open("rb")
        opened.append(handle)
        positions = {}
        while True:
            position = handle.tell()
            line = handle.readline()
            if not line:
                break
            row = json.loads(line)
            c012_check(row["token"] not in positions, "POSTING_DUPLICATE")
            positions[row["token"]] = position
        index_sources["POSTINGS"] = (handle, positions)
        vector_pins = c012_contract["query_vector_cache_pins"]
        vector_meta = json.loads((c012_root / vector_pins["metadata"]["path"]).read_bytes())
        vector_order = vector_meta["candidate_ids"]
        c012_check(len(vector_order) == len(set(vector_order)) == 1149
                   and set(c012_ids).issubset(vector_order), "VECTOR_CANDIDATE_BINDING")
        vector_positions = {cid: i for i, cid in enumerate(vector_order)}
        vectors = np_local.load(c012_root / vector_pins["array"]["path"],
                                mmap_mode="r", allow_pickle=False)
        c012_check(vectors.shape == (1149, 384) and vectors.dtype == np_local.float32,
                   "VECTOR_SHAPE_DTYPE")
        document_ids = c008_doc_ids
        document_lengths = np_local.asarray(c008_doc_lengths, dtype=np_local.float64)
        mean_length = float(c008_average_doc_length)
        c012_check(len(document_ids) == len(document_lengths) == 3488
                   and document_ids == c007_chunk_ids and mean_length > 0,
                   "SPARSE_DOCUMENT_CONTRACT")
        c012_timing["input_read"] += time.perf_counter() - started

        def get_row(label, key):
            handle, offsets = index_sources[label]
            c012_check(key in offsets, "SOURCE_MISSING:" + label + ":" + str(key))
            handle.seek(offsets[key])
            return json.loads(handle.readline())

        def ensure_model():
            nonlocal session, model_inputs
            if session is not None:
                return
            started = time.perf_counter()
            c012_check(ort_local.__version__ == "1.16.3", "ORT_VERSION")
            options = ort_local.SessionOptions()
            options.intra_op_num_threads = 4
            options.inter_op_num_threads = 1
            options.execution_mode = ort_local.ExecutionMode.ORT_SEQUENTIAL
            options.graph_optimization_level = ort_local.GraphOptimizationLevel.ORT_ENABLE_ALL
            session = ort_local.InferenceSession(str(model), sess_options=options,
                                                 providers=["CPUExecutionProvider"])
            model_inputs = {item.name for item in session.get_inputs()}
            c012_check(session.get_providers() == ["CPUExecutionProvider"]
                       and {"input_ids", "attention_mask"} <= model_inputs,
                       "ONNX_INPUT_CONTRACT")
            c012_timing["model_setup"] += time.perf_counter() - started

        def ensure_qdrant():
            nonlocal client, temp_index
            if client is not None:
                return
            started = time.perf_counter()
            stat = os.statvfs(str(c012_out))
            c012_check(stat.f_bavail * stat.f_frsize >= max(256 * 1024 * 1024,
                                                            2 * index_bytes),
                       "INSUFFICIENT_QDRANT_CLONE_DISK")
            temp_index = tempfile.TemporaryDirectory(
                prefix="NB22E-C012-", dir=str(c012_out))
            clone = Path(temp_index.name) / "qdrant"
            clone.mkdir()
            for source, stamp in index_stamps.items():
                guard()
                target = clone / source.relative_to(original)
                target.parent.mkdir(parents=True, exist_ok=True)
                with (source.open("rb") as f_in, target.open("xb") as f_out):
                    for data in iter(lambda: f_in.read(1024 * 1024), b""):
                        f_out.write(data)
                c012_check(c012_stamp(source) == stamp,
                           "INDEX_CHANGED_DURING_CLONE")
            client = QdrantClient(path=str(clone))
            c012_check(client.count(collection_name=c005_collection,
                                    exact=True).count == 3488,
                       "CLONED_QDRANT_POINT_COUNT")
            c012_timing["qdrant_setup"] += time.perf_counter() - started

        def sparse_for(cid, words):
            scores = np_local.zeros(3488, dtype=np_local.float64)
            for term, qtf in collections.Counter(words).items():
                if term not in index_sources["POSTINGS"][1]:
                    continue
                posting = get_row("POSTINGS", term)
                ids = np_local.asarray(posting["document_indexes"], dtype=np_local.int64)
                frequencies = np_local.asarray(posting["term_frequencies"], dtype=np_local.float64)
                c012_check(len(ids) == len(frequencies) == posting["document_frequency"]
                           and bool((ids[:-1] < ids[1:]).all()), "SPARSE_POSTINGS")
                df = len(ids)
                idf = math.log(1.0 + (3488 - df + 0.5) / (df + 0.5))
                scores[ids] += (idf * frequencies * 2.2
                                / (frequencies + 1.2 * (
                                    1.0 - 0.3 + 0.3 * document_lengths[ids] / mean_length
                                ))) * float(qtf)
            valid = np_local.flatnonzero(scores > 0)
            ranked = valid[np_local.lexsort((valid, -scores[valid]))][:100]
            return ([document_ids[i] for i in ranked],
                    [float(scores[i]) for i in ranked])

        def run_candidate(cid):
            global c012_active, c012_slow_consecutive
            nonlocal previous_fresh_onnx
            c012_active = cid
            guard()
            c012_check(not c012_path(cid, "stage2_done").exists(),
                       "ALREADY_DONE_CANDIDATE_SELECTED")
            started = time.perf_counter()
            hybrid_row = get_row("HYBRID", cid)
            pool_row = get_row("RERANKER", cid)
            rows100, rows50 = hybrid_row["candidates"], pool_row["candidates"]
            c012_check(len(rows100) == 100 and len(rows50) == 50
                       and [x["rank"] for x in rows100] == list(range(1, 101))
                       and [x["rank"] for x in rows50] == list(range(1, 51))
                       and hybrid_row.get("split") == "test"
                       and pool_row.get("split") in (None, "test"),
                       "HISTORICAL_RANK_SCHEMA:" + cid)
            ids100 = [row["chunk_id"] for row in rows100]
            ids50 = [row["chunk_id"] for row in rows50]
            previous_scores = {row["chunk_id"]: float(row["reranker_score"])
                               for row in rows50}
            missing50 = [chunk_id for chunk_id in ids100
                         if chunk_id not in previous_scores]
            c012_check(len(set(ids100)) == 100
                       and len(set(ids50)) == len(previous_scores) == 50
                       and set(ids50) == set(ids100[:50])
                       and len(missing50) == 50
                       and all(math.isfinite(x) for x in previous_scores.values()),
                       "FROZEN_POOL50:" + cid)
            record = c002_questions[cid]
            question = record["question"]
            c012_check(hashlib.sha256(question.encode("utf-8")).hexdigest()
                       == record["question_sha256"] == c012_members[cid],
                       "QUESTION_SHA:" + cid)
            question = question.strip()
            contents = {chunk_id: get_row("CHUNKS", chunk_id)["content"]
                        for chunk_id in missing50}
            c012_check(len(contents) == 50
                       and all(isinstance(s, str) for s in contents.values()),
                       "MISSING50_CONTENT")
            c012_timing["input_read"] += time.perf_counter() - started
            identity = hashlib.sha256(c012_canonical({
                "boundary": c012_boundary_sha, "candidate_id": cid,
                "question_sha256": record["question_sha256"],
                "historical_hybrid100": ids100,
                "historical_pool50": ids50,
                "historical_pool50_scores": previous_scores,
                "model_sha256": c012_model_sha,
            })).hexdigest()
            s1 = c012_path(cid, "stage1_started")
            d1 = c012_path(cid, "stage1_done")
            d2 = c012_path(cid, "stage2_done")
            s1_intent = {"status": "STARTED_UNCONFIRMED", "boundary": c012_boundary_sha,
                         "candidate_id": cid, "input_sha256": identity,
                         "query_limit": 1, "manual_retry_only_if_uncertain": True}
            if d1.is_file():
                intent, _ = c012_read(s1)
                finished, _ = c012_read(d1)
                c012_check(intent == s1_intent and finished["status"] == "STAGE1_PASS"
                           and finished["input_sha256"] == identity
                           and finished["observed_hybrid100"] == ids100
                           and finished["full_order_matches"] is True
                           and finished["pool50_membership_matches"] is True,
                           "REUSED_STAGE1_CONFLICT:" + cid)
            else:
                c012_check(not s1.exists(), "UNKNOWN_STAGE1_NO_RETRY:" + cid)
                ensure_qdrant()
                vector = np_local.asarray(vectors[vector_positions[cid]],
                                          dtype=np_local.float32)
                c012_check(vector.shape == (384,) and bool(np_local.isfinite(vector).all())
                           and bool(np_local.isclose(np_local.linalg.norm(vector), 1.0)),
                           "DENSE_VECTOR_CONTRACT")
                token_started = time.perf_counter()
                words = c008_tokens(question)
                c012_timing["tokenization"] += time.perf_counter() - token_started
                sparse_started = time.perf_counter()
                sparse_ids, sparse_scores = sparse_for(cid, words)
                c012_timing["input_read"] += time.perf_counter() - sparse_started
                guard()
                c012_check(c012_before["query_attempts"]
                           + c012_counts_run["new_queries"] + 1 <= 1112,
                           "QUERY_ATTEMPT_BUDGET")
                c012_write(s1, s1_intent)
                c012_counts_run["new_queries"] += 1
                query_started = time.perf_counter()
                response = client.query_points(
                    collection_name=c005_collection, query=vector.tolist(),
                    limit=100, with_payload=True, with_vectors=False)
                points = getattr(response, "points", response)
                dense_ids = [str(row.payload["chunk_id"]) for row in points]
                dense_scores = [float(row.score) for row in points]
                observed = c008_fuse(dense_ids, dense_scores,
                                     sparse_ids, sparse_scores)
                elapsed = time.perf_counter() - query_started
                c012_timing["query"] += elapsed
                order_ok = observed == ids100 and len(observed) == 100
                pool_ok = set(observed[:50]) == set(ids50)
                c012_write(d1, {
                    "status": "STAGE1_PASS" if order_ok and pool_ok else "STAGE1_BLOCKED",
                    "boundary": c012_boundary_sha, "candidate_id": cid,
                    "input_sha256": identity, "observed_hybrid100": observed,
                    "expected_hybrid100_sha256": hashlib.sha256(
                        c012_canonical(ids100)).hexdigest(),
                    "full_order_matches": order_ok,
                    "pool50_membership_matches": pool_ok,
                    "dense_count": len(dense_ids), "sparse_count": len(sparse_ids),
                    "query_count": 1, "elapsed_seconds": round(elapsed, 4),
                    "first_mismatch_rank": next((i + 1 for i in range(min(len(observed), 100))
                                                 if observed[i] != ids100[i]), None),
                })
                c012_check(order_ok and pool_ok, "HYBRID100_PARITY_STOP:" + cid)
                c012_check(elapsed <= c012_query_slow_limit_seconds,
                           "SEVERE_QUERY_SLOWDOWN:" + cid)
            fresh = {}
            batch_pins = []
            for start in range(0, 50, 4):
                batch_ids = missing50[start:start + 4]
                start_path = c012_path(cid, "stage2_batch_%02d_started" % start)
                done_path = c012_path(cid, "stage2_batch_%02d_done" % start)
                intent_core = {"status": "STARTED_UNCONFIRMED",
                               "boundary": c012_boundary_sha, "candidate_id": cid,
                               "input_sha256": identity, "batch_start": start,
                               "chunk_ids": batch_ids,
                               "no_automatic_retry_if_uncertain": True}
                c012_check(not (cid == c012_retry_id and start == 20),
                           "HISTORICAL_RETRY_CANNOT_RERUN")
                if done_path.is_file():
                    prev, _ = c012_read(start_path)
                    result, done_sha = c012_read(done_path)
                    c012_check(prev == intent_core and result["status"] == "BATCH_PASS"
                               and result["input_sha256"] == identity
                               and result["chunk_ids"] == batch_ids
                               and len(result["scores"]) == len(batch_ids)
                               and all(math.isfinite(x) for x in result["scores"]),
                               "EXISTING_STAGE2_BATCH_INVALID:" + cid)
                    scores = result["scores"]
                else:
                    c012_check(not start_path.exists(),
                               "STARTED_WITHOUT_DONE_NO_AUTORETRY:" + cid)
                    guard()
                    ensure_model()
                    c012_check(c012_before["pair_attempts"]
                               + c012_counts_run["new_onnx_attempts"]
                               + len(batch_ids) <= 55604,
                               "PAIR_ATTEMPT_LIMIT_55604")
                    tokenize_at = time.perf_counter()
                    encoded = c005_tokenizer(
                        [question] * len(batch_ids),
                        [contents[chunk] for chunk in batch_ids],
                        padding=True, truncation=True, max_length=512,
                        return_tensors="np")
                    c012_timing["tokenization"] += time.perf_counter() - tokenize_at
                    c012_check(model_inputs <= set(encoded), "ONNX_INPUT_TOKENS")
                    feed = {name: np_local.asarray(encoded[name], dtype=np_local.int64)
                            for name in model_inputs}
                    c012_check(all(value.ndim == 2
                                   and value.shape[0] == len(batch_ids)
                                   and value.shape[1] <= 512
                                   for value in feed.values()),
                               "ONNX_SEQUENCE_BOUNDS")
                    if previous_fresh_onnx:
                        pause_at = time.perf_counter()
                        time.sleep(c012_onnx_pause_seconds)
                        c012_timing["pause"] += time.perf_counter() - pause_at
                        guard()
                    c012_write(start_path, intent_core)
                    c012_counts_run["new_onnx_attempts"] += len(batch_ids)
                    c012_counts_run["onnx_calls"] += 1
                    compute_at = time.perf_counter()
                    outputs = session.run(None, feed)
                    compute = time.perf_counter() - compute_at
                    c012_timing["onnx"] += compute
                    scores_np = np_local.asarray(outputs[0], dtype=np_local.float64).reshape(-1)
                    c012_check(scores_np.shape == (len(batch_ids),)
                               and bool(np_local.isfinite(scores_np).all()),
                               "ONNX_OUTPUT_INVALID:" + cid)
                    scores = [float(value) for value in scores_np]
                    done_sha = c012_write(done_path, {
                        "status": "BATCH_PASS", "boundary": c012_boundary_sha,
                        "candidate_id": cid, "input_sha256": identity,
                        "chunk_ids": batch_ids, "scores": scores,
                        "elapsed_seconds": round(compute, 4),
                        "process_peak_rss_gib": c012_memory_peak(),
                        "onnx_calls": 1, "qdrant_queries": 0, "api_calls": 0,
                    })
                    c012_counts_run["new_onnx_completed"] += len(batch_ids)
                    previous_fresh_onnx = True
                    c012_recent_onnx.append(compute)
                    if compute > 4.0:
                        c012_slow_consecutive += 1
                    else:
                        c012_slow_consecutive = 0
                    c012_check(compute <= c012_onnx_slow_limit_seconds
                               and c012_slow_consecutive < 3,
                               "SUSTAINED_OR_SEVERE_ONNX_SLOWDOWN")
                    # Sample hardware only at a long interval, never per ONNX batch.
                    if time.perf_counter() - c012_last_sample_at >= c012_monitor_interval_seconds:
                        c012_sample_system()
                c012_check(not set(batch_ids).intersection(fresh),
                           "DUPLICATE_BATCH_SCORES")
                fresh.update(zip(batch_ids, scores))
                batch_pins.append({"path": str(done_path), "sha256": done_sha,
                                   "batch_start": start, "count": len(batch_ids)})
            combined = {**previous_scores, **fresh}
            c012_check(len(fresh) == 50 and len(combined) == 100
                       and set(combined) == set(ids100)
                       and all(math.isfinite(x) for x in combined.values()),
                       "LOCAL100_COMPLETENESS")
            rank = sorted(ids100, key=lambda x: (-combined[x], x))
            rank_sha = hashlib.sha256(c012_canonical({
                "boundary": c012_boundary_sha, "candidate_id": cid,
                "scores": combined, "order": rank})).hexdigest()
            c012_write(d2, {
                "status": "STAGE2_PASS", "boundary": c012_boundary_sha,
                "candidate_id": cid, "input_sha256": identity,
                "historical_hybrid100": ids100,
                "historical_pool50_scores": previous_scores,
                "new_scores": fresh, "local100_order": rank,
                "rank_artifact_content_sha256": rank_sha,
                "batch_receipts": batch_pins,
                "new_pairs": 50, "historical_reused_pairs": 50,
                "derived_order_quota_char_rrf_executed": False,
                "context_generated": False, "target_gate": "BLOCKED",
            })
            c012_write(c012_path(cid, "progress"), {
                "status": "CANDIDATE_CHECKPOINTED", "boundary": c012_boundary_sha,
                "candidate_id": cid, "stage1_path": str(d1),
                "stage2_path": str(d2),
            })
            completed, _ = c012_read(d2)
            progress, _ = c012_read(c012_path(cid, "progress"))
            c012_check(completed["status"] == "STAGE2_PASS"
                       and completed["rank_artifact_content_sha256"] == rank_sha
                       and progress["status"] == "CANDIDATE_CHECKPOINTED",
                       "CANDIDATE_RELOAD_INVALID:" + cid)
            c012_counts_run["new_candidates_completed"] += 1
            c012_active = None
            # Release the 50 texts and score maps before the next question.

        def close_original_batches():
            for batch in c012_batches:
                summary = c012_dir / ("C012.batch_%03d.done.json"
                                     % batch["batch_number"])
                if summary.is_file():
                    continue
                if not all(c012_path(cid, "stage2_done").is_file()
                           for cid in batch["candidate_ids"]):
                    continue
                rows = []
                for cid in batch["candidate_ids"]:
                    s1, s1sha = c012_read(c012_path(cid, "stage1_done"))
                    s2, s2sha = c012_read(c012_path(cid, "stage2_done"))
                    c012_check(s1["status"] == "STAGE1_PASS"
                               and s2["status"] == "STAGE2_PASS",
                               "ORIGINAL_BATCH_CLOSE_INVALID")
                    rows.append({"candidate_id": cid, "stage1_sha256": s1sha,
                                 "stage2_sha256": s2sha})
                c012_write(summary, {
                    "status": "BATCH_COMPLETE", "boundary": c012_boundary_sha,
                    "batch_number": batch["batch_number"],
                    "candidate_ids": batch["candidate_ids"],
                    "completed_in_batch": len(batch["candidate_ids"]),
                    "receipts": rows,
                })

        # ====================================================
        # Automatic bounded stages, no permission prompts
        # ====================================================
        c012_sample_system(force=True)
        queue = [cid for cid in c012_ids
                 if c012_before["states"][cid] != "COMPLETE"]
        c012_check(len(queue) == 1112 - c012_completed_before, "RESUME_QUEUE_IDENTITY")
        stage_index = 0
        while queue:
            stage_index += 1
            stage = queue[:c012_stage_size]
            stage_started = time.perf_counter()
            start_timing = dict(c012_timing)
            start_work = dict(c012_counts_run)
            for cid in stage:
                guard()
                if time.perf_counter() - c012_last_sample_at >= c012_monitor_interval_seconds:
                    c012_sample_system()
                run_candidate(cid)
            close_original_batches()
            # Verify each stage's completed questions from immutable receipts.
            for cid in stage:
                s2, _ = c012_read(c012_path(cid, "stage2_done"))
                p, _ = c012_read(c012_path(cid, "progress"))
                c012_check(s2["status"] == "STAGE2_PASS"
                           and p["status"] == "CANDIDATE_CHECKPOINTED",
                           "STAGE_CHECKPOINT_RELOAD:" + cid)
            del queue[:len(stage)]
            c012_sample_system()
            stage_elapsed = time.perf_counter() - stage_started
            delta = {key: round(c012_timing[key] - start_timing[key], 3)
                     for key in c012_timing}
            stage_measured = sum(delta.values())
            stage_other = round(max(0.0, stage_elapsed - stage_measured), 3)
            stage_record = {
                "stage": stage_index, "candidate_ids": stage,
                "completed_total": 1112 - len(queue),
                "remaining_candidates": len(queue),
                "pair_attempts_total": c012_attempts_before + c012_counts_run["new_onnx_attempts"],
                "unique_onnx_pairs_done_total": c012_pairs_before + c012_counts_run["new_onnx_completed"],
                "seconds": round(stage_elapsed, 3),
                "timing_seconds": delta, "other_unattributed_seconds": stage_other,
                "work_in_stage": {key: c012_counts_run[key] - start_work[key]
                                  for key in c012_counts_run},
                "memory": c012_system_samples[-1] if c012_system_samples else None,
            }
            c012_stage_history.append(stage_record)
            # Estimate from the latest five observed stages, not a fixed pilot.
            recent = c012_stage_history[-5:]
            seconds_per_candidate = statistics.median(
                item["seconds"] / len(item["candidate_ids"]) for item in recent)
            stage_record["eta_hours_current_rate"] = round(
                seconds_per_candidate * len(queue) / 3600, 2)
            stage_raw = c012_canonical(stage_record) + b"\n"
            stage_sha = hashlib.sha256(stage_raw).hexdigest()
            stage_path = c012_dir / ("C012.auto_stage_%03d.%s.json"
                                    % (stage_index, stage_sha))
            # Reuse the same atomic immutable publisher: stage report has no core receipt.
            if not stage_path.exists():
                temporary = None
                try:
                    with tempfile.NamedTemporaryFile("wb", dir=c012_dir,
                                                     prefix=".C012-auto-", suffix=".tmp",
                                                     delete=False) as stream:
                        temporary = Path(stream.name)
                        stream.write(stage_raw)
                        stream.flush()
                        os.fsync(stream.fileno())
                    try:
                        os.link(temporary, stage_path)
                    except FileExistsError:
                        pass
                finally:
                    if temporary is not None:
                        temporary.unlink(missing_ok=True)
            c012_check(stage_path.read_bytes() == stage_raw,
                       "STAGE_SUMMARY_IMMUTABILITY")
            stage_record["stage_report_sha256"] = stage_sha
            print(json.dumps({"cell": "NB22E-v4:C012", "phase": "AUTO_STAGE_COMPLETE",
                              **stage_record}, ensure_ascii=False, indent=2), flush=True)
        close_original_batches()
    finally:
        for stream in opened:
            stream.close()
        if client is not None:
            client.close()
        session = None
        if temp_index is not None:
            temp_index.cleanup()


try:
    c012_run_automatic()
except KeyboardInterrupt:
    c012_stop = {"kind": "KEYBOARD_INTERRUPT", "candidate_id": c012_active,
                 "action": "STOP_WITHOUT_AUTOMATIC_RETRY_OF_STARTED"}
except Exception as error:
    c012_stop = {"kind": type(error).__name__, "detail": str(error)[:900],
                 "candidate_id": c012_active}
    sys.excepthook(type(error), error, error.__traceback__)

# Final re-audit is read-only; failed receipt validation must never be hidden.
c012_after = None
c012_final_audit_error = None
try:
    c012_after = c012_audit()
except Exception as error:
    c012_final_audit_error = type(error).__name__ + ": " + str(error)[:700]

c012_total_elapsed = time.perf_counter() - c012_run_start
c012_report = {
    "cell": "NB22E-v4:C012",
    "status": ("STOPPED_WITH_CHECKPOINTS" if c012_stop or c012_final_audit_error
               else "STAGE1_STAGE2_EXECUTION_COMPLETED"),
    "execution_boundary_sha256": c012_boundary_sha,
    "starting_complete": c012_completed_before, "pilot_reused": 4,
    "completed_new_candidates": (c012_after["counts"]["COMPLETE"]
                                  if c012_after else None),
    "remaining_candidates": (c012_after["counts"]["NOT_STARTED"]
                             + c012_after["counts"]["PARTIAL"]
                             + c012_after["counts"]["UNCERTAIN"]
                             + c012_after["counts"]["BLOCKED"]
                             if c012_after else None),
    "unique_onnx_pairs_completed": (c012_after["pair_done"]
                                    if c012_after else None),
    "total_pair_attempts": (c012_after["pair_attempts"]
                            if c012_after else None),
    "pair_attempt_ceiling": 55604,
    "unknown": (c012_after["unknown"] if c012_after else None),
    "work_this_invocation": c012_counts_run,
    "stage_count_completed": len(c012_stage_history),
    "total_elapsed_seconds": round(c012_total_elapsed, 3),
    "timing_seconds": {**{key: round(value, 3) for key, value in c012_timing.items()},
                       "other_unattributed": round(max(0.0, c012_total_elapsed
                                                      - sum(c012_timing.values())), 3)},
    "last_system_sample": c012_system_samples[-1] if c012_system_samples else None,
    "system_sample_count": len(c012_system_samples),
    "stop_reason": c012_stop,
    "final_audit_error": c012_final_audit_error,
    "paid_api_calls": 0,
    "repository_modified": False,
    "generation_authorized": False,
    "derived_order_quota_char_rrf_executed": False,
    "R11_2_quality_accepted": False,
}
raw = c012_canonical(c012_report) + b"\n"
sha = hashlib.sha256(raw).hexdigest()
path = c012_dir / ("C012.run_summary." + sha + ".json")
if not path.exists():
    temporary = None
    try:
        with tempfile.NamedTemporaryFile("wb", dir=c012_dir,
                                         prefix=".C012-summary-", suffix=".tmp",
                                         delete=False) as stream:
            temporary = Path(stream.name)
            stream.write(raw)
            stream.flush()
            os.fsync(stream.fileno())
        try:
            os.link(temporary, path)
        except FileExistsError:
            pass
    finally:
        if temporary is not None:
            temporary.unlink(missing_ok=True)
c012_check(path.read_bytes() == raw, "FINAL_REPORT_RELOAD_FAILED")
c012_report["report_checkpoint"] = {"path": str(path), "sha256": sha}
NB22E_V4_POPULATION_EXECUTION = c012_report
print(json.dumps(c012_report, ensure_ascii=False, indent=2), flush=True)

{
  "cell": "NB22E-v4:C012",
  "phase": "AUTOMATIC_RESUME_PREFLIGHT",
  "completed": 9,
  "remaining": 1103,
  "pair_attempts": 454,
  "unique_pairs_remaining": 55150,
  "max_total_pair_attempts": 55604,
  "stage_size": 5,
  "onnx_pause_seconds": 0.8,
  "no_extra_retry": true
}


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

{
  "cell": "NB22E-v4:C012",
  "phase": "AUTO_STAGE_COMPLETE",
  "stage": 1,
  "candidate_ids": [
    "golden_v2_candidate_024bde300780ba3cb472936f",
    "golden_v2_candidate_024f8781eb4783b0ae854bbc",
    "golden_v2_candidate_037df66a958e7c81f03c0ef6",
    "golden_v2_candidate_039f9f42a79d212add2663f9",
    "golden_v2_candidate_0408bb31681308c71cc2a633"
  ],
  "completed_total": 14,
  "remaining_candidates": 1098,
  "pair_attempts_total": 704,
  "unique_onnx_pairs_done_total": 700,
  "seconds": 103.17,
  "timing_seconds": {
    "source_sha": 0.0,
    "index_sha": 0.0,
    "input_read": 0.061,
    "model_setup": 1.634,
    "qdrant_setup": 3.208,
    "query": 0.223,
    "tokenization": 1.341,
    "onnx": 44.452,
    "pause": 51.472,
    "checkpoint": 0.516,
    "system_monitor": 0.082
  },
  "other_unattributed_seconds": 0.181,
  "work_in_stage": {
    "new_queries": 5,
    "new_onnx_attempts": 250,
    "new_onnx_completed": 250,
    "onnx_calls": 65,
    "new_candidates_completed": 5,


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


{
  "cell": "NB22E-v4:C012",
  "phase": "AUTO_STAGE_COMPLETE",
  "stage": 2,
  "candidate_ids": [
    "golden_v2_candidate_04098e4135143c1430ec50db",
    "golden_v2_candidate_0424c8ef79eaab7c81ba32c6",
    "golden_v2_candidate_0477a54d2504de5ce0f0359b",
    "golden_v2_candidate_0477b187a45eb76763843cc2",
    "golden_v2_candidate_04a46058d3a357905665d81f"
  ],
  "completed_total": 19,
  "remaining_candidates": 1093,
  "pair_attempts_total": 954,
  "unique_onnx_pairs_done_total": 950,
  "seconds": 88.656,
  "timing_seconds": {
    "source_sha": 0.0,
    "index_sha": 0.0,
    "input_read": 0.074,
    "model_setup": 0.0,
    "qdrant_setup": 0.0,
    "query": 1.02,
    "tokenization": 0.812,
    "onnx": 33.571,
    "pause": 52.27,
    "checkpoint": 0.603,
    "system_monitor": 0.061
  },
  "other_unattributed_seconds": 0.245,
  "work_in_stage": {
    "new_queries": 5,
    "new_onnx_attempts": 250,
    "new_onnx_completed": 250,
    "onnx_calls": 65,
    "new_candidates_completed": 5,
    "n

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


{
  "cell": "NB22E-v4:C012",
  "phase": "AUTO_STAGE_COMPLETE",
  "stage": 4,
  "candidate_ids": [
    "golden_v2_candidate_059b226265e9b8a146978aac",
    "golden_v2_candidate_05acfe991a2b90c40ccaf873",
    "golden_v2_candidate_05d052d545a5eedefbd01558",
    "golden_v2_candidate_05e1c38a910abe497565084c",
    "golden_v2_candidate_065bbed1933859ad5ba020db"
  ],
  "completed_total": 29,
  "remaining_candidates": 1083,
  "pair_attempts_total": 1454,
  "unique_onnx_pairs_done_total": 1450,
  "seconds": 89.868,
  "timing_seconds": {
    "source_sha": 0.0,
    "index_sha": 0.0,
    "input_read": 0.17,
    "model_setup": 0.0,
    "qdrant_setup": 0.0,
    "query": 0.253,
    "tokenization": 0.517,
    "onnx": 35.432,
    "pause": 52.304,
    "checkpoint": 0.867,
    "system_monitor": 0.061
  },
  "other_unattributed_seconds": 0.264,
  "work_in_stage": {
    "new_queries": 5,
    "new_onnx_attempts": 250,
    "new_onnx_completed": 250,
    "onnx_calls": 65,
    "new_candidates_completed": 5,
   

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


RuntimeError: C012_MEASURED_THERMAL_CPU_SPEED_LIMIT

{
  "cell": "NB22E-v4:C012",
  "status": "STOPPED_WITH_CHECKPOINTS",
  "execution_boundary_sha256": "aa099a56e76054c30629a35a3acbd1ae9a86970367c9fb60677b907c2f0d2cbc",
  "starting_complete": 9,
  "pilot_reused": 4,
  "completed_new_candidates": 29,
  "remaining_candidates": 1083,
  "unique_onnx_pairs_completed": 1454,
  "total_pair_attempts": 1458,
  "pair_attempt_ceiling": 55604,
  "unknown": [],
  "work_this_invocation": {
    "new_queries": 21,
    "new_onnx_attempts": 1004,
    "new_onnx_completed": 1004,
    "onnx_calls": 261,
    "new_candidates_completed": 20,
    "new_retries": 0
  },
  "stage_count_completed": 4,
  "total_elapsed_seconds": 374.526,
  "timing_seconds": {
    "source_sha": 1.349,
    "index_sha": 0.236,
    "input_read": 2.398,
    "model_setup": 1.634,
    "qdrant_setup": 3.208,
    "query": 1.9,
    "tokenization": 3.128,
    "onnx": 141.127,
    "pause": 209.093,
    "checkpoint": 3.449,
    "system_monitor": 0.576,
    "other_unattributed": 6.429
  },
  "las

<div dir="rtl" align="right">

### NB22E-v4:C013 — R11.2: بررسی تخصصی Batch نامعلوم پس از Interrupt

**هدف:** بررسی وضعیت دقیق چهار جفت ONNX مربوط به Batch شمارهٔ ۲۰ سؤال نیمه‌تمام، بدون اجرای مجدد.

**ورودی:** ممیزی موفق C012، Manifest مستقل C011 و رسیدهای ذخیره‌شدهٔ C012.

**اقدامات:**
- تأیید وضعیت Stage 1 سؤال نیمه‌تمام.
- بازخوانی و اعتبارسنجی رسیدهای Batchهای موفق همان سؤال.
- بررسی STARTED بدون DONE در Batch شمارهٔ ۲۰.
- جست‌وجوی محدود فایل موقت احتمالی با محتوای قابل‌اعتبارسنجی.
- تفکیک نتیجهٔ قطعی از محاسبه‌ای که وضعیت آن نامعلوم است.

**محدودیت:** بدون Retrieval، ONNX، Tokenizer، API، تغییر Checkpoint یا نوشتن فایل.

**شرط پایان:** مشخص شدن شواهد قابل‌بازیابی یا نیاز به تصمیم صریح برای Batch نامعلوم. نتایج سالم قبلی دست‌نخورده باقی می‌مانند.

</div>

In [12]:
# NB22E-v4 | C013 | R11.2
# Read-only forensic audit of the interrupted ONNX batch; no retry or writes.

def c013_need(ok, code):
    if not ok:
        raise RuntimeError("C013_" + code)


c013_target = "golden_v2_candidate_0184f90d79f3dfc4744b6e29"
c013_unknown_start = 20

c013_need(
    c012_execute_new_work is False
    and c012_report["status"] == "AUDIT_ONLY_NO_EXECUTION"
    and c012_report["execution_boundary_sha256"]
    == c012_boundary_sha
    and len(c012_before["uncertain"]) == 1
    and c012_before["uncertain"][0]["candidate_id"]
    == c013_target
    and c012_before["uncertain"][0]["unit"]
    == "ONNX_BATCH"
    and c012_before["uncertain"][0]["batch_starts"]
    == [c013_unknown_start]
    and c013_target in c012_ids
    and c012_before["states"][c013_target]
    == "UNCERTAIN",
    "AUDIT_BASELINE_CONFLICT",
)

c013_stage1, c013_stage1_sha = c012_read(
    c012_path(c013_target, "stage1_done")
)

c013_need(
    c013_stage1["status"] == "STAGE1_PASS"
    and c013_stage1["candidate_id"] == c013_target
    and c013_stage1["boundary"] == c012_boundary_sha
    and c013_stage1["full_order_matches"] is True
    and c013_stage1["pool50_membership_matches"] is True,
    "STAGE1_NOT_VERIFIED",
)

c013_identity = c013_stage1["input_sha256"]

c013_receipts = []
c013_missing = []
c013_completed_pairs = 0

for start in range(0, 50, 4):
    started_path = c012_path(
        c013_target,
        "stage2_batch_%02d_started" % start,
    )
    done_path = c012_path(
        c013_target,
        "stage2_batch_%02d_done" % start,
    )

    started_exists = started_path.is_file()
    done_exists = done_path.is_file()

    c013_need(
        not done_exists or started_exists,
        "DONE_WITHOUT_STARTED:" + str(start),
    )

    started = None
    started_sha = None

    if started_exists:
        started, started_sha = c012_read(started_path)

        c013_need(
            started["candidate_id"] == c013_target
            and started["boundary"] == c012_boundary_sha
            and started["input_sha256"] == c013_identity
            and started["batch_start"] == start
            and started["status"] == "STARTED_UNCONFIRMED",
            "STARTED_RECEIPT_CONFLICT:" + str(start),
        )

    if done_exists:
        done, done_sha = c012_read(done_path)

        c013_need(
            done["status"] == "BATCH_PASS"
            and done["candidate_id"] == c013_target
            and done["boundary"] == c012_boundary_sha
            and done["input_sha256"] == c013_identity
            and done["chunk_ids"] == started["chunk_ids"]
            and len(done["scores"])
            == len(started["chunk_ids"])
            and all(
                math.isfinite(float(value))
                for value in done["scores"]
            ),
            "DONE_RECEIPT_CONFLICT:" + str(start),
        )

        c013_completed_pairs += len(done["scores"])

        c013_receipts.append({
            "batch_start": start,
            "status": "VERIFIED_DONE",
            "pair_count": len(done["scores"]),
            "file_sha256": done_sha,
        })

    elif started_exists:
        c013_missing.append({
            "batch_start": start,
            "status": "STARTED_WITHOUT_DONE",
            "pair_count": len(started["chunk_ids"]),
            "started_sha256": started_sha,
            "started_mtime_ns": started_path.stat().st_mtime_ns,
        })

    else:
        c013_receipts.append({
            "batch_start": start,
            "status": "NOT_STARTED",
        })

c013_need(
    len(c013_missing) == 1
    and c013_missing[0]["batch_start"] == 20
    and c013_missing[0]["pair_count"] == 4
    and c013_completed_pairs == 20
    and not c012_path(
        c013_target,
        "stage2_done",
    ).exists(),
    "INTERRUPTED_WORK_STATE_CHANGED",
)

# Inspect possible temporary receipts without modifying or promoting them.
c013_temp_files = []
c013_recoverable_candidates = []

if c012_dir.is_dir():
    c013_temp_files = sorted(
        p
        for p in c012_dir.iterdir()
        if p.name.startswith(".C012-")
        and p.name.endswith(".tmp")
        and p.is_file()
        and not p.is_symlink()
    )

c013_need(
    len(c013_temp_files) <= 500,
    "UNEXPECTED_TEMP_FILE_VOLUME",
)

c013_started_unknown, _ = c012_read(
    c012_path(
        c013_target,
        "stage2_batch_20_started",
    )
)

for temp_path in c013_temp_files:
    if temp_path.stat().st_size > 1024 * 1024:
        continue

    try:
        raw = temp_path.read_bytes()
        record = json.loads(raw)

    except (OSError, ValueError, UnicodeDecodeError):
        continue

    if not isinstance(record, dict):
        continue

    if record.get("candidate_id") != c013_target:
        continue

    if record.get("chunk_ids") != c013_started_unknown["chunk_ids"]:
        continue

    declared_sha = record.get("core_sha256")

    core = {
        key: value
        for key, value in record.items()
        if key != "core_sha256"
    }

    hash_valid = (
        isinstance(declared_sha, str)
        and hashlib.sha256(
            c012_json(core)
        ).hexdigest() == declared_sha
    )

    content_valid = (
        core.get("status") == "BATCH_PASS"
        and core.get("boundary") == c012_boundary_sha
        and core.get("input_sha256") == c013_identity
        and isinstance(core.get("scores"), list)
        and len(core["scores"]) == 4
        and all(
            isinstance(score, (int, float))
            and math.isfinite(score)
            for score in core["scores"]
        )
    )

    c013_recoverable_candidates.append({
        "filename": temp_path.name,
        "byte_sha256": hashlib.sha256(raw).hexdigest(),
        "core_sha256_valid": hash_valid,
        "content_schema_valid": content_valid,
        "requires_manual_adjudication": True,
    })

c013_report = {
    "cell": "NB22E-v4:C013",
    "status": "INTERRUPTED_BATCH_FORENSIC_AUDIT",
    "execution_boundary_sha256": c012_boundary_sha,
    "candidate_id": c013_target,
    "stage1_verified": True,
    "stage1_receipt_sha256": c013_stage1_sha,
    "candidate_completed_onnx_pairs": c013_completed_pairs,
    "verified_completed_batch_starts": [
        item["batch_start"]
        for item in c013_receipts
        if item["status"] == "VERIFIED_DONE"
    ],
    "uncertain_batch": c013_missing[0],
    "temp_files_found": len(c013_temp_files),
    "potential_temp_receipts": c013_recoverable_candidates,
    "automatic_retry_allowed": False,
    "automatic_checkpoint_promotion_allowed": False,
    "healthy_receipts_preserved": True,
    "new_retrieval_queries": 0,
    "new_onnx_calls": 0,
    "new_api_calls": 0,
    "repository_writes": 0,
    "artifact_writes": 0,
    "next": "MANUAL_ADJUDICATION_OF_SINGLE_UNCERTAIN_ONNX_BATCH",
}

print(
    json.dumps(
        c013_report,
        ensure_ascii=False,
        indent=2,
    ),
    flush=True,
)

{
  "cell": "NB22E-v4:C013",
  "status": "INTERRUPTED_BATCH_FORENSIC_AUDIT",
  "execution_boundary_sha256": "aa099a56e76054c30629a35a3acbd1ae9a86970367c9fb60677b907c2f0d2cbc",
  "candidate_id": "golden_v2_candidate_0184f90d79f3dfc4744b6e29",
  "stage1_verified": true,
  "stage1_receipt_sha256": "f40eb6ac904d87744da9191803bf8c9bb4773c3c39fc84c62988b378c554e821",
  "candidate_completed_onnx_pairs": 20,
  "verified_completed_batch_starts": [
    0,
    4,
    8,
    12,
    16
  ],
  "uncertain_batch": {
    "batch_start": 20,
    "status": "STARTED_WITHOUT_DONE",
    "pair_count": 4,
    "started_sha256": "4745577680cbd38d3a212e27ec67e3af98a42088cd2f3aace077fca563e4daf2",
    "started_mtime_ns": 1794138552775836347
  },
  "temp_files_found": 0,
  "potential_temp_receipts": [],
  "automatic_retry_allowed": false,
  "automatic_checkpoint_promotion_allowed": false,
  "healthy_receipts_preserved": true,
  "new_retrieval_queries": 0,
  "new_onnx_calls": 0,
  "new_api_calls": 0,
  "repository_


<div dir="rtl" align="right">

### NB22E-v4:C014 — اتصال نتایج نهایی آروان به ادامهٔ محلی

**هدف:** بازیابی وضعیت معتبر C012 از Artifactهای منتقل‌شده به مک، بدون بازاجرای محاسبات قبلی.

**ورودی‌ها:**
- Manifest فریز‌شدهٔ C011
- خروجی‌های کامل ۱۱۱۲ سؤال C012
- گزارش Audit و تأییدیهٔ SHA
- Runner منتقل‌شده از آروان

**معیار موفقیت:** تطابق هویت اجرا، وجود همهٔ خروجی‌های مورد انتظار و معتبر بودن شواهد تکمیل فنی.

**محدودیت:** فقط خواندن فایل‌ها؛ بدون ONNX، Retrieval، API، تغییر Git یا نوشتن Artifact.

**مرحلهٔ بعد:** تعیین بستهٔ اجرایی Rank نهایی، Quota، char_rrf و Context بر اساس قراردادهای موجود.

</div>


In [1]:
# NB22E-v4 | C014 | R11.2
# Local C012 handoff reconciliation; read-only, no API or model calls.

from pathlib import Path
import hashlib
import json

def c014_sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

# Locate the existing Mac project without executing earlier cells.
cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
candidates.append(
    Path.home()
    / "AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance"
)

repo = next(
    (
        p for p in candidates
        if (p / "notebooks/phase2/22e_retrieval_experiment_lab_v4.ipynb").is_file()
    ),
    None
)

if repo is None:
    raise RuntimeError("C014_PROJECT_ROOT_NOT_FOUND")

base = repo.parent
outputs = base / "_RAG_finance_NB22E_v4_outputs"

boundary = (
    "aa099a56e76054c30629a35a3acbd1ae9a86970367c9fb60677b907c2f0d2cbc"
)
manifest_sha = (
    "a3ffce372eab24e210bedae9b2b6087dc2493e45805c4c27836d223a6578fe80"
)

manifest_path = outputs / f"C011.population_boundary.{manifest_sha}.json"
c012_dir = outputs / f"C012_population_{boundary}"

transfer = (
    base / "_RAG_finance_Arvan_C012_backups"
    / "ARVAN_FINAL_RETURN/runner_inputs"
)
attestation_path = transfer / "C012_AUDIT_SHA_ATTESTATION_20261110T194001Z.json"
audit_log_path = transfer / "C012_AUDIT_PATCHED_20261110T194001Z.log"
runner_path = transfer / "c012_arvan_resumable_runner.py"

required = [manifest_path, attestation_path, audit_log_path, runner_path]
for path in required:
    if not path.is_file():
        raise RuntimeError(f"C014_MISSING_INPUT: {path.name}")

manifest = json.loads(manifest_path.read_bytes())
attestation = json.loads(attestation_path.read_bytes())
audit_text = audit_log_path.read_text(encoding="utf-8")

receipts = list(c012_dir.glob("C012.*.stage2_done.json"))

checks = {
    "MANIFEST_SHA": c014_sha256(manifest_path) == manifest_sha,
    "EXECUTION_BOUNDARY": manifest["execution_boundary_sha256"] == boundary,
    "POPULATION_1116": len(manifest["execution_contract"]["members"]) == 1116,
    "PILOTS_4": len(manifest["approved_pilot_provenance"]["pilot_ids"]) == 4,
    "LOGICAL_BATCHES_45": len(manifest["execution_queue"]["batches"]) == 45,
    "C012_RECEIPTS_1112": len(receipts) == 1112,
    "RUNNER_SHA": (
        c014_sha256(runner_path)
        == attestation["original_runner_sha256"]
    ),
    "AUDIT_LOG_SHA": (
        c014_sha256(audit_log_path)
        == attestation["original_audit_log_sha256"]
    ),
    "AUDIT_INTERNAL_PASS": (
        "C012_AUDIT_OK completed=1112/1112" in audit_text
        and "pairs_done=55600 attempts=55604" in audit_text
    ),
    "AUDIT_ATTESTATION": (
        attestation["status"]
        == "INTERNAL_AUDIT_PASS_WRAPPER_SHA_COUNT_ERROR"
        and attestation["completed"] == 1112
        and attestation["fresh_pairs"] == 55600
        and attestation["total_pair_attempts"] == 55604
        and attestation["new_work_during_audit"] == 0
    ),
}

for name, passed in checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")

if not all(checks.values()):
    raise RuntimeError("C014_HANDOFF_RECONCILIATION_FAILED")

NB22E_V4_C014_HANDOFF = {
    "status": "LOCAL_C012_HANDOFF_READY",
    "repository_root": str(repo),
    "output_directory": str(c012_dir),
    "execution_boundary_sha256": boundary,
    "completed_new_questions": 1112,
    "historical_pilots": 4,
    "completed_onnx_pairs": 55600,
    "total_pair_attempts": 55604,
    "new_api_calls": 0,
    "new_model_calls": 0,
    "artifact_writes": 0,
    "quality_gate": "BLOCKED",
    "next": "FROZEN_RANK_CONTEXT_CONTRACT_REVIEW",
}

print("\nC014_STATUS:", NB22E_V4_C014_HANDOFF["status"])
print("NEXT:", NB22E_V4_C014_HANDOFF["next"])


MANIFEST_SHA: PASS
EXECUTION_BOUNDARY: PASS
POPULATION_1116: PASS
PILOTS_4: PASS
LOGICAL_BATCHES_45: PASS
C012_RECEIPTS_1112: PASS
RUNNER_SHA: PASS
AUDIT_LOG_SHA: PASS
AUDIT_INTERNAL_PASS: PASS
AUDIT_ATTESTATION: PASS

C014_STATUS: LOCAL_C012_HANDOFF_READY
NEXT: FROZEN_RANK_CONTEXT_CONTRACT_REVIEW



<div dir="rtl" align="right">

### NB22E-v4:C015 — اتصال قراردادهای فریز‌شدهٔ Rank و Context

**هدف:** آماده‌سازی مسیر محاسباتی ۱۱۱۶عضوی پس از تکمیل C012، بدون بازاجرای مدل یا Retrieval.

**ورودی‌های معتبر:**
- Manifest فریز‌شدهٔ C011 با ۱۱۱۶ عضو
- چهار نتیجهٔ معتبر Pilot از C009
- نتایج کامل ۱۱۱۲ سؤال C012
- کد فریز‌شدهٔ `derived_order` در NB22C
- کد فریز‌شدهٔ `lexical_fusion` در NB22D
- شواهد C138/C139 و تابع `build_context` در Pipeline اصلی

**ترتیب اجرایی فریز‌شده:**

۱. ادغام رتبه‌های Hybrid100 و Local100 با RRF.

۲. اعمال Quota روی Top10، با حفظ ترتیب نسبی و سقف تعیین‌شدهٔ ورود اعضای خارج از Pool50.

۳. اعمال `char_rrf` روی ترتیب حاصل از Quota با استفاده از کانال Char.

۴. انتخاب Top10 نهایی و رندر Context با تابع تأییدشدهٔ Pipeline.

**اقدام این سل:** کنترل هویت کدهای Producer، منشأ تنظیمات، ارتباط C138/C139 و ساخت نقشهٔ ورودی‌های Pilot و C012 برای اجرای بعدی.

**شرط موفقیت:** شواهد معتبر و Schema ورودی‌ها با مسیر واقعی محاسبات سازگار باشند.

**مرزهای ممنوع:** هیچ ONNX، Embedding، Dense Query، Retrieval، Generation، API، تغییر Git یا نوشتن Artifact در این سل انجام نمی‌شود.

**وضعیت کیفیت:** `quality_gate = BLOCKED` تا زمان ارزیابی مربوط.

</div>


In [6]:
# NB22E-v4 | C015 | R11.2
# Frozen Rank/Context contracts and local 1116-member handoff.
# Read-only: no inference, retrieval, API, generation, Git, or writes.

import ast

if NB22E_V4_C014_HANDOFF["status"] != "LOCAL_C012_HANDOFF_READY":
    raise RuntimeError("C015_REQUIRES_ACCEPTED_C014")


def c015_read_pinned(path, expected_sha):
    path = Path(path)
    if not path.is_file():
        raise RuntimeError(f"C015_MISSING_SOURCE:{path}")
    raw = path.read_bytes()
    actual = hashlib.sha256(raw).hexdigest()
    if actual != expected_sha:
        raise RuntimeError(f"C015_SHA_CONFLICT:{path}:{actual}")
    return raw


def c015_notebook_code_hash(nb):
    sources = [
        "".join(c["source"]) if isinstance(c["source"], list)
        else c["source"]
        for c in nb["cells"] if c["cell_type"] == "code"
    ]
    raw = json.dumps(
        sources, ensure_ascii=False, sort_keys=True,
        separators=(",", ":"), allow_nan=False
    ).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()


def c015_frozen_function(source, name):
    tree = ast.parse(source)
    found = [
        node for node in tree.body
        if isinstance(node, ast.FunctionDef) and node.name == name
    ]
    if len(found) != 1:
        raise RuntimeError(f"C015_FUNCTION_NOT_UNIQUE:{name}")
    segment = ast.get_source_segment(source, found[0])
    if not segment:
        raise RuntimeError(f"C015_FUNCTION_SOURCE_MISSING:{name}")
    return segment


# ------------------------------------------------------------
# 1. Frozen C138/C139 evidence and product Pipeline
# ------------------------------------------------------------

c015_reports = (
    repo / "reports/phase2/quality_improvement/NB22E.lab.v3"
)

c015_c138_sha = (
    "1df8a9e4c0fcc6a4bfcce4edb4356f2557f71e312aed48f501b89a8c5464e910"
)
c015_c139_sha = (
    "3b5a43d5476dc28de79eed5fc35adc6e2a6834bbd9c6a341c8efa93b18669634"
)

c015_c138 = json.loads(c015_read_pinned(
    c015_reports / (
        "C138.R11_constructed_product_instance_local_rank_chain."
        + c015_c138_sha + ".json"
    ),
    c015_c138_sha
))

c015_c139 = json.loads(c015_read_pinned(
    c015_reports / (
        "C139.R11_persisted_opt_in_caller."
        + c015_c139_sha + ".json"
    ),
    c015_c139_sha
))

if not (
    c015_c138["status"]
    == "CONSTRUCTED_PRODUCT_INSTANCE_LOCAL_RANK_CHAIN_PASSED"
    and c015_c139["status"]
    == "OPT_IN_CALLER_PERSISTED_IN_PRODUCT_SOURCE"
    and c015_c139["input_file_hashes"]["C138"]
    == c015_c138_sha
):
    raise RuntimeError("C015_C138_C139_LINEAGE_CONFLICT")

c015_pipeline_sha = c015_c138["input_file_hashes"]["pipeline_source"]

if c015_c139["input_file_hashes"]["pipeline_source"] != c015_pipeline_sha:
    raise RuntimeError("C015_PIPELINE_PIN_CONFLICT")

c015_pipeline_source = c015_read_pinned(
    repo / "src/pipeline/rag_pipeline.py",
    c015_pipeline_sha
).decode("utf-8")

c015_route_pin = c015_c139["module"]
c015_route_path = (repo / c015_route_pin["path"]).resolve()

if not c015_route_path.is_relative_to(repo.resolve()):
    raise RuntimeError("C015_ROUTE_PATH_ESCAPE")

c015_read_pinned(
    c015_route_path,
    c015_route_pin["file_bytes_sha256"]
)


# ------------------------------------------------------------
# 2. Frozen NB22C / NB22D producer definitions
# ------------------------------------------------------------

c015_nb22c_path = (
    repo / "notebooks/phase2/"
    "22c_retrieval_diagnosis_and_improvement.ipynb"
)
c015_nb22d_path = (
    repo / "notebooks/phase2/"
    "22d_full_corpus_retrieval_expansion.ipynb"
)

c015_nb22c = json.loads(c015_nb22c_path.read_bytes())

c015_nb22d = json.loads(c015_read_pinned(
    c015_nb22d_path,
    "fff5bd072a4977820882045451b61d66f8621ef9a4918775e4534800f0ae1480"
))

for label, nb in (
    ("nb22c", c015_nb22c),
    ("nb22d", c015_nb22d),
):
    expected = (
        nb.get("metadata", {})
        .get(label, {})
        .get("code_source_sha256")
    )
    if c015_notebook_code_hash(nb) != expected:
        raise RuntimeError(
            f"C015_PRODUCER_CODE_IDENTITY_CONFLICT:{label}"
        )

c015_derived_source = c015_frozen_function(
    "".join(c015_nb22c["cells"][6]["source"]),
    "derived_order"
)

c015_lexical_source = c015_frozen_function(
    "".join(c015_nb22d["cells"][12]["source"]),
    "lexical_fusion"
)

c015_pipeline_tree = ast.parse(c015_pipeline_source)
c015_render_nodes = [
    method
    for cls in c015_pipeline_tree.body
    if isinstance(cls, ast.ClassDef) and cls.name == "RAGPipeline"
    for method in cls.body
    if isinstance(method, ast.FunctionDef)
    and method.name == "build_context"
]

if len(c015_render_nodes) != 1:
    raise RuntimeError("C015_CONTEXT_RENDERER_NOT_UNIQUE")

c015_renderer_source = ast.get_source_segment(
    c015_pipeline_source,
    c015_render_nodes[0]
)

c015_c138_code = (
    repo / "src/pipeline/"
    "char_rrf_development_local_startup.py"
).read_text(encoding="utf-8")


# ------------------------------------------------------------
# 3. Frozen algorithm and execution-order guards
# ------------------------------------------------------------

c015_guards = {
    "DERIVED_RRF_AND_TIEBREAK": (
        "key=lambda c: (-(w / (k + lr[c]) "
        "+ (1 - w) / (k + hr[c])), c)"
        in c015_derived_source
    ),

    "QUOTA_FROM_FROZEN_FUNCTION": (
        "spec['newcomer_limit']" in c015_derived_source
        and "if len(top) == 10:" in c015_derived_source
    ),

    "CHAR_RRF_FROM_FROZEN_FUNCTION": (
        "CONFIG['rrf_k']" in c015_lexical_source
        and "CONFIG['incumbent_rank_weight']"
        in c015_lexical_source
        and "return sorted(score,key=lambda c:(-score[c],c))"
        in c015_lexical_source
    ),

    "RUNTIME_DERIVED_THEN_QUOTA": (
        "quota_order = top + "
        "[cid for cid in ranked if cid not in chosen]"
        in c015_c138_code
    ),

    "RUNTIME_QUOTA_THEN_CHAR_RRF": (
        'quota_order, {"char": char_order}, '
        '{"channels": ["char"]}'
        in c015_c138_code
    ),

    "CONTEXT_EXACT_BUILDER": (
        'separator = "\\n\\n" + ("-" * 80) + "\\n\\n"'
        in c015_renderer_source
        and "return separator.join(context_parts)"
        in c015_renderer_source
    ),

    "NO_PRODUCT_PIPELINE_CALL": (
        "ask_with_char_rrf" not in c015_derived_source
        and "ask_with_char_rrf" not in c015_lexical_source
        and "ask_with_char_rrf" not in c015_renderer_source
    ),
}

for label, passed in c015_guards.items():
    print(f"{label}: {'PASS' if passed else 'FAIL'}")

if not all(c015_guards.values()):
    raise RuntimeError(
        "C015_FROZEN_COMPUTATION_CONTRACT_REVIEW_REQUIRED"
    )


# ------------------------------------------------------------
# 4. Reuse the four historical C009 Pilot receipts
# ------------------------------------------------------------

c015_pilot_pin = (
    manifest["approved_pilot_provenance"]["C009"]
)

c015_pilot_summary = json.loads(c015_read_pinned(
    outputs / "C009_stage2"
    / Path(c015_pilot_pin["path"]).name,
    c015_pilot_pin["file_bytes_sha256"]
))

c015_pilot_ids = (
    manifest["approved_pilot_provenance"]["pilot_ids"]
)

if not (
    c015_pilot_summary["status"] == "STAGE2_PASS"
    and set(c015_pilot_summary["candidate_ids"])
    == set(c015_pilot_ids)
    and len(c015_pilot_summary["candidate_results"]) == 4
):
    raise RuntimeError("C015_PILOT_SUMMARY_CONFLICT")

c015_pilot_files = {}

for ref in c015_pilot_summary["candidate_results"]:
    path = (
        outputs / "C009_stage2"
        / Path(ref["path"]).name
    )
    record = json.loads(c015_read_pinned(
        path, ref["file_bytes_sha256"]
    ))
    cid = record["candidate_id"]

    if (
        cid != ref["candidate_id"]
        or record["status"] != "CANDIDATE_PASS"
        or cid in c015_pilot_files
    ):
        raise RuntimeError(
            f"C015_PILOT_RECEIPT_CONFLICT:{cid}"
        )

    c015_pilot_files[cid] = path

if set(c015_pilot_files) != set(c015_pilot_ids):
    raise RuntimeError("C015_PILOT_MEMBERSHIP_CONFLICT")


# ------------------------------------------------------------
# 5. Exact population and corrected C012 receipt paths
# ------------------------------------------------------------

c015_execution_contract = manifest["execution_contract"]

c015_member_ids = [
    row["candidate_id"]
    for row in c015_execution_contract["members"]
]

c015_queue = [
    cid
    for batch in manifest["execution_queue"]["batches"]
    for cid in batch["candidate_ids"]
]

if not (
    len(c015_member_ids) == len(set(c015_member_ids)) == 1116
    and len(c015_queue) == len(set(c015_queue)) == 1112
    and set(c015_member_ids)
    == set(c015_queue) | set(c015_pilot_ids)
    and not (set(c015_queue) & set(c015_pilot_ids))
):
    raise RuntimeError("C015_POPULATION_MEMBERSHIP_CONFLICT")

c015_sample_id = c015_queue[0]

# Exact frozen C012 filename rule:
# SHA256(candidate_id encoded as UTF-8), not the raw candidate_id.
c015_sample_key = hashlib.sha256(
    c015_sample_id.encode("utf-8")
).hexdigest()

c015_s1_path = (
    c012_dir / f"C012.{c015_sample_key}.stage1_done.json"
)
c015_s2_path = (
    c012_dir / f"C012.{c015_sample_key}.stage2_done.json"
)

if not (c015_s1_path.is_file() and c015_s2_path.is_file()):
    raise RuntimeError(
        f"C015_SAMPLE_RECEIPTS_MISSING:{c015_sample_id}"
    )

c015_sample_s1 = json.loads(c015_s1_path.read_bytes())
c015_sample_s2 = json.loads(c015_s2_path.read_bytes())

if not (
    c015_sample_s1["status"] == "STAGE1_PASS"
    and c015_sample_s2["status"] == "STAGE2_PASS"
    and c015_sample_s1["candidate_id"] == c015_sample_id
    and c015_sample_s2["candidate_id"] == c015_sample_id
    and c015_sample_s2["boundary"]
    == manifest["execution_boundary_sha256"]
    and len(c015_sample_s2["local100_order"]) == 100
):
    raise RuntimeError("C015_C012_RECEIPT_SCHEMA_CONFLICT")


# ------------------------------------------------------------
# 6. Read-only C016 handoff
# ------------------------------------------------------------

c015_source_identity = {
    "execution_boundary_sha256": (
        manifest["execution_boundary_sha256"]
    ),
    "C138": c015_c138_sha,
    "C139": c015_c139_sha,
    "pipeline_source_sha256": c015_pipeline_sha,
    "nb22c_derived_source_sha256": hashlib.sha256(
        c015_derived_source.encode("utf-8")
    ).hexdigest(),
    "nb22d_lexical_source_sha256": hashlib.sha256(
        c015_lexical_source.encode("utf-8")
    ).hexdigest(),
}

c015_contract_sha = hashlib.sha256(
    json.dumps(
        c015_source_identity,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        allow_nan=False
    ).encode("utf-8")
).hexdigest()

C015_FROZEN_CONTRACT = {
    "status": "FROZEN_RANK_CONTEXT_CONTRACT_READY",
    "execution_boundary_sha256": (
        manifest["execution_boundary_sha256"]
    ),
    "population_count": len(c015_member_ids),
    "pilot_count": len(c015_pilot_files),
    "c012_count": len(c015_queue),
    "rank_and_context_producer_sha256": c015_contract_sha,
    "pilot_candidate_files": {
        cid: str(path)
        for cid, path in c015_pilot_files.items()
    },
    "source_pins": (
        c015_execution_contract["source_producer_pins"]
    ),
    "input_pins": c015_execution_contract["input_pins"],
    "next": "C016_LOCAL_RANK_QUOTA_CHAR_RRF_CONTEXT",
    "api_calls": 0,
    "model_calls": 0,
    "artifact_writes": 0,
    "quality_gate": "BLOCKED",
}

print("\nPILOTS_READY:", len(c015_pilot_files), "/ 4")
print("C012_QUEUE_MEMBERS:", len(c015_queue), "/ 1112")
print(
    "SOURCE_PIN_LABELS:",
    sorted(c015_execution_contract["source_producer_pins"])
)
print(
    "INPUT_PIN_LABELS:",
    sorted(c015_execution_contract["input_pins"])
)
print(
    "PILOT_CANDIDATE_FIELDS:",
    sorted(record)
)
print(
    "C012_STAGE1_FIELDS:",
    sorted(c015_sample_s1)
)
print(
    "C012_STAGE2_FIELDS:",
    sorted(c015_sample_s2)
)
print(
    "\nC015_STATUS:",
    C015_FROZEN_CONTRACT["status"]
)
print("NEXT:", C015_FROZEN_CONTRACT["next"])

DERIVED_RRF_AND_TIEBREAK: PASS
QUOTA_FROM_FROZEN_FUNCTION: PASS
CHAR_RRF_FROM_FROZEN_FUNCTION: PASS
RUNTIME_DERIVED_THEN_QUOTA: PASS
RUNTIME_QUOTA_THEN_CHAR_RRF: PASS
CONTEXT_EXACT_BUILDER: PASS
NO_PRODUCT_PIPELINE_CALL: PASS

PILOTS_READY: 4 / 4
C012_QUEUE_MEMBERS: 1112 / 1112
SOURCE_PIN_LABELS: ['C118', 'C119', 'C139', 'C148_POINTER', 'C186', 'C187_POINTER', 'C188', 'CALLER', 'NB16', 'NB22C', 'NB22D', 'PIPELINE']
INPUT_PIN_LABELS: ['CHUNKS', 'GOLDEN', 'HYBRID', 'RERANKER']
PILOT_CANDIDATE_FIELDS: ['all100_scores_by_chunk_id', 'api_calls', 'batch_receipts', 'candidate_id', 'cell', 'elapsed_seconds_this_execution', 'historical_local100_rank_equivalence', 'input_sha256', 'local100_order', 'local100_order_sha256', 'new_model_calls_this_execution', 'new_pair_count', 'new_pairs_this_execution', 'new_scores_by_chunk_id', 'original_hybrid100_order', 'process_peak_rss_gib_since_kernel_start', 'qdrant_queries', 'question_sha256', 'record_core_sha256', 'reused_pool50_pair_count', 'scope_sha256'


<div dir="rtl" align="right">

### NB22E-v4:C016 — تولید Rank و Quota از Scoring موجود و ثبت وضعیت Context

**هدف:** مصرف خروجی‌های معتبر C009 و C012 برای تمام ۱۱۱۶ عضو Manifest C011.

**ورودی‌ها:** امتیازهای فریز‌شدهٔ Pool50 و Local100، کد فریز‌شدهٔ `derived_order`، متن چانک‌های نسخه‌دار و هویت C015.

**اقدامات:**
1. بازیابی و کنترل امتیازها و ۱۰۰ چانک هر سؤال.
2. تولید ترتیب Derived با RRF فریز‌شده و Tie-break شناسهٔ چانک.
3. اعمال Quota با سقف یک عضو خارج از Pool50 در Top10.
4. ثبت هش ترتیب‌ها و اتصال به متن چانک‌های فریز‌شده.
5. ثبت مانع مشخص در صورت نبود رتبهٔ معتبر کانال Char.
6. ذخیرهٔ مستقل و قابل Resume برای هر سؤال و Manifest جمعیتی.

**معیار موفقیت مرحله:** تمام ۱۱۱۶ عضو دارای رسید خروجی یا مانع مشخص باشند. تکمیل این مرحله به‌تنهایی مجوز Generation نیست.

**محدودیت:** بازاجرای ONNX، Embedding، Retrieval، API، Generation، تغییر Gold و تغییر Git ممنوع است.

**وضعیت:** `quality_gate = BLOCKED` باقی می‌ماند.

</div>


In [10]:
# NB22E-v4 | C016 | R11.2
# Reuse frozen scores; checkpoint derived/quota rank and exact Test char blockers.
# Local, sequential, resumable; no ONNX, embedding, retrieval, API or Git writes.

if C015_FROZEN_CONTRACT['status'] != 'FROZEN_RANK_CONTEXT_CONTRACT_READY':
    raise RuntimeError('C016_REQUIRES_C015')
if C015_FROZEN_CONTRACT['quality_gate'] != 'BLOCKED':
    raise RuntimeError('C016_QUALITY_GATE_CHANGED')


def c016_canon(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True,
                      separators=(',', ':'), allow_nan=False).encode('utf-8')


def c016_sha(raw):
    return hashlib.sha256(raw).hexdigest()


def c016_verify_receipt(path, kind):
    raw = path.read_bytes()
    data = json.loads(raw)
    key = 'record_core_sha256' if kind == 'pilot' else 'core_sha256'
    core = {k: v for k, v in data.items() if k != key}
    if data.get(key) != c016_sha(c016_canon(core)):
        raise ValueError('INVALID_RECEIPT_CORE_SHA')
    return data, c016_sha(raw)


def c016_put_once(path, document):
    core_hash = c016_sha(c016_canon(document))
    payload = c016_canon({**document, 'record_core_sha256': core_hash}) + b'\n'
    if path.exists():
        if path.read_bytes() != payload:
            raise RuntimeError('C016_IMMUTABLE_OUTPUT_CONFLICT:' + path.name)
        return c016_sha(payload), False
    temp_path = None
    try:
        with tempfile.NamedTemporaryFile(mode='wb', dir=path.parent,
                                         prefix='.C016-', suffix='.tmp',
                                         delete=False) as stream:
            temp_path = Path(stream.name)
            stream.write(payload)
            stream.flush()
            os.fsync(stream.fileno())
        if temp_path.read_bytes() != payload:
            raise RuntimeError('C016_TEMP_RELOAD_CONFLICT')
        try:
            os.link(temp_path, path)
        except FileExistsError:
            pass
    finally:
        if temp_path is not None:
            temp_path.unlink(missing_ok=True)
    if path.read_bytes() != payload:
        raise RuntimeError('C016_ATOMIC_RELOAD_CONFLICT:' + path.name)
    return c016_sha(payload), True


# 1. Verify the frozen population before any write.

c016_boundary = manifest['execution_boundary_sha256']
if (c016_boundary != C015_FROZEN_CONTRACT['execution_boundary_sha256']
        or len(c015_member_ids) != 1116
        or len(c015_queue) != 1112
        or len(c015_pilot_files) != 4):
    raise RuntimeError('C016_POPULATION_BOUNDARY_CONFLICT')
if 'CHAR' in C015_FROZEN_CONTRACT['input_pins']:
    raise RuntimeError('C016_UNREVIEWED_CHAR_PIN_PRESENT')


# 2. Load frozen question identities once.

c016_question_sha = '9a4f0c044b459e78930ab6f0aded8f7d1bb75b677dbe170679333011cd94b5e3'
c016_questions_path = (base / '_RAG_finance_C341_question_binding_outputs'
                       / ('C341.question_bindings.' + c016_question_sha + '.jsonl'))
c016_questions_raw = c015_read_pinned(c016_questions_path, c016_question_sha)
c016_questions = {}

for line in c016_questions_raw.splitlines():
    if not line.strip():
        continue
    row = json.loads(line)
    cid, question, qsha = row['candidate_id'], row['question'], row['question_sha256']
    if cid in c016_questions or c016_sha(question.encode('utf-8')) != qsha:
        raise RuntimeError('C016_QUESTION_ID_OR_HASH_CONFLICT:' + cid)
    c016_questions[cid] = qsha

if set(c016_questions) != set(c015_member_ids):
    raise RuntimeError('C016_QUESTION_POPULATION_CONFLICT')


# 3. Load and verify the frozen chunk corpus once.

c016_chunk_pin = C015_FROZEN_CONTRACT['input_pins']['CHUNKS']
c016_chunks_relative = Path(c016_chunk_pin['path'])

if c016_chunks_relative.is_absolute() or '..' in c016_chunks_relative.parts:
    raise RuntimeError('C016_UNSAFE_CHUNK_SOURCE_PATH')

c016_chunk_raw = c015_read_pinned(
    repo / c016_chunks_relative, c016_chunk_pin['file_bytes_sha256'])

c016_chunks = {}

for line in c016_chunk_raw.splitlines():
    if not line.strip():
        continue
    row = json.loads(line)
    cid = row['chunk_id']
    if cid in c016_chunks or not isinstance(row.get('content'), str):
        raise RuntimeError('C016_CORPUS_SCHEMA_OR_DUPLICATE:' + cid)
    c016_chunks[cid] = c016_sha(row['content'].encode('utf-8'))

if len(c016_chunks) != 3488:
    raise RuntimeError('C016_CORPUS_POPULATION_CONFLICT')


# 4. Load only the hash-verified NB22C derived_order function.

c016_source_tree = ast.parse(c015_derived_source)
c016_nodes = [
    node for node in c016_source_tree.body
    if isinstance(node, ast.FunctionDef) and node.name == 'derived_order'
]

if len(c016_nodes) != 1:
    raise RuntimeError('C016_DERIVED_PRODUCER_NOT_UNIQUE')

c016_functions = {
    'require': lambda ok, reason: None if ok else (_ for _ in ()).throw(ValueError(reason))
}

exec(
    compile(
        ast.Module(body=c016_nodes, type_ignores=[]),
        'PINNED_NB22C_DERIVED',
        'exec'
    ),
    c016_functions
)

c016_derived = c016_functions['derived_order']

c016_raw_spec = {
    'family': 'rrf100',
    'k': 10,
    'weight': 0.75
}

c016_quota_spec = {
    'family': 'quota100',
    'k': 10,
    'weight': 0.75,
    'newcomer_limit': 1
}


# 5. Create an independent identity for new outputs.
# NB22D A03 contains Development lexical channels.
# It is not evidence of a frozen Test char order.
# No lexical_search or Retrieval is invoked here.

c016_identity = {
    'schema': 'NB22E_V4_C016_DERIVED_QUOTA_PRECHAR_V1',
    'c011_execution_boundary_sha256': c016_boundary,
    'c015_contract_sha256': C015_FROZEN_CONTRACT['rank_and_context_producer_sha256'],
    'c341_question_file_sha256': c016_question_sha,
    'chunks_file_sha256': c016_chunk_pin['file_bytes_sha256'],
    'derived_spec': c016_raw_spec,
    'quota_spec': c016_quota_spec,
    'char_source_policy': 'FROZEN_TEST_CHAR_ORDER_REQUIRED_NO_RETRIEVAL'
}

c016_identity_sha = c016_sha(c016_canon(c016_identity))
c016_out = (outputs / ('C016_rank_context_' + c016_identity_sha)).resolve()

if c016_out.is_relative_to(repo.resolve()) or c016_out.parent != outputs.resolve():
    raise RuntimeError('C016_OUTPUT_NOT_EXTERNAL_TO_REPOSITORY')

c016_out.mkdir(parents=True, exist_ok=True)


# 6. Process all 1116 members sequentially with immutable checkpoints.

c016_entries = []
c016_written = 0
c016_reused = 0
c016_counts = collections.Counter()

for c016_n, cid in enumerate(c015_member_ids, 1):
    c016_key = c016_sha(cid.encode('utf-8'))
    c016_path = c016_out / ('C016.' + c016_key + '.json')

    c016_source_kind = 'PILOT_C009' if cid in c015_pilot_files else 'C012'

    c016_score_path = (
        c015_pilot_files[cid]
        if c016_source_kind == 'PILOT_C009'
        else c012_dir / ('C012.' + c016_key + '.stage2_done.json')
    )

    c016_status = 'BLOCKED'
    c016_problem = None
    c016_result = {}
    c016_source_sha = None

    try:
        score_row, c016_source_sha = c016_verify_receipt(
            c016_score_path,
            'pilot' if c016_source_kind == 'PILOT_C009' else 'stage2'
        )

        if score_row['candidate_id'] != cid:
            raise ValueError('SCORE_CANDIDATE_ID_MISMATCH')

        if c016_source_kind == 'PILOT_C009':
            if (
                score_row['status'] != 'CANDIDATE_PASS'
                or score_row['question_sha256'] != c016_questions[cid]
            ):
                raise ValueError('PILOT_STATUS_OR_QUESTION_MISMATCH')

            hybrid = score_row['original_hybrid100_order']
            pool = score_row['stored_pool50_order']
            scores = score_row['all100_scores_by_chunk_id']
            local = score_row['local100_order']

            if scores != {
                **score_row['stored_pool50_scores'],
                **score_row['new_scores_by_chunk_id']
            }:
                raise ValueError('PILOT_SCORE_MERGE_MISMATCH')

        else:
            stage1_path = c012_dir / (
                'C012.' + c016_key + '.stage1_done.json'
            )
            s1, _ = c016_verify_receipt(stage1_path, 'stage2')

            if (
                s1['status'] != 'STAGE1_PASS'
                or s1['candidate_id'] != cid
                or s1['boundary'] != c016_boundary
                or score_row['status'] != 'STAGE2_PASS'
                or score_row['boundary'] != c016_boundary
                or s1['input_sha256'] != score_row['input_sha256']
                or s1['observed_hybrid100']
                != score_row['historical_hybrid100']
            ):
                raise ValueError('STAGE1_STAGE2_IDENTITY_CONFLICT')

            hybrid = score_row['historical_hybrid100']
            pool = hybrid[:50]
            scores = {
                **score_row['historical_pool50_scores'],
                **score_row['new_scores']
            }
            local = score_row['local100_order']

            if (
                len(score_row['new_scores']) != 50
                or len(score_row['historical_pool50_scores']) != 50
            ):
                raise ValueError('LOCAL100_PAIR_PARTITION_MISMATCH')

        if not (
            len(hybrid) == len(local) == len(set(hybrid)) == 100
            and set(local) == set(hybrid) == set(scores)
            and len(pool) == len(set(pool)) == 50
            and set(pool) == set(hybrid[:50])
            and all(
                isinstance(v, (int, float))
                and float('-inf') < v < float('inf')
                for v in scores.values()
            )
            and local == sorted(
                hybrid, key=lambda x: (-scores[x], x)
            )
            and all(chunk in c016_chunks for chunk in hybrid)
        ):
            raise ValueError('FROZEN_LOCAL100_OR_CHUNK_INTEGRITY_CONFLICT')

        derived = c016_derived(
            hybrid, local, pool, c016_raw_spec
        )
        quota = c016_derived(
            hybrid, local, pool, c016_quota_spec
        )

        pool_set = set(pool)

        if not (
            len(derived) == len(quota) == 100
            and set(derived) == set(quota) == set(hybrid)
            and sum(x not in pool_set for x in quota[:10]) <= 1
        ):
            raise ValueError('DERIVED_QUOTA_CONTRACT_CONFLICT')

        c016_result = {
            'hybrid100_order_sha256': c016_sha(c016_canon(hybrid)),
            'local100_order_sha256': c016_sha(c016_canon(local)),
            'pool50_order_sha256': c016_sha(c016_canon(pool)),
            'scores_by_chunk_id_sha256': c016_sha(c016_canon(scores)),
            'derived_order': derived,
            'derived_order_sha256': c016_sha(c016_canon(derived)),
            'quota100_order': quota,
            'quota100_order_sha256': c016_sha(c016_canon(quota)),
            'quota100_ordered_chunk_text_sha256': [
                c016_chunks[x] for x in quota
            ]
        }

        c016_status = 'DERIVED_QUOTA_READY_CHAR_RRF_BLOCKED'
        c016_problem = 'NO_FROZEN_TEST_CHAR_CHANNEL_RANK_IN_C011_BOUND_INPUTS'

    except (OSError, KeyError, ValueError, TypeError, json.JSONDecodeError) as exc:
        c016_problem = type(exc).__name__ + ':' + str(exc)

    c016_record = {
        'cell': 'NB22E-v4:C016',
        'schema_version': c016_identity['schema'],
        'c016_identity_sha256': c016_identity_sha,
        'c011_execution_boundary_sha256': c016_boundary,
        'candidate_id': cid,
        'question_sha256': c016_questions[cid],
        'score_receipt_kind': c016_source_kind,
        'score_receipt_path': str(c016_score_path),
        'score_receipt_file_sha256': c016_source_sha,
        'rank_state': c016_status,
        'blocker': c016_problem,
        'rank_partial': c016_result,
        'char_rrf_final_order_sha256': None,
        'rendered_context_sha256': None,
        'generation_input_ready': False,
        'quality_gate': 'BLOCKED',
        'new_model_calls': 0,
        'new_retrieval_calls': 0,
        'new_api_calls': 0
    }

    digest, written = c016_put_once(c016_path, c016_record)

    c016_written += int(written)
    c016_reused += int(not written)
    c016_counts[c016_status] += 1

    c016_entries.append({
        'candidate_id': cid,
        'status': c016_status,
        'blocker': c016_problem,
        'receipt': c016_path.name,
        'receipt_file_sha256': digest
    })

    if c016_n % 100 == 0 or c016_n == 1116:
        print(f'C016_PROGRESS: {c016_n}/1116')


# 7. Publish and reload the population manifest.

c016_summary = {
    'cell': 'NB22E-v4:C016',
    'status': 'PARTIAL_WITH_EXPLICIT_BLOCKERS',
    'identity': c016_identity,
    'identity_sha256': c016_identity_sha,
    'population_count': 1116,
    'per_candidate_status': c016_entries,
    'status_counts': dict(sorted(c016_counts.items())),
    'generation_ready_count': 0,
    'unresolved_count': 1116,
    'quality_gate': 'BLOCKED',
    'new_model_calls': 0,
    'new_retrieval_calls': 0,
    'new_api_calls': 0
}

c016_summary_sha = c016_sha(c016_canon(c016_summary))
c016_summary_path = c016_out / (
    'C016.summary.' + c016_summary_sha + '.json'
)

c016_summary_bytes_sha, _ = c016_put_once(
    c016_summary_path, c016_summary
)

C016_LOCAL_RESULTS = {
    'status': c016_summary['status'],
    'output_directory': str(c016_out),
    'summary_path': str(c016_summary_path),
    'summary_file_sha256': c016_summary_bytes_sha,
    'status_counts': dict(c016_counts),
    'generation_ready_count': 0,
    'quality_gate': 'BLOCKED'
}

print('C016_RECEIPTS_NEW:', c016_written)
print('C016_RECEIPTS_REUSED:', c016_reused)
print('C016_STATUS_COUNTS:', dict(sorted(c016_counts.items())))
print('C016_SUMMARY_SHA256:', c016_summary_bytes_sha)
print('C016_STATUS:', C016_LOCAL_RESULTS['status'])
print(
    'NEXT_BLOCKER:',
    'FROZEN_TEST_CHAR_CHANNEL_REQUIRED_BEFORE_FINAL_CONTEXT'
)


C016_PROGRESS: 100/1116
C016_PROGRESS: 200/1116
C016_PROGRESS: 300/1116
C016_PROGRESS: 400/1116
C016_PROGRESS: 500/1116
C016_PROGRESS: 600/1116
C016_PROGRESS: 700/1116
C016_PROGRESS: 800/1116
C016_PROGRESS: 900/1116
C016_PROGRESS: 1000/1116
C016_PROGRESS: 1100/1116
C016_PROGRESS: 1116/1116
C016_RECEIPTS_NEW: 1116
C016_RECEIPTS_REUSED: 0
C016_STATUS_COUNTS: {'DERIVED_QUOTA_READY_CHAR_RRF_BLOCKED': 1116}
C016_SUMMARY_SHA256: 7271622b2e6b770d186c114fc2ab35bd05ec472a3acac0334a5f716a9c216468
C016_STATUS: PARTIAL_WITH_EXPLICIT_BLOCKERS
NEXT_BLOCKER: FROZEN_TEST_CHAR_CHANNEL_REQUIRED_BEFORE_FINAL_CONTEXT


<div dir="rtl" align="right">

### NB22E-v4:C017 — تعیین پوشش کانال Char برای جمعیت C011

**هدف:** شناسایی ورودی‌های معتبر و موجود کانال Char برای ۱۱۱۶ عضو C011، بدون اجرای دوبارهٔ Retrieval.

**شواهد موجود:**
- خروجی C016: تعداد ۱۱۱۶ Rank و Quota آماده
- منبع `NB22D-A03`: رتبه‌های Lexical تاریخی Development
- هویت فریز‌شدهٔ C011 و قرارداد C015

**اقدام:** تطبیق شناسه‌های جمعیت با فایل نسخه‌دار کانال Char و کنترل SHA منبع.

**شرط موفقیت:** تعداد پوشش واقعی و وضعیت قابلیت استفادهٔ مجدد مشخص شود.

**محدودیت:** صرف حضور یک شناسه در منبع تاریخی، مجوز استفادهٔ آن برای Test نیست. بدون احراز هویت و دامنه، رتبه یا Context نهایی ساخته نمی‌شود.

**API، مدل، Retrieval و نوشتن Artifact:** صفر.

</div>

In [11]:
# NB22E-v4 | C017 | R11.2
# Check frozen NB22D char-channel coverage against C011 Test members.
# Read-only: no lexical search, retrieval, model, API, or artifact writes.

if C015_FROZEN_CONTRACT["status"] != "FROZEN_RANK_CONTEXT_CONTRACT_READY":
    raise RuntimeError("C017_C015_NOT_READY")

if C016_LOCAL_RESULTS["status"] != "PARTIAL_WITH_EXPLICIT_BLOCKERS":
    raise RuntimeError("C017_UNEXPECTED_C016_STATUS")

# 1. Consume the already persisted C016 summary.
c017_summary_path = Path(C016_LOCAL_RESULTS["summary_path"])
c017_summary_raw = c017_summary_path.read_bytes()

if hashlib.sha256(c017_summary_raw).hexdigest() != C016_LOCAL_RESULTS["summary_file_sha256"]:
    raise RuntimeError("C017_C016_SUMMARY_HASH_CONFLICT")

c017_summary = json.loads(c017_summary_raw)

c017_entries = c017_summary["per_candidate_status"]
c017_target_ids = set(c015_member_ids)
c017_record_ids = [entry["candidate_id"] for entry in c017_entries]

if not (
    len(c017_entries) == len(c017_record_ids) == 1116
    and len(set(c017_record_ids)) == 1116
    and set(c017_record_ids) == c017_target_ids
    and c017_summary["identity"]["c011_execution_boundary_sha256"]
        == C015_FROZEN_CONTRACT["execution_boundary_sha256"]
    and all(
        entry["status"] == "DERIVED_QUOTA_READY_CHAR_RRF_BLOCKED"
        for entry in c017_entries
    )
):
    raise RuntimeError("C017_C016_POPULATION_OR_STATUS_CONFLICT")

print("C016_DERIVED_QUOTA_COVERAGE: 1116/1116")

# 2. Inspect only the exact frozen NB22D A03 lexical artifact.
# Source identity comes from the historical NB22D producer output.
c017_a03_sha = (
    "41d1e2481910dee526c066ee807bc27ce22486d841b150bacace4c8c8ab0aa82"
)

c017_a03_path = (
    repo / "reports/phase2/quality_improvement/NB22D.v1"
    / ("A03.lexical_channels." + c017_a03_sha + ".jsonl")
)

c017_seen = set()
c017_matches = {}
c017_a03_rows = 0
c017_a03_state = "NOT_AVAILABLE_LOCALLY"

if c017_a03_path.is_file():
    hasher = hashlib.sha256()

    with c017_a03_path.open("rb") as stream:
        for raw_line in stream:
            hasher.update(raw_line)

            if not raw_line.strip():
                continue

            row = json.loads(raw_line)
            cid = row["candidate_id"]

            if cid in c017_seen:
                raise RuntimeError(
                    "C017_DUPLICATE_NB22D_CANDIDATE:" + cid
                )

            c017_seen.add(cid)
            c017_a03_rows += 1

            if cid in c017_target_ids:
                channels = row["channels"]

                if "char" not in channels:
                    raise RuntimeError(
                        "C017_MATCHED_ROW_WITHOUT_CHAR:" + cid
                    )

                order = channels["char"]

                if not (
                    isinstance(order, list)
                    and len(order) <= 800
                    and len(order) == len(set(order))
                    and all(isinstance(x, str) for x in order)
                ):
                    raise RuntimeError(
                        "C017_INVALID_MATCHED_CHAR_ORDER:" + cid
                    )

                c017_matches[cid] = order

    if hasher.hexdigest() != c017_a03_sha:
        raise RuntimeError("C017_NB22D_A03_SHA_CONFLICT")

    if c017_a03_rows != 2664:
        raise RuntimeError(
            "C017_NB22D_DEVELOPMENT_COUNT_CONFLICT"
        )

    c017_a03_state = "FROZEN_FILE_SHA_VERIFIED"

# 3. Classify the evidence without inferring missing Test ranks.
c017_matched_count = len(c017_matches)

C017_CHAR_EVIDENCE = {
    "cell": "NB22E-v4:C017",
    "c011_boundary": C015_FROZEN_CONTRACT["execution_boundary_sha256"],
    "test_population": 1116,
    "derived_quota_ready": 1116,
    "nb22d_a03_source_status": c017_a03_state,
    "nb22d_a03_rows": c017_a03_rows,
    "test_ids_matching_nb22d_a03": c017_matched_count,
    "test_char_channel_authorized": False,
    "final_char_rrf_executed": False,
    "final_context_executed": False,
    "new_retrieval_calls": 0,
    "new_model_calls": 0,
    "new_api_calls": 0,
    "artifact_writes": 0,
    "quality_gate": "BLOCKED",
}

print("NB22D_A03_SOURCE:", c017_a03_state)
print("NB22D_A03_ROWS:", c017_a03_rows)
print("TEST_CHAR_ID_MATCHES:", f"{c017_matched_count}/1116")
print("FINAL_CONTEXT_READY:", 0)

if c017_matched_count == 0:
    c017_status = "TEST_CHAR_CHANNEL_NOT_PROVEN"
elif c017_matched_count < 1116:
    c017_status = "PARTIAL_TEST_CHAR_SOURCE_MATCH_REQUIRES_REVIEW"
else:
    c017_status = "FULL_ID_MATCH_REQUIRES_SCOPE_VALIDATION"

C017_CHAR_EVIDENCE["status"] = c017_status

print("C017_STATUS:", c017_status)
print(
    "NEXT:",
    "REVIEW_FROZEN_TEST_CHAR_SOURCE_OR_REQUEST_SCOPED_DECISION"
)

C016_DERIVED_QUOTA_COVERAGE: 1116/1116
NB22D_A03_SOURCE: FROZEN_FILE_SHA_VERIFIED
NB22D_A03_ROWS: 2664
TEST_CHAR_ID_MATCHES: 0/1116
FINAL_CONTEXT_READY: 0
C017_STATUS: TEST_CHAR_CHANNEL_NOT_PROVEN
NEXT: REVIEW_FROZEN_TEST_CHAR_SOURCE_OR_REQUEST_SCOPED_DECISION


<div dir="rtl" align="right">

### NB22E-v4:C018 — ثبت مانع کانال Char و آماده‌سازی تصمیم ناظر

**نتیجهٔ C017:**
- Rank و Quota برای ۱۱۱۶ سؤال آماده است.
- فایل `NB22D-A03` از نظر SHA معتبر است.
- پوشش کانال Char برای جمعیت C011 در این فایل: صفر از ۱۱۱۶.

**اقدام:** کنترل رسیدهای ذخیره‌شدهٔ C016 و تولید Handoff نسخه‌دار، شامل شواهد، پوشش، محدودیت و تصمیم موردنیاز.

**تصمیم موردنیاز ناظر:** معرفی منبع فریز‌شدهٔ معتبر برای رتبهٔ Char جمعیت C011، یا صدور مجوز مشخص برای محاسبهٔ محلی کانال Char بدون مدل و بدون استفاده از Gold برای تنظیم.

**شرط موفقیت:** Handoff خارج از مخزن ذخیره و از دیسک بازخوانی شود. وضعیت Generation و Quality همچنان مسدود بماند.

**API، Retrieval، ONNX، Generation و تغییر Git:** صفر.

</div>

In [12]:
# NB22E-v4 | C018 | R11.2
# Persist the exact C016/C017 blocker evidence and scoped supervisor handoff.
# No retrieval, model, API, generation, Git changes, or historical rewrites.

if C017_CHAR_EVIDENCE["status"] != "TEST_CHAR_CHANNEL_NOT_PROVEN":
    raise RuntimeError("C018_UNEXPECTED_C017_STATUS")

if C016_LOCAL_RESULTS["status"] != "PARTIAL_WITH_EXPLICIT_BLOCKERS":
    raise RuntimeError("C018_UNEXPECTED_C016_STATUS")

if not (
    C017_CHAR_EVIDENCE["test_population"] == 1116
    and C017_CHAR_EVIDENCE["derived_quota_ready"] == 1116
    and C017_CHAR_EVIDENCE["test_ids_matching_nb22d_a03"] == 0
    and C017_CHAR_EVIDENCE["final_char_rrf_executed"] is False
    and C017_CHAR_EVIDENCE["final_context_executed"] is False
):
    raise RuntimeError("C018_CHAR_EVIDENCE_CONFLICT")

# Verify the persisted C016 population summary.
c018_summary_path = Path(C016_LOCAL_RESULTS["summary_path"])
c018_summary_raw = c018_summary_path.read_bytes()

if c016_sha(c018_summary_raw) != C016_LOCAL_RESULTS["summary_file_sha256"]:
    raise RuntimeError("C018_C016_SUMMARY_SHA_CONFLICT")

c018_summary = json.loads(c018_summary_raw)

if not (
    c018_summary["identity"]["c011_execution_boundary_sha256"]
    == C015_FROZEN_CONTRACT["execution_boundary_sha256"]
    and len(c018_summary["per_candidate_status"]) == 1116
    and c018_summary["status_counts"]
    == {"DERIVED_QUOTA_READY_CHAR_RRF_BLOCKED": 1116}
    and c018_summary["generation_ready_count"] == 0
):
    raise RuntimeError("C018_C016_SUMMARY_CONTENT_CONFLICT")

# Verify every referenced C016 receipt, without recalculation.
c018_ids = set()
c018_verified = 0

for entry in c018_summary["per_candidate_status"]:
    cid = entry["candidate_id"]

    if cid in c018_ids or cid not in c015_member_ids:
        raise RuntimeError("C018_POPULATION_DUPLICATE_OR_EXTRA")

    c018_ids.add(cid)

    receipt_path = Path(C016_LOCAL_RESULTS["output_directory"]) / entry["receipt"]
    raw = receipt_path.read_bytes()

    if c016_sha(raw) != entry["receipt_file_sha256"]:
        raise RuntimeError("C018_RECEIPT_FILE_SHA_CONFLICT:" + cid)

    receipt = json.loads(raw)
    core = {k: v for k, v in receipt.items() if k != "record_core_sha256"}

    if not (
        receipt["record_core_sha256"] == c016_sha(c016_canon(core))
        and receipt["candidate_id"] == cid
        and receipt["rank_state"] == "DERIVED_QUOTA_READY_CHAR_RRF_BLOCKED"
        and len(receipt["rank_partial"]["derived_order"]) == 100
        and len(receipt["rank_partial"]["quota100_order"]) == 100
        and receipt["generation_input_ready"] is False
    ):
        raise RuntimeError("C018_RECEIPT_CONTENT_CONFLICT:" + cid)

    c018_verified += 1

if c018_ids != set(c015_member_ids):
    raise RuntimeError("C018_INCOMPLETE_POPULATION")

# Freeze the exact evidence and a narrowly scoped decision request.
c018_handoff = {
    "cell": "NB22E-v4:C018",
    "status": "SCOPED_SUPERVISOR_DECISION_REQUIRED",
    "c011_execution_boundary_sha256": C015_FROZEN_CONTRACT[
        "execution_boundary_sha256"
    ],
    "c015_contract_sha256": C015_FROZEN_CONTRACT[
        "rank_and_context_producer_sha256"
    ],
    "c016_summary_path": str(c018_summary_path),
    "c016_summary_file_sha256": C016_LOCAL_RESULTS[
        "summary_file_sha256"
    ],
    "c016_verified_receipts": c018_verified,
    "population": {
        "total": 1116,
        "historical_pilots": 4,
        "c012_completed": 1112,
        "derived_and_quota_ready": 1116,
        "char_rrf_ready": 0,
        "context_ready": 0,
        "generation_ready": 0
    },
    "char_evidence": {
        "source": str(c017_a03_path),
        "source_sha256": c017_a03_sha,
        "source_status": C017_CHAR_EVIDENCE[
            "nb22d_a03_source_status"
        ],
        "historical_development_rows": 2664,
        "c011_candidate_matches": 0
    },
    "blocker": "FROZEN_TEST_CHAR_CHANNEL_NOT_PROVEN",
    "decision_options": [
        "PROVIDE_PINNED_C011_TEST_CHAR_RANK_ARTIFACT",
        "EXPLICITLY_AUTHORIZE_SCOPED_LOCAL_CPU_CHAR_LEXICAL_COMPUTATION"
    ],
    "prohibited_without_decision": [
        "NEW_LEXICAL_RETRIEVAL",
        "MODEL_INFERENCE",
        "GENERATION",
        "API_CALLS",
        "TEST_BASED_CONFIGURATION_TUNING"
    ],
    "quality_gate": "BLOCKED",
    "new_retrieval_calls": 0,
    "new_model_calls": 0,
    "new_api_calls": 0,
    "git_changes": False
}

c018_output = Path(C016_LOCAL_RESULTS["output_directory"])
c018_core_sha = c016_sha(c016_canon(c018_handoff))
c018_path = c018_output / ("C018.handoff." + c018_core_sha + ".json")

c018_file_sha, c018_created = c016_put_once(c018_path, c018_handoff)

# Independently reload and validate the published handoff.
c018_reloaded = json.loads(c018_path.read_bytes())
c018_reloaded_core = {
    k: v for k, v in c018_reloaded.items()
    if k != "record_core_sha256"
}

if not (
    c016_sha(c018_path.read_bytes()) == c018_file_sha
    and c018_reloaded_core == c018_handoff
    and c018_reloaded["record_core_sha256"] == c018_core_sha
):
    raise RuntimeError("C018_HANDOFF_RELOAD_CONFLICT")

C018_HANDOFF = {
    "status": "SCOPED_SUPERVISOR_DECISION_REQUIRED",
    "path": str(c018_path),
    "sha256": c018_file_sha,
    "verified_receipts": c018_verified,
    "quality_gate": "BLOCKED"
}

print("C018_C016_RECEIPTS_VERIFIED:", c018_verified, "/1116")
print("C018_CHAR_MATCHES:", 0, "/1116")
print("C018_HANDOFF_RELOAD: PASS")
print("C018_HANDOFF_SHA256:", c018_file_sha)
print("C018_STATUS:", C018_HANDOFF["status"])
print("NEXT: SUPERVISOR_DECISION_ON_TEST_CHAR_CHANNEL")

C018_C016_RECEIPTS_VERIFIED: 1116 /1116
C018_CHAR_MATCHES: 0 /1116
C018_HANDOFF_RELOAD: PASS
C018_HANDOFF_SHA256: 551dc91eed68fd2830618ab720b70eb7b836a376e804fa3e34aeedeb52176450
C018_STATUS: SCOPED_SUPERVISOR_DECISION_REQUIRED
NEXT: SUPERVISOR_DECISION_ON_TEST_CHAR_CHANNEL


<div dir="rtl" align="right">

### NB22E-v4:C019 — محاسبهٔ مجاز Char و ساخت Rank و Context نهایی

**مجوز:** تصمیم ناظر، گزینهٔ B، محدود به ۱۱۱۶ عضو Manifest C011.

**منابع فریز‌شده:**
- امتیازهای C012 و چهار Pilot تاریخی
- Derived Rank و Quota ذخیره‌شده در C016
- توابع `normalized_words`، `char_tokens`، `build_index`، `lexical_search` و `lexical_fusion` از NB22D
- تنظیمات تأییدشدهٔ NB22D/C138
- تابع `build_context` از Pipeline پین‌شدهٔ C015/C139

**ترتیب اجرا:**

۱. کنترل هویت Producer، تنظیمات، Corpus و سؤال‌ها.

۲. آماده‌سازی یک‌بارهٔ Index کانال Char از Corpus فریز‌شده، فقط روی CPU.

۳. کنترل چهار سؤال نخست، شامل زمان اجرا، حافظه، ترتیب و دامنهٔ شناسه‌ها.

۴. ادامهٔ ترتیبی همان اجرا در Batchهای C011، با امکان Resume.

۵. مصرف Quota موجود، اعمال `char_rrf` و ثبت منشأ رتبهٔ نهایی.

۶. اتصال Top10 به متن چانک‌ها و رندر Context بدون تغییر قالب یا اعمال محدودیت جدید.

۷. ذخیره و بازخوانی رسیدهای Char، Rank، Context، Manifest و Summary خارج از مخزن.

**شرط پایان:** ثبت خروجی معتبر یا مانع مشخص برای تمام ۱۱۱۶ سؤال و گزارش پوشش واقعی هر مرحله.

**محدودیت:** هیچ ONNX، Scoring مجدد، Dense Query، Generation، API، تغییر Gold، تنظیم روی Test یا تغییر Git مجاز نیست.

آمادگی Context صرفاً پیش‌نیاز آماده‌سازی Generation است؛ مجوز ارسال درخواست نیست.

`quality_gate = BLOCKED`

</div>

In [13]:
# NB22E-v4 | C019 | R11.2
# Supervisor option B: frozen NB22D CPU char -> C016 quota -> exact C139 Context.
# Sequential, resumable, outside repo. No ONNX, Dense, API, Generation or Git.

if C018_HANDOFF['status'] != 'SCOPED_SUPERVISOR_DECISION_REQUIRED':
    raise RuntimeError('C019_REQUIRES_C018_HANDOFF')
if C015_FROZEN_CONTRACT['status'] != 'FROZEN_RANK_CONTEXT_CONTRACT_READY':
    raise RuntimeError('C019_REQUIRES_C015')
if C016_LOCAL_RESULTS['quality_gate'] != 'BLOCKED':
    raise RuntimeError('C019_QUALITY_GATE_MUST_REMAIN_BLOCKED')


def c019_require(condition, reason):
    if not condition:
        raise ValueError(reason)


def c019_read_record(path, expected_sha=None):
    raw = path.read_bytes()
    actual = c016_sha(raw)
    c019_require(expected_sha is None or actual == expected_sha,
                 'RECEIPT_FILE_SHA_CONFLICT:' + path.name)
    row = json.loads(raw)
    core = {k: v for k, v in row.items() if k != 'record_core_sha256'}
    c019_require(row.get('record_core_sha256') == c016_sha(c016_canon(core)),
                 'RECEIPT_CORE_SHA_CONFLICT:' + path.name)
    return row, actual


def c019_peak_gib():
    peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    return round(peak / (1024 ** 3 if sys.platform == 'darwin' else 1024 ** 2), 3)


# 1. Freeze and cross-check the actual NB22D and C138 parameters.
c019_nb22d_cell = ''.join(c015_nb22d['cells'][12]['source'])
c019_nb22d_setup = ast.parse(''.join(c015_nb22d['cells'][4]['source']))
c019_config_nodes = [
    n.value for n in c019_nb22d_setup.body
    if isinstance(n, ast.Assign)
    and any(isinstance(t, ast.Name) and t.id == 'CONFIG' for t in n.targets)
]
c019_require(len(c019_config_nodes) == 1, 'NB22D_CONFIG_NOT_UNIQUE')
c019_nb22d_config = ast.literal_eval(c019_config_nodes[0])
c019_runtime_config = {
    key: c019_nb22d_config[key]
    for key in ('char_max_df_fraction', 'bm25', 'rrf_k', 'incumbent_rank_weight')
}
c019_c138_configs = [
    ast.literal_eval(kw.value)
    for node in ast.walk(ast.parse(c015_c138_code))
    if isinstance(node, ast.Call)
    and isinstance(node.func, ast.Attribute) and node.func.attr == 'update'
    for kw in node.keywords if kw.arg == 'CONFIG'
]
c019_require(len(c019_c138_configs) == 1 and
             c019_runtime_config == c019_c138_configs[0],
             'NB22D_C138_FROZEN_CONFIG_CONFLICT')
c019_require(c016_identity['quota_spec'] == {
    'family': 'quota100', 'k': c019_runtime_config['rrf_k'],
    'weight': c019_runtime_config['incumbent_rank_weight'],
    'newcomer_limit': 1
}, 'C016_QUOTA_C138_CONFLICT')

c019_required_functions = (
    'normalized_words', 'char_tokens', 'build_index',
    'lexical_search', 'lexical_fusion'
)
c019_func_nodes = [
    n for n in ast.parse(c019_nb22d_cell).body
    if isinstance(n, ast.FunctionDef) and n.name in c019_required_functions
]
c019_require(len(c019_func_nodes) == len(c019_required_functions)
             and {n.name for n in c019_func_nodes} == set(c019_required_functions),
             'NB22D_FROZEN_FUNCTIONS_MISSING')
c019_require(ast.get_source_segment(
    c019_nb22d_cell, next(n for n in c019_func_nodes if n.name == 'lexical_fusion')
) == c015_lexical_source, 'C015_LEXICAL_SOURCE_CHANGED')
c019_scope = {
    'collections': collections, 'unicodedata': unicodedata,
    're': re, 'math': math, 'CONFIG': c019_runtime_config,
    'require': c019_require
}
exec(compile(ast.Module(body=c019_func_nodes, type_ignores=[]),
             'PINNED_NB22D_C006_FUNCTIONS', 'exec'), c019_scope)
c019_render_scope = {}
exec(compile(ast.Module(body=[c015_render_nodes[0]], type_ignores=[]),
             'PINNED_PIPELINE_BUILD_CONTEXT', 'exec'), c019_render_scope)
c019_render = c019_render_scope['build_context']

# 2. Reuse C016's already verified corpus and C341 question bytes.
c019_chunks_sha = C015_FROZEN_CONTRACT['input_pins']['CHUNKS']['file_bytes_sha256']
c019_require(c016_sha(c016_chunk_raw) == c019_chunks_sha,
             'FROZEN_CORPUS_SHA_CONFLICT')
c019_corpus = {}
for c019_line in c016_chunk_raw.splitlines():
    if c019_line.strip():
        c019_chunk = json.loads(c019_line)
        c019_cid = c019_chunk['chunk_id']
        c019_require(c019_cid not in c019_corpus and
                     c016_sha(c019_chunk['content'].encode('utf-8')) == c016_chunks[c019_cid],
                     'FROZEN_CHUNK_CONTENT_CONFLICT:' + c019_cid)
        c019_corpus[c019_cid] = c019_chunk
c019_require(len(c019_corpus) == len(c016_chunks) == 3488,
             'FROZEN_CORPUS_POPULATION_CONFLICT')
c019_require(c016_sha(c016_questions_raw) == c016_question_sha,
             'C341_QUESTION_SOURCE_SHA_CONFLICT')
c019_questions = {}
for c019_line in c016_questions_raw.splitlines():
    if c019_line.strip():
        c019_row = json.loads(c019_line)
        c019_id, c019_question = c019_row['candidate_id'], c019_row['question']
        c019_require(c019_id not in c019_questions and
                     c016_sha(c019_question.encode('utf-8')) == c016_questions[c019_id],
                     'FROZEN_QUESTION_BINDING_CONFLICT:' + c019_id)
        c019_questions[c019_id] = c019_question
c019_require(set(c019_questions) == set(c015_member_ids),
             'C019_C011_QUESTION_POPULATION_CONFLICT')

# 3. Verify previous manifest and its immutable lineage, without reranking.
c019_c016_summary, c019_c016_summary_sha = c019_read_record(
    Path(C016_LOCAL_RESULTS['summary_path']),
    C016_LOCAL_RESULTS['summary_file_sha256']
)
c019_parent_entries = {
    row['candidate_id']: row for row in c019_c016_summary['per_candidate_status']
}
c019_require(len(c019_parent_entries) == 1116 and
             set(c019_parent_entries) == set(c019_questions) and
             c019_c016_summary['identity_sha256'] == c016_identity_sha and
             c019_c016_summary['status_counts'] ==
             {'DERIVED_QUOTA_READY_CHAR_RRF_BLOCKED': 1116},
             'C019_C016_MANIFEST_CONFLICT')

# 4. New identity and external immutable output directory.
c019_identity = {
    'schema': 'NB22E_V4_C019_SUPERVISOR_B_FROZEN_CHAR_CONTEXT_V1',
    'permission': 'SUPERVISOR_OPTION_B_CPU_CHAR_FOR_C011_1116',
    'c011_boundary_sha256': c016_boundary,
    'c016_summary_file_sha256': c019_c016_summary_sha,
    'c018_handoff_file_sha256': C018_HANDOFF['sha256'],
    'c015_contract_sha256': C015_FROZEN_CONTRACT['rank_and_context_producer_sha256'],
    'nb22d_notebook_code_sha256': c015_nb22d['metadata']['nb22d']['code_source_sha256'],
    'nb22d_functions_sha256': c016_sha(c019_nb22d_cell.encode('utf-8')),
    'c138_evidence_sha256': c015_c138_sha,
    'c139_evidence_sha256': c015_c139_sha,
    'context_pipeline_source_sha256': c015_pipeline_sha,
    'frozen_chunks_file_sha256': c019_chunks_sha,
    'frozen_questions_file_sha256': c016_question_sha,
    'char_config': c019_runtime_config,
    'final_rank_rule': {'channels': ['char'], 'top_k': 10},
}
c019_identity_sha = c016_sha(c016_canon(c019_identity))
c019_dir = (outputs / ('C019_char_rank_context_' + c019_identity_sha)).resolve()
c019_require(c019_dir.parent == outputs.resolve() and
             not c019_dir.is_relative_to(repo.resolve()),
             'C019_OUTPUT_MUST_BE_OUTSIDE_REPOSITORY')
c019_dir.mkdir(parents=True, exist_ok=True)

# 5. Same sequential run: four opening questions, then all C011 batches.
c019_work = [('C009_PILOT', cid) for cid in c015_pilot_ids]
for c019_batch_no, c019_batch in enumerate(manifest['execution_queue']['batches'], 1):
    c019_work.extend((f'C011_BATCH_{c019_batch_no:02}', cid)
                     for cid in c019_batch['candidate_ids'])
c019_require(len(c019_work) == 1116 and
             len({cid for _, cid in c019_work}) == 1116 and
             {cid for _, cid in c019_work} == set(c019_parent_entries),
             'C019_WORKLIST_SCOPE_CONFLICT')

c019_char_index = None
c019_index_seconds = None
c019_elapsed_start = time.monotonic()
c019_peak_start_gib = c019_peak_gib()
c019_counts = collections.Counter()
c019_stages = collections.Counter()
c019_writes = collections.Counter()
c019_reuse = collections.Counter()
c019_manifest_rows = []
c019_stop_reason = None
c019_last_batch = None

for c019_i, (c019_batch_label, cid) in enumerate(c019_work, 1):
    if c019_stop_reason is not None:
        c019_counts['DEFERRED_AFTER_PRECHECK_OR_TIME_CAP'] += 1
        c019_manifest_rows.append({
            'candidate_id': cid, 'batch': c019_batch_label,
            'status': 'DEFERRED_AFTER_PRECHECK_OR_TIME_CAP',
            'blocker': c019_stop_reason
        })
        continue

    if (time.monotonic() - c019_elapsed_start
            >= c019_nb22d_config['maximum_local_seconds']):
        c019_stop_reason = 'FROZEN_NB22D_LOCAL_TIME_CAP_REACHED_RESUME_REQUIRED'
        c019_counts['DEFERRED_AFTER_PRECHECK_OR_TIME_CAP'] += 1
        c019_manifest_rows.append({
            'candidate_id': cid, 'batch': c019_batch_label,
            'status': 'DEFERRED_AFTER_PRECHECK_OR_TIME_CAP',
            'blocker': c019_stop_reason
        })
        continue

    c019_stage = 'C016_BIND'
    c019_key = c016_sha(cid.encode('utf-8'))
    c019_parent = c019_parent_entries[cid]
    c019_parent_path = Path(C016_LOCAL_RESULTS['output_directory']) / c019_parent['receipt']
    c019_char_path = c019_dir / ('C019.' + c019_key + '.char.json')
    c019_rank_path = c019_dir / ('C019.' + c019_key + '.rank.json')
    c019_context_path = c019_dir / ('C019.' + c019_key + '.context.json')
    c019_case_t0 = time.monotonic()
    c019_stage_hashes = {}

    try:
        c019_c016, c019_parent_sha = c019_read_record(
            c019_parent_path, c019_parent['receipt_file_sha256'])
        c019_r = c019_c016['rank_partial']
        c019_quota = c019_r['quota100_order']
        c019_derived = c019_r['derived_order']
        c019_require(
            c019_c016['candidate_id'] == cid
            and c019_c016['question_sha256'] == c016_questions[cid]
            and c019_c016['c011_execution_boundary_sha256'] == c016_boundary
            and c019_c016['c016_identity_sha256'] == c016_identity_sha
            and c019_c016['rank_state'] == 'DERIVED_QUOTA_READY_CHAR_RRF_BLOCKED'
            and len(c019_derived) == len(c019_quota) == 100
            and len(set(c019_quota)) == 100
            and set(c019_quota) == set(c019_derived)
            and all(x in c019_corpus for x in c019_quota)
            and c016_sha(c016_canon(c019_quota)) == c019_r['quota100_order_sha256']
            and c016_sha(c016_canon(c019_derived)) == c019_r['derived_order_sha256']
            and [c016_chunks[x] for x in c019_quota]
                == c019_r['quota100_ordered_chunk_text_sha256'],
            'C016_QUOTA_BINDING_OR_TEXT_SHA_CONFLICT'
        )
        c019_stages['C016_DERIVED_QUOTA_BOUND'] += 1

        c019_stage = 'CHAR_CHANNEL'
        if c019_char_path.exists():
            c019_char, c019_char_sha = c019_read_record(c019_char_path)
            c019_reuse['char'] += 1
        else:
            if c019_char_index is None:
                c019_stage = 'FROZEN_CHAR_INDEX'
                c019_index_t0 = time.monotonic()
                c019_char_index = c019_scope['build_index'](c019_corpus, 'char')
                c019_index_seconds = round(time.monotonic() - c019_index_t0, 3)
                c019_require(set(c019_char_index['ids']) == set(c019_corpus)
                             and len(c019_char_index['ids']) == 3488,
                             'NB22D_INDEX_CORPUS_CONFLICT')
                print('C019_CHAR_INDEX_READY:',
                      'chunks=3488',
                      'seconds=' + str(c019_index_seconds),
                      'peak_rss_gib=' + str(c019_peak_gib()), flush=True)
            c019_stage = 'CHAR_CHANNEL'
            c019_char_ids = c019_scope['lexical_search'](
                c019_char_index, c019_questions[cid], 'char')
            c019_char = {
                'cell': 'NB22E-v4:C019', 'kind': 'CHAR_CHANNEL',
                'producer': 'NEW_CPU_COMPUTATION_FROZEN_NB22D_C006',
                'identity_sha256': c019_identity_sha,
                'candidate_id': cid, 'question_sha256': c016_questions[cid],
                'c016_receipt_file_sha256': c019_parent_sha,
                'frozen_corpus_sha256': c019_chunks_sha,
                'nb22d_code_sha256': c019_identity['nb22d_notebook_code_sha256'],
                'char_ranked_chunk_ids': c019_char_ids,
                'char_order_sha256': c016_sha(c016_canon(c019_char_ids)),
            }
            c019_char_sha, created = c016_put_once(c019_char_path, c019_char)
            c019_writes['char'] += int(created)

        c019_char_ids = c019_char['char_ranked_chunk_ids']
        c019_require(
            c019_char['identity_sha256'] == c019_identity_sha
            and c019_char['candidate_id'] == cid
            and c019_char['question_sha256'] == c016_questions[cid]
            and c019_char['c016_receipt_file_sha256'] == c019_parent_sha
            and c019_char['producer'] == 'NEW_CPU_COMPUTATION_FROZEN_NB22D_C006'
            and len(c019_char_ids) <= 800
            and len(c019_char_ids) == len(set(c019_char_ids))
            and set(c019_char_ids).issubset(c019_corpus)
            and c016_sha(c016_canon(c019_char_ids)) == c019_char['char_order_sha256'],
            'FROZEN_CHAR_CHANNEL_RECEIPT_CONFLICT'
        )
        c019_stage_hashes['char'] = c019_char_sha
        c019_stages['CHAR_CHANNEL_VALID'] += 1

        c019_stage = 'FINAL_CHAR_RRF_RANK'
        if c019_rank_path.exists():
            c019_rank, c019_rank_sha = c019_read_record(c019_rank_path)
            c019_reuse['rank'] += 1
        else:
            c019_final_ids = c019_scope['lexical_fusion'](
                c019_quota, {'char': c019_char_ids}, {'channels': ['char']})
            c019_qpos = {x: i for i, x in enumerate(c019_quota, 1)}
            c019_cpos = {x: i for i, x in enumerate(c019_char_ids, 1)}
            c019_top10_evidence = []
            for c019_final_pos, c019_chunk_id in enumerate(c019_final_ids[:10], 1):
                qpos = c019_qpos.get(c019_chunk_id)
                cpos = c019_cpos.get(c019_chunk_id)
                qterm = (c019_runtime_config['incumbent_rank_weight'] /
                         (c019_runtime_config['rrf_k'] + qpos)) if qpos else 0.0
                cterm = ((1 - c019_runtime_config['incumbent_rank_weight']) /
                         (c019_runtime_config['rrf_k'] + cpos)) if cpos else 0.0
                c019_top10_evidence.append({
                    'chunk_id': c019_chunk_id,
                    'final_rank': c019_final_pos,
                    'quota_rank': qpos, 'char_rank': cpos,
                    'quota_term': qterm, 'char_term': cterm,
                    'fused_score': qterm + cterm,
                    'chunk_text_sha256': c016_chunks[c019_chunk_id]
                })
            c019_rank = {
                'cell': 'NB22E-v4:C019', 'kind': 'FINAL_RANK',
                'identity_sha256': c019_identity_sha, 'candidate_id': cid,
                'question_sha256': c016_questions[cid],
                'c016_receipt_file_sha256': c019_parent_sha,
                'char_receipt_file_sha256': c019_char_sha,
                'quota100_order_sha256': c019_r['quota100_order_sha256'],
                'char_order_sha256': c019_char['char_order_sha256'],
                'selected_variant': 'char_rrf',
                'final_ranked_chunk_ids': c019_final_ids,
                'final_ranked_chunk_ids_sha256': c016_sha(c016_canon(c019_final_ids)),
                'top10_ordered_evidence': c019_top10_evidence,
                'top10_text_sha256': [c016_chunks[x] for x in c019_final_ids[:10]]
            }
            c019_rank_sha, created = c016_put_once(c019_rank_path, c019_rank)
            c019_writes['rank'] += int(created)

        c019_final_ids = c019_rank['final_ranked_chunk_ids']
        c019_require(
            c019_rank['identity_sha256'] == c019_identity_sha
            and c019_rank['candidate_id'] == cid
            and c019_rank['question_sha256'] == c016_questions[cid]
            and c019_rank['c016_receipt_file_sha256'] == c019_parent_sha
            and c019_rank['char_receipt_file_sha256'] == c019_char_sha
            and c019_rank['quota100_order_sha256'] == c019_r['quota100_order_sha256']
            and c019_rank['char_order_sha256'] == c019_char['char_order_sha256']
            and len(c019_final_ids) == len(set(c019_final_ids)) >= 100
            and set(c019_final_ids) == set(c019_quota) | set(c019_char_ids)
            and c016_sha(c016_canon(c019_final_ids)) == c019_rank['final_ranked_chunk_ids_sha256']
            and len(c019_rank['top10_ordered_evidence']) == 10
            and [x['chunk_id'] for x in c019_rank['top10_ordered_evidence']]
                == c019_final_ids[:10]
            and [c016_chunks[x] for x in c019_final_ids[:10]]
                == c019_rank['top10_text_sha256'],
            'FINAL_FROZEN_RANK_OR_LINEAGE_CONFLICT'
        )
        c019_stage_hashes['rank'] = c019_rank_sha
        c019_stages['FINAL_RANK_VALID'] += 1

        c019_stage = 'CONTEXT_RENDER'
        if c019_context_path.exists():
            c019_context, c019_context_sha = c019_read_record(c019_context_path)
            c019_reuse['context'] += 1
        else:
            c019_top_chunks = [
                {'chunk_id': x, 'text': c019_corpus[x]['content'],
                 'title': c019_corpus[x].get('title'),
                 'url': c019_corpus[x].get('url')}
                for x in c019_final_ids[:10]
            ]
            c019_rendered = c019_render(None, c019_top_chunks)
            c019_context = {
                'cell': 'NB22E-v4:C019', 'kind': 'RENDERED_CONTEXT',
                'identity_sha256': c019_identity_sha, 'candidate_id': cid,
                'question_sha256': c016_questions[cid],
                'rank_receipt_file_sha256': c019_rank_sha,
                'selected_variant': 'char_rrf',
                'ordered_chunk_ids': c019_final_ids[:10],
                'ordered_chunk_text_sha256': c019_rank['top10_text_sha256'],
                'rendered_context': c019_rendered,
                'rendered_context_sha256': c016_sha(c019_rendered.encode('utf-8')),
                'context_renderer_source_sha256': c015_pipeline_sha,
                'truncation_policy': 'EXACT_PINNED_BUILD_CONTEXT_NO_EXPLICIT_TRUNCATION',
                'generation_send_authorized': False
            }
            c019_context_sha, created = c016_put_once(c019_context_path, c019_context)
            c019_writes['context'] += int(created)

        c019_require(
            c019_context['identity_sha256'] == c019_identity_sha
            and c019_context['candidate_id'] == cid
            and c019_context['question_sha256'] == c016_questions[cid]
            and c019_context['rank_receipt_file_sha256'] == c019_rank_sha
            and c019_context['context_renderer_source_sha256'] == c015_pipeline_sha
            and c019_context['ordered_chunk_ids'] == c019_final_ids[:10]
            and c019_context['ordered_chunk_text_sha256'] == c019_rank['top10_text_sha256']
            and c016_sha(c019_context['rendered_context'].encode('utf-8'))
                == c019_context['rendered_context_sha256']
            and c019_context['generation_send_authorized'] is False,
            'RENDERED_CONTEXT_OR_LINEAGE_CONFLICT'
        )
        c019_stage_hashes['context'] = c019_context_sha
        c019_stages['CONTEXT_RENDERED'] += 1

        c019_counts['CONTEXT_PREPARED'] += 1
        c019_manifest_rows.append({
            'candidate_id': cid, 'batch': c019_batch_label,
            'status': 'CONTEXT_PREPARED',
            'question_sha256': c016_questions[cid],
            'scoring_receipt_file_sha256': c019_c016['score_receipt_file_sha256'],
            'c016_quota_receipt_file_sha256': c019_parent_sha,
            'char_receipt': c019_char_path.name, 'char_receipt_sha256': c019_char_sha,
            'rank_receipt': c019_rank_path.name, 'rank_receipt_sha256': c019_rank_sha,
            'context_receipt': c019_context_path.name,
            'context_receipt_sha256': c019_context_sha,
            'final_rank_sha256': c019_rank['final_ranked_chunk_ids_sha256'],
            'rendered_context_sha256': c019_context['rendered_context_sha256'],
            'generation_preparation_input_valid': True,
            'generation_send_authorized': False
        })

    except Exception as error:
        c019_reason = f'{c019_stage}:{type(error).__name__}:{error}'
        c019_counts['BLOCKED'] += 1
        c019_blocker = {
            'cell': 'NB22E-v4:C019', 'kind': 'BLOCKER',
            'identity_sha256': c019_identity_sha,
            'candidate_id': cid, 'question_sha256': c016_questions[cid],
            'stage': c019_stage, 'reason': c019_reason,
            'verified_partial_receipts': c019_stage_hashes,
            'generation_send_authorized': False
        }
        c019_blocker_path = c019_dir / (
            'C019.' + c019_key + '.blocker.'
            + c016_sha(c016_canon(c019_blocker)) + '.json'
        )
        c019_blocker_sha, created = c016_put_once(c019_blocker_path, c019_blocker)
        c019_writes['blocker'] += int(created)
        c019_manifest_rows.append({
            'candidate_id': cid, 'batch': c019_batch_label,
            'status': 'BLOCKED', 'blocker': c019_reason,
            'blocker_receipt': c019_blocker_path.name,
            'blocker_receipt_sha256': c019_blocker_sha,
            'partial_receipt_hashes': c019_stage_hashes
        })

    if c019_i == 4:
        c019_precheck_pass = c019_counts['CONTEXT_PREPARED'] == 4
        print('C019_FIRST4:', 'PASS' if c019_precheck_pass else 'BLOCKED',
              'prepared=' + str(c019_counts['CONTEXT_PREPARED']) + '/4',
              'seconds=' + str(round(time.monotonic() - c019_elapsed_start, 2)),
              'peak_rss_gib=' + str(c019_peak_gib()), flush=True)
        if not c019_precheck_pass:
            c019_stop_reason = 'FIRST_FOUR_PRODUCER_SHAPE_OR_LINEAGE_PRECHECK_FAILED'

    if (c019_i == 4 or c019_batch_label != c019_last_batch
            or c019_i == len(c019_work)):
        if c019_i > 4 and c019_batch_label != c019_last_batch:
            print('C019_BATCH:', c019_batch_label,
                  'processed=' + str(c019_i) + '/1116',
                  'context_ready=' + str(c019_counts['CONTEXT_PREPARED']),
                  'blocked=' + str(c019_counts['BLOCKED']),
                  'peak_rss_gib=' + str(c019_peak_gib()), flush=True)
        c019_last_batch = c019_batch_label

# 6. Persist complete coverage and linked hashes, then reload.
c019_require(len(c019_manifest_rows) == 1116 and
             len({x['candidate_id'] for x in c019_manifest_rows}) == 1116,
             'C019_INCOMPLETE_MANIFEST_COVERAGE')
c019_ready = c019_counts['CONTEXT_PREPARED']
c019_manifest = {
    'cell': 'NB22E-v4:C019', 'kind': 'POPULATION_LINEAGE_MANIFEST',
    'identity': c019_identity, 'identity_sha256': c019_identity_sha,
    'population_count': 1116,
    'scope': 'C011_ONLY_4_C009_PILOTS_PLUS_1112_C012',
    'status_counts': dict(c019_counts),
    'stage_coverage': dict(c019_stages),
    'entries': c019_manifest_rows,
    'generation_preparation_input_ready_count': c019_ready,
    'generation_send_authorized': False, 'quality_gate': 'BLOCKED'
}
c019_manifest_path = c019_dir / ('C019.manifest.' + c016_sha(c016_canon(c019_manifest)) + '.json')
c019_manifest_sha, _ = c016_put_once(c019_manifest_path, c019_manifest)
c019_reloaded_manifest, _ = c019_read_record(c019_manifest_path, c019_manifest_sha)
c019_require(c019_reloaded_manifest['entries'] == c019_manifest_rows,
             'C019_MANIFEST_RELOAD_CONFLICT')

c019_summary = {
    'cell': 'NB22E-v4:C019',
    'status': 'CONTEXT_PREPARED_ALL' if c019_ready == 1116 else 'PARTIAL_WITH_EXPLICIT_BLOCKERS',
    'c011_boundary_sha256': c016_boundary,
    'identity_sha256': c019_identity_sha,
    'population_count': 1116,
    'frozen_scoring_and_quota_count': 1116,
    'char_receipts_valid_and_context_prepared_count': c019_ready,
    'generation_preparation_input_ready_count': c019_ready,
    'generation_send_authorized': False,
    'quality_gate': 'BLOCKED',
    'counts': dict(c019_counts),
    'stage_coverage': dict(c019_stages),
    'new_artifacts': dict(c019_writes),
    'reused_artifacts': dict(c019_reuse),
    'first_four_precheck_pass': c019_counts['CONTEXT_PREPARED'] >= 4,
    'char_index_build_seconds': c019_index_seconds,
    'elapsed_seconds': round(time.monotonic() - c019_elapsed_start, 2),
    'rss_peak_start_gib': c019_peak_start_gib,
    'rss_peak_end_gib': c019_peak_gib(),
    'open_items': [
        {'candidate_id': row['candidate_id'], 'reason': row.get('blocker')}
        for row in c019_manifest_rows if row['status'] != 'CONTEXT_PREPARED'
    ],
    'manifest_path': str(c019_manifest_path),
    'manifest_file_sha256': c019_manifest_sha,
    'next_required_checks': [
        'GENERATION_MODEL_AND_REQUEST_CONTRACT',
        'CACHE_AND_ATTEMPT_CONTROL',
        'RELEVANT_QUALITY_EVALUATION'
    ],
    'new_model_calls': 0, 'new_dense_queries': 0,
    'new_api_calls': 0, 'new_generation_requests': 0
}
c019_summary_path = c019_dir / ('C019.summary.' + c016_sha(c016_canon(c019_summary)) + '.json')
c019_summary_sha, _ = c016_put_once(c019_summary_path, c019_summary)
c019_reloaded_summary, _ = c019_read_record(c019_summary_path, c019_summary_sha)
c019_require(c019_reloaded_summary['manifest_file_sha256'] == c019_manifest_sha,
             'C019_SUMMARY_RELOAD_CONFLICT')

C019_LOCAL_RESULTS = {
    'status': c019_summary['status'],
    'output_directory': str(c019_dir),
    'manifest_path': str(c019_manifest_path),
    'manifest_sha256': c019_manifest_sha,
    'summary_path': str(c019_summary_path),
    'summary_sha256': c019_summary_sha,
    'context_prepared': c019_ready,
    'open_count': 1116 - c019_ready,
    'generation_send_authorized': False,
    'quality_gate': 'BLOCKED'
}
print('C019_COUNTS:', dict(c019_counts))
print('C019_STAGE_COVERAGE:', dict(c019_stages))
print('C019_NEW_RECEIPTS:', dict(c019_writes))
print('C019_REUSED_RECEIPTS:', dict(c019_reuse))
print('C019_CONTEXT_PREPARED:', str(c019_ready) + '/1116')
print('C019_MANIFEST_SHA256:', c019_manifest_sha)
print('C019_SUMMARY_SHA256:', c019_summary_sha)
print('C019_STATUS:', C019_LOCAL_RESULTS['status'])
print('GENERATION_SEND_AUTHORIZED: False | quality_gate: BLOCKED')

C019_CHAR_INDEX_READY: chunks=3488 seconds=2.454 peak_rss_gib=0.29
C019_FIRST4: PASS prepared=4/4 seconds=2.54 peak_rss_gib=0.291
C019_BATCH: C011_BATCH_01 processed=5/1116 context_ready=5 blocked=0 peak_rss_gib=0.291
C019_BATCH: C011_BATCH_02 processed=30/1116 context_ready=30 blocked=0 peak_rss_gib=0.291
C019_BATCH: C011_BATCH_03 processed=55/1116 context_ready=55 blocked=0 peak_rss_gib=0.291
C019_BATCH: C011_BATCH_04 processed=80/1116 context_ready=80 blocked=0 peak_rss_gib=0.291
C019_BATCH: C011_BATCH_05 processed=105/1116 context_ready=105 blocked=0 peak_rss_gib=0.291
C019_BATCH: C011_BATCH_06 processed=130/1116 context_ready=130 blocked=0 peak_rss_gib=0.291
C019_BATCH: C011_BATCH_07 processed=155/1116 context_ready=155 blocked=0 peak_rss_gib=0.291
C019_BATCH: C011_BATCH_08 processed=180/1116 context_ready=180 blocked=0 peak_rss_gib=0.291
C019_BATCH: C011_BATCH_09 processed=205/1116 context_ready=205 blocked=0 peak_rss_gib=0.291
C019_BATCH: C011_BATCH_10 processed=230/1116 context

<div dir="rtl" align="right">

### NB22E-v4:C020 — آماده‌سازی آفلاین Generation، Cache و Attempt

**هدف:** ساخت بستهٔ کامل درخواست‌های آفلاین برای ۱۱۱۶ Context معتبر C019، بدون ارسال هیچ درخواست.

**هویت فریز‌شده:**
- Prompt و قالب مرجع از C340 با کنترل SHA
- مدل تاریخی: DeepSeek-V4-Flash
- `temperature = 0`
- `max_tokens = 600`
- `thinking = disabled`
- مرز C011 و هویت Manifest و Summary مربوط به C019

**مراحل:**

۱. کنترل Manifest، Summary و رسیدهای Rank و Context، بدون اجرای دوبارهٔ Pipeline.

۲. ساخت Payload آفلاین هر سؤال با Prompt فریز‌شده، متن سؤال و Context رندرشده.

۳. ساخت هویت مستقل درخواست بر اساس نسخهٔ قرارداد، مدل، تنظیمات، سؤال و SHAهای Context.

۴. بررسی Cache و Attempt فقط در دامنهٔ هویت جدید. نتیجهٔ ناسازگار یا تلاش با پایان نامعلوم Held می‌ماند و Retry خودکار نمی‌شود.

۵. نگهداری مستقل تاریخچه: ۲۹ نتیجهٔ محفوظ، دو هویت نامحرز و دو خطای قبلی. هیچ‌یک خودکار وارد جمعیت C011 نمی‌شوند.

۶. اندازه‌گیری حجم دقیق UTF-8 پیام‌ها و ثبت برآورد تقریبی توکن؛ تا زمان احراز Tokenizer و Context Window، کنترل قطعی ظرفیت PASS نمی‌شود.

۷. ثبت برآورد هزینهٔ مشروط، طرح Pilot سه‌سؤالی با انتخاب مستقل از Gold، Manifest درخواست‌ها، دفتر Cache/Attempt و Summary.

**شرط پایان:** هر ۱۱۱۶ سؤال دارای درخواست آفلاین معتبر یا مانع مشخص باشد. Artifactهای جدید خارج از مخزن ذخیره و بازخوانی شوند.

**ممنوعیت‌ها:** Generator، شبکه، API، ارسال Pilot، تغییر Git، تغییر Gold، Scoring و Retrieval.

**وضعیت:** `quality_gate = BLOCKED` و `dispatch_authorized = false`.

</div>

In [14]:
# NB22E-v4 | C020 | R11.2
# Offline generation envelope / cache-attempt ledger / conditional budget.
# No generator, network, model, retrieval, or repository writes.

if C019_LOCAL_RESULTS['status'] != 'CONTEXT_PREPARED_ALL':
    raise RuntimeError('C020_REQUIRES_C019_ALL')
if C019_LOCAL_RESULTS['quality_gate'] != 'BLOCKED':
    raise RuntimeError('C020_QUALITY_GATE_CONFLICT')


def c020_need(ok, reason):
    if not ok:
        raise RuntimeError('C020_' + reason)


def c020_load_core(path, digest):
    raw = Path(path).read_bytes()
    c020_need(c016_sha(raw) == digest, 'FILE_SHA_CONFLICT:' + Path(path).name)
    record = json.loads(raw)
    core = {k: v for k, v in record.items() if k != 'record_core_sha256'}
    c020_need(record.get('record_core_sha256') == c016_sha(c016_canon(core)),
              'CORE_SHA_CONFLICT:' + Path(path).name)
    return record


# A. Verify C019 immutable summary and manifest; consume, never regenerate.
c020_summary = c020_load_core(C019_LOCAL_RESULTS['summary_path'],
                              C019_LOCAL_RESULTS['summary_sha256'])
c020_manifest = c020_load_core(C019_LOCAL_RESULTS['manifest_path'],
                               C019_LOCAL_RESULTS['manifest_sha256'])
c020_need(c020_summary['status'] == 'CONTEXT_PREPARED_ALL'
          and c020_summary['manifest_file_sha256'] == C019_LOCAL_RESULTS['manifest_sha256']
          and c020_manifest['identity_sha256'] == c020_summary['identity_sha256']
          and c020_manifest['identity']['c011_boundary_sha256'] == c016_boundary
          and c020_manifest['population_count'] == 1116
          and c020_summary['generation_send_authorized'] is False
          and c020_summary['quality_gate'] == 'BLOCKED',
          'C019_FROZEN_LINEAGE_CONFLICT')
c020_prior_rows = c020_manifest['entries']
c020_need(len(c020_prior_rows) == 1116 and
          len({row['candidate_id'] for row in c020_prior_rows}) == 1116 and
          {row['candidate_id'] for row in c020_prior_rows} == set(c015_member_ids) and
          all(row['status'] == 'CONTEXT_PREPARED' for row in c020_prior_rows),
          'C019_COVERAGE_CONFLICT')

# B. Reopen exact frozen C340 historical template; do not inherit its target ID.
c020_template_sha = 'f72301a2309c5c66aa84b0a5ba3eda96dcee703e1d93d67c9df71e40942240d9'
c020_template_path = (repo.parent / '_RAG_finance_C340_preflight_outputs'
                      / ('C340.frozen_template.' + c020_template_sha + '.json')).resolve()
c020_need(c020_template_path.parent ==
          (repo.parent / '_RAG_finance_C340_preflight_outputs').resolve(),
          'TEMPLATE_PATH_ESCAPE')
c020_template_raw = c015_read_pinned(c020_template_path, c020_template_sha)
c020_need(len(c020_template_raw) == 47393, 'TEMPLATE_SIZE_CONFLICT')
c020_template = json.loads(c020_template_raw)
c020_ref = c020_template['reference_request']
c020_target = c020_template['target_identity']
c020_profile = {k: v for k, v in c020_ref.items() if k != 'messages'}
c020_need(c020_profile == {'model': 'DeepSeek-V4-Flash', 'temperature': 0,
                           'max_tokens': 600, 'thinking': {'type': 'disabled'}}
          and c020_target['profile'] == c020_profile
          and len(c020_ref['messages']) == 2
          and [m['role'] for m in c020_ref['messages']] == ['system', 'user']
          and c020_template['target_identity_sha256'] ==
              c016_sha(c016_canon(c020_target)) ==
              '8d5875bfee0cfe5f2dc0ce9bdf67e5f14301191b60891047a630c3fe65442065',
          'C340_FROZEN_PROFILE_OR_IDENTITY_CONFLICT')
c020_system = c020_ref['messages'][0]['content']
c020_need(isinstance(c020_system, str)
          and c016_sha(c020_system.encode('utf-8')) ==
              c020_template['system_prompt_sha256'] == c020_target['system_prompt_sha256'],
          'FROZEN_SYSTEM_PROMPT_SHA_CONFLICT')
c020_ref_user = c020_ref['messages'][1]['content']
c020_need(isinstance(c020_ref_user, str), 'REFERENCE_USER_MESSAGE_NOT_TEXT')

# Preserve exact historical punctuation and whitespace around variable spans.
c020_wrapper = re.fullmatch(
    r'(?s)(\s*سؤال:\n)(.+?)(\n\nمنابع:\n)(.*?)(\n\nپاسخ:\s*)',
    c020_ref_user
)
c020_wrapper_ok = c020_wrapper is not None
if c020_wrapper_ok:
    c020_parts = [c020_wrapper.group(1), c020_wrapper.group(3), c020_wrapper.group(5)]
    c020_need(bool(c020_wrapper.group(2).strip()) and
              bool(c020_wrapper.group(4).strip()), 'C340_REFERENCE_FIELDS_EMPTY')
else:
    c020_parts = None

# C. Record the effective mismatch in existing Generator, without calling it.
c020_generator_source = (repo / 'src/generation/arvan_generator.py').read_bytes()
c020_generator_text = c020_generator_source.decode('utf-8')
c020_generator_mismatch = (
    'Qwen3-30B-A3B' in c020_generator_text
    and 'temperature: float = 0.2' in c020_generator_text
    and 'max_tokens=' not in c020_generator_text
    and 'thinking=' not in c020_generator_text
)
c020_need(c020_generator_mismatch, 'LEGACY_GENERATOR_DIFFERENCE_NOT_PROVEN')
c020_legacy_sha = c016_sha(c020_generator_source)

# Never infer the Provider ID from the historical display model.
c020_provider = {
    'model_id': None,
    'route_identity_sha256': None,
    'context_window_tokens': None,
    'tokenizer_identity_sha256': None,
    'arvan_tariff': None,
    'dispatch_verified': False
}

# D. Historical jobs are evidence only; no cross-version cache matching.
c020_c342_sha = 'fe34926715265f4595dc10084ab18a3dcaca0fbbaf72b37de323466648616b3b'
c020_c342_path = repo.parent / '_RAG_finance_C342_rank_route_preflight_outputs' / (
    'C342.summary.' + c020_c342_sha + '.json')
c020_historical = json.loads(c015_read_pinned(c020_c342_path, c020_c342_sha))
c020_need(c020_historical['preserved'] == {
    'EXACT_PRESERVED': 29, 'MODEL_IDENTITY_HELD': 2,
    'RECORDED_ERROR_HELD': 2, 'OUTSIDE_PACKET': 1116
}, 'HISTORICAL_SEPARATION_CONFLICT')

# E. Versioned C020 identity and external output directories.
c020_identity = {
    'schema': 'NB22E_V4_C020_OFFLINE_GENERATION_PREPARATION_V1',
    'c011_boundary_sha256': c016_boundary,
    'c019_manifest_file_sha256': C019_LOCAL_RESULTS['manifest_sha256'],
    'c019_summary_file_sha256': C019_LOCAL_RESULTS['summary_sha256'],
    'c340_template_file_sha256': c020_template_sha,
    'system_prompt_sha256': c020_template['system_prompt_sha256'],
    'frozen_profile': c020_profile,
    'legacy_generator_source_sha256': c020_legacy_sha,
    'provider_model_binding': 'NOT_VERIFIED_NO_ALIAS_INFERRED',
    'context_window': 'NOT_VERIFIED',
    'retry_policy': 'ZERO_AUTOMATIC_RETRIES_UNKNOWN_ATTEMPT_HELD',
    'historical_target_identity_reused': False
}
c020_identity_sha = c016_sha(c016_canon(c020_identity))
c020_dir = (outputs / ('C020_offline_generation_' + c020_identity_sha)).resolve()
c020_need(c020_dir.parent == outputs.resolve()
          and not c020_dir.is_relative_to(repo.resolve()), 'OUTPUT_NOT_EXTERNAL')
c020_dir.mkdir(parents=True, exist_ok=True)
c020_request_dir = c020_dir / 'requests'
c020_request_dir.mkdir(exist_ok=True)

# Reserved for future explicit execution only; C020 never creates attempt events.
c020_attempt_dir = c020_dir / 'attempt_events'
c020_cache_dir = c020_dir / 'completed_cache'

# F. Read-only question bindings.
c020_need(set(c019_questions) == set(c015_member_ids), 'QUESTION_COVERAGE_CONFLICT')
c020_count = collections.Counter()
c020_bytes = collections.Counter()
c020_ledger = []
c020_blockers = []
c020_pilot_candidates = []
c020_request_writes = 0
c020_stale_cache = 0
c020_cache_reusable = 0
c020_held_unknown = 0
c020_max_request_bytes = 0

for c020_i, c020_case in enumerate(c020_prior_rows, 1):
    cid = c020_case['candidate_id']
    candidate_key = c016_sha(cid.encode('utf-8'))
    c020_case_stage = 'C019_CONTEXT_BIND'
    c020_problem = None
    c020_request_id = None
    c020_request_name = None
    c020_cached = False
    c020_attempt_state = 'NO_PREVIOUS_ATTEMPT_FOR_NEW_IDENTITY'
    c020_measured = None

    try:
        c020_need(c020_case['question_sha256'] == c016_questions[cid]
                  and c020_case['generation_send_authorized'] is False,
                  'QUESTION_OR_GENERATION_GATE_CONFLICT')

        c020_context_path = (
            Path(C019_LOCAL_RESULTS['output_directory'])
            / c020_case['context_receipt']
        )
        c020_rank_path = (
            Path(C019_LOCAL_RESULTS['output_directory'])
            / c020_case['rank_receipt']
        )

        c020_context = c020_load_core(
            c020_context_path, c020_case['context_receipt_sha256']
        )
        c020_rank = c020_load_core(
            c020_rank_path, c020_case['rank_receipt_sha256']
        )
        c020_rendered = c020_context['rendered_context']

        c020_need(
            c020_context['candidate_id'] == cid
            and c020_rank['candidate_id'] == cid
            and c020_context['rank_receipt_file_sha256']
                == c020_case['rank_receipt_sha256']
            and c020_context['rendered_context_sha256']
                == c020_case['rendered_context_sha256']
            and c016_sha(c020_rendered.encode('utf-8'))
                == c020_case['rendered_context_sha256']
            and c020_context['ordered_chunk_ids']
                == c020_rank['final_ranked_chunk_ids'][:10]
            and c020_context['ordered_chunk_text_sha256']
                == c020_rank['top10_text_sha256']
            and c020_context['question_sha256']
                == c020_case['question_sha256'],
            'CONTEXT_RANK_SHA_OR_ORDER_CONFLICT'
        )

        c020_count['context_verified'] += 1
        c020_case_stage = 'C340_WRAPPER_BIND'
        c020_need(c020_wrapper_ok, 'FROZEN_USER_WRAPPER_NOT_PROVEN')

        c020_question = c019_questions[cid]
        c020_user = (
            c020_parts[0] + c020_question + c020_parts[1]
            + c020_rendered + c020_parts[2]
        )
        c020_payload = {
            'model': c020_profile['model'],
            'temperature': c020_profile['temperature'],
            'max_tokens': c020_profile['max_tokens'],
            'thinking': c020_profile['thinking'],
            'messages': [
                {'role': 'system', 'content': c020_system},
                {'role': 'user', 'content': c020_user}
            ]
        }

        c020_need(
            set(c020_payload) == set(c020_ref)
            and [m['role'] for m in c020_payload['messages']]
                == ['system', 'user'],
            'PAYLOAD_PROFILE_CONFLICT'
        )

        c020_payload_sha = c016_sha(c016_canon(c020_payload))
        c020_request_id = c016_sha(c016_canon({
            'c020_identity_sha256': c020_identity_sha,
            'candidate_id': cid,
            'question_sha256': c016_questions[cid],
            'context_receipt_file_sha256': c020_case['context_receipt_sha256'],
            'context_sha256': c020_case['rendered_context_sha256'],
            'payload_sha256': c020_payload_sha
        }))

        c020_request_name = (
            'C020.' + candidate_key + '.'
            + c020_request_id + '.request.json'
        )
        c020_request_path = c020_request_dir / c020_request_name

        c020_req_doc = {
            'cell': 'NB22E-v4:C020',
            'kind': 'OFFLINE_REQUEST',
            'identity_sha256': c020_identity_sha,
            'request_id': c020_request_id,
            'candidate_id': cid,
            'question_sha256': c016_questions[cid],
            'c019_context_receipt_sha256':
                c020_case['context_receipt_sha256'],
            'c019_rank_receipt_sha256':
                c020_case['rank_receipt_sha256'],
            'c019_score_receipt_sha256':
                c020_case['scoring_receipt_file_sha256'],
            'c019_quota_receipt_sha256':
                c020_case['c016_quota_receipt_file_sha256'],
            'rendered_context_sha256':
                c020_case['rendered_context_sha256'],
            'payload_sha256': c020_payload_sha,
            'payload': c020_payload,
            'provider_binding_verified': False,
            'context_window_verified': False,
            'dispatch_authorized': False,
            'quality_gate': 'BLOCKED'
        }

        request_sha, created = c016_put_once(
            c020_request_path, c020_req_doc
        )
        c020_request_writes += int(created)

        c020_need(
            c020_load_core(c020_request_path, request_sha)['payload_sha256']
                == c020_payload_sha,
            'REQUEST_RELOAD_CONFLICT'
        )
        c020_count['offline_request_prepared'] += 1

        c020_case_stage = 'CACHE_ATTEMPT_RECONCILIATION'

        # No attempt files are created or changed.
        # Unknown attempts remain held.
        attempt_path = c020_attempt_dir / (c020_request_id + '.json')
        cache_path = c020_cache_dir / (c020_request_id + '.json')

        if attempt_path.exists():
            attempt = json.loads(attempt_path.read_bytes())

            c020_need(
                attempt.get('request_id') == c020_request_id
                and attempt.get('payload_sha256') == c020_payload_sha,
                'EXISTING_ATTEMPT_IDENTITY_CONFLICT'
            )

            c020_attempt_state = attempt.get('status', 'UNKNOWN')

            if c020_attempt_state in (
                'STARTED', 'SENT', 'TIMEOUT_UNKNOWN',
                'UNKNOWN', 'INTERRUPTED', 'UNCERTAIN'
            ):
                c020_held_unknown += 1
                c020_count['unknown_attempt_held'] += 1
            elif c020_attempt_state not in (
                'COMPLETED', 'FAILED_FINAL', 'NOT_STARTED'
            ):
                raise ValueError(
                    'UNKNOWN_ATTEMPT_STATE:' + str(c020_attempt_state)
                )

        if cache_path.exists():
            cache = json.loads(cache_path.read_bytes())

            c020_cached = (
                cache.get('request_id') == c020_request_id
                and cache.get('payload_sha256') == c020_payload_sha
                and cache.get('context_sha256')
                    == c020_case['rendered_context_sha256']
                and cache.get('generation_profile') == c020_profile
                and isinstance(cache.get('provider_model_id'), str)
                and bool(cache.get('provider_route_sha256'))
                and isinstance(cache.get('answer'), str)
                and c016_sha(cache['answer'].encode('utf-8'))
                    == cache.get('answer_sha256')
            )

            # A recorded Provider name without a verified binding
            # cannot certify compatibility with the new release.
            if c020_cached and c020_provider['dispatch_verified']:
                c020_cache_reusable += 1
                c020_count['compatible_cache_reused'] += 1
            else:
                c020_cached = False
                c020_stale_cache += 1
                c020_count['cache_unverified_held'] += 1

        c020_input_text_bytes = (
            len(c020_system.encode('utf-8'))
            + len(c020_user.encode('utf-8'))
        )
        c020_msg_bytes = len(c016_canon(c020_payload['messages']))
        c020_payload_bytes = len(c016_canon(c020_payload))

        c020_measured = {
            'context_utf8_bytes': len(c020_rendered.encode('utf-8')),
            'system_utf8_bytes': len(c020_system.encode('utf-8')),
            'user_utf8_bytes': len(c020_user.encode('utf-8')),
            'messages_json_utf8_bytes': c020_msg_bytes,
            'request_json_utf8_bytes': c020_payload_bytes,
            'text_bytes_without_transport_overhead': c020_input_text_bytes,
            'token_proxy_low_at_4_bytes_per_token':
                math.ceil(c020_msg_bytes / 4),
            'token_proxy_high_at_2_bytes_per_token':
                math.ceil(c020_msg_bytes / 2),
            'actual_provider_tokens': None,
            'requested_max_output_tokens': 600,
            'context_window_preflight': 'NOT_VERIFIED'
        }

        for key, val in c020_measured.items():
            if isinstance(val, int):
                c020_bytes[key] += val

        c020_max_request_bytes = max(
            c020_max_request_bytes, c020_payload_bytes
        )

        c020_count['provider_binding_pending'] += 1
        c020_count['context_window_pending'] += 1

        c020_pilot_candidates.append((
            c016_sha((c016_boundary + cid).encode('utf-8')),
            cid,
            c020_request_id
        ))

        c020_ledger.append({
            'candidate_id': cid,
            'request_id': c020_request_id,
            'request_file': c020_request_name,
            'request_file_sha256': request_sha,
            'status': 'OFFLINE_READY_DISPATCH_GATED',
            'cache_status': (
                'VERIFIED_REUSED' if c020_cached else
                'UNVERIFIED_HELD' if cache_path.exists() else
                'NO_MATCHING_SCOPED_CACHE'
            ),
            'attempt_status': c020_attempt_state,
            'retry_authorized': False,
            'dispatch_authorized': False,
            'token_measurement': c020_measured
        })

    except Exception as exc:
        c020_problem = (
            c020_case_stage + ':' + type(exc).__name__ + ':' + str(exc)
        )
        c020_count['case_blocked'] += 1
        c020_blockers.append({
            'candidate_id': cid,
            'reason': c020_problem
        })
        c020_ledger.append({
            'candidate_id': cid,
            'status': 'OFFLINE_PREPARATION_BLOCKED',
            'blocker': c020_problem,
            'request_id': c020_request_id,
            'attempt_status': c020_attempt_state,
            'dispatch_authorized': False,
            'retry_authorized': False
        })

    if c020_i % 200 == 0 or c020_i == 1116:
        print(
            'C020_PROGRESS:',
            str(c020_i) + '/1116',
            flush=True
        )

# G. Immutable ledger, small Pilot plan and conditional cost.
c020_need(
    len(c020_ledger) == 1116
    and len({v['candidate_id'] for v in c020_ledger}) == 1116,
    'LEDGER_COVERAGE_CONFLICT'
)

c020_ledger_document = {
    'cell': 'NB22E-v4:C020',
    'kind': 'CACHE_ATTEMPT_LEDGER_SNAPSHOT',
    'identity_sha256': c020_identity_sha,
    'entries': c020_ledger,
    'historical_separation': c020_historical['preserved'],
    'historical_cache_reuse_authorized': False,
    'unknown_attempt_auto_retry_authorized': False
}

c020_ledger_path = c020_dir / (
    'C020.ledger.'
    + c016_sha(c016_canon(c020_ledger_document)) + '.json'
)

c020_ledger_sha, _ = c016_put_once(
    c020_ledger_path, c020_ledger_document
)

c020_need(
    c020_load_core(c020_ledger_path, c020_ledger_sha)['entries']
        == c020_ledger,
    'LEDGER_RELOAD_CONFLICT'
)

c020_pilot = [
    {
        'candidate_id': cid,
        'request_id': request_id,
        'selection': 'DETERMINISTIC_CONTEXT_BLIND_SHA256_NOT_GOLD',
        'authorized_to_send': False
    }
    for _, cid, request_id in sorted(c020_pilot_candidates)[:3]
]

c020_pilot_id_set = {
    x['request_id'] for x in c020_pilot
}
c020_pilot_measures = [
    r['token_measurement']
    for r in c020_ledger
    if r.get('request_id') in c020_pilot_id_set
    and 'token_measurement' in r
]

c020_pilot_proxy_low = sum(
    x['token_proxy_low_at_4_bytes_per_token']
    for x in c020_pilot_measures
)
c020_pilot_proxy_high = sum(
    x['token_proxy_high_at_2_bytes_per_token']
    for x in c020_pilot_measures
)

c020_proxy_low = c020_bytes[
    'token_proxy_low_at_4_bytes_per_token'
]
c020_proxy_high = c020_bytes[
    'token_proxy_high_at_2_bytes_per_token'
]
c020_n = (
    c020_count['offline_request_prepared']
    - c020_cache_reusable
)

# These rates apply only conditionally to direct DeepSeek.
# They are NOT verified rates for Arvan or another intermediary.
# Token proxies are NOT actual model token counts.
c020_conditional_cost = {
    'status': 'CONDITIONAL_ONLY_PROVIDER_AND_TOKENIZER_UNVERIFIED',
    'provider': 'DIRECT_DEEPSEEK_ONLY_NOT_ARVAN_TARIFF',
    'source': 'https://api-docs.deepseek.com/quick_start/pricing/',
    'verified_for_external_reference_date': '2026-11-11',
    'currency': 'USD',
    'unit': 'per_1M_tokens',
    'off_peak_input_cache_miss': 0.15,
    'peak_input_cache_miss': 0.30,
    'off_peak_output': 0.60,
    'peak_output': 1.20,
    'request_max_output_tokens': 600,
    'automatic_retries_assumed': 0,
    'input_token_proxy_low': c020_proxy_low,
    'input_token_proxy_high': c020_proxy_high,
    'illustrative_offpeak_usd_low': round(
        (c020_proxy_low * 0.15 + c020_n * 600 * 0.60) / 1e6,
        6
    ),
    'illustrative_peak_usd_high': round(
        (c020_proxy_high * 0.30 + c020_n * 600 * 1.20) / 1e6,
        6
    ),
    'illustrative_three_case_pilot_offpeak_usd': round(
        (c020_pilot_proxy_low * 0.15
         + len(c020_pilot) * 600 * 0.60) / 1e6,
        6
    ),
    'illustrative_three_case_pilot_peak_usd': round(
        (c020_pilot_proxy_high * 0.30
         + len(c020_pilot) * 600 * 1.20) / 1e6,
        6
    ),
    'provider_actual_cost_usd': None,
    'notes': (
        'UTF8_BYTE_HEURISTIC_NOT_MODEL_TOKENS_'
        'NO_DISPATCH_BUDGET_GUARANTEE'
    )
}

c020_summary = {
    'cell': 'NB22E-v4:C020',
    'status': (
        'OFFLINE_PREPARED_PROVIDER_AND_CONTEXT_GATES_OPEN'
        if c020_count['offline_request_prepared'] == 1116
        else 'PARTIAL_OFFLINE_PREPARATION_WITH_EXPLICIT_BLOCKERS'
    ),
    'identity': c020_identity,
    'identity_sha256': c020_identity_sha,
    'population': {
        'c011': 1116,
        'historical_rank_pilots_included': 4,
        'historical_29_results_not_merged': True,
        'historical_2_model_identity_held_separate': True,
        'historical_2_errors_held_separate': True
    },
    'legacy_question_only_cache_policy':
        'DO_NOT_REUSE_NO_CONTEXT_PROFILE_IDENTITY',
    'counts': dict(sorted(c020_count.items())),
    'offline_requests_ready':
        c020_count['offline_request_prepared'],
    'provider_verified_dispatch_ready': 0,
    'cache_reusable_verified': c020_cache_reusable,
    'unknown_attempts_held': c020_held_unknown,
    'unverified_cache_held': c020_stale_cache,
    'blocked_preparation': c020_count['case_blocked'],
    'measured_input': dict(c020_bytes),
    'max_request_utf8_bytes': c020_max_request_bytes,
    'actual_tokenizer_verified': False,
    'provider_context_limit_verified': False,
    'request_message_format_verified': c020_wrapper_ok,
    'legacy_generator_effective_profile_mismatch':
        c020_generator_mismatch,
    'provider_binding': c020_provider,
    'conditional_cost': c020_conditional_cost,
    'pilot_proposal': {
        'count': len(c020_pilot),
        'items': c020_pilot,
        'scope': 'TRANSPORT_SCHEMA_AND_COST_ONLY_NO_TEST_TUNING',
        'max_new_attempts_on_authorized_pilot': len(c020_pilot),
        'automatic_retries': 0,
        'pilot_execution_authorized': False
    },
    'blockers': c020_blockers,
    'pending_global': [
        'PROVIDER_MODEL_ID_AND_ENDPOINT_BINDING_NOT_VERIFIED',
        'EFFECTIVE_GENERATOR_ADAPTER_DIFFERS_FROM_C340_FROZEN_PROFILE',
        'ACTUAL_TOKENIZER_AND_FULL_MESSAGE_CONTEXT_WINDOW_NOT_VERIFIED',
        'PROVIDER_TARIFF_OR_ARVAN_ACCOUNT_RATE_NOT_VERIFIED',
        'LEGACY_QUESTION_ONLY_CACHE_IS_NOT_VERSION_CONTEXT_BOUND',
        'GENERATION_PILOT_NOT_AUTHORIZED',
        'QUALITY_GATE_BLOCKED'
    ],
    'ledger_path': str(c020_ledger_path),
    'ledger_file_sha256': c020_ledger_sha,
    'requests_directory': str(c020_request_dir),
    'new_request_files': c020_request_writes,
    'generation_api_calls': 0,
    'generator_executed': False,
    'retry_authorized': False,
    'dispatch_authorized': False,
    'quality_gate': 'BLOCKED'
}

c020_summary_path = c020_dir / (
    'C020.summary.'
    + c016_sha(c016_canon(c020_summary)) + '.json'
)
c020_summary_sha, _ = c016_put_once(
    c020_summary_path, c020_summary
)

c020_need(
    c020_load_core(
        c020_summary_path,
        c020_summary_sha
    )['ledger_file_sha256'] == c020_ledger_sha,
    'SUMMARY_RELOAD_CONFLICT'
)

C020_OFFLINE_PREPARATION = {
    'status': c020_summary['status'],
    'summary_path': str(c020_summary_path),
    'summary_sha256': c020_summary_sha,
    'ledger_path': str(c020_ledger_path),
    'ledger_sha256': c020_ledger_sha,
    'offline_requests': c020_count['offline_request_prepared'],
    'blocked': c020_count['case_blocked'],
    'quality_gate': 'BLOCKED',
    'dispatch_authorized': False
}

print(
    'C020_OFFLINE_REQUESTS:',
    c020_count['offline_request_prepared'],
    '/1116'
)
print('C020_CACHE_COMPATIBLE:', c020_cache_reusable)
print('C020_UNKNOWN_ATTEMPTS_HELD:', c020_held_unknown)
print('C020_BLOCKED:', c020_count['case_blocked'])
print('C020_PILOT_PROPOSED:', len(c020_pilot), 'NOT_AUTHORIZED')
print(
    'C020_TOKENIZER: NOT_VERIFIED | '
    'PROVIDER_MODEL_BINDING: NOT_VERIFIED'
)
print('C020_LEDGER_SHA256:', c020_ledger_sha)
print('C020_SUMMARY_SHA256:', c020_summary_sha)
print('C020_STATUS:', C020_OFFLINE_PREPARATION['status'])
print('GENERATION_SEND_AUTHORIZED: False | quality_gate: BLOCKED')

C020_PROGRESS: 200/1116
C020_PROGRESS: 400/1116
C020_PROGRESS: 600/1116
C020_PROGRESS: 800/1116
C020_PROGRESS: 1000/1116
C020_PROGRESS: 1116/1116
C020_OFFLINE_REQUESTS: 1116 /1116
C020_CACHE_COMPATIBLE: 0
C020_UNKNOWN_ATTEMPTS_HELD: 0
C020_BLOCKED: 0
C020_PILOT_PROPOSED: 3 NOT_AUTHORIZED
C020_TOKENIZER: NOT_VERIFIED | PROVIDER_MODEL_BINDING: NOT_VERIFIED
C020_LEDGER_SHA256: 6101fc4c094c24fdbd6d2c42bd4262652f1370fcc30c908adf4d7b5b7a267f37
C020_SUMMARY_SHA256: a429a3087fccb91831987a522ea220c1716cfd6e4c579e5071f9aeec18441e11
C020_STATUS: OFFLINE_PREPARED_PROVIDER_AND_CONTEXT_GATES_OPEN
GENERATION_SEND_AUTHORIZED: False | quality_gate: BLOCKED


<div dir="rtl" align="right">

### NB22E-v4:C021 — تعیین‌تکلیف آفلاین Provider، ظرفیت و هزینهٔ Generation

**هدف:** بررسی نهایی پیش‌نیازهای ارسال، بدون ارسال حتی یک درخواست.

**ورودی‌های فریز‌شده:**
- Summary و Manifest موفق C019
- درخواست‌ها، Ledger و Summary نسخه‌دار C020
- Prompt و پروفایل فریز‌شدهٔ C340
- Adapter واقعی `ArvanGenerator`
- شواهد تاریخی C342، شامل ۲۹ نتیجهٔ محفوظ، دو هویت نامحرز و دو خطای جداگانه
- مستندات رسمی و به‌روز DeepSeek، صرفاً به‌عنوان مرجع سرویس مستقیم آن

**بررسی Provider و مدل:**

شناسهٔ قابل‌ارسال، نسخهٔ مدل، مسیر اتصال و پشتیبانی پارامترهای مؤثر جداگانه بررسی شوند. تغییر از مدل تاریخی V4-Flash به V4.1-Flash نباید به‌عنوان تطابق همان مدل ثبت شود. هیچ Secret، آدرس اتصال محرمانه یا مقدار کلید در خروجی ذخیره نشود.

**بررسی Token و ظرفیت:**

پیام‌های System و User و Payload موجود C020 بدون بازسازی خوانده شوند. اندازهٔ UTF-8 دقیق است؛ تقریب توکن با آن یکسان نیست. اگر Tokenizer سازگار و سربار Chat Template احراز نشد، ظرفیت قطعی Held بماند. Context نباید بریده یا تغییر داده شود.

**بررسی هزینه:**

سناریوهای Cache Miss صددرصد، خروجی حداکثر ۶۰۰ توکن و Retry خودکار صفر برای Pilot سه‌تایی و جمعیت ۱۱۱۶تایی محاسبه شوند. نرخ‌های مستقیم DeepSeek فقط در سناریوی مشروط همان Provider به‌کار روند. تبدیل تومان بدون نرخ معتبر و تاریخ‌دار انجام نشود.

**تحویل:**

- Ledger کنترل ظرفیت و وضعیت ارسال برای ۱۱۱۶ درخواست
- Summary وضعیت Provider، مدل و Adapter
- برآورد مشروط هزینهٔ Pilot و جمعیت
- موارد Held و اختلاف مؤثر قرارداد
- تصمیم واحد لازم برای ادامه

همهٔ خروجی‌های جدید خارج از مخزن، نسخه‌دار و بازخوانی‌شده باشند.

**ممنوعیت:** اجرای Generator، API، Pilot، تغییر C019/C020، تغییر Gold، تغییر Git و Retry.

`generation_send_authorized = False`

`quality_gate = BLOCKED`

</div>

In [15]:
# NB22E-v4 | C021 | R11.2
# Offline provider/model, capacity and tariff determination from pinned C020 inputs.
# No network, generator, retrieval, paid calls, secrets or edits to C019/C020.

if (
    C020_OFFLINE_PREPARATION['quality_gate'] != 'BLOCKED'
    or C020_OFFLINE_PREPARATION['dispatch_authorized']
    or C020_OFFLINE_PREPARATION['offline_requests'] != 1116
):
    raise RuntimeError('C021_C020_GATE_CONFLICT')


def c021_check(ok, reason):
    if not ok:
        raise RuntimeError('C021_' + reason)


# 1. Reload C020 and C019. Do not rebuild Generation requests.

c021_summary = c020_load_core(
    C020_OFFLINE_PREPARATION['summary_path'],
    C020_OFFLINE_PREPARATION['summary_sha256']
)
c021_ledger_doc = c020_load_core(
    C020_OFFLINE_PREPARATION['ledger_path'],
    C020_OFFLINE_PREPARATION['ledger_sha256']
)
c021_rows = c021_ledger_doc['entries']

c021_c019_manifest = c020_load_core(
    C019_LOCAL_RESULTS['manifest_path'],
    C019_LOCAL_RESULTS['manifest_sha256']
)
c021_c019_by_id = {
    row['candidate_id']: row
    for row in c021_c019_manifest['entries']
}

c021_check(
    c021_summary['identity_sha256'] == c020_identity_sha
    and c021_summary['ledger_file_sha256']
        == C020_OFFLINE_PREPARATION['ledger_sha256']
    and c021_summary['identity']['c019_manifest_file_sha256']
        == C019_LOCAL_RESULTS['manifest_sha256']
    and len(c021_rows) == len(c021_c019_by_id) == 1116
    and len({row['candidate_id'] for row in c021_rows}) == 1116
    and {row['candidate_id'] for row in c021_rows}
        == set(c015_member_ids)
    and c021_summary['generation_api_calls'] == 0
    and c021_summary['dispatch_authorized'] is False,
    'C020_SOURCE_OR_POPULATION_CONFLICT'
)


# 2. Inspect the real project Adapter and historical evidence.

c021_adapter_path = repo / 'src/generation/arvan_generator.py'
c021_adapter_bytes = c021_adapter_path.read_bytes()

c021_check(
    c016_sha(c021_adapter_bytes)
    == c021_summary['identity']['legacy_generator_source_sha256'],
    'GENERATOR_SOURCE_CHANGED_SINCE_C020'
)

c021_adapter_text = c021_adapter_bytes.decode('utf-8')
c021_adapter_tree = ast.parse(c021_adapter_text)

c021_method = [
    method
    for cls in c021_adapter_tree.body
    if isinstance(cls, ast.ClassDef) and cls.name == 'ArvanGenerator'
    for method in cls.body
    if isinstance(method, ast.FunctionDef)
    and method.name == 'generate_answer'
]

c021_check(
    len(c021_method) == 1,
    'ADAPTER_GENERATE_METHOD_NOT_UNIQUE'
)

c021_method_source = ast.get_source_segment(
    c021_adapter_text, c021_method[0]
)

c021_check(
    'chat.completions.create(' in c021_method_source
    and 'model=self.model' in c021_method_source
    and 'temperature=self.temperature' in c021_method_source
    and 'max_tokens=' not in c021_method_source
    and 'thinking=' not in c021_method_source
    and 'extra_body=' not in c021_method_source,
    'EFFECTIVE_ADAPTER_PROFILE_MUST_BE_REVIEWED'
)

c021_check(
    'ARVAN_CHAT_URL' in c021_adapter_text
    and 'ARVAN_CHAT_MODEL' in c021_adapter_text
    and 'Qwen3-30B-A3B' in c021_adapter_text
    and 'temperature: float = 0.2' in c021_adapter_text,
    'LEGACY_ADAPTER_DEFAULTS_CHANGED'
)

# Only classified statuses are retained.
# No secret, endpoint URL or raw environment model is printed or saved.

c021_base_url = os.environ.get('ARVAN_CHAT_URL', '')
c021_runtime_model = os.environ.get('ARVAN_CHAT_MODEL', '')
c021_env_host = (
    urlsplit(c021_base_url).hostname
    if c021_base_url else None
)

c021_endpoint_kind = (
    'DIRECT_DEEPSEEK_HOST_IN_ARVAN_ADAPTER'
    if c021_env_host == 'api.deepseek.com'
    else 'NON_DEEPSEEK_HOST_UNVERIFIED'
    if c021_env_host
    else 'ENDPOINT_NOT_CONFIGURED_IN_NOTEBOOK_PROCESS'
)

c021_runtime_model_kind = (
    'NO_OVERRIDE_QWEN_DEFAULT'
    if not c021_runtime_model
    else 'FROZEN_DISPLAY_LABEL_ONLY'
    if c021_runtime_model == 'DeepSeek-V4-Flash'
    else 'CURRENT_DEEPSEEK_ID_DIFFERENT_TARGET'
    if c021_runtime_model == 'deepseek-flash'
    else 'LEGACY_DEEPSEEK_ALIAS_DIFFERENT_TARGET'
    if c021_runtime_model == 'deepseek-v4-flash'
    else 'OTHER_MODEL_NOT_DISCLOSED'
)

c021_secret_presence = {
    'arvan_key_env_present': bool(
        os.environ.get('ARVAN_CHAT_API_KEY')
        or os.environ.get('ARVAN_AI_API_KEY')
    ),
    'endpoint_env_present': bool(c021_base_url)
}

del c021_base_url, c021_runtime_model, c021_env_host

c021_prior_raw = c015_read_pinned(
    c020_c342_path, c020_c342_sha
)
c021_prior = json.loads(c021_prior_raw)

c021_check(
    c021_prior['preserved'] == {
        'EXACT_PRESERVED': 29,
        'MODEL_IDENTITY_HELD': 2,
        'RECORDED_ERROR_HELD': 2,
        'OUTSIDE_PACKET': 1116
    },
    'HISTORICAL_CASES_MUST_REMAIN_SEPARATE'
)


# 3. Official direct-DeepSeek reference; not a verified Arvan tariff.

c021_reference = {
    'reference_date': '2026-11-11',
    'provider': 'DEEPSEEK_DIRECT_REFERENCE_ONLY',
    'current_sendable_model': 'deepseek-flash',
    'currently_served_version': 'DeepSeek-V4.1-Flash-1017',
    'historical_v4_flash_retired': True,
    'old_alias_routes_to_newer_model': True,
    'context_window_tokens': 1048576,
    'max_output_tokens': 393216,
    'nonthinking_chat_extra_body': {
        'thinking': {'type': 'disabled'}
    },
    'temperature_zero_supported_in_nonthinking_chat': True,
    'rates_per_million_usd_cache_miss': {
        'offpeak': {'input': 0.15, 'output': 0.60},
        'peak': {'input': 0.30, 'output': 1.20}
    },
    'rates_source':
        'https://api-docs.deepseek.com/quick_start/pricing/',
    'model_change_source':
        'https://api-docs.deepseek.com/news/news260910/',
    'thinking_source':
        'https://api-docs.deepseek.com/guides/thinking_mode/',
    'tokenization_source':
        'https://api-docs.deepseek.com/quick_start/token_usage/'
}

c021_profile = c021_summary['identity']['frozen_profile']

c021_check(
    c021_profile == {
        'model': 'DeepSeek-V4-Flash',
        'temperature': 0,
        'max_tokens': 600,
        'thinking': {'type': 'disabled'}
    },
    'FROZEN_C340_PROFILE_CHANGED'
)

c021_profile_mismatch = {
    'display_name_not_sendable_id': True,
    'direct_deepseek_v4_now_routes_to_v4_1': True,
    'effective_adapter_missing_max_tokens': True,
    'effective_adapter_missing_nonthinking_extra_body': True,
    'effective_adapter_temperature_default_differs': True,
    'provider_route_not_verified_for_frozen_model': True
}


# 4. Verify the existing C020 request bytes and capacity evidence.

c021_pilot_ids = [
    item['request_id']
    for item in c021_summary['pilot_proposal']['items']
]

c021_check(
    len(c021_pilot_ids) == len(set(c021_pilot_ids)) == 3,
    'PILOT_THREE_IDENTITIES_REQUIRED'
)

c021_per_request = []
c021_totals = collections.Counter()
c021_pilot_totals = collections.Counter()
c021_verify_count = 0
c021_proxy_within = 0
c021_held = 0
c021_case_blockers = []

c021_request_folder = Path(
    c021_summary['requests_directory']
).resolve()

c021_check(
    c021_request_folder.parent
    == Path(C020_OFFLINE_PREPARATION['summary_path']).parent.resolve(),
    'REQUEST_DIRECTORY_CHANGED'
)

for c021_i, item in enumerate(c021_rows, 1):
    cid = item['candidate_id']
    reasons = []
    request_id = item.get('request_id')
    token_low = None
    token_high = None

    if item.get('status') != 'OFFLINE_READY_DISPATCH_GATED':
        reasons.append('C020_REQUEST_NOT_READY')

    if not request_id or not item.get('request_file'):
        reasons.append('REQUEST_REFERENCE_MISSING')

    if not reasons:
        path = (
            c021_request_folder / item['request_file']
        ).resolve()

        c021_check(
            path.parent == c021_request_folder and path.is_file(),
            'REQUEST_PATH_UNSAFE_OR_MISSING:' + cid
        )

        request = c020_load_core(
            path, item['request_file_sha256']
        )
        payload = request['payload']

        c021_check(
            request['candidate_id'] == cid
            and request['request_id'] == request_id
            and request['identity_sha256'] == c020_identity_sha
            and request['payload_sha256']
                == c016_sha(c016_canon(payload))
            and request['rendered_context_sha256']
                == c021_c019_by_id[cid]['rendered_context_sha256']
            and request['c019_context_receipt_sha256']
                == c021_c019_by_id[cid]['context_receipt_sha256']
            and request['c019_rank_receipt_sha256']
                == c021_c019_by_id[cid]['rank_receipt_sha256']
            and request['dispatch_authorized'] is False
            and payload['model'] == c021_profile['model']
            and payload['temperature'] == 0
            and payload['max_tokens'] == 600
            and payload['thinking'] == {'type': 'disabled'}
            and len(payload['messages']) == 2
            and [m['role'] for m in payload['messages']]
                == ['system', 'user'],
            'REQUEST_PAYLOAD_OR_LINEAGE_CONFLICT:' + cid
        )

        measure = item['token_measurement']

        token_low = measure[
            'token_proxy_low_at_4_bytes_per_token'
        ]
        token_high = measure[
            'token_proxy_high_at_2_bytes_per_token'
        ]

        exact_message_bytes = len(
            c016_canon(payload['messages'])
        )

        c021_check(
            exact_message_bytes == measure['messages_json_utf8_bytes']
            and len(c016_canon(payload))
                == measure['request_json_utf8_bytes']
            and 0 <= token_low <= token_high,
            'MEASUREMENT_RECEIPT_CONFLICT:' + cid
        )

        c021_verify_count += 1

        for key in (
            'messages_json_utf8_bytes',
            'request_json_utf8_bytes',
            'context_utf8_bytes',
            'token_proxy_low_at_4_bytes_per_token',
            'token_proxy_high_at_2_bytes_per_token'
        ):
            c021_totals[key] += measure[key]

            if request_id in c021_pilot_ids:
                c021_pilot_totals[key] += measure[key]

        # This is only a heuristic comparison.
        # It is NOT an exact tokenizer-based capacity PASS.

        if token_high + 600 < c021_reference['context_window_tokens']:
            c021_proxy_within += 1
        else:
            reasons.append(
                'PROXY_AT_OR_ABOVE_DIRECT_DEEPSEEK_REFERENCE_WINDOW'
            )

        if item['attempt_status'] not in (
            'NO_PREVIOUS_ATTEMPT_FOR_NEW_IDENTITY',
            'NOT_STARTED',
            'FAILED_FINAL'
        ):
            reasons.append(
                'SCOPED_ATTEMPT_NOT_SAFE_FOR_AUTORETRY'
            )

        if item['cache_status'] != 'NO_MATCHING_SCOPED_CACHE':
            reasons.append(
                'SCOPED_CACHE_RECONCILIATION_HELD'
            )

    reasons.extend([
        'PROVIDER_AND_MODEL_BINDING_NOT_VERIFIED',
        'EXACT_COMPATIBLE_TOKENIZER_OR_CHAT_TEMPLATE_NOT_PINNED',
        'PROVIDER_CONTEXT_WINDOW_NOT_VERIFIED'
    ])

    c021_held += 1

    c021_per_request.append({
        'candidate_id': cid,
        'request_id': request_id,
        'source_request_receipt_sha256': item.get(
            'request_file_sha256'
        ),
        'input_token_proxy_low': token_low,
        'input_token_proxy_high': token_high,
        'max_output_tokens': 600,
        'capacity_verified': False,
        'dispatch_authorized': False,
        'status': 'HELD_PRE_DISPATCH',
        'reasons': reasons
    })

    if len(reasons) > 3:
        c021_case_blockers.append({
            'candidate_id': cid,
            'additional_reasons': reasons[:-3]
        })

    if c021_i % 250 == 0 or c021_i == 1116:
        print(
            'C021_PROGRESS:',
            str(c021_i) + '/1116',
            flush=True
        )

c021_check(
    c021_verify_count == c021_summary['offline_requests_ready'] == 1116,
    'NOT_ALL_OFFLINE_REQUESTS_RELOADED'
)

c021_check(
    len(c021_per_request) == c021_held == 1116,
    'CAPACITY_LEDGER_COVERAGE_CONFLICT'
)


# 5. Conditional DIRECT DeepSeek cost only.
# UTF8 proxies are NOT exact tokens or guaranteed strict bounds.

def c021_cost(source, count, tier):
    rate = c021_reference[
        'rates_per_million_usd_cache_miss'
    ][tier]

    low = source['token_proxy_low_at_4_bytes_per_token']
    high = source['token_proxy_high_at_2_bytes_per_token']
    output = count * 600

    return {
        'input_low_proxy_usd': round(
            low * rate['input'] / 1000000, 6
        ),
        'input_high_proxy_usd': round(
            high * rate['input'] / 1000000, 6
        ),
        'output_at_600_each_usd': round(
            output * rate['output'] / 1000000, 6
        ),
        'total_using_low_proxy_usd': round(
            (low * rate['input'] + output * rate['output'])
            / 1000000, 6
        ),
        'total_using_high_proxy_usd': round(
            (high * rate['input'] + output * rate['output'])
            / 1000000, 6
        )
    }


c021_costs = {
    'status': 'REFERENCE_ONLY_NOT_ACTUAL_PROVIDER_QUOTE',
    'currency': 'USD',
    'cache_miss_fraction_assumed': 1.0,
    'max_output_tokens_per_request': 600,
    'auto_retries_assumed': 0,
    'proxy_not_strict_token_bounds': True,
    'exact_provider_tokens': None,
    'pilot_3': {
        tier: c021_cost(c021_pilot_totals, 3, tier)
        for tier in ('offpeak', 'peak')
    },
    'population_1116': {
        tier: c021_cost(c021_totals, 1116, tier)
        for tier in ('offpeak', 'peak')
    },
    'actual_provider_tariff_and_currency': None,
    'toman_conversion': None
}


# 6. Versioned C021 identity and independent output directory.
# No changed model is silently substituted in C020 requests.

c021_identity = {
    'schema': 'NB22E_V4_C021_OFFLINE_PROVIDER_CAPACITY_COST_V1',
    'c011_boundary_sha256': c016_boundary,
    'c019_manifest_file_sha256':
        C019_LOCAL_RESULTS['manifest_sha256'],
    'c020_summary_file_sha256':
        C020_OFFLINE_PREPARATION['summary_sha256'],
    'c020_ledger_file_sha256':
        C020_OFFLINE_PREPARATION['ledger_sha256'],
    'adapter_source_sha256': c016_sha(c021_adapter_bytes),
    'reference_date': '2026-11-11',
    'reference_model': c021_reference['current_sendable_model'],
    'send_authorized': False
}

c021_identity_sha = c016_sha(c016_canon(c021_identity))

c021_dir = (
    outputs / ('C021_offline_generation_gates_' + c021_identity_sha)
).resolve()

c021_check(
    c021_dir.parent == outputs.resolve()
    and not c021_dir.is_relative_to(repo.resolve()),
    'OUTPUT_MUST_BE_OUTSIDE_REPOSITORY'
)

c021_dir.mkdir(parents=True, exist_ok=True)

c021_ledger_document = {
    'cell': 'NB22E-v4:C021',
    'kind': 'CAPACITY_AND_DISPATCH_GATE_LEDGER',
    'identity_sha256': c021_identity_sha,
    'entries': c021_per_request,
    'generator_executed': False,
    'dispatch_authorized': False,
    'quality_gate': 'BLOCKED'
}

c021_ledger_path = c021_dir / (
    'C021.ledger.'
    + c016_sha(c016_canon(c021_ledger_document))
    + '.json'
)

c021_ledger_sha, _ = c016_put_once(
    c021_ledger_path,
    c021_ledger_document
)

c021_check(
    c020_load_core(
        c021_ledger_path,
        c021_ledger_sha
    )['entries'] == c021_per_request,
    'LEDGER_RELOAD_CONFLICT'
)

c021_summary_out = {
    'cell': 'NB22E-v4:C021',
    'status': 'OFFLINE_GATE_REVIEW_COMPLETE_PILOT_NOT_AUTHORIZED',
    'identity': c021_identity,
    'identity_sha256': c021_identity_sha,
    'actual_adapter': {
        'class': 'ArvanGenerator',
        'method': 'generate_answer',
        'endpoint_class': c021_endpoint_kind,
        'model_environment_class': c021_runtime_model_kind,
        'secret_presence_flags_only': c021_secret_presence,
        'effective_profile_matches_C340': False,
        'source_sha256': c016_sha(c021_adapter_bytes)
    },
    'historical_held_and_preserved': c021_prior['preserved'],
    'provider_reference': c021_reference,
    'model_or_adapter_mismatches': c021_profile_mismatch,
    'provider_model_sendable_verified_for_this_route': False,
    'actual_tokenizer_compatible_verified': False,
    'messages_and_payload_bytes_exact_verified_count':
        c021_verify_count,
    'context_capacity_verified_count': 0,
    'estimated_under_direct_reference_window_count':
        c021_proxy_within,
    'capacity_held_count': c021_held,
    'pilot_3_case_ids': [
        item['candidate_id']
        for item in c021_summary['pilot_proposal']['items']
    ],
    'pilot_3_sendable_now': 0,
    'population_1116_sendable_now': 0,
    'max_tokens': 600,
    'cost_estimates': c021_costs,
    'additional_per_case_blockers': c021_case_blockers,
    'provider_token_count_method':
        'C020_UTF8_BYTE_HEURISTIC_4_AND_2_BYTES_PER_TOKEN_NOT_EXACT',
    'chat_template_overhead_tokens': None,
    'do_not_truncate_context': True,
    'derived_request_created': False,
    'derived_request_reason':
        'CHANGING_TO_V4_1_REQUIRES_EXPLICIT_MODEL_IDENTITY_DECISION',
    'single_next_decision': (
        'APPROVE_REAL_PROVIDER_ENDPOINT_MODEL_AND_V4_1_SUBSTITUTION_OR_PROVIDE_'
        'THE_ORIGINALLY_FROZEN_V4_EXACT_SERVING_ROUTE_PLUS_TOKENIZER_AND_TARIFF'
    ),
    'generation_api_calls': 0,
    'generator_executed': False,
    'dispatch_authorized': False,
    'quality_gate': 'BLOCKED',
    'ledger_path': str(c021_ledger_path),
    'ledger_file_sha256': c021_ledger_sha
}

c021_summary_path = c021_dir / (
    'C021.summary.'
    + c016_sha(c016_canon(c021_summary_out))
    + '.json'
)

c021_summary_sha, _ = c016_put_once(
    c021_summary_path,
    c021_summary_out
)

c021_check(
    c020_load_core(
        c021_summary_path,
        c021_summary_sha
    )['ledger_file_sha256'] == c021_ledger_sha,
    'SUMMARY_RELOAD_CONFLICT'
)

C021_OFFLINE_GATES = {
    'status': c021_summary_out['status'],
    'summary_path': str(c021_summary_path),
    'summary_sha256': c021_summary_sha,
    'ledger_sha256': c021_ledger_sha,
    'capacity_verified_count': 0,
    'pilot_send_authorized': False,
    'quality_gate': 'BLOCKED'
}

print('C021_REQUEST_BYTES_VERIFIED:', c021_verify_count, '/1116')
print('C021_PROVIDER_MODEL_VERIFIED: False')
print('C021_CAPACITY_VERIFIED: 0/1116')
print(
    'C021_CAPACITY_ESTIMATED_UNDER_DIRECT_REFERENCE:',
    c021_proxy_within, '/1116'
)
print('C021_HELD:', c021_held, '/1116')
print('C021_PILOT_3_USD_REFERENCE:', c021_costs['pilot_3'])
print(
    'C021_POPULATION_USD_REFERENCE:',
    c021_costs['population_1116']
)
print('C021_LEDGER_SHA256:', c021_ledger_sha)
print('C021_SUMMARY_SHA256:', c021_summary_sha)
print('C021_STATUS:', C021_OFFLINE_GATES['status'])
print('GENERATION_SEND_AUTHORIZED: False | quality_gate: BLOCKED')

C021_PROGRESS: 250/1116
C021_PROGRESS: 500/1116
C021_PROGRESS: 750/1116
C021_PROGRESS: 1000/1116
C021_PROGRESS: 1116/1116
C021_REQUEST_BYTES_VERIFIED: 1116 /1116
C021_PROVIDER_MODEL_VERIFIED: False
C021_CAPACITY_VERIFIED: 0/1116
C021_CAPACITY_ESTIMATED_UNDER_DIRECT_REFERENCE: 1116 /1116
C021_HELD: 1116 /1116
C021_PILOT_3_USD_REFERENCE: {'offpeak': {'input_low_proxy_usd': 0.002515, 'input_high_proxy_usd': 0.005029, 'output_at_600_each_usd': 0.00108, 'total_using_low_proxy_usd': 0.003595, 'total_using_high_proxy_usd': 0.006109}, 'peak': {'input_low_proxy_usd': 0.005029, 'input_high_proxy_usd': 0.010058, 'output_at_600_each_usd': 0.00216, 'total_using_low_proxy_usd': 0.007189, 'total_using_high_proxy_usd': 0.012218}}
C021_POPULATION_USD_REFERENCE: {'offpeak': {'input_low_proxy_usd': 0.959313, 'input_high_proxy_usd': 1.918545, 'output_at_600_each_usd': 0.40176, 'total_using_low_proxy_usd': 1.361073, 'total_using_high_proxy_usd': 2.320305}, 'peak': {'input_low_proxy_usd': 1.918625, 'input_h

<div dir="rtl" align="right">

### NB22E-v4:C022 — تطبیق محدود موجودی Generation و داوری تاریخی

**هدف:** تعیین پوشش واقعی پاسخ‌ها و بررسی‌های موجود در مقابل ۱۱۱۶ درخواست آفلاین C020.

**منابع اصلی:**
- Ledger و Summary فریز‌شدهٔ C020
- نتیجهٔ کنترل ارسال C021
- Handoff فریز‌شدهٔ NB19-C239
- ورودی، پاسخ و رسید بررسی داخلی NB19، فقط از ارجاعات همان Handoff
- راهنمای اتصال تاریخی ARVAN_GATEWAY
- تعرفهٔ مشاهده‌شدهٔ DeepSeek-V4-Flash در پنل مالک

**طبقه‌بندی Generation:**
- A: Cache با هویت اجرایی کاملاً احرازشده
- B: پاسخ تاریخی با اختلاف مؤثر مدل یا Prompt
- C: متن پاسخ موجود، اما هویت اجرای کامل نامحرز
- D: اتصال پاسخ، Cache یا Attempt حل‌نشده
- E: فقدان اثبات‌شده، فقط در محدودهٔ موجودی بررسی‌شده

**تفکیک داوری:**

بررسی داخلی تاریخی، داوری مستقل و پاسخ مرجع Golden سه دارایی متفاوت‌اند.

رسید بررسی داخلی NB19 فقط در دامنهٔ پاسخ و شواهد تاریخی خودش قابل‌مصرف است. تطابق شناسهٔ سؤال به‌تنهایی برای انتقال رأی به پاسخ نسخهٔ C020 کافی نیست.

**روش اجرا:**
۱. مصرف Handoff دقیق NB19 و کنترل SHA منبع.
۲. خواندن JSONLهای ارجاع‌شده به‌صورت ترتیبی.
۳. نگهداری شناسه و هش پاسخ؛ نه انباشت متن تمام پاسخ‌ها.
۴. اتصال سؤال، Prompt، پاسخ و رسید بررسی داخلی.
۵. طبقه‌بندی غیرهم‌پوشان ۱۱۱۶ درخواست.
۶. ثبت منابع ناموجود با مسیر دقیق.
۷. محاسبهٔ هزینهٔ مشروط Pilot و کل جمعیت با نرخ پنل آروان.
۸. انتشار Ledger ردیفی و Summary نسخه‌دار خارج از مخزن.

**محدودیت نتیجه:**

تفاوت اثبات‌شدهٔ درخواست تاریخی با C020، به‌خودی‌خود اثبات نبود هیچ Cache دیگری در پروژه نیست.

درخواست نامحرز وارد صف ارسال نمی‌شود. وضعیت Provider و Adapter تا کنترل قرارداد واقعی ارسال، مسدود می‌ماند.

**ممنوعیت:** Generation، Judge، API، Scoring، Retrieval، تغییر C019/C020/C021، تغییر Gold و Git.

`generation_send_authorized = False`

`quality_gate = BLOCKED`

</div>

In [19]:
# NB22E-v4 | C022 | R11.2
# Scoped Test answer/decision reconciliation from NB19-C239 and C020/C021 receipts.
# No Generation, Judge, API, Git changes, or recomputation of Rank/Context.

if (C021_OFFLINE_GATES['status'] != 'OFFLINE_GATE_REVIEW_COMPLETE_PILOT_NOT_AUTHORIZED'
        or C020_OFFLINE_PREPARATION['offline_requests'] != 1116
        or C019_LOCAL_RESULTS['status'] != 'CONTEXT_PREPARED_ALL'
        or C021_OFFLINE_GATES['quality_gate'] != 'BLOCKED'):
    raise RuntimeError('C022_UPSTREAM_BOUNDARY_CONFLICT')


def c022_require(ok, reason):
    if not ok:
        raise ValueError('C022_' + reason)


def c022_read_jsonl_pinned(path, ref, consume, expected=1149):
    hasher = hashlib.sha256()
    count = 0
    c022_require(not path.is_symlink() and path.is_file(),
                 'PINNED_FILE_MISSING:' + str(path))
    with path.open('rb') as source:
        for raw in source:
            hasher.update(raw)
            c022_require(raw.strip() and len(raw) <= 2 * 1024 * 1024,
                         'JSONL_ROW_INVALID_OR_TOO_LONG:' + path.name)
            row = json.loads(raw)
            c022_require(isinstance(row, dict),
                         'JSONL_ROW_NOT_OBJECT:' + path.name)
            consume(row)
            count += 1
            c022_require(count <= expected,
                         'EXCESS_RECORDS:' + path.name)
    c022_require(count == expected and hasher.hexdigest() == ref['sha256'],
                 'PINNED_JSONL_COUNT_OR_SHA_CONFLICT:' + path.name)
    if 'bytes' in ref:
        c022_require(path.stat().st_size == ref['bytes'],
                     'PINNED_JSONL_BYTES_CONFLICT:' + path.name)
    return {'path': str(path), 'sha256': ref['sha256'], 'rows': count}


# A. Current immutable requests and the separated older 33-case packet.
c022_summary = c020_load_core(
    C020_OFFLINE_PREPARATION['summary_path'],
    C020_OFFLINE_PREPARATION['summary_sha256']
)
c022_ledger = c020_load_core(
    C020_OFFLINE_PREPARATION['ledger_path'],
    C020_OFFLINE_PREPARATION['ledger_sha256']
)
c022_gate = c020_load_core(
    C021_OFFLINE_GATES['summary_path'],
    C021_OFFLINE_GATES['summary_sha256']
)
c022_current = c022_ledger['entries']

c022_require(
    len(c022_current) == 1116
    and len({r['candidate_id'] for r in c022_current}) == 1116
    and {r['candidate_id'] for r in c022_current} == set(c015_member_ids)
    and c022_summary['identity_sha256'] == c020_identity_sha
    and c022_summary['dispatch_authorized'] is False
    and c022_gate['population_1116_sendable_now'] == 0
    and c022_ledger['historical_separation'] == {
        'EXACT_PRESERVED': 29,
        'MODEL_IDENTITY_HELD': 2,
        'RECORDED_ERROR_HELD': 2,
        'OUTSIDE_PACKET': 1116
    },
    'C020_C021_CURRENT_IDENTITY_CONFLICT'
)

c022_question_sha = {
    cid: r['question_sha256']
    for cid, r in c002_questions.items()
}
c022_require(
    set(c022_question_sha) == set(c015_member_ids),
    'C341_QUESTION_SCOPE_CONFLICT'
)


# B. Exact NB19-C239 Test recovery handoff and its three source roles.
c022_nb19_handoff_rel = (
    'artifacts/phase2/generation_and_citation/work/outputs/'
    'nb19_test_handoff_chatgpt_recovery_v1.json'
)
c022_nb19_handoff_sha = (
    'c789c8a60ff7aae36e32272b878f834fef90c1bda608af986c53581f5b77c40a'
)
c022_nb19_handoff_path = (repo / c022_nb19_handoff_rel).resolve()

c022_missing = []
c022_nb19_status = 'PINNED_HANDOFF_NOT_PRESENT'
c022_nb19_source_receipts = {}
c022_nb19_input = {}
c022_nb19_answers = {}
c022_nb19_reviews = {}


def c022_safe_repo_ref(ref):
    rel = Path(ref['relative_path'])
    c022_require(
        not rel.is_absolute() and '..' not in rel.parts,
        'UNSAFE_HISTORICAL_REFERENCE'
    )
    path = (repo / rel).resolve()
    c022_require(
        path.is_relative_to(repo.resolve()),
        'HISTORICAL_REFERENCE_ESCAPES_REPOSITORY'
    )
    return path


def c022_find_nb19_slot(obj):
    if isinstance(obj, dict):
        if (
            isinstance(obj.get('question'), str)
            and isinstance(obj.get('evidence'), list)
            and obj['evidence']
            and isinstance(obj['evidence'][0], dict)
        ):
            return obj
        for value in obj.values():
            found = c022_find_nb19_slot(value)
            if found is not None:
                return found
    elif isinstance(obj, list):
        for value in obj:
            found = c022_find_nb19_slot(value)
            if found is not None:
                return found
    return None


if c022_nb19_handoff_path.is_file():
    try:
        c022_handoff_raw = c022_nb19_handoff_path.read_bytes()

        c022_require(
            c016_sha(c022_handoff_raw) == c022_nb19_handoff_sha,
            'NB19_C239_HANDOFF_SHA_CONFLICT'
        )

        c022_handoff = json.loads(c022_handoff_raw)

        c022_require(
            c022_handoff['schema_version']
                == 'nb19_test_recovery_handoff_v1'
            and c022_handoff['scope']['generation_origin']
                == 'CHATGPT_AGENT_RECOVERY'
            and c022_handoff['counts']['candidate_count'] == 1149
            and c022_handoff['scope'][
                'exact_frozen_messages_execution_claimed'
            ] is False,
            'NB19_C239_HANDOFF_SCOPE_CONFLICT'
        )

        c022_refs = {
            'inputs': c022_handoff['source_package_files'][
                'consolidated/inputs.jsonl'
            ],
            'answers': c022_handoff['artifact_roles'][
                'generation_citation'
            ],
            'reviews': c022_handoff['artifact_roles'][
                'internal_claim_gate'
            ]
        }

        c022_paths = {
            k: c022_safe_repo_ref(v)
            for k, v in c022_refs.items()
        }

        c022_missing = [
            str(path)
            for path in c022_paths.values()
            if not path.is_file()
        ]

        if not c022_missing:

            def c022_consume_input(row):
                cid = row['candidate_id']

                c022_require(
                    cid not in c022_nb19_input,
                    'NB19_DUPLICATE_INPUT:' + cid
                )

                messages = row['messages']

                c022_require(
                    isinstance(messages, list)
                    and c016_sha(c016_canon(messages))
                        == row['messages_sha256'],
                    'NB19_MESSAGE_SHA_CONFLICT:' + cid
                )

                systems = [
                    m['content'] for m in messages
                    if m.get('role') == 'system'
                ]
                users = [
                    m['content'] for m in messages
                    if m.get('role') == 'user'
                ]

                c022_require(
                    len(systems) == len(users) == 1
                    and all(
                        isinstance(x, str)
                        for x in systems + users
                    ),
                    'NB19_MESSAGES_NOT_SYSTEM_USER:' + cid
                )

                slot = c022_find_nb19_slot(json.loads(users[0]))

                c022_require(
                    slot is not None,
                    'NB19_QUESTION_SLOT_MISSING:' + cid
                )

                c022_nb19_input[cid] = {
                    'question_sha256': c016_sha(
                        slot['question'].encode('utf-8')
                    ),
                    'system_prompt_sha256': c016_sha(
                        systems[0].encode('utf-8')
                    ),
                    'messages_sha256': row['messages_sha256']
                }


            def c022_consume_answer(row):
                cid = row['candidate_id']

                c022_require(
                    cid not in c022_nb19_answers
                    and row['split'] == 'test'
                    and row['schema_version']
                        == 'nb19_test_recovery_output_v1',
                    'NB19_ANSWER_SCHEMA_OR_DUPLICATE:' + cid
                )

                meta = row['recovery_metadata']
                response = row['recovery_response']

                c022_require(
                    meta['generation_origin']
                        == 'CHATGPT_AGENT_RECOVERY'
                    and meta['model_identity'] == 'NOT_VERIFIED'
                    and c016_sha(c016_canon(response))
                        == meta['active_response_sha256']
                    and isinstance(
                        response.get('answer_text'), str
                    ),
                    'NB19_ANSWER_RESPONSE_SHA_CONFLICT:' + cid
                )

                c022_nb19_answers[cid] = {
                    'answer_text_sha256': c016_sha(
                        response['answer_text'].encode('utf-8')
                    ),
                    'active_response_core_sha256':
                        meta['active_response_sha256'],
                    'package_messages_sha256':
                        meta['package_messages_sha256'],
                    'core_record_sha256':
                        c016_sha(c016_canon(row)),
                    'answer_has_text':
                        bool(response['answer_text'].strip()),
                    'historical_route':
                        meta['recovery_status']
                }


            def c022_consume_review(row):
                cid = row['candidate_id']

                c022_require(
                    cid not in c022_nb19_reviews
                    and row['split'] == 'test'
                    and row['schema_version']
                        == 'nb19_test_recovery_internal_claim_gate_v1',
                    'NB19_REVIEW_SCHEMA_OR_DUPLICATE:' + cid
                )

                c022_nb19_reviews[cid] = {
                    'source_core_record_sha256':
                        row['source_core_record_sha256'],
                    'route': row['route'],
                    'independent':
                        row['recovery_metadata'][
                            'support_independent'
                        ],
                    'gate_text_is_none':
                        row['answer_text'] is None
                }


            for role, callback in (
                ('inputs', c022_consume_input),
                ('answers', c022_consume_answer),
                ('reviews', c022_consume_review)
            ):
                c022_nb19_source_receipts[role] = (
                    c022_read_jsonl_pinned(
                        c022_paths[role],
                        c022_refs[role],
                        callback
                    )
                )

            c022_require(
                set(c022_nb19_input)
                    == set(c022_nb19_answers)
                    == set(c022_nb19_reviews),
                'NB19_1149_ROW_SET_CONFLICT'
            )

            c022_nb19_status = (
                'PINNED_1149_INPUT_RESPONSE_REVIEW_CHAIN_VERIFIED'
            )
        else:
            c022_nb19_status = (
                'NB19_C239_REFERENCED_DATA_FILE_MISSING'
            )

    except (
        OSError, ValueError, KeyError, TypeError,
        json.JSONDecodeError
    ) as exc:
        c022_nb19_status = (
            'HISTORICAL_SOURCE_BLOCKED:'
            + type(exc).__name__ + ':' + str(exc)
        )

        c022_nb19_input.clear()
        c022_nb19_answers.clear()
        c022_nb19_reviews.clear()
else:
    c022_missing.append(str(c022_nb19_handoff_path))


# C. Narrow fallback only if the pinned Test recovery chain was unavailable.
c022_old_cache_path = (
    repo / 'data/cache/rag_cache.json'
).resolve()
c022_old_cache = {}
c022_old_cache_sha = None

if (
    c022_nb19_status
        != 'PINNED_1149_INPUT_RESPONSE_REVIEW_CHAIN_VERIFIED'
    and c022_old_cache_path.is_file()
):
    raw = c022_old_cache_path.read_bytes()
    c022_old_cache_sha = c016_sha(raw)
    old_doc = json.loads(raw)

    c022_require(
        isinstance(old_doc, dict),
        'OLD_APP_CACHE_SCHEMA'
    )

    for qsha, old in old_doc.items():
        if (
            isinstance(old, dict)
            and isinstance(old.get('question'), str)
            and c016_sha(old['question'].encode('utf-8')) == qsha
            and isinstance(old.get('answer'), str)
            and bool(old['answer'])
        ):
            c022_old_cache[qsha] = {
                'answer_sha256': c016_sha(
                    old['answer'].encode('utf-8')
                ),
                'model': old.get('generation_model')
            }

    del raw, old_doc


# D. Historical verdict summary: aggregate evidence, not a per-case verdict.
c022_judge_rel = (
    'reports/phase2/evaluation/NB21C.C018.action26_final/'
    'C018.summary_fa.49cc5c5c5f812718ee6a68bc4354bf146c03c721b415a0cd7b67b7b086cfaf01.md'
)
c022_judge_path = (repo / c022_judge_rel).resolve()

c022_judge_summary_sha = (
    c016_sha(c022_judge_path.read_bytes())
    if c022_judge_path.is_file()
    else None
)


# E. Historical Gateway contract, without loading any secret into an artifact.
c022_guide_rel = (
    'notebooks/data_load_read/ ورژن جدید ارسال/'
    '03_API_execution_and_RAG_profile_fa.md'
)
c022_guide_path = (repo / c022_guide_rel).resolve()

c022_require(
    c022_guide_path.is_file()
    and c022_guide_path.is_relative_to(repo.resolve()),
    'GATEWAY_GUIDE_NOT_FOUND'
)

c022_guide_bytes = c022_guide_path.read_bytes()

c022_require(
    all(
        term in c022_guide_bytes.decode('utf-8')
        for term in (
            'ARVAN_GATEWAY_URL',
            'ARVAN_GATEWAY_API_KEY',
            'DeepSeek-V4-Flash',
            'Authorization',
            'apikey'
        )
    ),
    'GATEWAY_GUIDE_CONTRACT_MISMATCH'
)

c022_guide_sha = c016_sha(c022_guide_bytes)
c022_env_keys = set()

if (repo / '.env').is_file():
    with (repo / '.env').open(
        'r', encoding='utf-8', errors='replace'
    ) as stream:
        for line in stream:
            field = line.split('=', 1)[0].strip()

            if field in (
                'ARVAN_GATEWAY_URL',
                'ARVAN_GATEWAY_API_KEY',
                'ARVAN_GATEWAY_KEY',
                'ARVAN_GATEWAY_MODEL'
            ):
                c022_env_keys.add(field)


# F. Versioned output identity.
c022_identity = {
    'schema': 'NB22E_V4_C022_BOUNDED_TEST_HISTORY_RECONCILIATION_V1',
    'c011_boundary_sha256': c016_boundary,
    'c020_ledger_file_sha256':
        C020_OFFLINE_PREPARATION['ledger_sha256'],
    'c021_summary_file_sha256':
        C021_OFFLINE_GATES['summary_sha256'],
    'nb19_handoff_sha256': c022_nb19_handoff_sha,
    'nb19_recovery_state': c022_nb19_status,
    'old_app_cache_sha256': c022_old_cache_sha,
    'gateway_guide_sha256': c022_guide_sha,
    'report_basis':
        'RAG_finance_phase2_API_history_and_current_state_fa',
    'history_report_upload_sha256':
        '0e363b40aa58da4a813e7d644dda4469c6f0266a49727db1135b55b5a7bfef87',
    'price_panel_upload_sha256':
        '49bdf8ed475bcf84b22c8068cf844b23627f5516c7a6f491156e8aa4c6762562',
    'no_new_generation': True
}

c022_identity_sha = c016_sha(c016_canon(c022_identity))

c022_dir = (
    outputs / ('C022_history_reconciliation_' + c022_identity_sha)
).resolve()

c022_require(
    c022_dir.parent == outputs.resolve()
    and not c022_dir.is_relative_to(repo.resolve()),
    'OUTPUT_NOT_EXTERNAL'
)

c022_dir.mkdir(parents=True, exist_ok=True)


# G. Classify every current question, independently of historical verdicts.
c022_counts = collections.Counter()
c022_verdict_counts = collections.Counter()
c022_rows = []
c022_totals = collections.Counter()

c022_pilot_ids = {
    x['request_id']
    for x in c022_summary['pilot_proposal']['items']
}

c022_require(
    len(c022_pilot_ids) == 3,
    'PILOT_THREE_COUNT_CONFLICT'
)

c022_legacy_bound = 0

for row in c022_current:
    cid = row['candidate_id']

    c022_require(
        row['status'] == 'OFFLINE_READY_DISPATCH_GATED'
        and cid in c022_question_sha,
        'C020_CASE_BINDING_CONFLICT:' + cid
    )

    question_sha = c022_question_sha[cid]
    legacy_answer_sha = None
    legacy_source = None
    historical_review = 'JUDGMENT_PER_CASE_NOT_BOUND'

    previous = c022_nb19_answers.get(cid)
    old_input = c022_nb19_input.get(cid)
    gate = c022_nb19_reviews.get(cid)

    if (
        row['attempt_status']
            != 'NO_PREVIOUS_ATTEMPT_FOR_NEW_IDENTITY'
        or row['cache_status']
            != 'NO_MATCHING_SCOPED_CACHE'
    ):
        category = 'D_SCOPED_CACHE_OR_ATTEMPT_UNRESOLVED'
        reason = (
            'C020_LOCAL_IDENTITY_REQUIRES_SEPARATE_RECONCILIATION'
        )

    elif (
        previous is not None
        and old_input is not None
        and gate is not None
    ):
        c022_legacy_bound += 1

        if old_input['question_sha256'] != question_sha:
            category = 'D_HISTORICAL_QUESTION_TEXT_CONFLICT'
            reason = (
                'NB19_C239_ID_MATCH_BUT_QUESTION_SHA_MISMATCH'
            )

        elif (
            previous['package_messages_sha256']
            != old_input['messages_sha256']
        ):
            category = 'D_HISTORICAL_MESSAGE_RECEIPT_CONFLICT'
            reason = (
                'NB19_C239_RESPONSE_INPUT_MESSAGE_SHA_MISMATCH'
            )

        elif (
            gate['source_core_record_sha256']
                != previous['core_record_sha256']
            or not gate['gate_text_is_none']
        ):
            category = 'D_HISTORICAL_RESPONSE_REVIEW_CHAIN_CONFLICT'
            reason = (
                'NB19_C239_INTERNAL_REVIEW_CORE_SHA_MISMATCH'
            )

        else:
            legacy_answer_sha = previous['answer_text_sha256']
            legacy_source = (
                c022_nb19_source_receipts['answers']['path']
            )

            if (
                old_input['system_prompt_sha256']
                != c022_summary['identity']['system_prompt_sha256']
            ):
                category = 'B_EFFECTIVE_SYSTEM_PROMPT_DIFFERENCE'
                reason = (
                    'NB19_RECOVERY_SYSTEM_DIFFERS_C340_FROZEN_SYSTEM'
                )
            else:
                category = (
                    'C_TEXT_AVAILABLE_EXECUTION_IDENTITY_UNVERIFIED'
                )
                reason = (
                    'NB19_AGENT_RECOVERY_NOT_A_PROVEN_DEEPSEEK_REQUEST'
                )

            if (
                gate['route'] == 'INTERNAL_SUPPORT_ELIGIBLE'
                and gate['independent'] is False
            ):
                historical_review = (
                    'MECHANICAL_REUSE_POSSIBLE_ONLY_FOR_OLD_AGENT_REVIEW_SCOPE'
                )
            elif gate['route'] == 'BLOCKED_FOR_REVIEW':
                historical_review = (
                    'HISTORICAL_INTERNAL_GATE_BLOCKED_NOT_TRANSFERABLE'
                )
            else:
                historical_review = (
                    'HISTORICAL_REVIEW_ROUTE_UNKNOWN'
                )

    elif question_sha in c022_old_cache:
        old = c022_old_cache[question_sha]

        legacy_answer_sha = old['answer_sha256']
        legacy_source = str(c022_old_cache_path)

        if (
            old['model']
            and old['model'] != 'DeepSeek-V4-Flash'
        ):
            category = 'B_EFFECTIVE_MODEL_DIFFERENCE_OLD_APP_CACHE'
            reason = (
                'QUESTION_MATCHES_OLD_APP_CACHE_BUT_MODEL_DIFFERS'
            )
        else:
            category = 'C_TEXT_AVAILABLE_EXECUTION_IDENTITY_UNVERIFIED'
            reason = (
                'OLD_APP_CACHE_MISSING_EXACT_PROMPT_CONTEXT_AND_ATTEMPT'
            )

    else:
        category = 'D_HISTORICAL_RESPONSE_BINDING_UNRESOLVED'
        reason = (
            'NB19_C239_ROWS_NOT_AVAILABLE_OR_NOT_MATCHED:'
            + c022_nb19_status
        )

    c022_counts[category] += 1
    c022_verdict_counts[historical_review] += 1

    measure = row['token_measurement']

    low = measure['token_proxy_low_at_4_bytes_per_token']
    high = measure['token_proxy_high_at_2_bytes_per_token']

    c022_totals['low_proxy'] += low
    c022_totals['high_proxy'] += high

    if row['request_id'] in c022_pilot_ids:
        c022_totals['pilot_low_proxy'] += low
        c022_totals['pilot_high_proxy'] += high

    c022_rows.append({
        'candidate_id': cid,
        'request_id': row['request_id'],
        'question_sha256': question_sha,
        'c020_request_file_sha256':
            row['request_file_sha256'],
        'generation_category': category,
        'reason': reason,
        'previous_answer_text_sha256': legacy_answer_sha,
        'previous_answer_source': legacy_source,
        'historical_review': historical_review,
        'current_target_verdict':
            'NOT_APPLICABLE_NO_C020_PRODUCT_ANSWER',
        'identical_generation_run_proven': False,
        'cache_reuse_authorized': False,
        'new_generation_required_globally_proven': False,
        'dispatch_authorized': False
    })

c022_require(
    len(c022_rows) == sum(c022_counts.values()) == 1116
    and sum(c022_verdict_counts.values()) == 1116,
    'NONOVERLAPPING_COUNTS_CONFLICT'
)


# H. Arvan-panel tariff; text-byte proxies are not model token counts.
def c022_cost(low, high, n):
    return {
        'input_proxy_low_tokens': low,
        'input_proxy_high_tokens': high,
        'output_ceiling_tokens': n * 600,
        'toman_using_low_proxy_and_600_output': round(
            (low * 45000 + n * 600 * 90000) / 1000000, 2
        ),
        'toman_using_high_proxy_and_600_output': round(
            (high * 45000 + n * 600 * 90000) / 1000000, 2
        ),
        'cache_miss_assumed': True,
        'automatic_retries': 0,
        'retry_reserve_toman': 0,
        'not_a_verified_spend_ceiling': True
    }


c022_tariff = {
    'source': 'USER_ARVAN_MODEL_BAZAAR_SCREENSHOT_2026_11_11',
    'screenshot_sha256':
        '49bdf8ed475bcf84b22c8068cf844b23627f5516c7a6f491156e8aa4c6762562',
    'display_model': 'DeepSeek-V4-Flash',
    'rial_per_million_input': 450000,
    'rial_per_million_output': 900000,
    'toman_per_million_input': 45000,
    'toman_per_million_output': 90000,
    'three_case_pilot_hypothetical': c022_cost(
        c022_totals['pilot_low_proxy'],
        c022_totals['pilot_high_proxy'],
        3
    ),
    'full_1116_hypothetical': c022_cost(
        c022_totals['low_proxy'],
        c022_totals['high_proxy'],
        1116
    ),
    'actual_generation_deficit_cost': None,
    'unknown_billing_components': [
        'TAX',
        'REASONING',
        'SERVER_CACHE_POLICY',
        'PROVIDER_ROUTE_ID',
        'ACTUAL_TOKENIZER'
    ]
}


# I. Publish one immutable row-wise ledger outside the repository.
c022_ledger_path = (
    c022_dir / ('C022.rows.' + c022_identity_sha + '.jsonl')
)
c022_expected_hash = hashlib.sha256()

if not c022_ledger_path.is_file():
    with tempfile.NamedTemporaryFile(
        mode='wb',
        dir=c022_dir,
        prefix='.C022.',
        suffix='.tmp',
        delete=False
    ) as f:
        c022_temp = Path(f.name)

        for r in c022_rows:
            data = c016_canon(r) + b'\n'
            c022_expected_hash.update(data)
            f.write(data)

        f.flush()
        os.fsync(f.fileno())

    try:
        os.link(c022_temp, c022_ledger_path)
    except FileExistsError:
        pass
    finally:
        c022_temp.unlink(missing_ok=True)

else:
    for r in c022_rows:
        c022_expected_hash.update(
            c016_canon(r) + b'\n'
        )

c022_ledger_sha = c022_expected_hash.hexdigest()
c022_check_hash = hashlib.sha256()

with c022_ledger_path.open('rb') as f:
    for line in f:
        c022_check_hash.update(line)

c022_require(
    c022_check_hash.hexdigest() == c022_ledger_sha,
    'IMMUTABLE_LEDGER_RELOAD_CONFLICT'
)


# J. Final bounded report: evidence, uncertainty, costs, and next blocker.
c022_result = {
    'cell': 'NB22E-v4:C022',
    'status':
        'BOUNDED_HISTORICAL_RECONCILIATION_COMPLETED_DISPATCH_BLOCKED',
    'identity': c022_identity,
    'identity_sha256': c022_identity_sha,
    'population_count': 1116,
    'generation_categories': dict(sorted(c022_counts.items())),
    'historical_review_categories':
        dict(sorted(c022_verdict_counts.items())),
    'target_current_verdicts_verified': 0,
    'target_generation_exact_cache_reuse_proven': 0,
    'historical_answer_row_bindings_checked': c022_legacy_bound,
    'new_generation_deficit_globally_proven': None,
    'current_scope_historical_effective_mismatch_count': sum(
        n for label, n in c022_counts.items()
        if label.startswith('B_')
    ),
    'current_scope_text_without_generation_identity_count': sum(
        n for label, n in c022_counts.items()
        if label.startswith('C_')
    ),
    'current_scope_unresolved_count': sum(
        n for label, n in c022_counts.items()
        if label.startswith('D_')
    ),
    'current_scope_exact_cache_hits': 0,
    'current_scope_proven_missing_after_exhaustive_inventory': 0,
    'eligible_for_generation_dispatch': 0,
    'historical_target_packet_outside_scope':
        c022_ledger['historical_separation'],
    'historical_sources': {
        'nb19_handoff_path': str(c022_nb19_handoff_path),
        'nb19_handoff_sha256': c022_nb19_handoff_sha,
        'nb19_state': c022_nb19_status,
        'verified_data_sources': c022_nb19_source_receipts,
        'missing_required_paths': c022_missing,
        'legacy_app_cache_path': str(c022_old_cache_path),
        'legacy_app_cache_sha256_if_read': c022_old_cache_sha,
        'nb21c_noanswer_summary_path': c022_judge_rel,
        'nb21c_noanswer_summary_sha256_if_present':
            c022_judge_summary_sha
    },
    'gateway': {
        'contract_family': 'ARVAN_GATEWAY',
        'guide_path': c022_guide_rel,
        'guide_sha256': c022_guide_sha,
        'url_setting_key_present':
            'ARVAN_GATEWAY_URL' in c022_env_keys,
        'credential_key_present':
            bool(c022_env_keys.intersection(
                ('ARVAN_GATEWAY_API_KEY', 'ARVAN_GATEWAY_KEY')
            )),
        'configured_display_model': 'DeepSeek-V4-Flash',
        'provider_sendable_model_id_verified': False,
        'actual_adapter_model_parameter_verified': False,
        'gateway_auth_contract_difference':
            'GUIDE_APIKEY_VS_NB19_C216_OPENAI_SDK_METADATA_ONLY_NOT_GENERATION_ROUTE_PROOF',
        'no_secret_or_private_url_stored': True
    },
    'owner_panel_tariff_scenario': c022_tariff,
    'next_single_blocker': (
        'EXACT_NB19_C239_SOURCE_PATHS_MISSING'
        if c022_missing
        else 'PROVIDER_ADAPTER_AND_DISPATCH_ATTEMPT_CONTRACT_STILL_UNVERIFIED'
    ),
    'ledger_path': str(c022_ledger_path),
    'ledger_file_sha256': c022_ledger_sha,
    'new_api_calls': 0,
    'new_judge_calls': 0,
    'generation_send_authorized': False,
    'quality_gate': 'BLOCKED'
}

c022_result_path = (
    c022_dir / (
        'C022.summary.'
        + c016_sha(c016_canon(c022_result))
        + '.json'
    )
)

c022_result_sha, _ = c016_put_once(
    c022_result_path, c022_result
)

c022_require(
    c020_load_core(
        c022_result_path, c022_result_sha
    )['ledger_file_sha256'] == c022_ledger_sha,
    'SUMMARY_RELOAD_CONFLICT'
)

C022_RECONCILIATION = {
    'status': c022_result['status'],
    'summary_path': str(c022_result_path),
    'summary_sha256': c022_result_sha,
    'ledger_sha256': c022_ledger_sha,
    'dispatch_authorized': False,
    'quality_gate': 'BLOCKED'
}

print('C022_POPULATION: 1116/1116')
print('C022_NB19_SOURCE:', c022_nb19_status)
print('C022_CATEGORIES:', dict(sorted(c022_counts.items())))
print('C022_HISTORICAL_REVIEW:', dict(sorted(c022_verdict_counts.items())))
print('C022_MISSING_EXACT_PATHS_COUNT:', len(c022_missing))

if c022_missing:
    print('C022_MISSING_EXACT_PATHS:', c022_missing[:4])

print(
    'C022_HYPOTHETICAL_PILOT_TOMAN:',
    c022_tariff['three_case_pilot_hypothetical']
)
print(
    'C022_HYPOTHETICAL_FULL_TOMAN:',
    c022_tariff['full_1116_hypothetical']
)
print('C022_LEDGER_SHA256:', c022_ledger_sha)
print('C022_SUMMARY_SHA256:', c022_result_sha)
print('C022_STATUS:', C022_RECONCILIATION['status'])
print('GENERATION_SEND_AUTHORIZED: False | quality_gate: BLOCKED')

C022_POPULATION: 1116/1116
C022_NB19_SOURCE: HISTORICAL_SOURCE_BLOCKED:KeyError:'source_package_files'
C022_CATEGORIES: {'D_HISTORICAL_RESPONSE_BINDING_UNRESOLVED': 1116}
C022_HISTORICAL_REVIEW: {'JUDGMENT_PER_CASE_NOT_BOUND': 1116}
C022_MISSING_EXACT_PATHS_COUNT: 0
C022_HYPOTHETICAL_PILOT_TOMAN: {'input_proxy_low_tokens': 16764, 'input_proxy_high_tokens': 33526, 'output_ceiling_tokens': 1800, 'toman_using_low_proxy_and_600_output': 916.38, 'toman_using_high_proxy_and_600_output': 1670.67, 'cache_miss_assumed': True, 'automatic_retries': 0, 'retry_reserve_toman': 0, 'not_a_verified_spend_ceiling': True}
C022_HYPOTHETICAL_FULL_TOMAN: {'input_proxy_low_tokens': 6395418, 'input_proxy_high_tokens': 12790303, 'output_ceiling_tokens': 669600, 'toman_using_low_proxy_and_600_output': 348057.81, 'toman_using_high_proxy_and_600_output': 635827.64, 'cache_miss_assumed': True, 'automatic_retries': 0, 'retry_reserve_toman': 0, 'not_a_verified_spend_ceiling': True}
C022_LEDGER_SHA256: cf6b7d85401ec3

<div dir="rtl" align="right">

### NB22E-v4:C023 — اصلاح اتصال Handoff و تطبیق تاریخی Test

**علت:** در C022، فیلد `source_package_files` به‌اشتباه از Handoff خوانده شد؛ این فیلد متعلق به Manifest ارجاع‌شده در Handoff است.

**اقدام:**
۱. خواندن Handoff فریز‌شدهٔ NB19-C239 و Manifest معرفی‌شده در آن.
۲. احراز SHA و تعداد ردیف‌های سه منبع لازم: ورودی تاریخی، پاسخ تاریخی و رسید بررسی داخلی.
۳. تطبیق با ۱۱۱۶ درخواست موجود C020، بدون بازسازی درخواست یا Context.
۴. طبقه‌بندی غیرهم‌پوشان پاسخ‌ها بر اساس سؤال، Prompt، پاسخ و منشأ اجرای واقعی.
۵. بررسی جداگانهٔ قابلیت مصرف رأی تاریخی و رأی نسخهٔ فعلی.
۶. ثبت موارد بدون اتصال، همراه با علت و مسیر منبع.
۷. ساخت Ledger و Summary نسخه‌دار خارج از مخزن، با اتصال به C022.

**محدودیت:** پاسخ‌های NB19 با منشأ `CHATGPT_AGENT_RECOVERY` خودکار اجرای DeepSeek محسوب نمی‌شوند. بررسی داخلی و غیرمستقل NB19 نیز جای رأی مستقل محصول جدید نیست.

**هزینه:** از قیمت پنل و برآورد مشروط C022 استفاده می‌شود؛ عدد کسری واقعی تا احراز موجودی ساخته نمی‌شود.

**ممنوعیت:** Generation، داوری تازه، شبکه، Retry، تغییر Artifactهای قبلی و Git.

`generation_send_authorized = False`

`quality_gate = BLOCKED`

</div>

In [20]:
# NB22E-v4 | C023 | R11.2
# Fix NB19 handoff -> manifest -> pinned inputs/answers/internal gate.
# Offline only; no model, API, judge, Git or changes to C019-C022.

if C022_RECONCILIATION['quality_gate'] != 'BLOCKED' or C022_RECONCILIATION['dispatch_authorized']:
    raise RuntimeError('C023_UPSTREAM_GATE_CONFLICT')


def c023_need(ok, reason):
    if not ok:
        raise RuntimeError('C023_' + reason)


def c023_safe_reference(ref, label):
    relative = Path(ref['relative_path'])
    c023_need(not relative.is_absolute() and '..' not in relative.parts,
              'UNSAFE_SOURCE_REFERENCE:' + label)
    unresolved = repo / relative
    c023_need(not unresolved.is_symlink(), 'SYMLINK_SOURCE:' + label)
    path = unresolved.resolve()
    c023_need(path.is_relative_to(repo.resolve()) and path != repo.resolve(),
              'SOURCE_OUTSIDE_REPO:' + label)
    if not path.is_file():
        print('C023_MISSING_EXACT_SOURCE:', str(path))
        raise FileNotFoundError('C023_REQUIRED_SOURCE_MISSING:' + str(path))
    return path


def c023_read_json_ref(ref, label):
    path = c023_safe_reference(ref, label)
    raw = path.read_bytes()
    c023_need(c016_sha(raw) == ref['sha256'] and
              ('bytes' not in ref or len(raw) == ref['bytes']),
              'PINNED_JSON_SHA_OR_SIZE:' + label)
    return json.loads(raw), path


def c023_unique_pairs(pairs):
    result = {}
    for k, v in pairs:
        c023_need(k not in result, 'DUPLICATE_JSON_KEY')
        result[k] = v
    return result


def c023_reject_constant(value):
    raise ValueError('C023_NON_FINITE_JSON_CONSTANT')


def c023_find_question_slot(obj):
    if isinstance(obj, dict):
        if (isinstance(obj.get('question'), str) and
                isinstance(obj.get('evidence'), list) and
                obj['evidence'] and isinstance(obj['evidence'][0], dict)):
            return obj
        for value in obj.values():
            found = c023_find_question_slot(value)
            if found is not None:
                return found
    elif isinstance(obj, list):
        for value in obj:
            found = c023_find_question_slot(value)
            if found is not None:
                return found
    return None


def c023_stream_pinned_jsonl(ref, label, handler):
    path = c023_safe_reference(ref, label)
    digest = hashlib.sha256()
    rows = 0

    with path.open('rb') as stream:
        for raw in stream:
            digest.update(raw)
            c023_need(raw.strip() and len(raw) <= 2 * 1024 * 1024,
                      'ROW_BLANK_OR_TOO_LARGE:' + label)
            row = json.loads(
                raw,
                object_pairs_hook=c023_unique_pairs,
                parse_constant=c023_reject_constant
            )
            c023_need(isinstance(row, dict), 'ROW_NOT_OBJECT:' + label)
            handler(row)
            rows += 1
            c023_need(rows <= 1149, 'TOO_MANY_ROWS:' + label)

    c023_need(
        rows == 1149
        and digest.hexdigest() == ref['sha256']
        and ('bytes' not in ref or path.stat().st_size == ref['bytes']),
        'SOURCE_ROW_COUNT_OR_SHA_CONFLICT:' + label
    )

    return {
        'path': str(path),
        'file_sha256': ref['sha256'],
        'records': rows
    }


# A. Consume existing C020/C022 lineage; do not reopen 1116 payloads.

c023_c022 = c020_load_core(
    C022_RECONCILIATION['summary_path'],
    C022_RECONCILIATION['summary_sha256']
)

c023_need(
    c023_c022['historical_sources']['nb19_state']
        == "HISTORICAL_SOURCE_BLOCKED:KeyError:'source_package_files'"
    and c023_c022['generation_categories']
        == {'D_HISTORICAL_RESPONSE_BINDING_UNRESOLVED': 1116},
    'EXPECTED_C022_SCHEMA_FAILURE_NOT_FOUND'
)

c023_c020 = c020_load_core(
    C020_OFFLINE_PREPARATION['summary_path'],
    C020_OFFLINE_PREPARATION['summary_sha256']
)

c023_c020_ledger = c020_load_core(
    C020_OFFLINE_PREPARATION['ledger_path'],
    C020_OFFLINE_PREPARATION['ledger_sha256']
)

c023_current = c023_c020_ledger['entries']

c023_need(
    len(c023_current) == 1116
    and len({x['candidate_id'] for x in c023_current}) == 1116
    and {x['candidate_id'] for x in c023_current} == set(c016_questions)
    and c023_c020['identity_sha256'] == c020_identity_sha
    and c023_c020['dispatch_authorized'] is False,
    'C020_IDENTITY_AND_POPULATION_CONFLICT'
)


# B. Correct hop: Handoff -> Manifest -> source_package_files.

c023_handoff_path = (
    repo /
    'artifacts/phase2/generation_and_citation/work/outputs/'
    'nb19_test_handoff_chatgpt_recovery_v1.json'
).resolve()

c023_need(
    c023_handoff_path.is_file() and not c023_handoff_path.is_symlink(),
    'NB19_HANDOFF_NOT_AVAILABLE:' + str(c023_handoff_path)
)

c023_handoff_raw = c023_handoff_path.read_bytes()

c023_handoff_sha = (
    'c789c8a60ff7aae36e32272b878f834fef90c1bda608af986c53581f5b77c40a'
)

c023_need(
    c016_sha(c023_handoff_raw) == c023_handoff_sha,
    'NB19_HANDOFF_SHA_CHANGED'
)

c023_handoff = json.loads(c023_handoff_raw)

c023_manifest, c023_manifest_path = c023_read_json_ref(
    c023_handoff['artifact_roles']['manifest'],
    'NB19_FROZEN_MANIFEST'
)

c023_need(
    c023_handoff['schema_version'] == 'nb19_test_recovery_handoff_v1'
    and c023_manifest['schema_version'] == 'nb19_test_recovery_manifest_v1'
    and c023_manifest['data_contract_validated'] is True
    and c023_handoff['counts'] == c023_manifest['counts']
    and c023_manifest['counts']['candidate_count'] == 1149
    and c023_handoff['scope'] == c023_manifest['scope']
    and c023_manifest['scope']['generation_origin']
        == 'CHATGPT_AGENT_RECOVERY'
    and c023_manifest['scope']['exact_frozen_messages_execution_claimed'] is False
    and c023_manifest['scope']['support_independent'] is False,
    'NB19_HANDOFF_MANIFEST_SCOPE_MISMATCH'
)

c023_refs = {
    'historical_input':
        c023_manifest['source_package_files']['consolidated/inputs.jsonl'],
    'historical_answer':
        c023_handoff['artifact_roles']['generation_citation'],
    'historical_internal_review':
        c023_handoff['artifact_roles']['internal_claim_gate']
}

c023_need(
    c023_manifest['artifacts']['generation_citation']
        == c023_refs['historical_answer']
    and c023_manifest['artifacts']['internal_claim_gate']
        == c023_refs['historical_internal_review'],
    'NB19_MANIFEST_ARTIFACT_ROLE_MISMATCH'
)


# C. Read pinned JSONL once; retain only hashes and statuses.

c023_inputs = {}
c023_answers = {}
c023_reviews = {}


def c023_take_input(row):
    cid = row['candidate_id']

    c023_need(
        cid not in c023_inputs and row['split'] == 'test',
        'DUPLICATE_OR_INVALID_INPUT:' + cid
    )

    messages = row['messages']

    c023_need(
        isinstance(messages, list)
        and c016_sha(c016_canon(messages)) == row['messages_sha256'],
        'HISTORICAL_MESSAGE_SHA_CONFLICT:' + cid
    )

    systems = [
        m['content'] for m in messages if m.get('role') == 'system'
    ]
    users = [
        m['content'] for m in messages if m.get('role') == 'user'
    ]

    c023_need(
        len(systems) == len(users) == 1
        and all(isinstance(x, str) for x in systems + users),
        'HISTORICAL_MESSAGE_SHAPE:' + cid
    )

    slot = c023_find_question_slot(json.loads(users[0]))

    c023_need(
        slot is not None,
        'HISTORICAL_QUESTION_SLOT_NOT_FOUND:' + cid
    )

    c023_inputs[cid] = {
        'question_sha256':
            c016_sha(slot['question'].encode('utf-8')),
        'system_prompt_sha256':
            c016_sha(systems[0].encode('utf-8')),
        'message_sha256': row['messages_sha256'],
        'user_message_format': 'NB19_JSON_EVIDENCE_PACKET'
    }


def c023_take_answer(row):
    cid = row['candidate_id']

    c023_need(
        cid not in c023_answers
        and row['split'] == 'test'
        and row['schema_version'] == 'nb19_test_recovery_output_v1',
        'DUPLICATE_OR_INVALID_ANSWER:' + cid
    )

    meta = row['recovery_metadata']
    response = row['recovery_response']

    c023_need(
        meta['generation_origin'] == 'CHATGPT_AGENT_RECOVERY'
        and meta['model_identity'] == 'NOT_VERIFIED'
        and c016_sha(c016_canon(response)) == meta['active_response_sha256']
        and isinstance(response['answer_text'], str),
        'HISTORICAL_RESPONSE_SCHEMA_OR_SHA_CONFLICT:' + cid
    )

    c023_answers[cid] = {
        'answer_text_sha256':
            c016_sha(response['answer_text'].encode('utf-8')),
        'has_nonempty_text':
            bool(response['answer_text'].strip()),
        'input_message_sha256':
            meta['package_messages_sha256'],
        'core_record_sha256':
            c016_sha(c016_canon(row)),
        'recovery_status': meta['recovery_status'],
        'model_execution_identity': meta['model_identity']
    }


def c023_take_review(row):
    cid = row['candidate_id']

    c023_need(
        cid not in c023_reviews
        and row['split'] == 'test'
        and row['schema_version']
            == 'nb19_test_recovery_internal_claim_gate_v1',
        'DUPLICATE_OR_INVALID_REVIEW:' + cid
    )

    c023_reviews[cid] = {
        'source_core_sha256': row['source_core_record_sha256'],
        'route': row['route'],
        'independent':
            row['recovery_metadata']['support_independent'],
        'gate_has_no_answer_text': row['answer_text'] is None
    }


c023_source_receipts = {}

for name, consumer in (
    ('historical_input', c023_take_input),
    ('historical_answer', c023_take_answer),
    ('historical_internal_review', c023_take_review)
):
    c023_source_receipts[name] = c023_stream_pinned_jsonl(
        c023_refs[name], name, consumer
    )

c023_need(
    set(c023_inputs) == set(c023_answers) == set(c023_reviews)
    and len(c023_inputs) == 1149,
    'NB19_THREE_ROLE_POPULATIONS_DIFFER'
)

print('C023_NB19_SOURCES_LOADED: 1149/1149')


# D. One nonoverlapping category per C020 request.

c023_rows = []
c023_counts = collections.Counter()
c023_review_counts = collections.Counter()

c023_target_system_sha = (
    c023_c020['identity']['system_prompt_sha256']
)

c023_need(
    c023_c020['request_message_format_verified'] is True,
    'C020_FROZEN_WRAPPER_NOT_VERIFIED'
)

for current in c023_current:
    cid = current['candidate_id']

    c023_need(
        current['status'] == 'OFFLINE_READY_DISPATCH_GATED'
        and cid in c016_questions,
        'C020_CURRENT_CASE_NOT_READY:' + cid
    )

    incoming = c023_inputs.get(cid)
    answer = c023_answers.get(cid)
    review = c023_reviews.get(cid)

    reasons = []
    old_answer_sha = None
    verdict = 'HISTORICAL_VERDICT_NOT_BOUND'
    target_verdict = 'NOT_DETERMINED_NO_CURRENT_GENERATION'

    if (
        current['attempt_status']
            != 'NO_PREVIOUS_ATTEMPT_FOR_NEW_IDENTITY'
        or current['cache_status']
            != 'NO_MATCHING_SCOPED_CACHE'
    ):
        category = 'D_SCOPED_ATTEMPT_OR_CACHE_REQUIRES_RECONCILIATION'
        reasons.append('C020_LOCAL_ATTEMPT_OR_CACHE_NOT_CLEAN')

    elif incoming is None and answer is None and review is None:
        category = 'E_NOT_FOUND_IN_PINNED_NB19_1149_ONLY'
        reasons.append('NOT_GLOBALLY_PROVEN_MISSING')

    elif incoming is None or answer is None or review is None:
        category = 'D_INCOMPLETE_NB19_ROLE_BINDING'
        reasons.append('NB19_INPUT_ANSWER_REVIEW_SET_PARTIAL')

    elif incoming['question_sha256'] != c016_questions[cid]:
        category = 'D_SAME_ID_DIFFERENT_QUESTION_TEXT'
        reasons.append('QUESTION_SHA256_CONFLICT')

    elif answer['input_message_sha256'] != incoming['message_sha256']:
        category = 'D_RESPONSE_NOT_BOUND_TO_HISTORICAL_INPUT'
        reasons.append('NB19_INPUT_MESSAGE_SHA256_CONFLICT')

    elif (
        review['source_core_sha256'] != answer['core_record_sha256']
        or not review['gate_has_no_answer_text']
        or review['independent'] is not False
    ):
        category = 'D_HISTORICAL_REVIEW_LINK_CONFLICT'
        reasons.append('NB19_INTERNAL_GATE_CORE_OR_SCOPE_CONFLICT')

    else:
        old_answer_sha = answer['answer_text_sha256']

        if review['route'] == 'INTERNAL_SUPPORT_ELIGIBLE':
            verdict = (
                'MECHANICAL_ADAPTER_POSSIBLE_FOR_HISTORICAL_ANSWER_ONLY'
            )
        elif review['route'] == 'BLOCKED_FOR_REVIEW':
            verdict = 'HISTORICAL_REVIEW_HELD_NOT_TRANSFERABLE'
        else:
            verdict = 'HISTORICAL_REVIEW_ROUTE_UNRECOGNIZED'

        if not answer['has_nonempty_text']:
            category = 'D_HISTORICAL_TEXT_EMPTY_NOT_REUSABLE'
            reasons.append('NO_NONEMPTY_PRODUCT_ANSWER_TEXT')

        elif incoming['system_prompt_sha256'] != c023_target_system_sha:
            category = 'B_EFFECTIVE_SYSTEM_PROMPT_MISMATCH'
            reasons.extend([
                'SYSTEM_PROMPT_SHA256_DIFFERS',
                'NB19_JSON_USER_MESSAGE_VS_C340_FROZEN_WRAPPER'
            ])
            target_verdict = (
                'NOT_TRANSFERABLE_DIFFERENT_PROMPT_AND_EVIDENCE'
            )

        else:
            category = 'B_EFFECTIVE_USER_PAYLOAD_FORMAT_MISMATCH'
            reasons.append(
                'NB19_JSON_USER_MESSAGE_VS_C340_FROZEN_WRAPPER'
            )
            target_verdict = (
                'NOT_TRANSFERABLE_DIFFERENT_REQUEST_STRUCTURE'
            )

    c023_counts[category] += 1
    c023_review_counts[verdict] += 1

    c023_rows.append({
        'candidate_id': cid,
        'c020_request_id': current['request_id'],
        'c020_request_file_sha256':
            current['request_file_sha256'],
        'question_sha256': c016_questions[cid],
        'generation_inventory_category': category,
        'reasons': reasons,
        'historical_answer_text_sha256': old_answer_sha,
        'historical_answer_provenance':
            'NB19_CHATGPT_AGENT_RECOVERY'
            if old_answer_sha else None,
        'historical_review_reuse_scope': verdict,
        'current_generation_verdict_transfer': target_verdict,
        'independent_reference':
            'SEPARATE_ASSET_NOT_REEVALUATED_IN_C023',
        'cache_same_execution_proven': False,
        'global_generation_deficit_proven': False,
        'automatic_retry_permitted': False,
        'generation_send_authorized': False
    })

c023_need(
    len(c023_rows)
        == sum(c023_counts.values())
        == sum(c023_review_counts.values())
        == 1116,
    'NONOVERLAPPING_RECONCILIATION_COUNTS_CONFLICT'
)


# E. New immutable outputs; never overwrite C022.

c023_identity = {
    'schema': 'NB22E_V4_C023_NB19_MANIFEST_HOP_REPAIR_V1',
    'c011_boundary_sha256': c016_boundary,
    'c020_ledger_file_sha256':
        C020_OFFLINE_PREPARATION['ledger_sha256'],
    'c022_summary_file_sha256':
        C022_RECONCILIATION['summary_sha256'],
    'nb19_handoff_file_sha256': c023_handoff_sha,
    'nb19_manifest_file_sha256':
        c023_handoff['artifact_roles']['manifest']['sha256'],
    'source_file_sha256': {
        k: v['sha256'] for k, v in c023_refs.items()
    },
    'category_scope': 'NB19_C239_PINNED_1149_NOT_GLOBAL_INVENTORY',
    'generation_send_authorized': False
}

c023_identity_sha = c016_sha(c016_canon(c023_identity))

c023_out = (
    outputs / ('C023_scoped_generation_history_' + c023_identity_sha)
).resolve()

c023_need(
    c023_out.parent == outputs.resolve()
    and not c023_out.is_relative_to(repo.resolve()),
    'OUTPUT_MUST_BE_OUTSIDE_REPO'
)

c023_out.mkdir(parents=True, exist_ok=True)

c023_ledger_path = (
    c023_out / ('C023.rows.' + c023_identity_sha + '.jsonl')
)

c023_expected = hashlib.sha256()

if not c023_ledger_path.exists():
    with tempfile.NamedTemporaryFile(
        mode='wb',
        dir=c023_out,
        prefix='.C023.',
        suffix='.tmp',
        delete=False
    ) as stream:
        c023_temp = Path(stream.name)

        for r in c023_rows:
            line = c016_canon(r) + b'\n'
            c023_expected.update(line)
            stream.write(line)

        stream.flush()
        os.fsync(stream.fileno())

    try:
        os.link(c023_temp, c023_ledger_path)
    except FileExistsError:
        pass
    finally:
        c023_temp.unlink(missing_ok=True)

else:
    for r in c023_rows:
        c023_expected.update(c016_canon(r) + b'\n')

c023_ledger_sha = c023_expected.hexdigest()
c023_reload_hash = hashlib.sha256()
c023_reload_count = 0

with c023_ledger_path.open('rb') as stream:
    for line in stream:
        c023_reload_hash.update(line)
        c023_reload_count += 1

c023_need(
    c023_reload_hash.hexdigest() == c023_ledger_sha
    and c023_reload_count == 1116,
    'C023_LEDGER_RELOAD_SHA_OR_COUNT_CONFLICT'
)


# F. Versioned summary, including the exact C022 correction.

c023_summary = {
    'cell': 'NB22E-v4:C023',
    'status':
        'PINNED_NB19_1149_RECONCILED_CURRENT_GENERATION_STILL_HELD',
    'identity': c023_identity,
    'identity_sha256': c023_identity_sha,
    'population': 1116,
    'historical_source_population': 1149,
    'generation_categories': dict(sorted(c023_counts.items())),
    'historical_review_categories':
        dict(sorted(c023_review_counts.items())),
    'reusable_exact_generation_cache_proven': 0,
    'global_generation_deficit': None,
    'independent_reference_asset_transfer':
        'NOT_ASSESSED_NO_AUTO_INVALIDATION',
    'historical_33_case_packet_outside_c011':
        c023_c020_ledger['historical_separation'],
    'pinned_sources': c023_source_receipts,
    'nb19_handoff_path': str(c023_handoff_path),
    'nb19_manifest_path': str(c023_manifest_path),
    'superseded_c022_classification_reason':
        'SOURCE_PACKAGE_FILES_BELONG_TO_MANIFEST_NOT_HANDOFF',
    'previous_c022_summary_sha256':
        C022_RECONCILIATION['summary_sha256'],
    'panel_tariff_scenario_from_c022':
        c023_c022['owner_panel_tariff_scenario'],
    'generation_send_authorized': False,
    'quality_gate': 'BLOCKED',
    'new_model_calls': 0,
    'new_judge_calls': 0,
    'new_api_calls': 0,
    'ledger_path': str(c023_ledger_path),
    'ledger_file_sha256': c023_ledger_sha,
    'next_required_boundary':
        'PROVIDER_ADAPTER_CAPACITY_ATTEMPT_COST_AND_PILOT_SCOPE'
}

c023_summary_path = c023_out / (
    'C023.summary.'
    + c016_sha(c016_canon(c023_summary))
    + '.json'
)

c023_summary_sha, _ = c016_put_once(
    c023_summary_path,
    c023_summary
)

c023_saved = c020_load_core(
    c023_summary_path,
    c023_summary_sha
)

c023_need(
    c023_saved['ledger_file_sha256'] == c023_ledger_sha
    and c023_saved['generation_categories'] == dict(c023_counts),
    'C023_SUMMARY_RELOAD_CONFLICT'
)

C023_RECONCILIATION = {
    'status': c023_summary['status'],
    'summary_path': str(c023_summary_path),
    'summary_sha256': c023_summary_sha,
    'ledger_sha256': c023_ledger_sha,
    'generation_send_authorized': False,
    'quality_gate': 'BLOCKED'
}

print('C023_NB19_CHAIN_VERIFIED: 1149/1149')
print('C023_CURRENT_COVERAGE: 1116/1116')
print('C023_GENERATION_CATEGORIES:', dict(sorted(c023_counts.items())))
print('C023_HISTORICAL_REVIEW:', dict(sorted(c023_review_counts.items())))
print('C023_GLOBAL_GENERATION_DEFICIT: NOT_PROVEN')
print('C023_LEDGER_SHA256:', c023_ledger_sha)
print('C023_SUMMARY_SHA256:', c023_summary_sha)
print('C023_STATUS:', C023_RECONCILIATION['status'])
print('GENERATION_SEND_AUTHORIZED: False | quality_gate: BLOCKED')

C023_NB19_SOURCES_LOADED: 1149/1149
C023_NB19_CHAIN_VERIFIED: 1149/1149
C023_CURRENT_COVERAGE: 1116/1116
C023_GENERATION_CATEGORIES: {'B_EFFECTIVE_SYSTEM_PROMPT_MISMATCH': 1116}
C023_HISTORICAL_REVIEW: {'HISTORICAL_REVIEW_HELD_NOT_TRANSFERABLE': 70, 'MECHANICAL_ADAPTER_POSSIBLE_FOR_HISTORICAL_ANSWER_ONLY': 1046}
C023_GLOBAL_GENERATION_DEFICIT: NOT_PROVEN
C023_LEDGER_SHA256: 321db21e9a59e26864962aca11a2a49d67af9973d06ff92587a573af44ca2fb1
C023_SUMMARY_SHA256: c8f78000ac8bbbca0cdd6276c5f083a1bf98f39de52492be49a55d07cf6d19dd
C023_STATUS: PINNED_NB19_1149_RECONCILED_CURRENT_GENERATION_STILL_HELD
GENERATION_SEND_AUTHORIZED: False | quality_gate: BLOCKED


<div dir="rtl" align="right">

### NB22E-v4:C024 — تثبیت محدود Gateway، موجودی و پیش‌نیازهای Pilot

**مبنای پذیرفته‌شده:** C019 تا C023، بدون تکرار Scoring، Rank، Context، درخواست‌سازی یا اتصال NB19.

**شواهد تاریخی:**
- C023: تعداد ۱۱۱۶ پاسخ تاریخی با اختلاف مؤثر Prompt
- ۱۰۴۶ بررسی داخلی با امکان Adapter مکانیکی، محدود به پاسخ تاریخی
- ۷۰ بررسی داخلی Held
- C342: بستهٔ ۳۳تایی NB22E خارج از جمعیت فعلی C011
- API-DIAG-C07/C08: شاهد تاریخی مسیر، مدل و خطر مصرف Reasoning بدون پاسخ نهایی

**اقدام اجرایی:**

۱. مصرف Summary و Ledger فریز‌شدهٔ C020/C023 و مرز پذیرفته‌شدهٔ C342.

۲. بررسی آفلاین تنظیمات محدود `ARVAN_GATEWAY` از `.env` پروژه. مقدار Secret و URL کامل چاپ یا ذخیره نمی‌شود.

۳. مقایسهٔ اثر انگشت مسیر فعلی با مسیر آزمون تاریخی C07، بدون نتیجه‌گیری دربارهٔ دسترسی زنده.

۴. تثبیت پروفایل درخواست C340: `DeepSeek-V4-Flash`، دمای صفر، خروجی ۶۰۰ و Thinking غیرفعال. هر تنظیم یا پشتیبانی احرازنشده جدا Held می‌ماند.

۵. کنترل تخمینی ظرفیت با Context یک‌میلیونی اعلام‌شده در پنل؛ شمارش تقریبی توکن، تأیید قطعی ظرفیت نیست.

۶. تعیین شمارهای تفکیک‌شدهٔ Cache سازگار، Attempt نامعلوم، نامزدهای ارسال در موجودی محدود و موارد دارای مانع فنی ارسال.

۷. مصرف تعرفهٔ پنل آروان و برآورد محفوظ C022/C023 برای Pilot سه‌تایی و جمعیت ۱۱۱۶تایی؛ بدون ساختن توکن یا قیمت جدید.

۸. تهیهٔ قرارداد یک Worker، صفر Retry خودکار، ثبت Attempt قبل از ارسال، ذخیرهٔ Raw و Usage پیش از Parse و Resume بدون ارسال مجدد نتیجهٔ نامعلوم.

**خروجی:** Ledger ردیفی و Summary نسخه‌دار خارج از مخزن با SHA و شمارهای غیرهم‌پوشان.

**شرط Pilot:** فقط پس از احراز Provider، Adapter، ظرفیت و کنترل Attempt/Cache. مجوز Pilot به معنای مجوز اجرای جمعیتی نیست.

**ممنوعیت:** Generation، Judge، API، اجرای Pilot، تغییر Artifactهای موفق و Git.

`generation_send_authorized = False`

`quality_gate = BLOCKED`

</div>

In [21]:
# NB22E-v4 | C024 | R11.2
# Scoped C023/C342 inventory, historical ARVAN_GATEWAY routing and pre-dispatch plan.
# Offline only: no Generation, Judge, metadata network call, Git or old-artifact edits.

c024_required = (
    'C023_RECONCILIATION', 'C020_OFFLINE_PREPARATION',
    'C021_OFFLINE_GATES', 'NB22E_V4_TRANSFER',
    'c002_summary', 'c002_questions', 'c015_member_ids',
    'c016_boundary', 'repo', 'outputs', 'c020_load_core',
    'c016_sha', 'c016_canon', 'c016_put_once'
)
if any(name not in globals() for name in c024_required):
    raise RuntimeError('C024_REQUIRED_PRIOR_NOTEBOOK_STATE_MISSING')
if (C023_RECONCILIATION['quality_gate'] != 'BLOCKED'
        or C023_RECONCILIATION['generation_send_authorized'] is not False
        or C020_OFFLINE_PREPARATION['offline_requests'] != 1116
        or C021_OFFLINE_GATES['pilot_send_authorized'] is not False):
    raise RuntimeError('C024_UPSTREAM_SCOPE_OR_DISPATCH_GATE_CHANGED')


def c024_need(ok, reason):
    if not ok:
        raise RuntimeError('C024_' + reason)


# 1. Consume already verified boundary and archived results; do not re-open NB19.
c024_c023 = c020_load_core(
    C023_RECONCILIATION['summary_path'],
    C023_RECONCILIATION['summary_sha256']
)
c024_c020 = c020_load_core(
    C020_OFFLINE_PREPARATION['summary_path'],
    C020_OFFLINE_PREPARATION['summary_sha256']
)
c024_c021 = c020_load_core(
    C021_OFFLINE_GATES['summary_path'],
    C021_OFFLINE_GATES['summary_sha256']
)
c024_current_doc = c020_load_core(
    C020_OFFLINE_PREPARATION['ledger_path'],
    C020_OFFLINE_PREPARATION['ledger_sha256']
)
c024_current = {r['candidate_id']: r for r in c024_current_doc['entries']}
c024_pilot = c024_c020['pilot_proposal']['items']
c024_pilot_ids = {r['request_id'] for r in c024_pilot}

c024_need(
    c024_c023['population'] == 1116
    and c024_c023['generation_categories']
        == {'B_EFFECTIVE_SYSTEM_PROMPT_MISMATCH': 1116}
    and c024_c023['historical_review_categories'] == {
        'MECHANICAL_ADAPTER_POSSIBLE_FOR_HISTORICAL_ANSWER_ONLY': 1046,
        'HISTORICAL_REVIEW_HELD_NOT_TRANSFERABLE': 70
    }
    and c024_c023['global_generation_deficit'] is None
    and c024_c020['identity']['c011_boundary_sha256'] == c016_boundary
    and c024_c021['capacity_held_count'] == 1116
    and c024_c021['context_capacity_verified_count'] == 0
    and len(c024_current) == 1116
    and set(c024_current) == set(c015_member_ids)
    and len(c024_pilot_ids) == len(c024_pilot) == 3,
    'PREVIOUS_VERIFIED_COUNT_OR_IDENTITY_CONFLICT'
)

c024_historical_packet = c002_summary['preserved']
c024_need(
    c024_historical_packet == {
        'EXACT_PRESERVED': 29,
        'MODEL_IDENTITY_HELD': 2,
        'RECORDED_ERROR_HELD': 2,
        'OUTSIDE_PACKET': 1116
    }
    and c002_summary['existing_final_rank_coverage'] == 'C189_ONLY_33_VERIFIED'
    and set(c002_questions) == set(c015_member_ids)
    and NB22E_V4_TRANSFER['candidate_count'] == 1116
    and NB22E_V4_TRANSFER['artifact_sha256']['summary']
        == 'fe34926715265f4595dc10084ab18a3dcaca0fbbaf72b37de323466648616b3b',
    'C342_33_CASE_COMPLEMENT_OR_C011_BINDING_CONFLICT'
)
# C342 establishes that its 29+2+2 cases are outside this exact C011 population.
# Do not read or reclassify the 33 cases and do not count them as C020 Cache.


# 2. Read only existing saved API-DIAG-C07/C08, if that exact notebook is present.
c024_diag_path = repo / 'notebooks/phase2/API_connection_diagnostics_v1.ipynb'
c024_diag_source_sha = None
c024_diag_c07 = None
c024_diag_c08 = None
c024_diag_historical_route_sha = None
c024_diag_state = 'SAVED_DIAGNOSTIC_NOTEBOOK_NOT_FOUND'


def c024_diagnostic_output(cell, required):
    stdout = ''.join(
        ''.join(x.get('text', []))
        for x in cell.get('outputs', [])
        if x.get('output_type') == 'stream'
    )
    decoder = json.JSONDecoder()
    for pos, ch in enumerate(stdout):
        if ch == '{':
            try:
                parsed, _ = decoder.raw_decode(stdout[pos:])
            except ValueError:
                continue
            if isinstance(parsed, dict) and required.issubset(parsed):
                return parsed
    return None


if c024_diag_path.is_file() and not c024_diag_path.is_symlink():
    c024_diag_bytes = c024_diag_path.read_bytes()
    c024_diag_source_sha = c016_sha(c024_diag_bytes)
    c024_diag_json = json.loads(c024_diag_bytes)
    c024_cells = {}
    for cell in c024_diag_json['cells']:
        first = ''.join(cell.get('source', [])).split('\n', 1)[0]
        if first.startswith('# API-DIAG-C07'):
            c024_need('C07' not in c024_cells, 'DUPLICATE_DIAGNOSTIC_C07')
            c024_cells['C07'] = cell
        elif first.startswith('# API-DIAG-C08'):
            c024_need('C08' not in c024_cells, 'DUPLICATE_DIAGNOSTIC_C08')
            c024_cells['C08'] = cell
    c024_need(set(c024_cells) == {'C07', 'C08'}, 'DIAGNOSTIC_C07_C08_MISSING')
    c024_diag_c07 = c024_diagnostic_output(
        c024_cells['C07'], {'status', 'configured_model', 'response_model', 'http_status'}
    )
    c024_diag_c08 = c024_diagnostic_output(
        c024_cells['C08'], {'status', 'results', 'api_request_attempts'}
    )
    c024_need(c024_diag_c07 is not None and c024_diag_c08 is not None,
              'SAVED_DIAGNOSTIC_OUTPUTS_NOT_DECODABLE')
    c024_need(
        c024_diag_c07['status'] == 'DEEPSEEK_SMALL_CHAT_PASS'
        and c024_diag_c07['configured_model'] == 'DeepSeek-V4-Flash'
        and c024_diag_c07['response_model'] == 'DeepSeek-V4-Flash'
        and c024_diag_c07['http_status'] == 200
        and c024_diag_c07['authorization_scheme'] == 'apikey'
        and c024_diag_c07['automatic_retries'] == 0,
        'HISTORICAL_C07_MODEL_OR_ROUTE_NOT_VERIFIED'
    )
    c024_diag_c08_deepseek = [
        r for r in c024_diag_c08['results']
        if r.get('model') == 'DeepSeek-V4-Flash'
    ]
    c024_need(
        len(c024_diag_c08_deepseek) == 1
        and c024_diag_c08_deepseek[0].get('http_status') == 200
        and c024_diag_c08_deepseek[0].get('finish_reason') == 'length'
        and c024_diag_c08_deepseek[0].get('usage', {})
            .get('completion_tokens_details', {}).get('reasoning_tokens') == 4096,
        'HISTORICAL_C08_REASONING_RESULT_CHANGED'
    )
    c024_c07_code = ''.join(c024_cells['C07']['source'])
    c024_match = re.search(r'if base != [\"\']([^\"\']+)[\"\']:', c024_c07_code)
    c024_need(c024_match is not None, 'HISTORICAL_DIAGNOSTIC_ROUTE_NOT_PINNED')
    c024_historical_base = c024_match.group(1).rstrip('/')
    c024_diag_historical_route_sha = c016_sha(
        (c024_historical_base + '/chat/completions').encode('utf-8')
    )
    c024_diag_state = 'HISTORICAL_C07_PASS_AND_C08_REASONING_RISK_CONFIRMED'
    del c024_historical_base, c024_diag_bytes, c024_diag_json


# 3. Inspect only the current project .env gateway settings, never emit secrets.
c024_env_path = repo / '.env'
c024_env_fields = {}
c024_credential_ok = False
c024_credential_alias_ok = False
c024_env_duplicate = False
c024_env_error = None
c024_env_location = 'PROJECT_DOTENV'

if c024_env_path.is_file() and not c024_env_path.is_symlink():
    with c024_env_path.open('r', encoding='utf-8-sig') as source:
        for line in source:
            match = re.match(
                r'^\s*(?:export\s+)?'
                r'(ARVAN_GATEWAY_URL|ARVAN_GATEWAY_API_KEY|ARVAN_GATEWAY_KEY|ARVAN_GATEWAY_MODEL)'
                r'\s*=\s*(.*)$', line.rstrip('\n')
            )
            if match is None:
                continue
            name, value = match.groups()
            if name in c024_env_fields:
                c024_env_duplicate = True
                continue
            value = value.strip()
            if value[:1] in ('\"', "'"):
                if len(value) < 2 or value[-1] != value[0]:
                    c024_env_error = 'ENV_VALUE_QUOTE_INVALID'
                    value = ''
                else:
                    value = value[1:-1]
            else:
                value = value.split(' #', 1)[0].strip()
            if name in ('ARVAN_GATEWAY_API_KEY', 'ARVAN_GATEWAY_KEY'):
                is_valid = bool(value) and '*' not in value and not value.startswith('<')
                c024_env_fields[name] = 'PRESENT' if is_valid else 'MISSING_OR_MASKED'
                if name == 'ARVAN_GATEWAY_API_KEY':
                    c024_credential_ok = is_valid
                else:
                    c024_credential_alias_ok = is_valid
            else:
                c024_env_fields[name] = value
            del value
else:
    c024_env_location = 'PROJECT_DOTENV_NOT_AVAILABLE'

c024_model_env = c024_env_fields.pop('ARVAN_GATEWAY_MODEL', None)
c024_model_override_class = (
    'MATCHES_C340' if c024_model_env == 'DeepSeek-V4-Flash'
    else 'UNSET_REQUEST_BODY_MODEL_IS_AUTHORITATIVE' if c024_model_env is None
    else 'DIFFERS_FROM_C340_REQUEST_BODY_NOT_USED'
)
del c024_model_env

c024_configured_url = c024_env_fields.pop('ARVAN_GATEWAY_URL', None)
c024_endpoint_sha = None
c024_endpoint_shape = 'URL_NOT_CONFIGURED'
c024_route_matches_historical_c07 = False

if c024_env_duplicate:
    c024_env_error = 'DUPLICATE_GATEWAY_SETTINGS'
elif c024_configured_url:
    parts = urlsplit(c024_configured_url)
    if (parts.scheme != 'https' or not parts.hostname or parts.username
            or parts.password or parts.query or parts.fragment):
        c024_env_error = 'HTTPS_URL_OR_CREDENTIAL_COMPONENT_INVALID'
    else:
        base_path = parts.path.rstrip('/')
        if base_path.endswith('/chat/completions'):
            endpoint_path = base_path
            c024_endpoint_shape = 'EXPLICIT_CHAT_COMPLETIONS'
        elif base_path == '/v1':
            endpoint_path = '/v1/chat/completions'
            c024_endpoint_shape = 'V1_PLUS_CHAT_COMPLETIONS'
        elif base_path == '':
            endpoint_path = '/v1/chat/completions'
            c024_endpoint_shape = 'ROOT_PLUS_V1_CHAT_COMPLETIONS'
        else:
            c024_env_error = 'UNRECOGNIZED_GATEWAY_PATH'
            endpoint_path = None
        if endpoint_path is not None:
            endpoint = urllib.parse.urlunsplit((
                parts.scheme, parts.netloc, endpoint_path, '', ''
            ))
            c024_endpoint_sha = c016_sha(endpoint.encode('utf-8'))
            c024_route_matches_historical_c07 = (
                c024_diag_historical_route_sha is not None
                and c024_endpoint_sha == c024_diag_historical_route_sha
            )
            del endpoint
    del parts

del c024_configured_url
c024_env_fields.clear()


# 4. Derive bounded per-case inventory from C023 only; do not reopen NB19.
c024_c023_rows_path = Path(c024_c023['ledger_path']).resolve()
c024_need(c024_c023_rows_path.is_file()
          and c024_c023_rows_path.parent
            == Path(C023_RECONCILIATION['summary_path']).resolve().parent,
          'C023_LEDGER_PATH_CHANGED')

c024_current_counts = collections.Counter()
c024_pilot_planned = []
c024_provisional_capacity = 0
c024_max_proxy_tokens = 0
c024_ledger_rows = []
c024_seen = set()
c024_c023_hasher = hashlib.sha256()

with c024_c023_rows_path.open('rb') as stream:
    for raw in stream:
        c024_c023_hasher.update(raw)
        record = json.loads(raw)
        cid = record['candidate_id']
        c024_need(cid in c024_current and cid not in c024_seen,
                  'C023_C020_QUESTION_OR_DUPLICATE_CONFLICT')
        c024_seen.add(cid)
        source = c024_current[cid]
        c024_need(
            record['c020_request_id'] == source['request_id']
            and record['c020_request_file_sha256'] == source['request_file_sha256']
            and record['question_sha256'] == c002_questions[cid]['question_sha256']
            and record['generation_inventory_category']
                == 'B_EFFECTIVE_SYSTEM_PROMPT_MISMATCH',
            'C023_LEDGER_REQUEST_OR_HISTORICAL_MISMATCH_CONFLICT'
        )
        attempt_status = source['attempt_status']
        cache_status = source['cache_status']
        current_safe = (
            attempt_status == 'NO_PREVIOUS_ATTEMPT_FOR_NEW_IDENTITY'
            and cache_status == 'NO_MATCHING_SCOPED_CACHE'
        )
        if attempt_status != 'NO_PREVIOUS_ATTEMPT_FOR_NEW_IDENTITY':
            state = 'HELD_SCOPED_ATTEMPT_RECONCILIATION'
            c024_current_counts['scoped_unknown_or_nonempty_attempts'] += 1
        elif cache_status != 'NO_MATCHING_SCOPED_CACHE':
            state = 'HELD_SCOPED_CACHE_RECONCILIATION'
            c024_current_counts['scoped_cache_needs_review'] += 1
        else:
            state = 'SCOPED_GENERATION_CANDIDATE_NOT_DISPATCH_AUTHORIZED'
            c024_current_counts['bounded_inventory_candidates'] += 1

        measure = source['token_measurement']
        upper_proxy = measure['token_proxy_high_at_2_bytes_per_token']
        c024_need(isinstance(upper_proxy, int) and upper_proxy >= 0,
                  'C020_TOKEN_PROXY_NOT_INTEGER')
        c024_max_proxy_tokens = max(c024_max_proxy_tokens, upper_proxy + 600)
        if upper_proxy + 600 < 1000000:
            c024_provisional_capacity += 1
        c024_current_counts['historical_NB19_prompt_mismatch'] += 1
        if source['request_id'] in c024_pilot_ids:
            c024_pilot_planned.append({
                'candidate_id': cid,
                'request_id': source['request_id'],
                'request_sha256': source['request_file_sha256'],
                'eligible_by_bounded_inventory': current_safe,
                'dispatch_authorized': False
            })
        c024_ledger_rows.append({
            'candidate_id': cid,
            'request_id': source['request_id'],
            'request_file_sha256': source['request_file_sha256'],
            'c023_generation_inventory': record['generation_inventory_category'],
            'c342_prior_33_scope': 'OUTSIDE_CURRENT_C011_POPULATION',
            'c020_cache': cache_status,
            'c020_attempt': attempt_status,
            'bounded_inventory_status': state,
            'generation_receipt_same_execution_proven': False,
            'attempt_auto_retry_allowed': False,
            'provider_capacity_checked_exactly': False,
            'dispatch_authorized': False
        })

c024_need(
    c024_c023_hasher.hexdigest() == C023_RECONCILIATION['ledger_sha256']
    and len(c024_seen) == 1116
    and len(c024_ledger_rows) == 1116
    and len(c024_pilot_planned) == 3,
    'C023_LEDGER_SHA_OR_COVERAGE_CONFLICT'
)


# 5. Existing tariff and C021 proxy estimates; no guessed new tokens.
c024_tariff = c024_c023['panel_tariff_scenario_from_c022']
c024_need(
    c024_tariff['display_model'] == 'DeepSeek-V4-Flash'
    and c024_tariff['rial_per_million_input'] == 450000
    and c024_tariff['rial_per_million_output'] == 900000
    and c024_tariff['toman_per_million_input'] == 45000
    and c024_tariff['toman_per_million_output'] == 90000
    and c024_c020['offline_requests_ready'] == 1116,
    'ARVAN_PANEL_TARIFF_OR_C020_REQUESTS_CONFLICT'
)
c024_pilot_cost = c024_tariff['three_case_pilot_hypothetical']
c024_all_cost = c024_tariff['full_1116_hypothetical']
c024_need(
    c024_pilot_cost['output_ceiling_tokens'] == 3 * 600
    and c024_all_cost['output_ceiling_tokens'] == 1116 * 600
    and c024_pilot_cost['automatic_retries'] == 0
    and c024_all_cost['automatic_retries'] == 0,
    'C022_COST_SCENARIO_OR_RETRY_CONFLICT'
)
c024_full_upper_toman = c024_all_cost['toman_using_high_proxy_and_600_output']
c024_pilot_upper_toman = c024_pilot_cost['toman_using_high_proxy_and_600_output']
c024_cost_status = (
    'FULL_EXECUTION_REQUIRES_PRIOR_APPROVAL_IF_THIS_ESTIMATE_IS_USED'
    if c024_full_upper_toman > 500000
    else 'FULL_EXECUTION_COST_ESTIMATE_BELOW_500K_NOT_A_GUARANTEE'
)


# 6. Gates and delivery plan: do not create actual attempt or raw-response files.
c024_gateway = {
    'family': 'ARVAN_GATEWAY',
    'configuration_source': c024_env_location,
    'gateway_url_fingerprint_sha256': c024_endpoint_sha,
    'gateway_url_shape': c024_endpoint_shape,
    'env_parse_issue': c024_env_error,
    'credential_api_key_present_and_unmasked': c024_credential_ok,
    'credential_legacy_alias_present_and_unmasked': c024_credential_alias_ok,
    'environment_model_override_class': c024_model_override_class,
    'endpoint_matches_saved_C07_route': c024_route_matches_historical_c07,
    'historical_diagnostic_state': c024_diag_state,
    'historical_diagnostic_source_path': str(c024_diag_path),
    'historical_diagnostic_notebook_sha256': c024_diag_source_sha,
    'historical_C07_response_model': (
        'DeepSeek-V4-Flash' if c024_diag_c07 is not None else None
    ),
    'historical_C08_reasoning_tokens': (
        4096 if c024_diag_c08 is not None else None
    ),
    'request_model': c024_c020['identity']['frozen_profile']['model'],
    'provider_model_id_verified_today': False,
    'model_version_backend_verified_today': False,
    'effective_thinking_disabled_at_gateway_verified': False,
    'temperature_zero_effect_at_gateway_verified': False,
    'effective_max_tokens_600_verified_at_gateway': False,
    'adapter_status': 'OFFLINE_CONTRACT_ONLY_NOT_A_VERIFIED_SEND_IMPLEMENTATION',
    'authorization_scheme_historical': 'apikey',
    'method_and_route': 'POST_CHAT_COMPLETIONS',
    'parallel_workers': 1,
    'automatic_retries': 0,
    'allow_redirects': False,
    'request_body_source': 'EXACT_C020_PAYLOAD_NO_MUTATION',
    'request_profile': c024_c020['identity']['frozen_profile'],
    'secret_values_saved': False,
    'network_calls': 0
}
c024_need(c024_gateway['request_profile'] == {
    'model': 'DeepSeek-V4-Flash',
    'temperature': 0,
    'max_tokens': 600,
    'thinking': {'type': 'disabled'}
}, 'C340_FROZEN_MODEL_PROFILE_CHANGED')

c024_capacity = {
    'source': 'USER_ARVAN_BAZAAR_SCREENSHOT_DISPLAY_CONTEXT_1M',
    'advertised_context_tokens': 1000000,
    'compared_with_C020_UTF8_HEURISTIC_only': True,
    'estimated_under_advertised_window': c024_provisional_capacity,
    'largest_upper_proxy_plus_600': c024_max_proxy_tokens,
    'provider_exact_tokenizer_pinned': False,
    'provider_message_template_overhead_pinned': False,
    'actual_provider_capacity_verified': 0,
    'automatic_context_truncation': False,
    'dispatch_capacity_gate': 'HELD_NOT_EXACT'
}
c024_attempt_plan = {
    'pilot_count_max': 3,
    'pilot_selected_in_C020': c024_pilot_planned,
    'new_attempts_allowed_now': 0,
    'future_attempt_protocol': [
        'RECHECK_EXACT_REQUEST_ID_AND_COMPATIBLE_CACHE',
        'FAIL_CLOSED_IF_ANY_UNKNOWN_PRIOR_ATTEMPT',
        'ATOMICALLY_PERSIST_UNIQUE_ATTEMPT_INTENT_AND_ONE_ATTEMPT_RESERVATION',
        'ONE_WORKER_ZERO_SDK_RETRIES_ZERO_REDIRECTS',
        'SAVE_RAW_HTTP_BYTES_AND_RESPONSE_HEADERS_BEFORE_PARSE',
        'SAVE_USAGE_RESPONSE_MODEL_REASONING_TOKENS_AND_FINISH_REASON',
        'SAVE_FINAL_TERMINAL_STATE_OR_HOLD_UNKNOWN_WITHOUT_AUTO_RESEND'
    ],
    'pre_dispatch_attempt_intents_created': 0,
    'raw_response_files_created': 0,
    'unknown_outcome_retry_automatically_authorized': False,
    'pilot_authorized_after_all_gates_only': True,
    'population_run_authorized': False
}

c024_cost = {
    'price_source': 'USER_ARVAN_BAZAAR_SCREENSHOT_2026_11_11',
    'model_display': 'DeepSeek-V4-Flash',
    'unit': 'TOMAN_PER_MILLION_TOKENS',
    'input_price': 45000,
    'output_price': 90000,
    'cache_miss_assumed': True,
    'max_output_tokens_per_request': 600,
    'automatic_retry_attempt_reserve': 0,
    'reserved_first_attempt_count_pilot_if_authorized': 3,
    'reserved_first_attempt_count_all_if_authorized': 1116,
    'additional_unknown_attempt_cost_reserve': 'UNPRICED_NO_UNKNOWN_IN_C020_SCOPE',
    'pilot_3_lower_proxy_toman': c024_pilot_cost['toman_using_low_proxy_and_600_output'],
    'pilot_3_upper_proxy_toman': c024_pilot_upper_toman,
    'all_1116_lower_proxy_toman': c024_all_cost['toman_using_low_proxy_and_600_output'],
    'all_1116_upper_proxy_toman': c024_full_upper_toman,
    'estimated_token_cost_is_not_a_strict_ceiling': True,
    'unknown_tax_reasoning_and_provider_cache_terms': True,
    'full_exposure_policy': c024_cost_status,
    'full_population_authorization': False,
    'pilot_current_authorization': False
}


# 7. Immutable external ledger and summary with scope-labeled counts.
c024_identity = {
    'schema': 'NB22E_V4_C024_BOUNDED_PROVIDER_AND_HISTORY_PRE_DISPATCH_V1',
    'c011_execution_boundary_sha256': c016_boundary,
    'c020_ledger_sha256': C020_OFFLINE_PREPARATION['ledger_sha256'],
    'c023_summary_sha256': C023_RECONCILIATION['summary_sha256'],
    'c023_ledger_sha256': C023_RECONCILIATION['ledger_sha256'],
    'c342_summary_sha256': NB22E_V4_TRANSFER['artifact_sha256']['summary'],
    'saved_diagnostic_notebook_sha256': c024_diag_source_sha,
    'gateway_endpoint_fingerprint': c024_endpoint_sha,
    'tariff': {'input_rial': 450000, 'output_rial': 900000},
    'no_network_and_no_new_attempts': True
}
c024_identity_sha = c016_sha(c016_canon(c024_identity))
c024_dir = (outputs / ('C024_gateway_bounded_inventory_' + c024_identity_sha)).resolve()
c024_need(
    c024_dir.parent == outputs.resolve()
    and not c024_dir.is_relative_to(repo.resolve()),
    'OUTPUT_MUST_BE_OUTSIDE_REPOSITORY'
)
c024_dir.mkdir(parents=True, exist_ok=True)
c024_ledger_path = c024_dir / ('C024.rows.' + c024_identity_sha + '.jsonl')
c024_payload_sha = hashlib.sha256()

if not c024_ledger_path.exists():
    with tempfile.NamedTemporaryFile(
        mode='wb', dir=c024_dir, prefix='.C024.', suffix='.tmp', delete=False
    ) as stream:
        c024_temp_path = Path(stream.name)
        for row in c024_ledger_rows:
            payload = c016_canon(row) + b'\n'
            c024_payload_sha.update(payload)
            stream.write(payload)
        stream.flush()
        os.fsync(stream.fileno())
    try:
        os.link(c024_temp_path, c024_ledger_path)
    except FileExistsError:
        pass
    finally:
        c024_temp_path.unlink(missing_ok=True)
else:
    for row in c024_ledger_rows:
        c024_payload_sha.update(c016_canon(row) + b'\n')

c024_ledger_sha = c024_payload_sha.hexdigest()
c024_reloaded_hasher = hashlib.sha256()
c024_reloaded_count = 0
with c024_ledger_path.open('rb') as stream:
    for raw in stream:
        c024_reloaded_hasher.update(raw)
        c024_reloaded_count += 1
c024_need(
    c024_reloaded_hasher.hexdigest() == c024_ledger_sha
    and c024_reloaded_count == 1116,
    'NEW_LEDGER_RELOAD_FAILED'
)

c024_summary = {
    'cell': 'NB22E-v4:C024',
    'status': 'BOUNDED_HISTORY_AND_GATEWAY_PREFLIGHT_COMPLETE_PILOT_HELD',
    'identity': c024_identity,
    'identity_sha256': c024_identity_sha,
    'history_scope': {
        'nb19': 'C023_VERIFIED_1149_CHAIN_1116_EFFECTIVE_PROMPT_MISMATCH',
        'nb22e': 'C342_ALREADY_VERIFIED_29_EXACT_2_IDENTITY_HELD_2_ERRORS_ALL_OUTSIDE_CURRENT_C011',
        'c020': 'ONLY_NEW_IDENTITY_SCOPED_CACHE_AND_ATTEMPT_LEDGER',
        'global_missing_answer_claimed': False
    },
    'historical_NB19_reviewer': c024_c023['historical_review_categories'],
    'scoped_generation_inventory_counts': dict(sorted(c024_current_counts.items())),
    'provider_and_adapter': c024_gateway,
    'capacity': c024_capacity,
    'attempt_plan': c024_attempt_plan,
    'cost_estimates': c024_cost,
    'currently_dispatch_authorized_count': 0,
    'currently_compatible_cache_proven_in_checked_scope': 0,
    'currently_unknown_attempts_in_C020_identity':
        c024_current_counts['scoped_unknown_or_nonempty_attempts'],
    'unresolved_provider_adapter_capacity_count': 1116,
    'bounded_inventory_new_generation_candidates_count':
        c024_current_counts['bounded_inventory_candidates'],
    'globally_proven_new_generation_deficit': None,
    'preconditions_for_pilot': [
        'VERIFY_CURRENT_GATEWAY_MODEL_ID_AND_EFFECTIVE_PROFILE',
        'VERIFY_GATEWAY_1M_CAPACITY_USING_COMPATIBLE_TOKENIZER_OR_AUTHORITATIVE_BOUND',
        'IMPLEMENT_AND_VALIDATE_ATOMIC_ATTEMPT_RAW_USAGE_ADAPTER',
        'RECHECK_CACHE_AND_UNKNOWN_ATTEMPT_FOR_THREE_SELECTED_REQUEST_IDS',
        'APPLY_COST_RULE_TO_ENTIRE_PROPOSED_SCOPE'
    ],
    'next_decision': 'NO_PAID_DISPATCH_NOW; CLOSE_PROVIDER_ADAPTER_AND_CAPACITY_PRECONDITIONS',
    'ledger_path': str(c024_ledger_path),
    'ledger_file_sha256': c024_ledger_sha,
    'new_generation_requests': 0,
    'new_judge_requests': 0,
    'new_network_requests': 0,
    'generation_send_authorized': False,
    'quality_gate': 'BLOCKED'
}

c024_summary_path = c024_dir / (
    'C024.summary.' + c016_sha(c016_canon(c024_summary)) + '.json'
)
c024_summary_sha, _ = c016_put_once(c024_summary_path, c024_summary)
c024_reloaded_summary = c020_load_core(c024_summary_path, c024_summary_sha)
c024_need(c024_reloaded_summary['ledger_file_sha256'] == c024_ledger_sha,
          'NEW_SUMMARY_RELOAD_FAILED')

C024_PREFLIGHT = {
    'status': c024_summary['status'],
    'summary_path': str(c024_summary_path),
    'summary_sha256': c024_summary_sha,
    'ledger_path': str(c024_ledger_path),
    'ledger_sha256': c024_ledger_sha,
    'pilot_dispatch_authorized': False,
    'quality_gate': 'BLOCKED'
}

print('C024_NB19_REVIEW: 1046 MECHANICAL_ONLY; 70 HISTORICAL_HELD')
print('C024_NB22E_33: 29 EXACT; 2 MODEL_HELD; 2 ERRORS; ALL_OUTSIDE_C011')
print('C024_BOUNDED_INVENTORY_COUNTS:', dict(sorted(c024_current_counts.items())))
print('C024_HISTORICAL_DIAGNOSTIC:', c024_diag_state)
if c024_diag_state == 'SAVED_DIAGNOSTIC_NOTEBOOK_NOT_FOUND':
    print('C024_MISSING_EXACT_SOURCE:', str(c024_diag_path))
print('C024_GATEWAY_PATH_CLASS:', c024_endpoint_shape)
print('C024_GATEWAY_ROUTE_MATCHES_OLD_C07:', c024_route_matches_historical_c07)
print('C024_GATEWAY_SECRET_PRESENT:', c024_credential_ok)
print('C024_MODEL_SENDABLE_TODAY_VERIFIED: False')
print('C024_EXACT_CAPACITY: 0/1116')
print('C024_PROVISIONAL_UNDER_PANEL_1M:', c024_provisional_capacity, '/1116')
print('C024_PILOT_ESTIMATE_TOMAN:', c024_cost['pilot_3_lower_proxy_toman'],
      '-', c024_cost['pilot_3_upper_proxy_toman'])
print('C024_FULL_ESTIMATE_TOMAN:', c024_cost['all_1116_lower_proxy_toman'],
      '-', c024_cost['all_1116_upper_proxy_toman'])
print('C024_FULL_COST_POLICY:', c024_cost_status)
print('C024_LEDGER_SHA256:', c024_ledger_sha)
print('C024_SUMMARY_SHA256:', c024_summary_sha)
print('C024_STATUS:', C024_PREFLIGHT['status'])
print('GENERATION_SEND_AUTHORIZED: False | quality_gate: BLOCKED')

C024_NB19_REVIEW: 1046 MECHANICAL_ONLY; 70 HISTORICAL_HELD
C024_NB22E_33: 29 EXACT; 2 MODEL_HELD; 2 ERRORS; ALL_OUTSIDE_C011
C024_BOUNDED_INVENTORY_COUNTS: {'bounded_inventory_candidates': 1116, 'historical_NB19_prompt_mismatch': 1116}
C024_HISTORICAL_DIAGNOSTIC: HISTORICAL_C07_PASS_AND_C08_REASONING_RISK_CONFIRMED
C024_GATEWAY_PATH_CLASS: V1_PLUS_CHAT_COMPLETIONS
C024_GATEWAY_ROUTE_MATCHES_OLD_C07: True
C024_GATEWAY_SECRET_PRESENT: True
C024_MODEL_SENDABLE_TODAY_VERIFIED: False
C024_EXACT_CAPACITY: 0/1116
C024_PROVISIONAL_UNDER_PANEL_1M: 1116 /1116
C024_PILOT_ESTIMATE_TOMAN: 916.38 - 1670.67
C024_FULL_ESTIMATE_TOMAN: 348057.81 - 635827.64
C024_FULL_COST_POLICY: FULL_EXECUTION_REQUIRES_PRIOR_APPROVAL_IF_THIS_ESTIMATE_IS_USED
C024_LEDGER_SHA256: 79d4d38b90ff5310343857a85cf35bf56f92581dd40fe77d34da82ca0e19b2af
C024_SUMMARY_SHA256: c49c90c2b66bc5195ba1178f402234c4b54b6bfb9a53aa0e2ab587e626465213
C024_STATUS: BOUNDED_HISTORY_AND_GATEWAY_PREFLIGHT_COMPLETE_PILOT_HELD
GENERATION_SEND_AUTHORI

<div dir="rtl" align="right">

### NB22E-v4:C025 — کنترل آفلاین Adapter و پروتکل امن Pilot

**هدف:** تکمیل بخش محلی پیش‌نیازهای Pilot، بدون اتصال به شبکه یا اجرای مدل.

**ورودی‌ها:** Summary موفق C024، Ledger و سه درخواست منتخب C020 و قرارداد فریز‌شدهٔ C340.

**عملیات:**

۱. کنترل هویت سه درخواست Pilot، اتصال آن‌ها به C020 و نبود Cache یا Attempt نامعلوم در همان دامنه.

۲. بررسی دقیق بدنهٔ درخواست شامل مدل `DeepSeek-V4-Flash`، دمای صفر، سقف خروجی ۶۰۰ و `thinking=disabled`، بدون تغییر Payload.

۳. شبیه‌سازی محلی ثبت انحصاری Attempt قبل از ارسال، جلوگیری از Attempt تکراری و ذخیرهٔ Raw قبل از Parse.

۴. آزمون ثبت `usage`، `finish_reason` و مدل برگشتی با پاسخ ساختگی کاملاً مشخص‌شده؛ نه پاسخ واقعی محصول.

۵. مصرف برآورد هزینهٔ پذیرفته‌شدهٔ C024، بدون محاسبهٔ مجدد جمعیت.

۶. انتشار Plan و Summary نسخه‌دار خارج از مخزن با کنترل بازخوانی و SHA.

**مرز اعتبار:** موفقیت آزمون محلی Adapter، قابل‌ارسال بودن فعلی مدل، پشتیبانی عملی Thinking یا شمارش دقیق توکن را ثابت نمی‌کند.

**ممنوعیت:** API، Generation، Judge، اجرای Pilot، Retry، ویرایش C019 تا C024 و هرگونه تغییر Git.

`generation_send_authorized = False`

`quality_gate = BLOCKED`

</div>

In [23]:
# NB22E-v4 | C025 | R11.2
# Offline three-request wire/attempt/raw/usage adapter dry-run; no network or Generation.
# Consume C024 and C020 only. Never overwrite existing requests or historical outputs.

if not all(name in globals() for name in (
    'C024_PREFLIGHT', 'C020_OFFLINE_PREPARATION', 'C023_RECONCILIATION',
    'c020_load_core', 'c016_canon', 'c016_sha', 'c016_put_once',
    'outputs', 'repo', 'c016_boundary'
)):
    raise RuntimeError('C025_UPSTREAM_NOTEBOOK_STATE_MISSING')


def c025_need(ok, code):
    if not ok:
        raise RuntimeError('C025_' + code)


# A. Consume accepted summaries. No NB19/C342 rescan or 1116-request rebuild.
c025_prev = c020_load_core(
    C024_PREFLIGHT['summary_path'], C024_PREFLIGHT['summary_sha256']
)
c025_c020 = c020_load_core(
    C020_OFFLINE_PREPARATION['summary_path'],
    C020_OFFLINE_PREPARATION['summary_sha256']
)
c025_ledger = c020_load_core(
    C020_OFFLINE_PREPARATION['ledger_path'],
    C020_OFFLINE_PREPARATION['ledger_sha256']
)
c025_need(
    c025_prev['status'] == 'BOUNDED_HISTORY_AND_GATEWAY_PREFLIGHT_COMPLETE_PILOT_HELD'
    and c025_prev['identity']['c011_execution_boundary_sha256'] == c016_boundary
    and c025_prev['identity']['c023_summary_sha256'] == C023_RECONCILIATION['summary_sha256']
    and c025_prev['identity']['c020_ledger_sha256'] == C020_OFFLINE_PREPARATION['ledger_sha256']
    and c025_prev['currently_dispatch_authorized_count'] == 0
    and c025_prev['currently_unknown_attempts_in_C020_identity'] == 0
    and c025_prev['bounded_inventory_new_generation_candidates_count'] == 1116
    and c025_prev['currently_compatible_cache_proven_in_checked_scope'] == 0
    and c025_prev['generation_send_authorized'] is False
    and c025_prev['quality_gate'] == 'BLOCKED'
    and len(c025_ledger['entries']) == 1116,
    'C024_SCOPE_OR_GATE_CHANGED'
)
c025_gateway = c025_prev['provider_and_adapter']
c025_need(
    c025_gateway['endpoint_matches_saved_C07_route'] is True
    and c025_gateway['credential_api_key_present_and_unmasked'] is True
    and c025_gateway['historical_diagnostic_state'] ==
       'HISTORICAL_C07_PASS_AND_C08_REASONING_RISK_CONFIRMED'
    and c025_gateway['request_model'] == 'DeepSeek-V4-Flash'
    and c025_gateway['provider_model_id_verified_today'] is False
    and c025_prev['capacity']['actual_provider_capacity_verified'] == 0,
    'MODEL_ROUTE_OR_CAPACITY_EVIDENCE_CHANGED'
)

# B. Exactly the three already proposed C020 candidates, not new selections.
c025_pilot = c025_prev['attempt_plan']['pilot_selected_in_C020']
c025_need(len(c025_pilot) == 3 and len({x['request_id'] for x in c025_pilot}) == 3,
          'PILOT_SCOPE_NOT_THREE_UNIQUE')
c025_index = {r['request_id']: r for r in c025_ledger['entries']}
c025_need(len(c025_index) == 1116, 'C020_REQUEST_ID_DUPLICATE')
c025_requests_dir = Path(c025_c020['requests_directory']).resolve()
c025_need(
    c025_requests_dir.parent == Path(C020_OFFLINE_PREPARATION['summary_path']).resolve().parent,
    'REQUEST_DIRECTORY_ESCAPE'
)
c025_prepared = []
c025_total = collections.Counter()
for item in c025_pilot:
    request_id = item['request_id']
    c025_need(request_id in c025_index, 'PILOT_REQUEST_NOT_IN_C020')
    prior = c025_index[request_id]
    c025_need(
        prior['candidate_id'] == item['candidate_id']
        and prior['request_file_sha256'] == item['request_sha256']
        and prior['status'] == 'OFFLINE_READY_DISPATCH_GATED'
        and prior['attempt_status'] == 'NO_PREVIOUS_ATTEMPT_FOR_NEW_IDENTITY'
        and prior['cache_status'] == 'NO_MATCHING_SCOPED_CACHE'
        and item['eligible_by_bounded_inventory'] is True,
        'PILOT_ATTEMPT_OR_CACHE_HELD'
    )
    file_path = (c025_requests_dir / prior['request_file']).resolve()
    c025_need(file_path.parent == c025_requests_dir and file_path.is_file()
              and not file_path.is_symlink(), 'PILOT_REQUEST_PATH_MISSING_OR_UNSAFE')
    req = c020_load_core(file_path, prior['request_file_sha256'])
    payload = req['payload']
    c025_need(
        req['request_id'] == request_id
        and req['candidate_id'] == item['candidate_id']
        and req['identity_sha256'] == c025_c020['identity_sha256']
        and req['payload_sha256'] == c016_sha(c016_canon(payload))
        and req['dispatch_authorized'] is False
        and set(payload) == {'model', 'messages', 'temperature', 'max_tokens', 'thinking'}
        and payload['model'] == 'DeepSeek-V4-Flash'
        and payload['max_tokens'] == 600
        and payload['temperature'] == 0
        and payload['thinking'] == {'type': 'disabled'}
        and [m['role'] for m in payload['messages']] == ['system', 'user']
        and all(isinstance(m.get('content'), str) for m in payload['messages']),
        'PILOT_FROZEN_WIRE_PAYLOAD_CONFLICT'
    )
    body = c016_canon(payload)
    measured = prior['token_measurement']
    c025_need(
        len(body) == measured['request_json_utf8_bytes']
        and len(c016_canon(payload['messages'])) == measured['messages_json_utf8_bytes'],
        'C020_REQUEST_BYTE_MEASUREMENT_CONFLICT'
    )
    c025_total['request_body_utf8_bytes'] += len(body)
    c025_total['messages_utf8_bytes'] += measured['messages_json_utf8_bytes']
    c025_total['lower_proxy_tokens'] += measured['token_proxy_low_at_4_bytes_per_token']
    c025_total['upper_proxy_tokens'] += measured['token_proxy_high_at_2_bytes_per_token']
    c025_prepared.append({
        'candidate_id': item['candidate_id'],
        'request_id': request_id,
        'request_receipt_sha256': prior['request_file_sha256'],
        'wire_payload_sha256': c016_sha(body),
        'wire_payload_utf8_bytes': len(body),
        'message_body_utf8_bytes': measured['messages_json_utf8_bytes'],
        'max_output_tokens': 600,
        'effective_thinking_setting': 'disabled_requested_not_verified_on_gateway',
        'model_binding_status': 'HISTORICAL_C07_PASS_CURRENT_MODEL_NOT_VERIFIED',
        'attempt_state': 'NOT_STARTED_NO_WRITE_IN_REAL_ATTEMPT_LEDGER',
        'dispatch_authorized': False
    })
    del payload, body, req

# C. Local simulation: prove write-ahead intent, raw-before-parse, no auto-resend.
# This is NOT a real attempt, API request, cache hit or synthetic product answer.
c025_identity = {
    'schema': 'NB22E_V4_C025_OFFLINE_DISPATCH_ADAPTER_DRYRUN_V1',
    'c011_boundary_sha256': c016_boundary,
    'c024_summary_file_sha256': C024_PREFLIGHT['summary_sha256'],
    'c020_summary_file_sha256': C020_OFFLINE_PREPARATION['summary_sha256'],
    'c020_ledger_file_sha256': C020_OFFLINE_PREPARATION['ledger_sha256'],
    'pilot_request_ids': sorted(x['request_id'] for x in c025_prepared),
    'route_fingerprint': c025_gateway['gateway_url_fingerprint_sha256'],
    'provider_model_binding': 'CURRENT_UNVERIFIED',
    'live_network': False
}
c025_identity_sha = c016_sha(c016_canon(c025_identity))
c025_out = (outputs / ('C025_offline_adapter_' + c025_identity_sha)).resolve()
c025_need(c025_out.parent == outputs.resolve()
          and not c025_out.is_relative_to(repo.resolve()),
          'C025_EXTERNAL_OUTPUT_BOUNDARY')
c025_out.mkdir(parents=True, exist_ok=True)


def c025_exclusive_write(path, data):
    # Never modify/replace an existing object. For a real attempt this
    # would be a durable pre-send intent or an exact raw HTTP response.
    fd = os.open(str(path), os.O_WRONLY | os.O_CREAT | os.O_EXCL, 0o600)
    try:
        with os.fdopen(fd, 'wb') as f:
            f.write(data)
            f.flush()
            os.fsync(f.fileno())
    except BaseException:
        # An ambiguous or incomplete write must NOT be silently retried.
        raise
    return c016_sha(path.read_bytes())


c025_simulated = {
    'test_only': True,
    'real_attempts_created': 0,
    'network_requests': 0,
    'write_ahead_intent': False,
    'raw_persisted_before_parse': False,
    'duplicate_attempt_prevented': False,
    'raw_usage_preserved': False
}
with tempfile.TemporaryDirectory(prefix='C025_fixture_', dir=c025_out) as fixture:
    scratch = Path(fixture)
    fake_id = c016_sha(('SIMULATED_ONLY:' + c025_prepared[0]['request_id']).encode())
    intent = {
        'kind': 'SYNTHETIC_TEST_NOT_REAL_ATTEMPT',
        'attempt_id': fake_id,
        'request_id': c025_prepared[0]['request_id'],
        'status': 'INTENT_WRITTEN_NO_NETWORK'
    }
    intent_path = scratch / (fake_id + '.intent.json')
    intent_sha = c025_exclusive_write(intent_path, c016_canon(intent))
    c025_simulated['write_ahead_intent'] = intent_sha == c016_sha(c016_canon(intent))
    try:
        c025_exclusive_write(intent_path, c016_canon(intent))
    except FileExistsError:
        c025_simulated['duplicate_attempt_prevented'] = True
    fake_response = {
        'id': 'C025_SYNTHETIC_NO_PROVIDER_RESPONSE',
        'model': 'DeepSeek-V4-Flash',
        'choices': [{'index': 0, 'message': {'role': 'assistant',
                     'content': 'SYNTHETIC_FIXTURE_ONLY'}, 'finish_reason': 'stop'}],
        'usage': {'prompt_tokens': 12, 'completion_tokens': 5,
                  'total_tokens': 17, 'completion_tokens_details': {'reasoning_tokens': 0}}
    }
    fake_bytes = c016_canon(fake_response)
    raw_path = scratch / (fake_id + '.raw.http.body')
    raw_sha = c025_exclusive_write(raw_path, fake_bytes)
    c025_simulated['raw_persisted_before_parse'] = raw_path.is_file()
    received = json.loads(raw_path.read_bytes())
    c025_simulated['raw_usage_preserved'] = (
        c016_sha(raw_path.read_bytes()) == raw_sha
        and received['usage'] == fake_response['usage']
        and received['model'] == 'DeepSeek-V4-Flash'
        and received['choices'][0]['finish_reason'] == 'stop'
    )

c025_need(all(c025_simulated[k] for k in (
    'write_ahead_intent', 'raw_persisted_before_parse',
    'duplicate_attempt_prevented', 'raw_usage_preserved'
)), 'LOCAL_ADAPTER_DRY_RUN_FAILED')

# D. Bounded dispatch economics and unresolved live gates.
c025_tariff = c025_prev['cost_estimates']
c025_need(
    c025_tariff['pilot_3_upper_proxy_toman'] < 100000
    and c025_tariff['all_1116_upper_proxy_toman'] > 500000
    and c025_tariff['max_output_tokens_per_request'] == 600
    and c025_tariff['automatic_retry_attempt_reserve'] == 0
    and c025_total['upper_proxy_tokens'] == sum(
        c025_index[x['request_id']]['token_measurement']
        ['token_proxy_high_at_2_bytes_per_token'] for x in c025_pilot
    ),
    'C024_COST_AND_PILOT_PROXY_CONFLICT'
)
c025_plan = {
    'cell': 'NB22E-v4:C025',
    'kind': 'OFFLINE_WIRE_AND_ATTEMPT_PLAN',
    'identity_sha256': c025_identity_sha,
    'pilot_cases': c025_prepared,
    'pilot_summary': dict(c025_total),
    'gateway_compatibility': {
        'historical_model_and_route_verified': True,
        'current_route_fingerprint_matches_C07': True,
        'current_model_inventory_or_live_support_verified': False,
        'C340_thinking_disabled_on_wire': True,
        'gateway_effective_thinking_disabled_verified': False,
        'temperature_0_on_wire': True,
        'max_tokens_600_on_wire': True,
        'provider_tokenizer_exact_pinned': False,
        'capacity_verification': 'HELD_EXACT_COUNT_AND_ROUTE_CAPACITY_UNVERIFIED',
        'provider_backend_model_version_verified': False
    },
    'write_ahead_simulation': c025_simulated,
    'production_attempt_protocol': [
        'RELOAD_IDENTICAL_C020_REQUEST_AND_SHA',
        'RECHECK_SAME_IDENTITY_CACHE_AND_UNCERTAIN_ATTEMPTS',
        'CREATE_EXCLUSIVE_DURABLE_INTENT_BEFORE_NETWORK',
        'RESERVE_TOTAL_SCOPE_BUDGET_BEFORE_DISPATCH',
        'ONE_WORKER_NO_AUTO_RETRY_NO_REDIRECT',
        'PERSIST_EXACT_RAW_HTTP_RESPONSE_AND_HEADERS_BEFORE_PARSE',
        'PERSIST_USAGE_FINISH_REASON_RETURNED_MODEL_AND_REASONING_TOKENS',
        'TIMEOUT_AFTER_DISPATCH_IS_OUTCOME_UNKNOWN_NO_AUTO_RESEND'
    ],
    'pending': [
        'CURRENT_PROVIDER_MODEL_ID_AND_EFFECTIVE_THINKING_UNVERIFIED',
        'EXACT_TOKENIZER_AND_PROVEN_ROUTE_CONTEXT_CAP_UNVERIFIED',
        'REAL_ATTEMPT_ADAPTER_NOT_NETWORK_TESTED',
        'PILOT_NOT_DISPATCHED',
        'FULL_POPULATION_DISPATCH_NOT_AUTHORIZED'
    ],
    'cost_toman': {
        'pilot3_low': c025_tariff['pilot_3_lower_proxy_toman'],
        'pilot3_high': c025_tariff['pilot_3_upper_proxy_toman'],
        'all1116_low': c025_tariff['all_1116_lower_proxy_toman'],
        'all1116_high': c025_tariff['all_1116_upper_proxy_toman'],
        'cache_miss_assumed': True, 'automatic_retries': 0,
        'pilot_first_attempt_reserve_count': 3,
        'entire_package_approval_needed_if_above_500000': True,
        'cost_is_not_a_guaranteed_upper_bound': True
    },
    'pilot_dispatch_authorized': False,
    'population_dispatch_authorized': False,
    'quality_gate': 'BLOCKED'
}
c025_plan_path = c025_out / ('C025.plan.' + c016_sha(c016_canon(c025_plan)) + '.json')
c025_plan_sha, _ = c016_put_once(c025_plan_path, c025_plan)
c025_need(c020_load_core(c025_plan_path, c025_plan_sha)['pilot_cases'] == c025_prepared,
          'PLAN_RELOAD_CONFLICT')

c025_summary = {
    'cell': 'NB22E-v4:C025',
    'status': 'OFFLINE_WIRE_AND_WRITE_AHEAD_DRYRUN_PASS_LIVE_GATES_HELD',
    'identity': c025_identity,
    'identity_sha256': c025_identity_sha,
    'pilot_plan_cases_validated': 3,
    'current_population_c020': 1116,
    'bounded_inventory_candidates': 1116,
    'current_live_sendable_count': 0,
    'confirmed_same_execution_cache_in_bounded_scope': 0,
    'unknown_attempts_in_new_c020_identity': 0,
    'live_model_verified_today': False,
    'exact_capacity_verified_count': 0,
    'historical_1046_mechanical_70_held_unchanged': True,
    'prior_C342_33_outside_C011_unchanged': True,
    'local_adapter_simulation_pass': True,
    'local_adapter_plan_path': str(c025_plan_path),
    'local_adapter_plan_sha256': c025_plan_sha,
    'network_calls': 0, 'generation_calls': 0, 'judge_calls': 0,
    'production_attempts_created': 0,
    'new_generation_send_authorized': False,
    'quality_gate': 'BLOCKED',
    'next_required_evidence': 'ARVAN_CURRENT_MODEL_CAPABILITY_AND_COMPATIBLE_TOKEN_CAPACITY'
}
c025_summary_path = c025_out / ('C025.summary.' + c016_sha(c016_canon(c025_summary)) + '.json')
c025_summary_sha, _ = c016_put_once(c025_summary_path, c025_summary)
c025_need(c020_load_core(c025_summary_path, c025_summary_sha)['local_adapter_plan_sha256'] == c025_plan_sha,
          'SUMMARY_RELOAD_CONFLICT')
C025_OFFLINE_ADAPTER = {
    'status': c025_summary['status'],
    'summary_path': str(c025_summary_path),
    'summary_sha256': c025_summary_sha,
    'plan_path': str(c025_plan_path),
    'plan_sha256': c025_plan_sha,
    'pilot_dispatch_authorized': False,
    'quality_gate': 'BLOCKED'
}
print('C025_PILOT_WIRE_VALIDATED: 3/3')
print('C025_DRYRUN_WRITE_AHEAD_AND_RAW_BEFORE_PARSE: PASS')
print('C025_CURRENT_MODEL_VERIFIED: False')
print('C025_EXACT_CAPACITY_VERIFIED: 0/1116')
print('C025_PILOT_TOMAN_PROXY:', c025_tariff['pilot_3_lower_proxy_toman'], '-',
      c025_tariff['pilot_3_upper_proxy_toman'])
print('C025_FULL_TOMAN_PROXY:', c025_tariff['all_1116_lower_proxy_toman'], '-',
      c025_tariff['all_1116_upper_proxy_toman'])
print('C025_PLAN_SHA256:', c025_plan_sha)
print('C025_SUMMARY_SHA256:', c025_summary_sha)
print('C025_STATUS:', C025_OFFLINE_ADAPTER['status'])
print('GENERATION_SEND_AUTHORIZED: False | quality_gate: BLOCKED')

C025_PILOT_WIRE_VALIDATED: 3/3
C025_DRYRUN_WRITE_AHEAD_AND_RAW_BEFORE_PARSE: PASS
C025_CURRENT_MODEL_VERIFIED: False
C025_EXACT_CAPACITY_VERIFIED: 0/1116
C025_PILOT_TOMAN_PROXY: 916.38 - 1670.67
C025_FULL_TOMAN_PROXY: 348057.81 - 635827.64
C025_PLAN_SHA256: 364e083bf2631735cb83493f6b56af57f117ac6a78b146601ebcfb37ac12488f
C025_SUMMARY_SHA256: 4aeb8b583a9179358e677b707592695697d73b34ce5a17316e02401ac6645b82
C025_STATUS: OFFLINE_WIRE_AND_WRITE_AHEAD_DRYRUN_PASS_LIVE_GATES_HELD
GENERATION_SEND_AUTHORIZED: False | quality_gate: BLOCKED


<div dir="rtl" align="right">

### NB22E-v4:C026 — احراز Gateway و Pilot مشروط سه‌درخواستی

**مبنا:** Plan و Summary موفق C025 و قراردادهای پذیرفته‌شدهٔ C020 و C024.

**مرحلهٔ اول — احراز پیش از ارسال**

۱. کنترل هویت و SHA سه درخواست فریز‌شدهٔ C020، بدون بازسازی Payload.

۲. خواندن فقط تنظیمات ضروری `ARVAN_GATEWAY` از `.env` و تطبیق اثر انگشت Endpoint با C024؛ بدون ذخیره یا نمایش Secret.

۳. یک درخواست فقط‌خواندنی به `/v1/models`، در صورت پشتیبانی Gateway، بدون Retry و Redirect.

۴. احراز شناسهٔ دقیق `DeepSeek-V4-Flash` و قابلیت‌های Thinking، دمای صفر و سقف خروجی ۶۰۰ توکن.

۵. کنترل ظرفیت هر سه درخواست با احتساب کل Payload، سربار قالب پیام و ۶۰۰ توکن خروجی. تقریب بایتی جای شمارش دقیق یا کران مستند نیست.

۶. کنترل مجدد Cache و Attempt هر سه شناسه، و محاسبهٔ Reserve برای کل Pilot، بدون تقسیم مصنوعی بسته.

**مرحلهٔ دوم — فقط در صورت عبور تمام شروط**

حداکثر سه درخواست، به‌صورت ترتیبی و با یک Worker ارسال می‌شوند. برای هر درخواست، Intent پایدار قبل از شبکه ثبت شده و پاسخ خام قبل از Parse ذخیره می‌شود.

Usage، مدل برگشتی، Reasoning و علت پایان پاسخ جداگانه ثبت می‌شوند. هر خطای مسیر، خروجی نامعتبر یا نتیجهٔ نامعلوم اجرای موارد بعدی را متوقف می‌کند. Retry خودکار ممنوع است.

**شروط توقف:**

اگر `/models` فقط نام مدل را برگرداند و قابلیت‌های لازم یا کران معتبر توکن را اعلام نکند، Pilot مسدود می‌ماند. مستندات API مستقیم DeepSeek به‌جای سند Gateway آروان استفاده نمی‌شوند.

**تعرفهٔ مبنا:** ورودی ۴۵٬۰۰۰ و خروجی ۹۰٬۰۰۰ تومان برای هر میلیون توکن، مطابق تصویر پنل مالک؛ Cache Miss کامل و Retry خودکار صفر.

**مجوز:** فقط Pilot حداکثر سه‌درخواستی، مشروط به احراز تمام پیش‌نیازها. اجرای جمعیتی و داوری تازه مجاز نیستند.

**خروجی:** رسیدهای جدید C026 و Summary با SHA، خارج از مخزن. C019 تا C025 تغییر نمی‌کنند.

`quality_gate = BLOCKED`

`population_dispatch_authorized = False`

</div>

In [24]:
# NB22E-v4 | C026 | R11.2
# Scoped Arvan capability check and conditional three-request Pilot.
# Fail closed; no population dispatch, automatic retries or historical rewrites.

if not all(k in globals() for k in (
    'C025_OFFLINE_ADAPTER', 'C024_PREFLIGHT',
    'C020_OFFLINE_PREPARATION', 'c020_load_core',
    'c016_sha', 'c016_canon', 'c016_put_once',
    'repo', 'outputs', 'c016_boundary'
)):
    raise RuntimeError('C026_REQUIRED_PRIOR_STATE_MISSING')


def c026_need(ok, reason):
    if not ok:
        raise RuntimeError('C026_' + reason)


def c026_exclusive(path, payload):
    fd = os.open(
        str(path),
        os.O_CREAT | os.O_EXCL | os.O_WRONLY,
        0o600
    )
    with os.fdopen(fd, 'wb') as stream:
        stream.write(payload)
        stream.flush()
        os.fsync(stream.fileno())

    digest = c016_sha(payload)
    c026_need(
        c016_sha(path.read_bytes()) == digest,
        'DURABLE_WRITE_RELOAD_CONFLICT'
    )
    return digest


def c026_exists(path):
    return path.exists() or path.is_symlink()


# A. Consume the successful C025 plan and C024 boundary.
c026_c025 = c020_load_core(
    C025_OFFLINE_ADAPTER['summary_path'],
    C025_OFFLINE_ADAPTER['summary_sha256']
)
c026_plan = c020_load_core(
    C025_OFFLINE_ADAPTER['plan_path'],
    C025_OFFLINE_ADAPTER['plan_sha256']
)
c026_c024 = c020_load_core(
    C024_PREFLIGHT['summary_path'],
    C024_PREFLIGHT['summary_sha256']
)
c026_c020 = c020_load_core(
    C020_OFFLINE_PREPARATION['summary_path'],
    C020_OFFLINE_PREPARATION['summary_sha256']
)

c026_need(
    c026_c025['status']
        == 'OFFLINE_WIRE_AND_WRITE_AHEAD_DRYRUN_PASS_LIVE_GATES_HELD'
    and c026_c025['local_adapter_simulation_pass'] is True
    and c026_c025['local_adapter_plan_sha256']
        == C025_OFFLINE_ADAPTER['plan_sha256']
    and c026_c024['quality_gate'] == 'BLOCKED'
    and c026_c020['quality_gate'] == 'BLOCKED'
    and c026_c024['bounded_inventory_new_generation_candidates_count']
        == 1116
    and c026_c024['currently_unknown_attempts_in_C020_identity'] == 0
    and c026_c024['currently_compatible_cache_proven_in_checked_scope'] == 0
    and c026_plan['identity_sha256']
        == c026_c025['identity_sha256']
    and len(c026_plan['pilot_cases']) == 3,
    'FROZEN_PLAN_OR_BOUNDARY_CHANGED'
)

c026_pilot = c026_plan['pilot_cases']

c026_need(
    len({p['request_id'] for p in c026_pilot}) == 3
    and len({p['candidate_id'] for p in c026_pilot}) == 3,
    'PILOT_NOT_THREE_UNIQUE_REQUESTS'
)

c026_requests_dir = Path(
    c026_c020['requests_directory']
).resolve()

c026_need(
    c026_requests_dir.parent
    == Path(C020_OFFLINE_PREPARATION['summary_path']).resolve().parent,
    'C020_REQUEST_DIRECTORY_CONFLICT'
)


# B. New independent identity and private output directory.
c026_identity = {
    'schema': 'NB22E_V4_C026_GATEWAY_PILOT_V1',
    'c011_boundary_sha256': c016_boundary,
    'c025_plan_sha256': C025_OFFLINE_ADAPTER['plan_sha256'],
    'c025_summary_sha256': C025_OFFLINE_ADAPTER['summary_sha256'],
    'c024_summary_sha256': C024_PREFLIGHT['summary_sha256'],
    'pilot_request_ids': [
        p['request_id'] for p in c026_pilot
    ],
    'endpoint_sha256':
        c026_c024['provider_and_adapter'][
            'gateway_url_fingerprint_sha256'
        ],
    'permission':
        'CONDITIONAL_THREE_REQUEST_PILOT_NO_AUTOMATIC_RETRIES',
    'population_dispatch_authorized': False
}

c026_identity_sha = c016_sha(c016_canon(c026_identity))

c026_out = (
    outputs / ('C026_gateway_pilot_' + c026_identity_sha)
).resolve()

c026_need(
    c026_out.parent == outputs.resolve()
    and not c026_out.is_relative_to(repo.resolve()),
    'OUTPUT_NOT_EXTERNAL'
)

c026_out.mkdir(parents=True, exist_ok=True)
c026_attempts = c026_out / 'attempts'
c026_attempts.mkdir(exist_ok=True)

c026_blockers = []
c026_results = []
c026_new_attempts = 0
c026_unknown_results = 0

c026_metadata = {
    'status': 'NOT_CHECKED',
    'http_status': None,
    'model_listed': False,
    'capabilities_verified': False,
    'metadata_requests': 0
}


# C. Read only the two actual Gateway values.
# Never print, hash into a published artifact, or persist the secret.
c026_env_path = repo / '.env'
c026_url = None
c026_key = None
c026_env_issue = None

if not c026_env_path.is_file() or c026_env_path.is_symlink():
    c026_env_issue = 'PROJECT_DOTENV_MISSING'
else:
    c026_settings = {}

    with c026_env_path.open(
        'r', encoding='utf-8-sig'
    ) as stream:
        for line in stream:
            match = re.match(
                r'^\s*(?:export\s+)?'
                r'(ARVAN_GATEWAY_URL|ARVAN_GATEWAY_API_KEY)'
                r'\s*=\s*(.*)$',
                line.rstrip('\n')
            )

            if match is None:
                continue

            name, value = match.groups()

            if name in c026_settings:
                c026_env_issue = 'DUPLICATE_GATEWAY_SETTING'
                break

            value = value.strip()

            if value.startswith(("'", '"')):
                if len(value) < 2 or value[-1] != value[0]:
                    c026_env_issue = 'INVALID_DOTENV_QUOTING'
                    break
                value = value[1:-1]
            else:
                value = value.split(' #', 1)[0].strip()

            c026_settings[name] = value

    if c026_env_issue is None:
        c026_url = c026_settings.get('ARVAN_GATEWAY_URL')
        c026_key = c026_settings.get('ARVAN_GATEWAY_API_KEY')

        if (
            not c026_url
            or not c026_key
            or '*' in c026_key
            or c026_key.startswith('<')
            or any(c.isspace() for c in c026_key)
        ):
            c026_env_issue = 'GATEWAY_URL_OR_CREDENTIAL_INVALID'

    c026_settings.clear()

c026_endpoint = None

if c026_env_issue is None:
    try:
        parts = urllib.parse.urlsplit(c026_url)

        c026_need(
            parts.scheme == 'https'
            and bool(parts.hostname)
            and not parts.username
            and not parts.password
            and not parts.query
            and not parts.fragment
            and parts.port is None,
            'GATEWAY_HTTPS_URL_INVALID'
        )

        base_path = parts.path.rstrip('/')

        if base_path == '/v1':
            prefix = '/v1'
        elif base_path.endswith('/chat/completions'):
            prefix = base_path[:-len('/chat/completions')]
        elif base_path == '':
            prefix = '/v1'
        else:
            prefix = None

        c026_need(
            prefix == '/v1',
            'GATEWAY_ROUTE_NOT_RECOGNIZED'
        )

        c026_endpoint = urllib.parse.urlunsplit((
            parts.scheme,
            parts.netloc,
            '/v1/chat/completions',
            '',
            ''
        ))

        c026_need(
            c016_sha(c026_endpoint.encode('utf-8'))
                == c026_identity['endpoint_sha256']
            and c026_c024['provider_and_adapter'][
                'endpoint_matches_saved_C07_route'
            ] is True,
            'GATEWAY_ENDPOINT_DIFFERS_FROM_PINNED_C07'
        )

    except Exception as exc:
        c026_env_issue = type(exc).__name__

if c026_env_issue:
    c026_blockers.append(
        'GATEWAY_CONFIG_NOT_VERIFIED:' + c026_env_issue
    )


# D. Optional read-only model inventory: one GET, no redirect/retry.
class C026NoRedirect(urllib.request.HTTPRedirectHandler):
    def redirect_request(
        self, request, fp, code, message, headers, newurl
    ):
        return None


c026_model_metadata = None

if not c026_blockers:
    c026_models_url = (
        c026_endpoint[:-len('/chat/completions')] + '/models'
    )

    c026_opener = urllib.request.build_opener(
        C026NoRedirect()
    )

    request = urllib.request.Request(
        c026_models_url,
        method='GET',
        headers={
            'Authorization': 'apikey ' + c026_key,
            'Accept': 'application/json'
        }
    )

    c026_metadata['metadata_requests'] = 1

    try:
        with c026_opener.open(request, timeout=15) as response:
            c026_metadata['http_status'] = response.status
            raw = response.read(1048577)

        c026_need(
            len(raw) <= 1048576,
            'MODEL_METADATA_RESPONSE_TOO_LARGE'
        )

        document = json.loads(raw)

        c026_need(
            isinstance(document, dict)
            and isinstance(document.get('data'), list),
            'MODEL_METADATA_SCHEMA_NOT_RECOGNIZED'
        )

        matches = [
            item for item in document['data']
            if isinstance(item, dict)
            and item.get('id') == 'DeepSeek-V4-Flash'
        ]

        c026_need(
            len(matches) <= 1,
            'DUPLICATE_MODEL_ID'
        )

        c026_model_metadata = matches[0] if matches else None

        c026_metadata['model_listed'] = bool(matches)
        c026_metadata['status'] = (
            'EXACT_MODEL_LISTED'
            if matches else 'MODEL_NOT_LISTED'
        )

    except urllib.error.HTTPError as exc:
        c026_metadata['http_status'] = exc.code
        c026_metadata['status'] = 'MODELS_ENDPOINT_HTTP_ERROR'

    except Exception as exc:
        c026_metadata['status'] = (
            'METADATA_CHECK_FAILED_' + type(exc).__name__
        )

    if not c026_metadata['model_listed']:
        c026_blockers.append(
            'CURRENT_MODEL_ID_NOT_VERIFIED:'
            + c026_metadata['status']
        )


# E. Require authoritative Gateway capabilities.
# Fields absent from provider metadata are NOT invented.
c026_context_limit = None
c026_output_limit = None
c026_tokens_per_byte_bound = None
c026_message_overhead_bound = None

if c026_model_metadata is not None:
    model = c026_model_metadata

    parameters = model.get('supported_parameters')
    thinking_modes = model.get('thinking_modes')

    c026_context_limit = model.get('context_length')
    c026_output_limit = model.get('max_output_tokens')

    c026_tokens_per_byte_bound = model.get(
        'documented_max_tokens_per_utf8_byte'
    )
    c026_message_overhead_bound = model.get(
        'documented_chat_template_overhead_tokens'
    )

    c026_metadata['capabilities_verified'] = (
        isinstance(parameters, list)
        and all(isinstance(x, str) for x in parameters)
        and {'thinking', 'temperature', 'max_tokens'}.issubset(
            set(parameters)
        )
        and isinstance(thinking_modes, list)
        and all(isinstance(x, str) for x in thinking_modes)
        and 'disabled' in thinking_modes
        and type(c026_context_limit) is int
        and c026_context_limit > 600
        and type(c026_output_limit) is int
        and c026_output_limit >= 600
        and type(c026_tokens_per_byte_bound) in (int, float)
        and 0 < c026_tokens_per_byte_bound <= 100
        and type(c026_message_overhead_bound) is int
        and 0 <= c026_message_overhead_bound <= 100000
    )

    if not c026_metadata['capabilities_verified']:
        c026_blockers.append(
            'ARVAN_THINKING_OR_TOKEN_CAPACITY_CONTRACT_NOT_PROVEN'
        )


# F. Recheck exactly three prior C020 requests and their local attempt/cache.
c026_payloads = []
c026_token_bounds = []

c026_c020_root = c026_requests_dir.parent
c026_old_attempts = c026_c020_root / 'attempt_events'
c026_old_cache = c026_c020_root / 'completed_cache'

for entry in c026_pilot:
    cid = entry['candidate_id']
    request_id = entry['request_id']
    candidate_key = c016_sha(cid.encode('utf-8'))

    request_name = (
        'C020.' + candidate_key
        + '.' + request_id
        + '.request.json'
    )

    request_path = (
        c026_requests_dir / request_name
    ).resolve()

    c026_need(
        request_path.parent == c026_requests_dir
        and request_path.is_file()
        and not request_path.is_symlink(),
        'C020_PILOT_REQUEST_FILE_MISSING'
    )

    request = c020_load_core(
        request_path,
        entry['request_receipt_sha256']
    )

    payload = request['payload']
    body = c016_canon(payload)

    c026_need(
        request['candidate_id'] == cid
        and request['request_id'] == request_id
        and request['payload_sha256'] == c016_sha(body)
        and entry['wire_payload_sha256']
            == request['payload_sha256']
        and payload['model'] == 'DeepSeek-V4-Flash'
        and payload['temperature'] == 0
        and payload['max_tokens'] == 600
        and payload['thinking'] == {'type': 'disabled'}
        and [m['role'] for m in payload['messages']]
            == ['system', 'user']
        and request['dispatch_authorized'] is False,
        'PILOT_PAYLOAD_IDENTITY_CHANGED'
    )

    if (
        c026_exists(c026_old_attempts / (request_id + '.json'))
        or c026_exists(c026_old_cache / (request_id + '.json'))
    ):
        c026_blockers.append(
            'EXISTING_SCOPED_ATTEMPT_OR_CACHE_REQUIRES_RECONCILIATION:'
            + cid
        )

    intent_path = (
        c026_attempts / (request_id + '.intent.json')
    )

    if c026_exists(intent_path):
        c026_blockers.append(
            'PRIOR_C026_ATTEMPT_REQUIRES_RECONCILIATION:'
            + cid
        )

    if c026_metadata['capabilities_verified']:
        upper_bound = (
            math.ceil(
                len(body) * c026_tokens_per_byte_bound
            )
            + c026_message_overhead_bound
        )

        c026_token_bounds.append(upper_bound)

        if upper_bound + 600 > c026_context_limit:
            c026_blockers.append(
                'PILOT_EXCEEDS_VERIFIED_CONTEXT_CAPACITY:' + cid
            )

    c026_payloads.append((entry, body))


# G. Pricing and full Pilot reserve.
c026_prices = c026_c024['cost_estimates']

c026_need(
    c026_prices['input_price'] == 45000
    and c026_prices['output_price'] == 90000
    and c026_prices['max_output_tokens_per_request'] == 600
    and c026_prices['automatic_retry_attempt_reserve'] == 0,
    'ARVAN_PANEL_PRICE_OR_RETRY_CONTRACT_CHANGED'
)

c026_pilot_proxy = c026_prices[
    'pilot_3_upper_proxy_toman'
]
c026_full_proxy = c026_prices[
    'all_1116_upper_proxy_toman'
]

# Conditional exposure based on a verified provider token window.
# It is not a billing guarantee while taxes/extra fees remain unknown.
c026_pilot_reserve = (
    round(
        3 * (
            c026_context_limit * 45000
            + 600 * 90000
        ) / 1000000,
        2
    )
    if c026_metadata['capabilities_verified']
    else None
)

if c026_pilot_reserve is None:
    c026_blockers.append(
        'PILOT_RESERVE_NOT_CERTIFIED_WITHOUT_PROVIDER_CAPACITY'
    )
elif c026_pilot_reserve > 500000:
    c026_blockers.append(
        'PILOT_EXPOSURE_ABOVE_OWNER_APPROVAL_THRESHOLD'
    )


# H. Conditional Pilot. No request is sent if any blocker remains.
c026_send_allowed = not c026_blockers

if c026_send_allowed:
    if c026_pilot_reserve >= 100000:
        print(
            'C026_OWNER_COST_NOTICE_TOMAN_CONDITIONAL:',
            c026_pilot_reserve
        )

    c026_opener = urllib.request.build_opener(
        C026NoRedirect()
    )

    for entry, body in c026_payloads:
        cid = entry['candidate_id']
        request_id = entry['request_id']

        try:
            # Durable write-ahead intent BEFORE network dispatch.
            c026_exclusive(
                c026_attempts / (request_id + '.intent.json'),
                c016_canon({
                    'kind': 'REAL_ATTEMPT_INTENT',
                    'candidate_id': cid,
                    'request_id': request_id,
                    'payload_sha256': c016_sha(body),
                    'status':
                        'OUTCOME_UNKNOWN_UNTIL_TERMINAL_EVIDENCE',
                    'automatic_retry': False,
                    'pilot_reserve_toman': c026_pilot_reserve
                })
            )

            c026_exclusive(
                c026_attempts / (
                    request_id + '.dispatch_started.json'
                ),
                c016_canon({
                    'request_id': request_id,
                    'state': 'DISPATCH_STARTED'
                })
            )

        except Exception as exc:
            c026_blockers.append(
                'DURABLE_INTENT_FAILED:'
                + cid + ':' + type(exc).__name__
            )
            break

        request = urllib.request.Request(
            c026_endpoint,
            data=body,
            method='POST',
            headers={
                'Authorization': 'apikey ' + c026_key,
                'Content-Type': 'application/json',
                'Accept': 'application/json'
            }
        )

        http_status = None
        safe_headers = {}

        try:
            try:
                with c026_opener.open(
                    request, timeout=180
                ) as response:
                    http_status = response.status

                    safe_headers = {
                        key: response.headers.get(key)
                        for key in (
                            'x-request-id', 'date',
                            'server', 'retry-after'
                        )
                        if response.headers.get(key)
                    }

                    raw = response.read(4 * 1024 * 1024 + 1)

            except urllib.error.HTTPError as exc:
                http_status = exc.code

                safe_headers = {
                    key: exc.headers.get(key)
                    for key in (
                        'x-request-id', 'date',
                        'server', 'retry-after'
                    )
                    if exc.headers.get(key)
                }

                raw = exc.read(4 * 1024 * 1024 + 1)

            c026_need(
                len(raw) <= 4 * 1024 * 1024,
                'RAW_RESPONSE_EXCEEDS_SAFE_LIMIT'
            )

            # Persist exact raw bytes before JSON Parse.
            raw_path = (
                c026_attempts
                / (request_id + '.raw.http.body')
            )

            raw_sha = c026_exclusive(raw_path, raw)

            c026_exclusive(
                c026_attempts
                / (request_id + '.transport.json'),
                c016_canon({
                    'request_id': request_id,
                    'http_status': http_status,
                    'safe_response_headers': safe_headers,
                    'raw_file_sha256': raw_sha
                })
            )

            c026_new_attempts += 1

            if not 200 <= http_status < 300:
                c026_blockers.append(
                    'PILOT_HTTP_ERROR_STOP:'
                    + cid + ':' + str(http_status)
                )
                break

            response_data = json.loads(raw_path.read_bytes())
            usage = response_data.get('usage')

            c026_need(
                isinstance(usage, dict),
                'RESPONSE_USAGE_MISSING'
            )

            # Persist Usage before validating answer structure.
            c026_exclusive(
                c026_attempts
                / (request_id + '.usage.json'),
                c016_canon({
                    'request_id': request_id,
                    'usage': usage,
                    'raw_file_sha256': raw_sha
                })
            )

            choices = response_data.get('choices')

            c026_need(
                isinstance(choices, list)
                and len(choices) == 1,
                'RESPONSE_CHOICES_INVALID'
            )

            choice = choices[0]
            returned_model = response_data.get('model')

            reasoning = (
                usage.get('completion_tokens_details') or {}
            )
            reasoning_tokens = reasoning.get('reasoning_tokens')

            c026_need(
                type(usage.get('prompt_tokens')) is int
                and type(usage.get('completion_tokens')) is int
                and type(reasoning_tokens) is int
                and reasoning_tokens == 0
                and isinstance(returned_model, str)
                and returned_model.split('/')[-1].lower() in (
                    'deepseek-v4-flash',
                    'deepseek-v4-flash-0731'
                )
                and choice.get('finish_reason') == 'stop'
                and isinstance(
                    choice.get('message', {}).get('content'),
                    str
                )
                and bool(
                    choice['message']['content'].strip()
                ),
                'MODEL_THINKING_USAGE_OR_FINISH_MISMATCH'
            )

            receipt = {
                'request_id': request_id,
                'candidate_id': cid,
                'raw_file_sha256': raw_sha,
                'returned_model': returned_model,
                'usage': usage,
                'finish_reason': choice['finish_reason'],
                'response_content_sha256': c016_sha(
                    choice['message']['content'].encode('utf-8')
                ),
                'status': 'TERMINAL_SUCCESS_NO_JUDGMENT'
            }

            c026_exclusive(
                c026_attempts
                / (request_id + '.verified.json'),
                c016_canon(receipt)
            )

            c026_results.append({
                'candidate_id': cid,
                'request_id': request_id,
                'status': 'PILOT_RETURNED_SUCCESS',
                'usage': usage,
                'raw_file_sha256': raw_sha
            })

        except Exception as exc:
            # The remote request might have consumed tokens.
            # No automatic resend is possible.
            c026_unknown_results += 1

            c026_blockers.append(
                'PILOT_OUTCOME_UNKNOWN_OR_INVALID_STOP:'
                + cid + ':' + type(exc).__name__
            )
            break


# I. Publish a new immutable C026 summary.
c026_status = (
    'PILOT_THREE_DONE_REVIEW_REQUIRED'
    if len(c026_results) == 3
    else 'PILOT_PARTIAL_OR_HELD'
    if c026_new_attempts
    else 'PREFLIGHT_HELD_NO_GENERATION'
)

c026_summary = {
    'cell': 'NB22E-v4:C026',
    'status': c026_status,
    'identity': c026_identity,
    'model_metadata': c026_metadata,
    'pilot_count_planned': 3,
    'pilot_generation_attempts': c026_new_attempts,
    'pilot_successful_raw_receipts': len(c026_results),
    'unknown_or_invalid_results': c026_unknown_results,
    'blockers': list(dict.fromkeys(c026_blockers)),
    'capacity_verified_for_pilot': (
        len(c026_token_bounds) == 3
        and c026_metadata['capabilities_verified']
        and not any(
            'EXCEEDS_VERIFIED_CONTEXT_CAPACITY' in x
            for x in c026_blockers
        )
    ),
    'pilot_input_token_upper_bounds': c026_token_bounds,
    'pilot_proxy_upper_toman': c026_pilot_proxy,
    'pilot_conditional_reserve_toman': c026_pilot_reserve,
    'full_population_proxy_upper_toman': c026_full_proxy,
    'pilot_receipts': c026_results,
    'population_dispatch_authorized': False,
    'automatic_retries': 0,
    'quality_gate': 'BLOCKED'
}

c026_summary_path = c026_out / (
    'C026.summary.'
    + c016_sha(c016_canon(c026_summary))
    + '.json'
)

c026_summary_sha, _ = c016_put_once(
    c026_summary_path,
    c026_summary
)

c026_reloaded = c020_load_core(
    c026_summary_path,
    c026_summary_sha
)

c026_need(
    c026_reloaded['pilot_generation_attempts']
        == c026_new_attempts,
    'C026_SUMMARY_RELOAD_CONFLICT'
)

C026_RESULT = {
    'status': c026_status,
    'summary_path': str(c026_summary_path),
    'summary_sha256': c026_summary_sha,
    'generation_calls': c026_new_attempts,
    'population_dispatch_authorized': False,
    'quality_gate': 'BLOCKED'
}

c026_key = None
c026_url = None

print('C026_GATEWAY_MODELS:', c026_metadata['status'])
print(
    'C026_CAPABILITIES_VERIFIED:',
    c026_metadata['capabilities_verified']
)
print(
    'C026_PILOT_CAPACITY_BOUNDS:',
    len(c026_token_bounds), '/3'
)
print('C026_BLOCKERS:', c026_summary['blockers'][:8])
print('C026_GENERATION_ATTEMPTS:', c026_new_attempts)
print('C026_SUCCESSFUL_RAW_RECEIPTS:', len(c026_results))
print('C026_PILOT_PROXY_TOMAN:', c026_pilot_proxy)
print('C026_SUMMARY_SHA256:', c026_summary_sha)
print('C026_STATUS:', C026_RESULT['status'])
print(
    'POPULATION_DISPATCH_AUTHORIZED: False | '
    'quality_gate: BLOCKED'
)

C026_GATEWAY_MODELS: EXACT_MODEL_LISTED
C026_CAPABILITIES_VERIFIED: False
C026_PILOT_CAPACITY_BOUNDS: 0 /3
C026_BLOCKERS: ['ARVAN_THINKING_OR_TOKEN_CAPACITY_CONTRACT_NOT_PROVEN', 'PILOT_RESERVE_NOT_CERTIFIED_WITHOUT_PROVIDER_CAPACITY']
C026_GENERATION_ATTEMPTS: 0
C026_SUCCESSFUL_RAW_RECEIPTS: 0
C026_PILOT_PROXY_TOMAN: 1670.67
C026_SUMMARY_SHA256: 4a2558b669e9fbb7d022ac0d1152a407484f4fa0ca5ebe3229e41d9901edd431
C026_STATUS: PREFLIGHT_HELD_NO_GENERATION
POPULATION_DISPATCH_AUTHORIZED: False | quality_gate: BLOCKED


<div dir="rtl" align="right">

### NB22E-v4:C027 — تکمیل شواهد Thinking و ظرفیت سه درخواست Pilot

**هدف:** تعیین دقیق شواهد موجود و کسری قرارداد Gateway بدون تکرار بسته‌های موفق.

**ورودی‌های پذیرفته‌شده:**
- Plan و Summary موفق C025
- Summary و نتیجهٔ `/models` در C026، فقط در محدودهٔ اطلاعات واقعی
- Ledger آفلاین C020
- قرارداد Gateway و تعرفهٔ ثبت‌شدهٔ C024

**کنترل‌ها:**

۱. مصرف تأیید شناسهٔ `DeepSeek-V4-Flash` از C026.

۲. بررسی فقط فیلدهای مرتبط با Thinking، Context، سقف خروجی و Tokenizer در Metadata موجود. فیلدِ غایب به معنی «پشتیبانی‌نشدن» نیست.

۳. تفکیک مستندات API مستقیم DeepSeek از قابلیت احرازشدهٔ Gateway آروان.

۴. کنترل سه شناسهٔ درخواست، حجم دقیق پیام‌ها و Payload، سقف ۶۰۰ توکن خروجی و برآوردهای توکن؛ بدون تبدیل تقریب به شمارش قطعی.

۵. کنترل مجدد مسیرهای مشخص Cache و Attempt برای همان سه شناسه؛ بدون پیمایش عمومی.

۶. نگهداری تعرفهٔ پنل و برآورد هزینهٔ مشروط Pilot، با صفر Retry خودکار.

۷. معرفی آزمون مستقل کوچک Thinking، فقط در صورت نیاز و فقط برای تصمیم ناظر؛ بدون ارسال خودکار.

**شواهد لازم برای عبور از Gate:** قرارداد آروان برای `thinking.type=disabled`، سقف مؤثر Context و خروجی، نسخهٔ Tokenizer یا کران بالای مستند شامل سربار پیام‌ها، و پذیرش اجرایی Adapter.

**خروجی:** Ledger سه‌ردیفی و Summary نسخه‌دار خارج از مخزن، با SHA و فهرست دقیق موارد نامحرز.

**ممنوعیت:** Generation، Pilot، Judge، تغییر Artifactهای C019 تا C026 و Git.

`generation_send_authorized = False`

`quality_gate = BLOCKED`

</div>

In [25]:
# NB22E-v4 | C027 | R11.2
# Constrained capability evidence, exact Pilot-source binding and cost; no dispatch.
# Consume C025/C026 receipts. No imports, API, model, Judge, or prior-artifact writes.

if not all(name in globals() for name in (
    'C025_OFFLINE_ADAPTER', 'C026_RESULT', 'C020_OFFLINE_PREPARATION',
    'C024_PREFLIGHT', 'c020_load_core', 'c016_sha', 'c016_canon',
    'c016_put_once', 'repo', 'outputs', 'c016_boundary'
)):
    raise RuntimeError('C027_REQUIRED_ACCEPTED_STATE_MISSING')


def c027_need(condition, reason):
    if not condition:
        raise RuntimeError('C027_' + reason)


# 1. Verify only the accepted Plan/Summary and C026 no-dispatch receipt.
c027_c025 = c020_load_core(
    C025_OFFLINE_ADAPTER['summary_path'], C025_OFFLINE_ADAPTER['summary_sha256']
)
c027_plan = c020_load_core(
    C025_OFFLINE_ADAPTER['plan_path'], C025_OFFLINE_ADAPTER['plan_sha256']
)
c027_c026 = c020_load_core(
    C026_RESULT['summary_path'], C026_RESULT['summary_sha256']
)
c027_c024 = c020_load_core(
    C024_PREFLIGHT['summary_path'], C024_PREFLIGHT['summary_sha256']
)
c027_c020 = c020_load_core(
    C020_OFFLINE_PREPARATION['summary_path'],
    C020_OFFLINE_PREPARATION['summary_sha256']
)
c027_ledger = c020_load_core(
    C020_OFFLINE_PREPARATION['ledger_path'],
    C020_OFFLINE_PREPARATION['ledger_sha256']
)

c027_need(
    c027_c025['status'] == 'OFFLINE_WIRE_AND_WRITE_AHEAD_DRYRUN_PASS_LIVE_GATES_HELD'
    and c027_c026['status'] == 'PREFLIGHT_HELD_NO_GENERATION'
    and c027_c026['model_metadata']['status'] == 'EXACT_MODEL_LISTED'
    and c027_c026['model_metadata']['capabilities_verified'] is False
    and c027_c026['pilot_generation_attempts'] == 0
    and c027_c026['pilot_successful_raw_receipts'] == 0
    and c027_c026['quality_gate'] == 'BLOCKED'
    and c027_c025['local_adapter_simulation_pass'] is True
    and c027_c025['local_adapter_plan_sha256']
        == C025_OFFLINE_ADAPTER['plan_sha256']
    and c027_c026['identity']['c025_plan_sha256']
        == C025_OFFLINE_ADAPTER['plan_sha256']
    and c027_c026['identity']['c011_boundary_sha256'] == c016_boundary
    and c027_c024['provider_and_adapter']['request_model']
        == 'DeepSeek-V4-Flash'
    and c027_c020['offline_requests_ready'] == 1116,
    'UPSTREAM_RECEIPTS_OR_NO_DISPATCH_CONFLICT'
)

c027_pilot = c027_plan['pilot_cases']

c027_need(
    len(c027_pilot) == len({r['request_id'] for r in c027_pilot}) == 3,
    'PILOT_POPULATION_CONFLICT'
)

c027_by_request = {
    r['request_id']: r for r in c027_ledger['entries']
}

c027_need(
    len(c027_by_request) == 1116,
    'C020_REQUEST_ID_DUPLICATE'
)


# 2. C026 retained the matching /models item in kernel.
# Its complete raw response was not persisted in the C026 disk summary.
# Do not invent capabilities or claim the full metadata was archived.

c027_model_record = globals().get('c026_model_metadata')

c027_model_source = (
    'C026_IN_KERNEL_UNPERSISTED_MODELS_ITEM'
    if isinstance(c027_model_record, dict)
    else 'C026_SUMMARY_ONLY_RAW_MODELS_ITEM_NOT_PERSISTED'
)

c027_public_metadata = {}

if isinstance(c027_model_record, dict):
    c027_need(
        c027_model_record.get('id') == 'DeepSeek-V4-Flash',
        'C026_IN_MEMORY_MODEL_ID_MISMATCH'
    )

    for field in (
        'context_length',
        'context_window',
        'max_model_len',
        'max_output_tokens',
        'max_tokens',
        'input_token_limit',
        'output_token_limit',
        'supported_parameters',
        'thinking_modes',
        'tokenizer_id'
    ):
        value = c027_model_record.get(field)

        if type(value) is int and 0 < value <= 100000000:
            c027_public_metadata[field] = value

        elif isinstance(value, list) and len(value) <= 40 and all(
            isinstance(item, str) and len(item) <= 80
            for item in value
        ):
            c027_public_metadata[field] = value

        elif (
            field == 'tokenizer_id'
            and isinstance(value, str)
            and len(value) <= 120
        ):
            c027_public_metadata[field] = value

    for container_name in (
        'capabilities', 'limits', 'features'
    ):
        nested = c027_model_record.get(container_name)

        if isinstance(nested, dict):
            c027_public_metadata[
                container_name + '_known_field_names'
            ] = sorted(
                name for name in nested
                if name in (
                    'thinking',
                    'thinking_disabled',
                    'temperature',
                    'max_tokens',
                    'max_output_tokens',
                    'context_length',
                    'context_window',
                    'tokenizer',
                    'tokenizer_id',
                    'supported_parameters'
                )
            )


# 3. Separate the direct DeepSeek documentation from Arvan evidence.
# The current Gateway model ID is listed, but behavior is not attested.

c027_thinking = 'NOT_ATTESTED_FOR_ARVAN_GATEWAY'
c027_tokenizer = 'NO_PROVIDER_MATCHED_TOKENIZER_AND_CHAT_TEMPLATE_PIN'
c027_gateway_window = 'DISPLAYED_1M_NOT_BACKEND_ATTESTED'
c027_gateway_output = 'REQUEST_600_NOT_BACKEND_ATTESTED'
c027_production_adapter = 'C025_SYNTHETIC_DRYRUN_NOT_LIVE_ACCEPTANCE'


# 4. Check only the three existing C020 request identities.
# Do not reconstruct their prompts or contexts.

c027_c020_root = Path(
    c027_c020['requests_directory']
).resolve().parent

c027_need(
    c027_c020_root
        == Path(C020_OFFLINE_PREPARATION['summary_path']).resolve().parent,
    'C020_REQUEST_ROOT_CHANGED'
)

c027_c026_root = Path(
    C026_RESULT['summary_path']
).resolve().parent

c027_need(
    c027_c026_root.parent == outputs.resolve(),
    'C026_OUTPUT_NOT_EXTERNAL'
)

c027_rows = []
c027_totals = collections.Counter()
c027_attempt_held = 0
c027_cache_held = 0

for pilot in c027_pilot:
    rid = pilot['request_id']
    cid = pilot['candidate_id']

    c027_need(
        rid in c027_by_request,
        'PILOT_REQUEST_NOT_IN_C020'
    )

    source = c027_by_request[rid]

    c027_need(
        pilot['request_receipt_sha256']
            == source['request_file_sha256']
        and source['candidate_id'] == cid
        and source['status'] == 'OFFLINE_READY_DISPATCH_GATED'
        and source['request_id'] == rid,
        'PILOT_C020_LINEAGE_CONFLICT'
    )

    # C025 already checked the complete wire bytes and their SHA.
    measure = source['token_measurement']

    c027_need(
        pilot['wire_payload_utf8_bytes']
            == measure['request_json_utf8_bytes']
        and pilot['message_body_utf8_bytes']
            == measure['messages_json_utf8_bytes']
        and pilot['max_output_tokens'] == 600,
        'PILOT_MESSAGE_BYTE_MEASUREMENT_CONFLICT'
    )

    # Check exact known paths. No directory-wide cache inventory.
    previous_paths = (
        c027_c020_root / 'attempt_events' / (rid + '.json'),
        c027_c020_root / 'completed_cache' / (rid + '.json'),
        c027_c026_root / 'attempts' / (rid + '.intent.json'),
        c027_c026_root / 'attempts' / (rid + '.dispatch_started.json'),
        c027_c026_root / 'attempts' / (rid + '.raw.http.body'),
        c027_c026_root / 'attempts' / (rid + '.transport.json'),
        c027_c026_root / 'attempts' / (rid + '.usage.json'),
        c027_c026_root / 'attempts' / (rid + '.verified.json')
    )

    c027_attempt = (
        source['attempt_status']
            != 'NO_PREVIOUS_ATTEMPT_FOR_NEW_IDENTITY'
        or any(
            p.exists() or p.is_symlink()
            for p in (previous_paths[0],) + previous_paths[2:]
        )
    )

    c027_cache = (
        source['cache_status']
            != 'NO_MATCHING_SCOPED_CACHE'
        or previous_paths[1].exists()
        or previous_paths[1].is_symlink()
    )

    c027_attempt_held += int(c027_attempt)
    c027_cache_held += int(c027_cache)

    for key in (
        'token_proxy_low_at_4_bytes_per_token',
        'token_proxy_high_at_2_bytes_per_token'
    ):
        c027_totals[key] += measure[key]

    c027_rows.append({
        'candidate_id': cid,
        'request_id': rid,
        'c020_request_file_sha256': source['request_file_sha256'],
        'messages_json_utf8_bytes_exact':
            measure['messages_json_utf8_bytes'],
        'request_json_utf8_bytes_exact':
            measure['request_json_utf8_bytes'],
        'input_token_proxy_low_not_a_bound':
            measure['token_proxy_low_at_4_bytes_per_token'],
        'input_token_proxy_high_not_a_bound':
            measure['token_proxy_high_at_2_bytes_per_token'],
        'tokenizer_count_verified': None,
        'chat_template_overhead_verified': None,
        'max_output_tokens_requested': 600,
        'gateway_context_cap_verified': None,
        'capacity_pass': False,
        'scoped_attempt_or_outcome_requires_reconciliation':
            bool(c027_attempt),
        'scoped_cache_requires_reconciliation':
            bool(c027_cache),
        'provider_route_thinking_verified': False,
        'dispatch_authorized': False
    })

c027_need(
    len(c027_rows) == 3,
    'PILOT_THREE_ROW_COVERAGE_CONFLICT'
)


# 5. Preserve the Arvan panel tariff and conditional C025 estimate.
# UTF-8 byte proxies are not actual billable token counts.

c027_cost = c027_c024['cost_estimates']

c027_need(
    c027_cost['input_price'] == 45000
    and c027_cost['output_price'] == 90000
    and c027_cost['max_output_tokens_per_request'] == 600
    and c027_cost['automatic_retry_attempt_reserve'] == 0
    and c027_totals['token_proxy_low_at_4_bytes_per_token'] > 0
    and c027_totals['token_proxy_high_at_2_bytes_per_token'] > 0,
    'C024_TARIFF_OR_PILOT_MEASUREMENT_CONFLICT'
)

# Optional one-request capability probe, for supervisor decision only.
# The 1000 input tokens are a pricing assumption, not an upper bound.
c027_probe_input_tokens_assumed = 1000
c027_probe_output_ceiling = 32

c027_probe_toman_conditional = round(
    (
        c027_probe_input_tokens_assumed * 45000
        + c027_probe_output_ceiling * 90000
    ) / 1000000,
    2
)

c027_blockers = [
    'ARVAN_MODEL_LIST_HAS_ID_BUT_NOT_A_VERIFIED_THINKING_DISABLED_CONTRACT',
    'ARVAN_BACKEND_TOKENIZER_AND_CHAT_TEMPLATE_OVERHEAD_NOT_PINNED',
    'ARVAN_EFFECTIVE_CONTEXT_AND_OUTPUT_LIMITS_NOT_ATTESTED',
    'LIVE_ADAPTER_ACCEPTANCE_AND_ACTUAL_BILLABLE_USAGE_NOT_YET_PROVEN'
]

if c027_attempt_held:
    c027_blockers.append(
        'PILOT_PREVIOUS_ATTEMPT_OR_UNKNOWN_OUTCOME_FOUND'
    )

if c027_cache_held:
    c027_blockers.append(
        'PILOT_PREVIOUS_CACHE_REQUIRES_IDENTITY_RECONCILIATION'
    )


# 6. Immutable C027 evidence outside the repository.
c027_identity = {
    'schema': 'NB22E_V4_C027_GATEWAY_CAPACITY_EVIDENCE_V1',
    'c011_boundary_sha256': c016_boundary,
    'c025_plan_sha256': C025_OFFLINE_ADAPTER['plan_sha256'],
    'c026_summary_sha256': C026_RESULT['summary_sha256'],
    'c024_summary_sha256': C024_PREFLIGHT['summary_sha256'],
    'c020_ledger_sha256': C020_OFFLINE_PREPARATION['ledger_sha256'],
    'c026_model_metadata_source': c027_model_source,
    'public_metadata_subset_sha256':
        c016_sha(c016_canon(c027_public_metadata)),
    'no_generation_request': True
}

c027_identity_sha = c016_sha(c016_canon(c027_identity))

c027_out = (
    outputs / (
        'C027_model_capacity_evidence_' + c027_identity_sha
    )
).resolve()

c027_need(
    c027_out.parent == outputs.resolve()
    and not c027_out.is_relative_to(repo.resolve()),
    'OUTPUT_NOT_EXTERNAL'
)

c027_out.mkdir(parents=True, exist_ok=True)

c027_lines = b''.join(
    c016_canon(row) + b'\n'
    for row in c027_rows
)

c027_rows_sha = c016_sha(c027_lines)

c027_rows_path = c027_out / (
    'C027.pilot3.' + c027_rows_sha + '.jsonl'
)

if not c027_rows_path.exists():
    with tempfile.NamedTemporaryFile(
        mode='wb',
        dir=c027_out,
        prefix='.C027.',
        suffix='.tmp',
        delete=False
    ) as stream:
        temp = Path(stream.name)
        stream.write(c027_lines)
        stream.flush()
        os.fsync(stream.fileno())

    try:
        os.link(temp, c027_rows_path)
    except FileExistsError:
        pass
    finally:
        temp.unlink(missing_ok=True)

c027_need(
    c016_sha(c027_rows_path.read_bytes()) == c027_rows_sha,
    'IMMUTABLE_PILOT_LEDGER_RELOAD_MISMATCH'
)


# 7. Publish the bounded findings and the exact missing evidence.
c027_summary = {
    'cell': 'NB22E-v4:C027',
    'status': 'GATEWAY_CAPABILITY_EVIDENCE_INCOMPLETE_PILOT_HELD',
    'identity': c027_identity,
    'identity_sha256': c027_identity_sha,
    'c026_models_check': 'EXACT_MODEL_ID_LISTED_C026',
    'models_metadata_source': c027_model_source,
    'models_metadata_whitelisted_fields_only': c027_public_metadata,
    'models_does_not_prove_unsupported_parameters': True,
    'arvan_thinking_disabled_support': c027_thinking,
    'arvan_effective_context_limit': c027_gateway_window,
    'arvan_effective_max_output': c027_gateway_output,
    'compatible_tokenizer_and_chat_overhead': c027_tokenizer,
    'adapter_validation_scope': c027_production_adapter,
    'pilot_context_capacity_exact_verified': 0,
    'population_context_capacity_exact_verified': 0,
    'pilot_request_count': 3,
    'pilot_scoped_unknown_or_nonterminal_attempts': c027_attempt_held,
    'pilot_scoped_cache_to_reconcile': c027_cache_held,
    'pilot_generation_calls': 0,
    'billing_rate_source': 'OWNER_ARVAN_SCREENSHOT_2026_11_11',
    'billing_currency': 'TOMAN',
    'price_input_per_1M': 45000,
    'price_output_per_1M': 90000,
    'pilot_input_proxy_low_not_strict':
        c027_totals['token_proxy_low_at_4_bytes_per_token'],
    'pilot_input_proxy_high_not_strict':
        c027_totals['token_proxy_high_at_2_bytes_per_token'],
    'pilot_output_token_ceiling': 1800,
    'pilot_proxy_low_toman':
        c027_cost['pilot_3_lower_proxy_toman'],
    'pilot_proxy_high_toman':
        c027_cost['pilot_3_upper_proxy_toman'],
    'pilot_guaranteed_spend_ceiling_toman': None,
    'retry_reserve':
        'ZERO_AUTOMATIC_RETRIES_UNKNOWN_OUTCOMES_SEPARATELY_HELD',
    'probe_proposal': {
        'kind': 'ONE_EXTRA_SYNTHETIC_NON_GOLD_CHAT_COMPLETION',
        'model': 'DeepSeek-V4-Flash',
        'thinking': {'type': 'disabled'},
        'max_tokens': c027_probe_output_ceiling,
        'temperature': 0,
        'messages': [
            {
                'role': 'system',
                'content': 'Return only one digit.'
            },
            {
                'role': 'user',
                'content': 'What is 2 + 2?'
            }
        ],
        'assumed_input_tokens_for_pricing_only':
            c027_probe_input_tokens_assumed,
        'conditional_reference_toman':
            c027_probe_toman_conditional,
        'actual_cost_ceiling_verified': False,
        'supervisor_authorization_required': True,
        'not_part_of_original_C020_Pilot_3': True,
        'check_response_model_and_usage_reasoning_tokens': True,
        'single_observation_is_not_universal_backend_guarantee': True
    },
    'evidence_needed': [
        'ARVAN_DOCUMENT_OR_CONTRACT:PASS_THROUGH_OF_thinking.type_disabled',
        'ARVAN_MODEL_BACKEND_CONTEXT_LENGTH_AND_MAX_OUTPUT_TOKENS',
        'TOKENIZER_ID_VERSION_AND_CHAT_ENCODING_MATCHING_ACTUAL_ARVAN_BACKEND',
        'OR_PROVIDER_AUTHORIZED_TOKEN_COUNT_OR_DOCUMENTED_STRICT_UPPER_BOUND',
        'FINAL_ATTEMPT_RAW_USAGE_ADAPTER_AND_SCOPE_BUDGET_RESERVATION'
    ],
    'deepseek_direct_api_docs_are_not_arvan_gateway_capabilities': True,
    'official_deepseek_direct_source':
        'https://api-docs.deepseek.com/api/create-chat-completion/',
    'official_v4_encoding_reference':
        'https://huggingface.co/deepseek-ai/DeepSeek-V4-Flash',
    'blockers': c027_blockers,
    'pilot_entries_path': str(c027_rows_path),
    'pilot_entries_file_sha256': c027_rows_sha,
    'generation_send_authorized': False,
    'population_dispatch_authorized': False,
    'quality_gate': 'BLOCKED'
}

c027_summary_path = c027_out / (
    'C027.summary.'
    + c016_sha(c016_canon(c027_summary))
    + '.json'
)

c027_summary_sha, _ = c016_put_once(
    c027_summary_path,
    c027_summary
)

c027_need(
    c020_load_core(
        c027_summary_path,
        c027_summary_sha
    )['pilot_entries_file_sha256'] == c027_rows_sha,
    'SUMMARY_RELOAD_CONFLICT'
)

C027_EVIDENCE = {
    'status': c027_summary['status'],
    'summary_path': str(c027_summary_path),
    'summary_sha256': c027_summary_sha,
    'pilot_send_authorized': False,
    'quality_gate': 'BLOCKED'
}

print('C027_MODEL_ID_IN_C026_MODELS: PASS')
print('C027_MODELS_METADATA_SOURCE:', c027_model_source)
print('C027_MODELS_PUBLIC_FIELDS:', sorted(c027_public_metadata))
print('C027_ARVAN_THINKING_DISABLED: NOT_ATTESTED')
print('C027_PILOT_CAPACITY_VERIFIED: 0/3')
print('C027_POPULATION_CAPACITY_VERIFIED: 0/1116')
print('C027_PILOT_ATTEMPT_HELD:', c027_attempt_held)
print('C027_PILOT_CACHE_RECONCILIATION:', c027_cache_held)
print(
    'C027_PILOT_PROXY_TOMAN:',
    c027_cost['pilot_3_lower_proxy_toman'],
    '-',
    c027_cost['pilot_3_upper_proxy_toman']
)
print(
    'C027_ONE_NEW_CAPABILITY_PROBE_CONDITIONAL_TOMAN:',
    c027_probe_toman_conditional
)
print('C027_SUMMARY_SHA256:', c027_summary_sha)
print('C027_STATUS:', C027_EVIDENCE['status'])
print('GENERATION_SEND_AUTHORIZED: False | quality_gate: BLOCKED')

C027_MODEL_ID_IN_C026_MODELS: PASS
C027_MODELS_METADATA_SOURCE: C026_IN_KERNEL_UNPERSISTED_MODELS_ITEM
C027_MODELS_PUBLIC_FIELDS: []
C027_ARVAN_THINKING_DISABLED: NOT_ATTESTED
C027_PILOT_CAPACITY_VERIFIED: 0/3
C027_POPULATION_CAPACITY_VERIFIED: 0/1116
C027_PILOT_ATTEMPT_HELD: 0
C027_PILOT_CACHE_RECONCILIATION: 0
C027_PILOT_PROXY_TOMAN: 916.38 - 1670.67
C027_ONE_NEW_CAPABILITY_PROBE_CONDITIONAL_TOMAN: 47.88
C027_SUMMARY_SHA256: 80f2b1b66e08467e3bf0e64d20367777c85158796be17dd155a380b463f74efd
C027_STATUS: GATEWAY_CAPABILITY_EVIDENCE_INCOMPLETE_PILOT_HELD
GENERATION_SEND_AUTHORIZED: False | quality_gate: BLOCKED


<div dir="rtl" align="right">

### NB22E-v4:C028 — آزمون واقعی تک‌درخواستی Thinking در ARVAN_GATEWAY

**مبنای پذیرفته‌شده:** C027، C026، C025 و C024؛ بدون اجرای مجدد بسته‌های قبلی.

**دامنهٔ مجوز:** فقط یک درخواست Chat Completions با متن ساختگی، مدل `DeepSeek-V4-Flash`، مقدار `thinking={"type":"disabled"}`، دمای صفر و `max_tokens=32`.

**کنترل پیش از ارسال:** تطبیق SHA مسیر Gateway، بررسی Secret بدون افشا، کنترل تعرفه و Reserve، و بررسی نبود Attempt قبلی همان هویت.

**قواعد اجرا:** ثبت Intent انحصاری و پایدار پیش از Dispatch، یک Worker، بدون Redirect یا Retry خودکار، ذخیرهٔ Raw قبل از Parse، سپس ثبت HTTP، مدل برگشتی، Usage، Reasoning و Finish Reason.

**تفسیر نتیجه:**
- HTTP 200: پذیرش درخواست، نه اثبات غیرفعال‌شدن Thinking.
- Reasoning صفر: فقط مشاهدهٔ همین درخواست.
- Reasoning ناموجود: نامحرز، نه صفر.
- Reasoning مثبت، اختلاف مدل یا نتیجهٔ نامعلوم: ثبت و توقف بدون Retry.

**ظرفیت Pilot:** بررسی جداگانهٔ سه Payload موجود C027، با احتساب کل پیام‌ها و سقف ۶۰۰ توکن خروجی. نبود Tokenizer سازگار یا محدودیت مستند Gateway به‌صورت نامحرز ثبت می‌شود.

**هزینه:** تعرفهٔ پنل مالک؛ ۴۵٬۰۰۰ تومان ورودی و ۹۰٬۰۰۰ تومان خروجی برای هر میلیون توکن. هزینهٔ مبتنی بر Usage با صورتحساب نهایی یکی نیست.

**خروجی:** Attempt، Raw، Telemetry و Summary نسخه‌دار، خارج از مخزن. هیچ Artifact پذیرفته‌شده‌ای بازنویسی نمی‌شود.

`pilot_send_authorized = False`

`population_dispatch_authorized = False`

`quality_gate = BLOCKED`

</div>

In [26]:
# NB22E-v4 | C028 | R11.2
# ONE synthetic Gateway diagnostic only; pinned C027; no three-case Pilot or Judge.
# No new imports. One worker, zero retries, durable intent and raw-before-parse.

c028_required = (
    'C027_EVIDENCE', 'C026_RESULT', 'C025_OFFLINE_ADAPTER',
    'C024_PREFLIGHT', 'c020_load_core', 'c016_canon', 'c016_sha',
    'c016_put_once', 'repo', 'outputs', 'c016_boundary'
)
if any(name not in globals() for name in c028_required):
    raise RuntimeError('C028_REQUIRED_ACCEPTED_STATE_MISSING')


def c028_need(ok, reason):
    if not ok:
        raise RuntimeError('C028_' + reason)


def c028_write_exclusive(path, obj):
    data = obj if isinstance(obj, bytes) else c016_canon(obj)
    fd = os.open(str(path), os.O_WRONLY | os.O_CREAT | os.O_EXCL, 0o600)
    with os.fdopen(fd, 'wb') as stream:
        stream.write(data)
        stream.flush()
        os.fsync(stream.fileno())
    c028_need(c016_sha(path.read_bytes()) == c016_sha(data),
              'DURABLE_RECEIPT_VERIFY_FAILED:' + path.name)
    return c016_sha(data)


# A. Consume existing acceptance evidence; no source rebuild or historical scan.
c028_prev = c020_load_core(
    C027_EVIDENCE['summary_path'], C027_EVIDENCE['summary_sha256']
)
c028_c026 = c020_load_core(
    C026_RESULT['summary_path'], C026_RESULT['summary_sha256']
)
c028_c024 = c020_load_core(
    C024_PREFLIGHT['summary_path'], C024_PREFLIGHT['summary_sha256']
)
c028_plan = c020_load_core(
    C025_OFFLINE_ADAPTER['plan_path'], C025_OFFLINE_ADAPTER['plan_sha256']
)
c028_need(
    C027_EVIDENCE['summary_sha256']
        == '80f2b1b66e08467e3bf0e64d20367777c85158796be17dd155a380b463f74efd'
    and c028_prev['status'] == 'GATEWAY_CAPABILITY_EVIDENCE_INCOMPLETE_PILOT_HELD'
    and c028_prev['pilot_generation_calls'] == 0
    and c028_prev['pilot_scoped_unknown_or_nonterminal_attempts'] == 0
    and c028_prev['pilot_scoped_cache_to_reconcile'] == 0
    and c028_prev['pilot_request_count'] == 3
    and c028_prev['pilot_context_capacity_exact_verified'] == 0
    and c028_prev['generation_send_authorized'] is False
    and c028_c026['status'] == 'PREFLIGHT_HELD_NO_GENERATION'
    and c028_c026['pilot_generation_attempts'] == 0
    and c028_c026['model_metadata']['status'] == 'EXACT_MODEL_LISTED'
    and c028_c024['provider_and_adapter']['endpoint_matches_saved_C07_route'] is True
    and c028_c024['provider_and_adapter']['request_model'] == 'DeepSeek-V4-Flash'
    and c028_prev['identity']['c011_boundary_sha256'] == c016_boundary
    and c028_prev['identity']['c025_plan_sha256'] == C025_OFFLINE_ADAPTER['plan_sha256']
    and len(c028_plan['pilot_cases']) == 3,
    'UPSTREAM_BOUNDARY_OR_PREVIOUS_DISPATCH_CONFLICT'
)

c028_pilot_rows_path = Path(c028_prev['pilot_entries_path']).resolve()
c028_need(
    c028_pilot_rows_path.parent
        == Path(C027_EVIDENCE['summary_path']).resolve().parent
    and c028_pilot_rows_path.is_file() and not c028_pilot_rows_path.is_symlink(),
    'C027_THREE_ROW_LEDGER_PATH_CONFLICT'
)
c028_pilot_raw = c028_pilot_rows_path.read_bytes()
c028_need(c016_sha(c028_pilot_raw) == c028_prev['pilot_entries_file_sha256'],
          'C027_THREE_ROW_LEDGER_SHA_CONFLICT')
c028_pilot_rows = [json.loads(line) for line in c028_pilot_raw.splitlines()]
c028_need(
    len(c028_pilot_rows) == 3
    and {r['request_id'] for r in c028_pilot_rows}
        == {r['request_id'] for r in c028_plan['pilot_cases']}
    and all(r['max_output_tokens_requested'] == 600 for r in c028_pilot_rows),
    'C027_PILOT_CAPACITY_INPUTS_CONFLICT'
)
c028_capacity_three = [
    {
        'request_id': row['request_id'],
        'messages_json_utf8_bytes': row['messages_json_utf8_bytes_exact'],
        'request_json_utf8_bytes': row['request_json_utf8_bytes_exact'],
        'input_token_proxy_not_a_bound': row['input_token_proxy_high_not_a_bound'],
        'output_cap_requested': 600,
        'effective_gateway_context_limit_tokens': None,
        'effective_gateway_output_limit_tokens': None,
        'provider_matched_token_count_or_strict_upper_bound': None,
        'capacity_verified': False
    }
    for row in c028_pilot_rows
]
del c028_pilot_raw


# B. Fixed diagnostic: no Gold, Context, production question, or C020 request.
c028_model_id = 'DeepSeek-V4-Flash'
c028_diag_payload = {
    'model': c028_model_id,
    'messages': [
        {'role': 'system', 'content': 'Reply with exactly OK.'},
        {'role': 'user', 'content': 'Synthetic connection test. Reply OK.'}
    ],
    'temperature': 0,
    'max_tokens': 32,
    'thinking': {'type': 'disabled'},
    'stream': False
}
c028_body = c016_canon(c028_diag_payload)
c028_need(len(c028_body) < 2048, 'DIAGNOSTIC_BODY_LARGER_THAN_EXPECTED')

# Owner-panel prices; 1000 input tokens is a proxy, NOT a cap.
c028_tariff = c028_c024['cost_estimates']
c028_need(
    c028_tariff['input_price'] == 45000
    and c028_tariff['output_price'] == 90000
    and c028_tariff['max_output_tokens_per_request'] == 600
    and c028_tariff['automatic_retry_attempt_reserve'] == 0,
    'OWNER_ARVAN_PANEL_TARIFF_CONFLICT'
)
c028_input_proxy = 1000
c028_input_reserve = 1000000  # Panel context scenario; NOT a verified strict limit.
c028_output_limit = 32
c028_proxy_toman = round(
    (c028_input_proxy * 45000 + c028_output_limit * 90000) / 1000000, 2
)
c028_reserve_toman = round(
    (c028_input_reserve * 45000 + c028_output_limit * 90000) / 1000000, 2
)
c028_need(
    c028_proxy_toman == 47.88
    and c028_reserve_toman == 45002.88
    and c028_reserve_toman < 100000,
    'DIAGNOSTIC_PRICE_RESERVE_NOT_UNDER_OWNER_THRESHOLD'
)


# C. Read exactly two Gateway .env settings; never persist the secret or URL.
c028_env_path = repo / '.env'
c028_config_problem = None
c028_config = {}
if not c028_env_path.is_file() or c028_env_path.is_symlink():
    c028_config_problem = 'PROJECT_DOTENV_MISSING_OR_SYMLINK'
else:
    with c028_env_path.open('r', encoding='utf-8-sig') as handle:
        for line in handle:
            match = re.match(
                r'^\s*(?:export\s+)?(ARVAN_GATEWAY_URL|ARVAN_GATEWAY_API_KEY)\s*=\s*(.*)$',
                line.rstrip('\n')
            )
            if match is None:
                continue
            key, value = match.groups()
            if key in c028_config:
                c028_config_problem = 'DUPLICATE_GATEWAY_CONFIG_KEY'
                break
            value = value.strip()
            if value[:1] in ('"', "'"):
                if len(value) < 2 or value[-1] != value[0]:
                    c028_config_problem = 'INVALID_GATEWAY_CONFIG_QUOTING'
                    break
                value = value[1:-1]
            else:
                value = value.split(' #', 1)[0].strip()
            c028_config[key] = value
            del value

c028_url = c028_config.pop('ARVAN_GATEWAY_URL', None)
c028_secret = c028_config.pop('ARVAN_GATEWAY_API_KEY', None)
c028_config.clear()
c028_endpoint = None
if c028_config_problem is None:
    if not c028_url or not c028_secret or '*' in c028_secret or c028_secret.startswith('<') or any(x.isspace() for x in c028_secret):
        c028_config_problem = 'URL_OR_UNMASKED_GATEWAY_CREDENTIAL_MISSING'
    else:
        try:
            parts = urllib.parse.urlsplit(c028_url)
            path = parts.path.rstrip('/')
            if (parts.scheme != 'https' or not parts.hostname or parts.username
                or parts.password or parts.query or parts.fragment or parts.port is not None
                or path not in ('', '/v1', '/v1/chat/completions')):
                c028_config_problem = 'UNRECOGNIZED_OR_UNSAFE_GATEWAY_ROUTE'
            else:
                c028_endpoint = urllib.parse.urlunsplit((
                    parts.scheme, parts.netloc, '/v1/chat/completions', '', ''
                ))
                if (
                    c016_sha(c028_endpoint.encode('utf-8'))
                    != c028_c026['identity']['endpoint_sha256']
                ):
                    c028_config_problem = 'ENDPOINT_DIFFERS_FROM_C026_PIN'
        except (ValueError, TypeError):
            c028_config_problem = 'UNREADABLE_GATEWAY_ENDPOINT'


# D. Stable identity and exclusive real-attempt marker prevent re-execution.
c028_identity = {
    'schema': 'NB22E_V4_C028_ONE_SYNTHETIC_THINKING_PROBE_V1',
    'c027_summary_sha256': C027_EVIDENCE['summary_sha256'],
    'c026_summary_sha256': C026_RESULT['summary_sha256'],
    'c025_plan_sha256': C025_OFFLINE_ADAPTER['plan_sha256'],
    'endpoint_fingerprint_sha256': c028_c026['identity']['endpoint_sha256'],
    'model_id_sent': c028_model_id,
    'body_sha256': c016_sha(c028_body),
    'maximum_new_generation_requests': 1,
    'no_retries': True
}
c028_identity_sha = c016_sha(c016_canon(c028_identity))
c028_dir = (
    outputs / ('C028_one_diagnostic_' + c028_identity_sha)
).resolve()
c028_need(
    c028_dir.parent == outputs.resolve()
    and not c028_dir.is_relative_to(repo.resolve()),
    'OUTPUT_DIRECTORY_NOT_EXTERNAL'
)
c028_dir.mkdir(mode=0o700, parents=True, exist_ok=True)
c028_attempt_dir = c028_dir / 'attempt'
c028_attempt_dir.mkdir(mode=0o700, exist_ok=True)
c028_intent_path = c028_attempt_dir / 'diagnostic.intent.json'
c028_transport_path = c028_attempt_dir / 'diagnostic.transport.json'
c028_raw_path = c028_attempt_dir / 'diagnostic.raw.http.body'
c028_telemetry_path = c028_attempt_dir / 'diagnostic.telemetry.json'
c028_outcome_path = c028_attempt_dir / 'diagnostic.outcome.json'

c028_blockers = []
if c028_config_problem:
    c028_blockers.append('CURRENT_GATEWAY_CONFIG:' + c028_config_problem)
if c028_prev['pilot_scoped_unknown_or_nonterminal_attempts'] != 0:
    c028_blockers.append('ORIGINAL_THREE_PILOT_ATTEMPTS_UNRESOLVED')
if any((x.exists() or x.is_symlink()) for x in (
    c028_intent_path, c028_transport_path, c028_raw_path,
    c028_telemetry_path, c028_outcome_path
)):
    c028_blockers.append('PREVIOUS_C028_DIAGNOSTIC_ATTEMPT_OR_ARTIFACT_EXISTS_NO_RESEND')

c028_send_attempted = 0
c028_http_status = None
c028_http_headers = {}
c028_raw_sha = None
c028_raw_capture_complete = None
c028_returned_model = None
c028_usage = None
c028_reasoning_tokens = None
c028_reasoning_tokens_source = 'NOT_REPORTED'
c028_reasoning_content_present = None
c028_finish_reason = None
c028_thinking_field_accepted = 'NOT_OBSERVED'
c028_thinking_behavior = 'NOT_OBSERVED'
c028_diag_status = 'PREFLIGHT_HELD_NO_NETWORK'
c028_actual_usage_price_toman = None
c028_response_message_sha = None

if not c028_blockers:
    # The 1M-token reserve is a conservative scenario, not a billing guarantee.
    print('C028_PRE_SEND_REQUESTS_MAX: 1')
    print('C028_PRE_SEND_TOMAN_PROXY_NOT_CEILING:', c028_proxy_toman)
    print('C028_PRE_SEND_TOMAN_CONSERVATIVE_SCENARIO_NOT_CEILING:', c028_reserve_toman)

    # O_EXCL prevents a rerun/concurrent invocation from sending twice.
    c028_write_exclusive(c028_intent_path, {
        'kind': 'REAL_ONE_REQUEST_DIAGNOSTIC_INTENT',
        'identity_sha256': c028_identity_sha,
        'request_body_sha256': c016_sha(c028_body),
        'endpoint_fingerprint_sha256': c028_identity['endpoint_fingerprint_sha256'],
        'model_id_sent': c028_model_id,
        'thinking_requested': {'type': 'disabled'},
        'max_tokens_requested': 32,
        'attempt_status': 'OUTCOME_UNKNOWN_UNTIL_RECEIPT',
        'request_limit': 1,
        'automatic_retry': False,
        'input_proxy_tokens': c028_input_proxy,
        'input_exposure_scenario_tokens_not_hard_bound': c028_input_reserve,
        'output_ceiling_requested': c028_output_limit,
        'conditional_toman_reserve_not_a_billing_guarantee': c028_reserve_toman
    })

    class C028NoRedirect(urllib.request.HTTPRedirectHandler):
        def redirect_request(self, request, fp, code, msg, headers, newurl):
            return None

    request = urllib.request.Request(
        c028_endpoint,
        data=c028_body,
        method='POST',
        headers={
            'Authorization': 'apikey ' + c028_secret,
            'Content-Type': 'application/json',
            'Accept': 'application/json'
        }
    )
    c028_opener = urllib.request.build_opener(C028NoRedirect())
    c028_raw = None
    c028_headers_obj = None
    c028_send_attempted = 1

    try:
        try:
            with c028_opener.open(request, timeout=180) as response:
                c028_http_status = response.status
                c028_headers_obj = response.headers
                c028_raw = response.read(1024 * 1024 + 1)
        except urllib.error.HTTPError as error:
            c028_http_status = error.code
            c028_headers_obj = error.headers
            c028_raw = error.read(1024 * 1024 + 1)

        c028_raw_capture_complete = len(c028_raw) <= 1024 * 1024
        c028_raw_sha = c028_write_exclusive(c028_raw_path, c028_raw)
        c028_http_headers = {
            header: c028_headers_obj.get(header)
            for header in ('x-request-id', 'date', 'server', 'retry-after')
            if c028_headers_obj is not None and c028_headers_obj.get(header) is not None
        }
        c028_write_exclusive(c028_transport_path, {
            'http_status': c028_http_status,
            'safe_headers': c028_http_headers,
            'raw_sha256': c028_raw_sha,
            'raw_complete': c028_raw_capture_complete,
            'parse_performed_before_raw_persistence': False
        })

        if not c028_raw_capture_complete:
            c028_diag_status = 'RAW_LIMIT_EXCEEDED_STOP_NO_RETRY'
        elif not (200 <= c028_http_status < 300):
            c028_diag_status = 'HTTP_NON_2XX_STOP_NO_RETRY'
        else:
            c028_thinking_field_accepted = 'HTTP_2XX_ACCEPTED_REQUEST_NOT_EFFECT_PROOF'

            try:
                parsed = json.loads(c028_raw_path.read_bytes())
                if not isinstance(parsed, dict):
                    raise ValueError('Response not object')

                c028_returned_model = parsed.get('model')
                c028_usage = parsed.get('usage') if isinstance(parsed.get('usage'), dict) else None
                choices = parsed.get('choices')

                if isinstance(choices, list) and len(choices) == 1 and isinstance(choices[0], dict):
                    choice = choices[0]
                    c028_finish_reason = choice.get('finish_reason')
                    message = choice.get('message')
                    if isinstance(message, dict):
                        content = message.get('content')
                        if isinstance(content, str):
                            c028_response_message_sha = c016_sha(content.encode('utf-8'))
                        c028_reasoning_content_present = any(
                            isinstance(message.get(key), str) and bool(message[key])
                            for key in ('reasoning_content', 'reasoning')
                        )

                if c028_usage is not None:
                    detail = c028_usage.get('completion_tokens_details')
                    nested = detail.get('reasoning_tokens') if isinstance(detail, dict) else None
                    top = c028_usage.get('reasoning_tokens')

                    if type(nested) is int and nested >= 0:
                        c028_reasoning_tokens = nested
                        c028_reasoning_tokens_source = 'usage.completion_tokens_details.reasoning_tokens'
                        if type(top) is int and top >= 0 and top != nested:
                            c028_blockers.append('REASONING_USAGE_FIELDS_DISAGREE')
                    elif type(top) is int and top >= 0:
                        c028_reasoning_tokens = top
                        c028_reasoning_tokens_source = 'usage.reasoning_tokens'

                    pt = c028_usage.get('prompt_tokens')
                    ct = c028_usage.get('completion_tokens')
                    if type(pt) is int and pt >= 0 and type(ct) is int and ct >= 0:
                        c028_actual_usage_price_toman = round(
                            (pt * 45000 + ct * 90000) / 1000000, 4
                        )

                # Save telemetry before semantic model/finish validation.
                c028_write_exclusive(c028_telemetry_path, {
                    'http_status': c028_http_status,
                    'response_model': c028_returned_model,
                    'usage': c028_usage,
                    'reasoning_tokens': c028_reasoning_tokens,
                    'reasoning_tokens_source': c028_reasoning_tokens_source,
                    'reasoning_text_nonempty': c028_reasoning_content_present,
                    'finish_reason': c028_finish_reason,
                    'response_content_sha256': c028_response_message_sha,
                    'raw_file_sha256': c028_raw_sha,
                    'estimate_toman_from_usage_not_final_invoice': c028_actual_usage_price_toman
                })

                if not isinstance(c028_returned_model, str) or c028_returned_model.lower().split('/')[-1] not in (
                    'deepseek-v4-flash', 'deepseek-v4-flash-0731'
                ):
                    c028_diag_status = 'RETURNED_MODEL_MISMATCH_STOP'
                elif c028_reasoning_tokens is None:
                    c028_thinking_behavior = 'NOT_REPORTED'
                    c028_diag_status = 'DIAGNOSTIC_RECEIVED_REASONING_TELEMETRY_MISSING'
                elif c028_reasoning_tokens > 0:
                    c028_thinking_behavior = 'REASONING_OBSERVED_DESPITE_DISABLED_REQUEST'
                    c028_diag_status = 'DIAGNOSTIC_RECEIVED_REASONING_NOT_DISABLED'
                elif c028_reasoning_content_present is True:
                    c028_thinking_behavior = 'REPORTED_ZERO_BUT_REASONING_TEXT_PRESENT'
                    c028_diag_status = 'DIAGNOSTIC_REASONING_TELEMETRY_CONFLICT'
                else:
                    c028_thinking_behavior = 'ZERO_REASONING_TOKENS_THIS_DIAGNOSTIC_ONLY'
                    c028_diag_status = 'DIAGNOSTIC_ZERO_REASONING_OBSERVED_ONLY'

                if c028_response_message_sha is None or c028_finish_reason != 'stop':
                    c028_diag_status += '_OUTPUT_STRUCTURE_OR_FINISH_NOT_VALID'

                if isinstance(c028_usage, dict) and type(c028_usage.get('completion_tokens')) is int and c028_usage['completion_tokens'] > 32:
                    c028_diag_status += '_OUTPUT_TOKEN_REQUEST_CAP_EXCEEDED'

            except (ValueError, TypeError, KeyError) as parse_error:
                c028_diag_status = 'RAW_SAVED_PARSE_OR_SCHEMA_ERROR_' + type(parse_error).__name__

    except Exception as transport_error:
        # The remote request may have executed even without a local raw receipt.
        c028_diag_status = 'OUTCOME_UNKNOWN_NO_RETRY_' + type(transport_error).__name__

    finally:
        c028_write_exclusive(c028_outcome_path, {
            'status': c028_diag_status,
            'identity_sha256': c028_identity_sha,
            'http_status': c028_http_status,
            'raw_sha256_if_saved': c028_raw_sha,
            'outcome_unknown': c028_diag_status.startswith('OUTCOME_UNKNOWN'),
            'automatic_retry_permitted': False
        })

    del request, c028_opener, c028_raw

# Do not retain the credential in stable globals longer than needed.
c028_secret = None
c028_url = None
c028_endpoint = None


# E. The diagnostic cannot establish three production Payload capacities.
c028_summary = {
    'cell': 'NB22E-v4:C028',
    'status': c028_diag_status if not c028_blockers else (
        'DIAGNOSTIC_PREFLIGHT_HELD' if c028_send_attempted == 0 else c028_diag_status
    ),
    'identity': c028_identity,
    'current_gateway_model_id_listed_from_C026': True,
    'synthetic_diagnostic_request_limit': 1,
    'new_generation_requests_attempted': c028_send_attempted,
    'http_status': c028_http_status,
    'response_model': c028_returned_model,
    'usage': c028_usage,
    'reasoning_tokens': c028_reasoning_tokens,
    'reasoning_tokens_source': c028_reasoning_tokens_source,
    'reasoning_text_nonempty': c028_reasoning_content_present,
    'finish_reason': c028_finish_reason,
    'thinking_parameter_http_acceptance': c028_thinking_field_accepted,
    'observed_thinking_behavior': c028_thinking_behavior,
    'backend_effective_thinking_contract_for_all_requests': 'NOT_ATTESTED',
    'raw_saved_before_parse': c028_raw_sha is not None,
    'raw_file_sha256': c028_raw_sha,
    'raw_capture_complete': c028_raw_capture_complete,
    'attempt_intent_path': str(c028_intent_path) if c028_intent_path.is_file() else None,
    'raw_path': str(c028_raw_path) if c028_raw_path.is_file() else None,
    'transport_path': str(c028_transport_path) if c028_transport_path.is_file() else None,
    'telemetry_path': str(c028_telemetry_path) if c028_telemetry_path.is_file() else None,
    'outcome_path': str(c028_outcome_path) if c028_outcome_path.is_file() else None,
    'pricing': {
        'source': 'OWNER_ARVAN_PANEL',
        'input_toman_per_million': 45000,
        'output_toman_per_million': 90000,
        'input_proxy_not_upper_bound': c028_input_proxy,
        'requested_output_tokens_max': c028_output_limit,
        'proxy_cost_toman_not_guaranteed': c028_proxy_toman,
        'conservative_input_exposure_scenario_not_hard_bound': c028_input_reserve,
        'pre_send_reserve_scenario_toman_not_hard_bound': c028_reserve_toman,
        'cost_estimate_from_returned_usage_toman_if_available': c028_actual_usage_price_toman,
        'other_billing_components_unknown': True,
        'retry_attempt_reserve': 0
    },
    'original_pilot_three_capacity_rows': c028_capacity_three,
    'verified_pilot_capacity_count': 0,
    'verified_population_capacity_count': 0,
    'effective_arvan_backend_context_tokens': None,
    'effective_arvan_backend_max_output_tokens': None,
    'provider_matched_tokenizer_or_strict_upper_bound': None,
    'capacity_evidence_missing': [
        'ARVAN_GATEWAY_DEEPSEEK_V4_FLASH_EFFECTIVE_CONTEXT_LIMIT',
        'ARVAN_GATEWAY_DEEPSEEK_V4_FLASH_EFFECTIVE_MAX_OUTPUT_LIMIT',
        'ARVAN_BACKEND_MATCHED_TOKENIZER_AND_CHAT_TEMPLATE_OR_DOCUMENTED_TOKEN_UPPER_BOUND'
    ],
    'future_pilot_generation_authorized': False,
    'population_generation_authorized': False,
    'quality_gate': 'BLOCKED',
    'blockers': c028_blockers,
    'automatic_retries': 0
}
c028_summary_path = c028_dir / (
    'C028.summary.' + c016_sha(c016_canon(c028_summary)) + '.json'
)
c028_summary_sha, _ = c016_put_once(c028_summary_path, c028_summary)
c028_need(
    c020_load_core(c028_summary_path, c028_summary_sha)['identity'] == c028_identity,
    'C028_SUMMARY_RELOAD_CONFLICT'
)

C028_DIAGNOSTIC = {
    'status': c028_summary['status'],
    'summary_path': str(c028_summary_path),
    'summary_sha256': c028_summary_sha,
    'new_generation_attempts': c028_send_attempted,
    'pilot_send_authorized': False,
    'quality_gate': 'BLOCKED'
}

print('C028_STATUS:', C028_DIAGNOSTIC['status'])
print('C028_HTTP_STATUS:', c028_http_status)
print('C028_RESPONSE_MODEL:', c028_returned_model)
print('C028_USAGE:', c028_usage)
print('C028_REASONING_TOKENS:', c028_reasoning_tokens if c028_reasoning_tokens is not None else 'NOT_REPORTED')
print('C028_THINKING_HTTP_ACCEPTANCE:', c028_thinking_field_accepted)
print('C028_THINKING_BEHAVIOR:', c028_thinking_behavior)
print('C028_GENERATION_ATTEMPTS:', c028_send_attempted, '/1')
print('C028_RAW_SAVED_BEFORE_PARSE:', c028_raw_sha is not None)
print('C028_PILOT_CONTEXT_CAPACITY_VERIFIED: 0/3')
print('C028_COST_PROXY_TOMAN:', c028_proxy_toman)
print('C028_COST_RESERVE_SCENARIO_NOT_CEILING_TOMAN:', c028_reserve_toman)
print('C028_ACTUAL_USAGE_BASED_COST_TOMAN:', c028_actual_usage_price_toman)
print('C028_BLOCKERS:', c028_blockers)
print('C028_SUMMARY_SHA256:', c028_summary_sha)
print('PILOT_SEND_AUTHORIZED: False | quality_gate: BLOCKED')

C028_PRE_SEND_REQUESTS_MAX: 1
C028_PRE_SEND_TOMAN_PROXY_NOT_CEILING: 47.88
C028_PRE_SEND_TOMAN_CONSERVATIVE_SCENARIO_NOT_CEILING: 45002.88
C028_STATUS: DIAGNOSTIC_ZERO_REASONING_OBSERVED_ONLY
C028_HTTP_STATUS: 200
C028_RESPONSE_MODEL: DeepSeek-V4-Flash
C028_USAGE: {'completion_tokens': 1, 'completion_tokens_details': {'reasoning_tokens': 0}, 'prompt_tokens': 18, 'prompt_tokens_details': {'cached_tokens': 0}, 'total_tokens': 19}
C028_REASONING_TOKENS: 0
C028_THINKING_HTTP_ACCEPTANCE: HTTP_2XX_ACCEPTED_REQUEST_NOT_EFFECT_PROOF
C028_THINKING_BEHAVIOR: ZERO_REASONING_TOKENS_THIS_DIAGNOSTIC_ONLY
C028_GENERATION_ATTEMPTS: 1 /1
C028_RAW_SAVED_BEFORE_PARSE: True
C028_PILOT_CONTEXT_CAPACITY_VERIFIED: 0/3
C028_COST_PROXY_TOMAN: 47.88
C028_COST_RESERVE_SCENARIO_NOT_CEILING_TOMAN: 45002.88
C028_ACTUAL_USAGE_BASED_COST_TOMAN: 0.9
C028_BLOCKERS: []
C028_SUMMARY_SHA256: 7c4d5a8867ef4ac62cbf7171185d92e74f1be6f6214a35abbcf04be0de77b37e
PILOT_SEND_AUTHORIZED: False | quality_gate: BLOCKED


<div dir="rtl" align="right">

### NB22E-v4 | C029 | R11.2 — کنترل ظرفیت و تصمیم ادامهٔ Generation

**هدف:** استفاده از شواهد واقعی Gateway و سوابق موفق NB22 برای کوتاه‌کردن مسیر اجرای Generation، بدون تکرار C019 تا C028.

#### شواهد پذیرفته‌شده

- C025: سه Payload فریز‌شده و بررسی موفق سازوکار Attempt و Raw.
- C026: تأیید شناسهٔ `DeepSeek-V4-Flash` در Gateway.
- C028: یک درخواست واقعی موفق، HTTP 200، مدل صحیح و صفر توکن Reasoning.
- NB22E-v3/C042: دریافت ۳۴ پاسخ از ۳۴ درخواست DeepSeek.
- NB22B/C010: سابقهٔ ۵۸ درخواست واقعی با تنظیم ۱۸ Worker و محدودیت ۸۰ درخواست در دقیقه.

#### کنترل ظرفیت

پنجرهٔ Context یک‌میلیونی در اطلاعات مدل و پنل نمایش داده شده است؛ معنای دقیق سقف مؤثر Gateway و شمارش سازگار توکن همچنان جداگانه ثبت می‌شود.

برای سه Payload، همهٔ پیام‌ها، سربار پیام و سقف ۶۰۰ توکن خروجی باید منظور شوند. تخمین بایتی به‌عنوان شمارش قطعی اعلام نمی‌شود.

#### کنترل هزینه

تعرفهٔ پنل مالک:

- ورودی: ۴۵٬۰۰۰ تومان به‌ازای یک میلیون توکن
- خروجی: ۹۰٬۰۰۰ تومان به‌ازای یک میلیون توکن
- Pilot سه‌تایی: برآورد ۹۱۶٫۳۸ تا ۱٬۶۷۰٫۶۷ تومان
- بستهٔ ۱۱۱۶تایی: برآورد ۳۴۸٬۰۵۷٫۸۱ تا ۶۳۵٬۸۲۷٫۶۴ تومان

#### تصمیم اجرایی

از تکرار آزمون Thinking و بررسی‌های تاریخی خودداری شود.

گام بعد، اجرای محدود سه درخواست با ثبت پایدار Attempt قبل از Dispatch، ذخیرهٔ Raw قبل از Parse، یک Worker و صفر Retry خودکار است؛ مشروط به تعیین‌تکلیف شروط واقعی ظرفیت و هزینه.

برای هر پاسخ، مدل برگشتی، Usage، Reasoning Tokens، Finish Reason و کیفیت متن بررسی شود.

در صورت موفقیت Pilot و پذیرش کیفیت، اجرای دسته‌ای با Cache و Resume و کنترل سقف هزینه برنامه‌ریزی شود. عبور احتمالی هزینهٔ کل از ۵۰۰ هزار تومان نیازمند مجوز جداگانهٔ مالک است.

**این مرحله مجوز اجرای جمعیتی یا داوری تازه نیست.**

</div>

In [27]:
# NB22E-v4 | C029 | R11.2
# Bounded capacity audit of three pinned C025 payloads; NO network, model or Judge.
# C028 diagnostic is consumed, not repeated. No changes to C019-C028 or Git.

c029_required = (
    'C028_DIAGNOSTIC', 'C027_EVIDENCE', 'C025_OFFLINE_ADAPTER',
    'C024_PREFLIGHT', 'C020_OFFLINE_PREPARATION', 'c020_load_core',
    'c016_canon', 'c016_sha', 'c016_put_once', 'repo', 'outputs',
    'c016_boundary'
)
if any(name not in globals() for name in c029_required):
    raise RuntimeError('C029_ACCEPTED_NOTEBOOK_STATE_MISSING')


def c029_need(ok, code):
    if not ok:
        raise RuntimeError('C029_' + code)


# A. Consume receipts only; no historical inventory or prior test rerun.
c029_c028 = c020_load_core(
    C028_DIAGNOSTIC['summary_path'],
    C028_DIAGNOSTIC['summary_sha256']
)
c029_c027 = c020_load_core(
    C027_EVIDENCE['summary_path'],
    C027_EVIDENCE['summary_sha256']
)
c029_c025 = c020_load_core(
    C025_OFFLINE_ADAPTER['summary_path'],
    C025_OFFLINE_ADAPTER['summary_sha256']
)
c029_plan = c020_load_core(
    C025_OFFLINE_ADAPTER['plan_path'],
    C025_OFFLINE_ADAPTER['plan_sha256']
)
c029_c024 = c020_load_core(
    C024_PREFLIGHT['summary_path'],
    C024_PREFLIGHT['summary_sha256']
)
c029_c020 = c020_load_core(
    C020_OFFLINE_PREPARATION['summary_path'],
    C020_OFFLINE_PREPARATION['summary_sha256']
)
c029_ledger = c020_load_core(
    C020_OFFLINE_PREPARATION['ledger_path'],
    C020_OFFLINE_PREPARATION['ledger_sha256']
)

c029_need(
    C028_DIAGNOSTIC['summary_sha256']
        == '7c4d5a8867ef4ac62cbf7171185d92e74f1be6f6214a35abbcf04be0de77b37e'
    and c029_c028['status']
        == 'DIAGNOSTIC_ZERO_REASONING_OBSERVED_ONLY'
    and c029_c028['new_generation_requests_attempted'] == 1
    and c029_c028['http_status'] == 200
    and c029_c028['response_model'] == 'DeepSeek-V4-Flash'
    and c029_c028['reasoning_tokens'] == 0
    and c029_c028['raw_saved_before_parse'] is True
    and c029_c028['future_pilot_generation_authorized'] is False
    and c029_c028['effective_arvan_backend_context_tokens'] is None
    and c029_c027['status']
        == 'GATEWAY_CAPABILITY_EVIDENCE_INCOMPLETE_PILOT_HELD'
    and c029_c027['pilot_request_count'] == 3
    and c029_c027['pilot_context_capacity_exact_verified'] == 0
    and c029_c025['status']
        == 'OFFLINE_WIRE_AND_WRITE_AHEAD_DRYRUN_PASS_LIVE_GATES_HELD'
    and c029_c025['local_adapter_plan_sha256']
        == C025_OFFLINE_ADAPTER['plan_sha256']
    and c029_c020['identity']['c011_boundary_sha256']
        == c016_boundary
    and c029_c024['provider_and_adapter']['request_model']
        == 'DeepSeek-V4-Flash',
    'C028_C027_C025_C020_RECEIPT_OR_SCOPE_CHANGED'
)

c029_pilots = c029_plan['pilot_cases']

c029_need(
    len(c029_pilots) == 3
    and len({p['request_id'] for p in c029_pilots}) == 3,
    'THREE_C025_REQUESTS_NOT_UNIQUE'
)

c029_rows_by_rid = {
    r['request_id']: r
    for r in c029_ledger['entries']
}

c029_need(
    len(c029_rows_by_rid) == 1116,
    'C020_REQUEST_ID_DUPLICATE'
)

c029_requests_dir = Path(
    c029_c020['requests_directory']
).resolve()

c029_need(
    c029_requests_dir.parent
        == Path(
            C020_OFFLINE_PREPARATION['summary_path']
        ).resolve().parent,
    'C020_REQUEST_DIRECTORY_CHANGED'
)


# B. Evidence boundary.
# Manufacturer and owner panel show 1M context, but do not establish
# the effective Arvan Chat route input-vs-total limit or max output.
c029_nominal_model_context = 1000000
c029_arvan_panel_context = 1000000

c029_gateway_context_rule = (
    'NOT_STATED_INPUT_ONLY_OR_INPUT_PLUS_OUTPUT'
)

c029_gateway_effective_context = None
c029_gateway_effective_output = None
c029_provider_matched_tokenizer = None
c029_chat_template_overhead_bound = None

c029_source = {
    'deepseek_author_model_card':
        'https://huggingface.co/deepseek-ai/DeepSeek-V4-Flash',
    'deepseek_encoding_instructions':
        'https://huggingface.co/deepseek-ai/DeepSeek-V4-Flash/tree/main/encoding',
    'arvan_owner_model_bazaar':
        'SCREENSHOT_C024_DEEPSEEK_V4_FLASH_1M',
    'arvan_model_id':
        'EXACT_ID_LISTED_C026',
    'arvan_thinking_observed':
        'C028_ONLY_REASONING_TOKENS_0',
    'arvan_context_semantics':
        c029_gateway_context_rule,
    'arvan_max_output_and_tokenizer':
        'NOT_ATTESTED_BY_AVAILABLE_EVIDENCE'
}


# C. Read exactly three pinned C025 request files.
# JSON bytes and the historical 2/4-byte proxies are not token bounds.
c029_rows = []
c029_totals = collections.Counter()
c029_held_cache = 0
c029_held_attempt = 0

c029_rate_in = (
    c029_c024['cost_estimates']['input_price']
)
c029_rate_out = (
    c029_c024['cost_estimates']['output_price']
)

c029_need(
    c029_rate_in == 45000
    and c029_rate_out == 90000
    and c029_c024['cost_estimates']['max_output_tokens_per_request']
        == 600,
    'OWNER_PANEL_TARIFF_OR_REQUEST_OUTPUT_CHANGED'
)

for entry in c029_pilots:
    rid = entry['request_id']
    cid = entry['candidate_id']

    c029_need(
        rid in c029_rows_by_rid,
        'REQUEST_ID_NOT_FOUND_IN_C020'
    )

    source = c029_rows_by_rid[rid]

    c029_need(
        source['candidate_id'] == cid
        and source['request_file_sha256']
            == entry['request_receipt_sha256']
        and source['status'] == 'OFFLINE_READY_DISPATCH_GATED',
        'C025_C020_CASE_IDENTITY_CONFLICT'
    )

    raw_path = (
        c029_requests_dir / source['request_file']
    )

    c029_need(
        not raw_path.is_symlink()
        and raw_path.is_file()
        and raw_path.resolve().parent == c029_requests_dir,
        'PINNED_REQUEST_PATH_MISSING_OR_UNSAFE'
    )

    saved_request = c020_load_core(
        raw_path.resolve(),
        source['request_file_sha256']
    )

    payload = saved_request['payload']
    messages = payload['messages']

    c029_need(
        saved_request['request_id'] == rid
        and saved_request['candidate_id'] == cid
        and saved_request['payload_sha256']
            == entry['wire_payload_sha256']
        and c016_sha(c016_canon(payload))
            == saved_request['payload_sha256']
        and payload['model'] == 'DeepSeek-V4-Flash'
        and payload['max_tokens'] == 600
        and payload['thinking'] == {'type': 'disabled'}
        and payload['temperature'] == 0
        and isinstance(messages, list)
        and len(messages) == 2
        and [m['role'] for m in messages] == ['system', 'user'],
        'C025_FROZEN_PAYLOAD_CHANGED'
    )

    measure = source['token_measurement']

    c029_need(
        all(
            isinstance(m['content'], str)
            for m in messages
        ),
        'MESSAGE_CONTENT_NOT_STRING'
    )

    content_bytes = [
        len(m['content'].encode('utf-8'))
        for m in messages
    ]

    roles_bytes = [
        len(m['role'].encode('utf-8'))
        for m in messages
    ]

    wire_messages_bytes = len(
        c016_canon(messages)
    )
    wire_payload_bytes = len(
        c016_canon(payload)
    )

    c029_need(
        wire_messages_bytes == entry['message_body_utf8_bytes']
        and wire_payload_bytes == entry['wire_payload_utf8_bytes']
        and wire_payload_bytes
            == measure['request_json_utf8_bytes']
        and wire_messages_bytes
            == measure['messages_json_utf8_bytes'],
        'PILOT_MESSAGE_OR_WIRE_BYTE_RECEIPT_CHANGED'
    )

    low = measure[
        'token_proxy_low_at_4_bytes_per_token'
    ]
    high = measure[
        'token_proxy_high_at_2_bytes_per_token'
    ]

    c029_need(
        type(low) is int
        and type(high) is int
        and 0 < low <= high,
        'INVALID_C020_TOKEN_PROXY'
    )

    c029_totals['proxy_low'] += low
    c029_totals['proxy_high'] += high
    c029_totals['wire_messages_bytes'] += wire_messages_bytes
    c029_totals['wire_payload_bytes'] += wire_payload_bytes

    c029_held_attempt += int(
        source['attempt_status']
            != 'NO_PREVIOUS_ATTEMPT_FOR_NEW_IDENTITY'
    )

    c029_held_cache += int(
        source['cache_status']
            != 'NO_MATCHING_SCOPED_CACHE'
    )

    c029_rows.append({
        'candidate_id': cid,
        'request_id': rid,
        'request_file_sha256':
            source['request_file_sha256'],
        'messages_count': len(messages),
        'roles': [m['role'] for m in messages],
        'message_content_utf8_bytes': content_bytes,
        'message_role_utf8_bytes': roles_bytes,
        'messages_json_utf8_bytes_exact':
            wire_messages_bytes,
        'whole_payload_json_utf8_bytes_exact':
            wire_payload_bytes,
        'input_proxy_4_bytes_per_token_not_bound': low,
        'input_proxy_2_bytes_per_token_not_bound': high,
        'provider_matched_prompt_tokens': None,
        'provider_chat_template_overhead_tokens': None,
        'documented_billable_input_token_upper_bound': None,
        'requested_max_output_tokens': 600,
        'gateway_max_output_at_least_600_proven': False,
        'gateway_effective_context_tokens': None,
        'gateway_context_input_vs_total_rule':
            c029_gateway_context_rule,
        'nominal_1m_context_under_proxy_only':
            high + 600 < 1000000,
        'input_plus_600_capacity_pass': False,
        'capacity_state':
            'HELD_GATEWAY_LIMIT_AND_MATCHED_TOKEN_COUNT_MISSING',
        'case_proxy_low_toman': round(
            (low * c029_rate_in + 600 * c029_rate_out)
            / 1000000,
            2
        ),
        'case_proxy_high_toman': round(
            (high * c029_rate_in + 600 * c029_rate_out)
            / 1000000,
            2
        ),
        'case_verified_max_billable_cost_toman': None,
        'dispatch_authorized': False
    })

    del saved_request, payload, messages

c029_need(
    len(c029_rows) == 3
    and c029_held_attempt
        == c029_c027[
            'pilot_scoped_unknown_or_nonterminal_attempts'
        ] == 0
    and c029_held_cache
        == c029_c027[
            'pilot_scoped_cache_to_reconcile'
        ] == 0,
    'C027_CACHE_ATTEMPT_OR_PILOT_SET_CONFLICT'
)


# D. Keep proxies and hypothetical 1M-context exposure separate.
c029_low_toman = round(
    (
        c029_totals['proxy_low'] * c029_rate_in
        + 1800 * c029_rate_out
    ) / 1000000,
    2
)

c029_high_toman = round(
    (
        c029_totals['proxy_high'] * c029_rate_in
        + 1800 * c029_rate_out
    ) / 1000000,
    2
)

c029_need(
    c029_low_toman
        == c029_c024['cost_estimates'][
            'pilot_3_lower_proxy_toman'
        ]
    and c029_high_toman
        == c029_c024['cost_estimates'][
            'pilot_3_upper_proxy_toman'
        ],
    'PREVIOUS_PILOT_PROXY_PRICING_CONFLICT'
)

c029_if_one_million_hard_input_each = round(
    (
        3 * 1000000 * c029_rate_in
        + 1800 * c029_rate_out
    ) / 1000000,
    2
)

c029_spend_reserve_proven = None
c029_capacity_verified = 0


# E. Immutable external audit artifacts; no Generation in C029.
c029_identity = {
    'schema':
        'NB22E_V4_C029_PINNED_PILOT_CAPACITY_EVIDENCE_V1',
    'c011_boundary_sha256': c016_boundary,
    'c025_plan_sha256':
        C025_OFFLINE_ADAPTER['plan_sha256'],
    'c027_summary_sha256':
        C027_EVIDENCE['summary_sha256'],
    'c028_summary_sha256':
        C028_DIAGNOSTIC['summary_sha256'],
    'c020_ledger_sha256':
        C020_OFFLINE_PREPARATION['ledger_sha256'],
    'three_request_ids': sorted(
        row['request_id']
        for row in c029_rows
    ),
    'new_api_calls_authorized_in_c029': 0
}

c029_identity_sha = c016_sha(
    c016_canon(c029_identity)
)

c029_out = (
    outputs / (
        'C029_pilot_capacity_' + c029_identity_sha
    )
).resolve()

c029_need(
    c029_out.parent == outputs.resolve()
    and not c029_out.is_relative_to(repo.resolve()),
    'OUTPUT_DIRECTORY_NOT_OUTSIDE_REPOSITORY'
)

c029_out.mkdir(
    parents=True,
    exist_ok=True
)

c029_rows_bytes = b''.join(
    c016_canon(row) + b'\n'
    for row in c029_rows
)

c029_rows_sha = c016_sha(c029_rows_bytes)

c029_rows_path = c029_out / (
    'C029.pilot3.' + c029_rows_sha + '.jsonl'
)

if not c029_rows_path.exists():
    with tempfile.NamedTemporaryFile(
        mode='wb',
        dir=c029_out,
        prefix='.C029.',
        suffix='.tmp',
        delete=False
    ) as temp_stream:
        temp_path = Path(temp_stream.name)

        temp_stream.write(c029_rows_bytes)
        temp_stream.flush()
        os.fsync(temp_stream.fileno())

    try:
        os.link(temp_path, c029_rows_path)
    except FileExistsError:
        pass
    finally:
        temp_path.unlink(missing_ok=True)

c029_need(
    not c029_rows_path.is_symlink()
    and c016_sha(c029_rows_path.read_bytes())
        == c029_rows_sha,
    'IMMUTABLE_PILOT_LEDGER_RELOAD_CONFLICT'
)


# F. Save the precise missing provider evidence and future boundary.
c029_summary = {
    'cell': 'NB22E-v4:C029',
    'status':
        'MODEL_1M_DOCUMENTED_GATEWAY_CAPACITY_SEMANTICS_UNATTESTED_PILOT_HELD',
    'identity': c029_identity,
    'identity_sha256': c029_identity_sha,
    'sources': c029_source,
    'manufacturer_nominal_model_context_tokens':
        c029_nominal_model_context,
    'owner_arvan_panel_display_context_tokens':
        c029_arvan_panel_context,
    'arvan_context_window_input_vs_total':
        c029_gateway_context_rule,
    'effective_arvan_route_context_window_tokens':
        c029_gateway_effective_context,
    'effective_arvan_route_max_output_tokens':
        c029_gateway_effective_output,
    'provider_matched_tokenizer':
        c029_provider_matched_tokenizer,
    'provider_chat_template_overhead_bound':
        c029_chat_template_overhead_bound,
    'pilot_cases_checked_from_exact_payload': 3,
    'pilot_capacity_verified': c029_capacity_verified,
    'population_capacity_verified': 0,
    'pilot_cache_unknown_or_held': c029_held_cache,
    'pilot_attempt_unknown_or_held': c029_held_attempt,
    'pilot_token_proxy_low': c029_totals['proxy_low'],
    'pilot_token_proxy_high': c029_totals['proxy_high'],
    'pilot_max_requested_output_tokens': 1800,
    'cost_toman': {
        'price_input_per_1M': c029_rate_in,
        'price_output_per_1M': c029_rate_out,
        'pilot_proxy_low_not_bound': c029_low_toman,
        'pilot_proxy_high_not_bound': c029_high_toman,
        'three_1m_inputs_plus_1800_outputs_scenario_only':
            c029_if_one_million_hard_input_each,
        'verified_pilot_spend_reserve':
            c029_spend_reserve_proven,
        'price_is_owner_panel_not_guaranteed_invoice':
            True,
        'unknown_tax_reasoning_cache_terms': True,
        'automatic_retries': 0
    },
    'exact_missing_evidence_from_arvan': [
        'DeepSeek-V4-Flash at configured ARVAN_GATEWAY chat/completions: effective context window token limit',
        'Does context limit cover input only or input plus generated output?',
        'Effective per-request max_output_tokens; explicit support for max_tokens=600',
        'Actual serving tokenizer ID/version and chat message encoding/template overhead',
        'Or provider-issued strict billable input token bound for each of three pinned requests'
    ],
    'next_procedure':
        'ON_ARVAN_EVIDENCE_RECHECK_THREE_CASES_THEN_SEPARATE_PILOT_EXECUTION_GATE',
    'c028_diagnostic_repeated': False,
    'new_api_calls': 0,
    'new_generation_calls': 0,
    'pilot_dispatch_authorized': False,
    'population_dispatch_authorized': False,
    'quality_gate': 'BLOCKED',
    'pilot_ledger_path': str(c029_rows_path),
    'pilot_ledger_sha256': c029_rows_sha
}

c029_summary_path = c029_out / (
    'C029.summary.'
    + c016_sha(c016_canon(c029_summary))
    + '.json'
)

c029_summary_sha, _ = c016_put_once(
    c029_summary_path,
    c029_summary
)

c029_need(
    c020_load_core(
        c029_summary_path,
        c029_summary_sha
    )['pilot_ledger_sha256'] == c029_rows_sha,
    'C029_SUMMARY_RELOAD_CONFLICT'
)

C029_CAPACITY = {
    'status': c029_summary['status'],
    'summary_path': str(c029_summary_path),
    'summary_sha256': c029_summary_sha,
    'pilot_capacity_verified': 0,
    'pilot_dispatch_authorized': False,
    'quality_gate': 'BLOCKED'
}

print('C029_SOURCES: DEEPSEEK_MODEL_CARD_1M + OWNER_ARVAN_PANEL_1M')
print('C029_ARVAN_CONTEXT_INPUT_OR_TOTAL: NOT_DOCUMENTED')
print('C029_ARVAN_MAX_OUTPUT_600: NOT_VERIFIED')

for row in c029_rows:
    print(
        'C029_CASE:', row['candidate_id'],
        'CONTENT_UTF8_BYTES:',
        row['message_content_utf8_bytes'],
        'MESSAGES_JSON_BYTES:',
        row['messages_json_utf8_bytes_exact'],
        'INPUT_TOKEN_PROXY_NOT_BOUND:',
        row['input_proxy_4_bytes_per_token_not_bound'],
        '-',
        row['input_proxy_2_bytes_per_token_not_bound'],
        'PLUS_OUTPUT:',
        row['requested_max_output_tokens'],
        'CAPACITY: HELD'
    )

print('C029_PILOT_CAPACITY_VERIFIED: 0/3')
print(
    'C029_PILOT_PROXY_TOMAN_NOT_BOUND:',
    c029_low_toman, '-', c029_high_toman
)
print(
    'C029_HARD_1M_PER_CASE_SCENARIO_ONLY_TOMAN:',
    c029_if_one_million_hard_input_each
)
print(
    'C029_VERIFIED_COST_RESERVE:',
    c029_spend_reserve_proven
)
print('C029_SUMMARY_SHA256:', c029_summary_sha)
print('C029_STATUS:', C029_CAPACITY['status'])
print('PILOT_DISPATCH_AUTHORIZED: False | quality_gate: BLOCKED')

C029_SOURCES: DEEPSEEK_MODEL_CARD_1M + OWNER_ARVAN_PANEL_1M
C029_ARVAN_CONTEXT_INPUT_OR_TOTAL: NOT_DOCUMENTED
C029_ARVAN_MAX_OUTPUT_600: NOT_VERIFIED
C029_CASE: golden_v2_candidate_62b1f8c24a843d4ea58cdcc9 CONTENT_UTF8_BYTES: [309, 22973] MESSAGES_JSON_BYTES: 23557 INPUT_TOKEN_PROXY_NOT_BOUND: 5890 - 11779 PLUS_OUTPUT: 600 CAPACITY: HELD
C029_CASE: golden_v2_candidate_dd8fed5954d9c09933ffbda6 CONTENT_UTF8_BYTES: [309, 18243] MESSAGES_JSON_BYTES: 18751 INPUT_TOKEN_PROXY_NOT_BOUND: 4688 - 9376 PLUS_OUTPUT: 600 CAPACITY: HELD
C029_CASE: golden_v2_candidate_f94a71bc57baf4968e4072ba CONTENT_UTF8_BYTES: [309, 24168] MESSAGES_JSON_BYTES: 24742 INPUT_TOKEN_PROXY_NOT_BOUND: 6186 - 12371 PLUS_OUTPUT: 600 CAPACITY: HELD
C029_PILOT_CAPACITY_VERIFIED: 0/3
C029_PILOT_PROXY_TOMAN_NOT_BOUND: 916.38 - 1670.67
C029_HARD_1M_PER_CASE_SCENARIO_ONLY_TOMAN: 135162.0
C029_VERIFIED_COST_RESERVE: None
C029_SUMMARY_SHA256: d2b1643c6681077cd09e914501810a3ebf6986e0576a11eb75e219109e1ad21a
C029_STATUS: MODEL_1M_DOC